# GRACE Imputation and Spatial Downscaling Pipeline

Colab-ready reproducible workflow for GRACE/GRACE-FO gap reconstruction and bias-aware 0.25°→0.1° spatial downscaling in the Bug River Basin.

The analytical workflow is retained from the original executed notebook. Repository-specific changes are limited to runtime bootstrap, structured local input paths, and local ZIP/CSV loading.


## 0. Runtime configuration

The repository is cloned/synchronized in Colab before imports. All analytical inputs are stored under structured `data/` subfolders. Model settings, validation design, feature definitions, balancing, mass conservation, and post-processing follow the original notebook.


In [ ]:
# Repository bootstrap and runtime controls
from pathlib import Path
import shutil
import subprocess
import sys

PIPELINE_BUILD = '2026-08-14-structured-repo-v1'
REPO_NAME = 'bug_river_area_tws_downscaling'
REPO_URL = f'https://github.com/VytautasSam/{REPO_NAME}.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_ROOT = Path('/content') / REPO_NAME
    if (REPO_ROOT / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'], check=True)
    else:
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
        check=True,
    )
else:
    cwd = Path.cwd().resolve()
    REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'GRACE_processing_pipeline.ipynb').exists()), cwd)

LOW_MEMORY_MODE = True
RUN_PUBLICATION_FIGURES = True
RUN_DIAGNOSTIC_ANALYTICS = False
RUN_SUPPLEMENTAL_VIDEOS = True
DISPLAY_FIGURES = True
SHOW_INTERMEDIATE_ANALYTICS = False
EXPORT_GEOPARQUET = True
EXPORT_FULL_FEATURE_OUTPUTS = False

N_JOBS = 1
EXPORT_DPI = 300
SHAP_SAMPLE_SIZE = 400
VIDEO_FPS = 3
VIDEO_DPI = 100

FINAL_RF_TREES = 600
OUTER_TEST_RF_TREES = 250
CV_RF_TREES = 60
PSEUDO_GAP_RF_TREES = 200
CV_SELECTION_FOLDS = 5
RF_PARAMETER_CANDIDATES = [
    {'max_features': 'sqrt', 'min_samples_leaf': 1, 'max_depth': None},
    {'max_features': 'sqrt', 'min_samples_leaf': 2, 'max_depth': None},
    {'max_features': 'sqrt', 'min_samples_leaf': 4, 'max_depth': None},
    {'max_features': 0.50, 'min_samples_leaf': 1, 'max_depth': None},
    {'max_features': 0.70, 'min_samples_leaf': 2, 'max_depth': None},
    {'max_features': 1.00, 'min_samples_leaf': 2, 'max_depth': None},
]
PSEUDO_GAP_LENGTHS = [1, 2, 3, 11]
PSEUDO_GAP_WINDOWS_PER_LENGTH = 100
PSEUDO_GAP_TEMPORAL_STRATA = 3
BOOTSTRAP_ITERATIONS = 300
UNCERTAINTY_Z = 1.96

print('Pipeline build:', PIPELINE_BUILD)
print('Repository root:', REPO_ROOT)


In [ ]:
from pathlib import Path
import gc
import os
import random
import sys
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from shapely.geometry import box
from itertools import combinations

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)
plt.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': EXPORT_DPI,
    'figure.max_open_warning': 10,
})

MIN_PYTHON = (3, 10)
if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        f'Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]}+ is required; active: {sys.version}'
    )

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

TARGET_COL = 'grace_or_imputed'
BIAS_METHOD = 'proportional'
BIAS_WEIGHT_COL = 'Q'
MASS_BLOCKS = 10

N_SPATIAL_BLOCKS = 5
HOLDOUT_YEARS = [2022, 2023]
BALANCE_REFERENCE_YEAR = min(HOLDOUT_YEARS) - 1

OUTER_BLOCK_COMBINATIONS = list(combinations(range(N_SPATIAL_BLOCKS), 2))

ROOT_DIR = REPO_ROOT / 'work'
RAW_CACHE_DIR = ROOT_DIR / 'raw_cache'
OUTPUT_DIR = REPO_ROOT / 'outputs'
MANUSCRIPT_DIR = OUTPUT_DIR / 'manuscript'
SUPPLEMENT_DIR = OUTPUT_DIR / 'supplementary'
DIAGNOSTIC_DIR = OUTPUT_DIR / 'diagnostics'
VIDEO_DIR = SUPPLEMENT_DIR / 'videos'

for directory in (RAW_CACHE_DIR, OUTPUT_DIR, MANUSCRIPT_DIR, SUPPLEMENT_DIR, DIAGNOSTIC_DIR, VIDEO_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def memory_report(label=''):
    """Print process RSS when psutil is available."""
    try:
        import psutil
        rss_gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        print(f'[memory] {label}: {rss_gb:.2f} GB')
    except Exception:
        pass


def save_show_close(fig, path, dpi=EXPORT_DPI, **kwargs):
    """Save, optionally display, and close a Matplotlib figure."""
    fig.savefig(
        path,
        dpi=dpi,
        bbox_inches=kwargs.pop('bbox_inches', 'tight'),
        **kwargs,
    )
    if DISPLAY_FIGURES:
        plt.show()
    plt.close(fig)


def display_if_requested(obj):
    """Display intermediate analytics only when explicitly requested."""
    if SHOW_INTERMEDIATE_ANALYTICS:
        display(obj)


def print_if_requested(*args, **kwargs):
    """Print intermediate diagnostics only when explicitly requested."""
    if SHOW_INTERMEDIATE_ANALYTICS:
        print(*args, **kwargs)


def require_columns(frame, columns, frame_name='DataFrame'):
    """Raise a clear error when required columns are absent."""
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f'{frame_name} is missing required columns: {missing}')


def to_month_start(values):
    """Convert timestamps to monthly period starts."""
    return pd.to_datetime(values, errors='coerce').dt.to_period('M').dt.to_timestamp()


def add_panel_label(axis, label, x=-0.06, y=1.02, fontsize=15):
    axis.text(
        x, y, label, transform=axis.transAxes, fontsize=fontsize,
        fontweight='bold', ha='left', va='bottom',
    )


def as_geodataframe(frame, geometry, crs=None):
    return gpd.GeoDataFrame(frame.copy(), geometry=geometry,
                            crs=crs if crs is not None else getattr(frame, 'crs', None))


def _get_random_forest_step(model_pipeline):
    """Return the fitted Random Forest from either pipeline naming convention."""
    for step_name in ('rf', 'model'):
        estimator = model_pipeline.named_steps.get(step_name)
        if estimator is not None:
            return estimator
    raise KeyError('No fitted Random Forest step was found in the pipeline.')


def predict_rf_distribution(
    model_pipeline,
    features,
    z_value=UNCERTAINTY_Z,
):
    """Predict the RF mean and tree-ensemble dispersion without a large matrix."""
    preprocessor = model_pipeline.named_steps['prep']
    random_forest = _get_random_forest_step(model_pipeline)
    transformed = preprocessor.transform(features)

    running_mean = np.zeros(len(features), dtype=float)
    running_m2 = np.zeros(len(features), dtype=float)

    for tree_number, tree in enumerate(random_forest.estimators_, start=1):
        tree_prediction = tree.predict(transformed)
        delta = tree_prediction - running_mean
        running_mean += delta / tree_number
        running_m2 += delta * (tree_prediction - running_mean)

    if len(random_forest.estimators_) > 1:
        prediction_std = np.sqrt(
            running_m2 / (len(random_forest.estimators_) - 1)
        )
    else:
        prediction_std = np.zeros(len(features), dtype=float)

    lower = running_mean - z_value * prediction_std
    upper = running_mean + z_value * prediction_std
    return running_mean, prediction_std, lower, upper


def bootstrap_regression_metrics(
    y_true,
    y_pred,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    random_state=RANDOM_STATE,
):
    """Return point estimates and paired-bootstrap 95% confidence intervals."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    point = {
        'r2': (
            np.nan
            if len(y_true) < 2 or np.allclose(y_true, y_true[0])
            else float(r2_score(y_true, y_pred))
        ),
        'mae': (
            np.nan
            if len(y_true) == 0
            else float(mean_absolute_error(y_true, y_pred))
        ),
        'rmse': (
            np.nan
            if len(y_true) == 0
            else float(np.sqrt(mean_squared_error(y_true, y_pred)))
        ),
    }

    if len(y_true) < 2:
        return {
            metric: {
                'estimate': value,
                'ci_lower': np.nan,
                'ci_upper': np.nan,
            }
            for metric, value in point.items()
        }

    rng = np.random.default_rng(random_state)
    values = {'r2': [], 'mae': [], 'rmse': []}

    for _ in range(n_bootstrap):
        sample_index = rng.integers(0, len(y_true), size=len(y_true))
        true_sample = y_true[sample_index]
        pred_sample = y_pred[sample_index]

        if not np.allclose(true_sample, true_sample[0]):
            values['r2'].append(r2_score(true_sample, pred_sample))
        values['mae'].append(mean_absolute_error(true_sample, pred_sample))
        values['rmse'].append(
            np.sqrt(mean_squared_error(true_sample, pred_sample))
        )

    output = {}
    for metric, estimate in point.items():
        metric_values = np.asarray(values[metric], dtype=float)
        output[metric] = {
            'estimate': estimate,
            'ci_lower': (
                float(np.nanquantile(metric_values, 0.025))
                if metric_values.size
                else np.nan
            ),
            'ci_upper': (
                float(np.nanquantile(metric_values, 0.975))
                if metric_values.size
                else np.nan
            ),
        }
    return output


def summarize_repeated_metrics(frame, metric_columns):
    """Summarize sensitivity to the choice of spatial holdout blocks."""
    rows = []
    for column in metric_columns:
        values = pd.to_numeric(frame[column], errors='coerce').dropna()
        rows.append({
            'metric': column,
            'mean': values.mean(),
            'standard_deviation': values.std(ddof=1),
            'median': values.median(),
            'empirical_2.5_percentile': values.quantile(0.025),
            'empirical_97.5_percentile': values.quantile(0.975),
            'minimum': values.min(),
            'maximum': values.max(),
            'number_of_outer_tests': len(values),
        })
    return pd.DataFrame(rows)


print('Python:', sys.version)
print('Output directory:', OUTPUT_DIR)
print('Manuscript figures:', MANUSCRIPT_DIR)
print('Supplementary material:', SUPPLEMENT_DIR)


## 1. Structured repository inputs

Primary data are grouped by function under `data/spatial`, `data/hydroclimate`, `data/grace`, and `data/benchmarks`. `data/reproducibility` contains local CSV representations used by the original Google-Sheets-based workflow. `data/auxiliary` contains supplied files that are retained for provenance but are not required by the canonical run.


In [ ]:
DATA_DIR = REPO_ROOT / 'data'
SPATIAL_DIR = DATA_DIR / 'spatial'
HYDRO_DIR = DATA_DIR / 'hydroclimate'
GRACE_DIR = DATA_DIR / 'grace'
BENCHMARK_DIR = DATA_DIR / 'benchmarks'
REPRO_DIR = DATA_DIR / 'reproducibility'

RAW_SOURCES = {
    'elevation_zip': SPATIAL_DIR / 'dem.zip',
    'lithology_zip': SPATIAL_DIR / 'lithology.zip',
    'land_cover_grid_zip': SPATIAL_DIR / 'land_cover_grid_PL_UA_BY.zip',
    'land_cover_zip': SPATIAL_DIR / 'land_cover.zip',
    'hydroclimate_zip': HYDRO_DIR / '2013_2023_data.zip',
    'new_runoff_zip': HYDRO_DIR / 'R_0.1.zip',
    'land_surface_temperature_zip': HYDRO_DIR / 'LST_ERA5.zip',
    'grid_025_zip': SPATIAL_DIR / 'grid_0.25.zip',
    'grace_original_sheet': REPRO_DIR / 'GRACE_025deg_orig.csv',
    'grace_filled_sheet': REPRO_DIR / 'GRACE_025deg_filled.csv',
    'soil_moisture_sheet': REPRO_DIR / 'SMS_GLDAS_025_2013-2023.csv',
    'wghm_original_05_sheet': REPRO_DIR / 'WGHM_Bug_original.csv',
    'wghm_interpolated_01_sheet': REPRO_DIR / 'WGHM_Bug_01deg.csv',
    'grace_seda_original_05_sheet': REPRO_DIR / 'GRACE_SeDA_Bug_original_05deg.csv',
    'grace_seda_interpolated_01_sheet': REPRO_DIR / 'GRACE_SeDA_Bug_01deg.csv',
}

missing = [path.relative_to(REPO_ROOT).as_posix() for path in RAW_SOURCES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing structured repository inputs: ' + ', '.join(missing))

GITHUB_BLOB_BASE = 'https://github.com/VytautasSam/bug_river_area_tws_downscaling/blob/main/'
RAW_SOURCE_LINKS = {
    key: GITHUB_BLOB_BASE + path.relative_to(REPO_ROOT).as_posix()
    for key, path in RAW_SOURCES.items()
}
print(f'Validated {len(RAW_SOURCES)} analytical inputs in structured data folders.')


### 1.1 Local input utilities

ZIP archives are extracted to `work/raw_cache`. Tabular analytical inputs are read from the repository CSVs; no Google Drive download layer is required.


In [ ]:
def extract_repo_zip(archive_path: Path, cache_name: str) -> Path:
    """Extract one repository ZIP once and return its extraction directory."""
    extract_root = RAW_CACHE_DIR / cache_name
    marker = extract_root / '.extracted'
    if not marker.exists():
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_root)
        marker.touch()
    return extract_root


def find_shapefile(root: Path, filename: str | None = None) -> Path:
    """Find one shapefile within an extracted archive."""
    candidates = list(root.rglob(Path(filename).name if filename else "*.shp"))
    if filename and not candidates:
        raise FileNotFoundError(f"Could not find {filename!r} below {root}")
    if not filename and len(candidates) != 1:
        raise ValueError(f"Expected one shapefile below {root}; found {len(candidates)}")
    return candidates[0]


def harmonize_crs(*gdfs, target="EPSG:4326"):
    out = []
    for gdf in gdfs:
        if gdf.crs is None:
            gdf = gdf.set_crs(target)
        elif str(gdf.crs) != target:
            gdf = gdf.to_crs(target)
        out.append(gdf)
    return out


def add_grid_polygons(df, lon_col="lon", lat_col="lat", res=0.25, crs="EPSG:4326"):
    lon = pd.to_numeric(df[lon_col], errors="coerce").to_numpy(dtype=float)
    lat = pd.to_numeric(df[lat_col], errors="coerce").to_numpy(dtype=float)
    eps = 1e-12
    x0 = np.floor((lon + eps) / res) * res
    y0 = np.floor((lat + eps) / res) * res
    geometry = [box(x, y, x + res, y + res) for x, y in zip(x0, y0)]
    out = gpd.GeoDataFrame(df.copy(), geometry=geometry, crs=crs)
    out["grid_lon0"] = np.round(x0, 6)
    out["grid_lat0"] = np.round(y0, 6)
    return out


def load_grace_cached(csv_path: Path, value_name: str) -> pd.DataFrame:
    wide = pd.read_csv(csv_path).rename(columns={
        "Unnamed: 0": "grid",
        "Unnamed: 1": "lat",
        "Unnamed: 2": "lon",
    })
    long = wide.melt(
        id_vars=["grid", "lat", "lon"],
        var_name="date",
        value_name=value_name,
    )
    for col in ["lon", "lat", value_name]:
        long[col] = pd.to_numeric(
            long[col].astype(str)
            .str.replace("\u00a0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False),
            errors="coerce",
        )
    return long


### 1.2 Prepare analytical sources

All required repository inputs are resolved to local paths before feature engineering.


In [ ]:
ZIP_SOURCE_KEYS = [
    'elevation_zip', 'lithology_zip', 'land_cover_grid_zip', 'land_cover_zip',
    'hydroclimate_zip', 'new_runoff_zip', 'land_surface_temperature_zip', 'grid_025_zip',
]
SHEET_SOURCE_KEYS = [
    'grace_original_sheet', 'grace_filled_sheet', 'soil_moisture_sheet',
    'wghm_original_05_sheet', 'wghm_interpolated_01_sheet',
    'grace_seda_original_05_sheet', 'grace_seda_interpolated_01_sheet',
]

RAW_LOCAL = {key: extract_repo_zip(RAW_SOURCES[key], key) for key in ZIP_SOURCE_KEYS}
RAW_LOCAL.update({key: RAW_SOURCES[key] for key in SHEET_SOURCE_KEYS})

print('All analytical sources are ready from the repository.')
memory_report('after input preparation')


### 1.3 Input inventory


In [ ]:
rows = []
for key, path in RAW_LOCAL.items():
    path = Path(path)
    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file()]
        size_bytes = sum(p.stat().st_size for p in files)
        n_files = len(files)
    else:
        size_bytes = path.stat().st_size
        n_files = 1
    rows.append({
        "source_key": key,
        "cached_path": str(path),
        "files": n_files,
        "size_mb": size_bytes / 1024**2,
    })

raw_data_inventory = pd.DataFrame(rows).sort_values("source_key").reset_index(drop=True)
display(raw_data_inventory)


## 2. Feature assembly

Sources are loaded sequentially, geometry is stored once per grid, static intersections are computed once per unique coarse cell, and intermediate objects are released after use.

### 2.1 Static grids and GRACE tables

In [ ]:
elevation_gdf = gpd.read_file(find_shapefile(RAW_LOCAL["elevation_zip"], "dem.shp"))
lithology_gdf = gpd.read_file(find_shapefile(
    RAW_LOCAL["lithology_zip"], "lithology.shp"
))
land_cover_grid_gdf = gpd.read_file(find_shapefile(
    RAW_LOCAL["land_cover_grid_zip"], "land_cover_grid_PL_UA_BY.shp"
))
gdf_grid = gpd.read_file(find_shapefile(RAW_LOCAL["grid_025_zip"]))
gdf_grid = gdf_grid.rename(columns={"ID": "grid"})

(
    elevation_gdf,
    lithology_gdf,
    land_cover_grid_gdf,
    gdf_grid,
) = harmonize_crs(
    elevation_gdf,
    lithology_gdf,
    land_cover_grid_gdf,
    gdf_grid,
)

df_grace_or = load_grace_cached(RAW_LOCAL["grace_original_sheet"], "grace_or")
df_grace_fl = load_grace_cached(RAW_LOCAL["grace_filled_sheet"], "grace_fl")
grace_or_gdf = add_grid_polygons(df_grace_or, res=0.25)
grace_fl_gdf = add_grid_polygons(df_grace_fl, res=0.25)

del df_grace_or, df_grace_fl
gc.collect()

print("Static 0.1° cells:", len(land_cover_grid_gdf))
print("GRACE original rows:", len(grace_or_gdf))
print("GRACE filled rows:", len(grace_fl_gdf))
memory_report("after loading static + GRACE")


### 2.2 Static land cover, lithology, and elevation

The 0.1° land-cover grid provides the reference geometry. Lithology and elevation are joined by the shared grid identifier.

In [ ]:
for frame in (land_cover_grid_gdf, lithology_gdf, elevation_gdf):
    frame["id"] = frame["id"].astype(str)

lith_attrs = (
    pd.DataFrame(lithology_gdf.drop(columns=lithology_gdf.geometry.name, errors="ignore"))
    .drop_duplicates("id")
)
elev_attrs = (
    pd.DataFrame(elevation_gdf.drop(columns=elevation_gdf.geometry.name, errors="ignore"))
    .drop_duplicates("id")
)

static_feature_gdf = land_cover_grid_gdf.merge(
    lith_attrs, on="id", how="left", suffixes=("", "_lith")
).merge(
    elev_attrs, on="id", how="left", suffixes=("", "_elev")
)
static_feature_gdf = gpd.GeoDataFrame(
    static_feature_gdf,
    geometry=land_cover_grid_gdf.geometry.name,
    crs=land_cover_grid_gdf.crs,
).rename(columns={
    "id": "grid",
    "name": "lc_name",
    "percent": "lc_percent",
    "class": "lc_class",
    "class_geom": "lc_class_geom",
    "class_lith": "lith_class",
    "percent_lith": "lith_percent",
    "value": "elevation",
})

# Raw right-hand frames are no longer needed.
del lithology_gdf, elevation_gdf, lith_attrs, elev_attrs, land_cover_grid_gdf
gc.collect()
print("Static merged grid:", static_feature_gdf.shape)


### 2.3 Dynamic predictors

Yearly source files are reshaped to monthly tables. Geometry is maintained in a separate lookup and attached after the temporal merges.

In [ ]:
MONTH_NAMES = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
MONTH_TO_NUM = {name: i + 1 for i, name in enumerate(MONTH_NAMES)}


def reshape_p_q_et_year(gdf: gpd.GeoDataFrame, year: int):
    value_cols = (
        [f"P_{i}" for i in range(1, 13)]
        + [f"Q_{i}" for i in range(1, 13)]
        + [f"ET_{i}" for i in range(1, 13)]
    )
    attrs = pd.DataFrame(gdf.drop(columns=gdf.geometry.name, errors="ignore"))
    attrs["year"] = year
    long = attrs.melt(
        id_vars=["year", "grid"],
        value_vars=value_cols,
        var_name="variable",
        value_name="value",
    )
    long["type"] = long["variable"].str.extract(r"([A-Z]+)")
    long["month"] = long["variable"].str.extract(r"_(\d+)").astype("int8")
    monthly = long.pivot_table(
        index=["year", "grid", "month"],
        columns="type",
        values="value",
        aggfunc="first",
    ).reset_index()
    monthly["date"] = pd.to_datetime(
        dict(year=monthly["year"], month=monthly["month"], day=1)
    )
    return monthly


def reshape_named_months_year(gdf: gpd.GeoDataFrame, year: int, value_name: str):
    attrs = pd.DataFrame(gdf.drop(columns=gdf.geometry.name, errors="ignore"))
    attrs = attrs.rename(columns={"id": "grid"})
    rename_map = {
        col: f"{value_name}_{MONTH_TO_NUM[col.lower()]}"
        for col in attrs.columns
        if col.lower() in MONTH_TO_NUM
    }
    attrs = attrs.rename(columns=rename_map)
    attrs["year"] = year
    monthly_cols = [f"{value_name}_{i}" for i in range(1, 13)]
    long = attrs.melt(
        id_vars=["year", "grid"],
        value_vars=monthly_cols,
        var_name="variable",
        value_name=value_name,
    )
    long["month"] = long["variable"].str.extract(r"_(\d+)").astype("int8")
    long["date"] = pd.to_datetime(
        dict(year=long["year"], month=long["month"], day=1)
    )
    return long[["year", "grid", "month", "date", value_name]]


# P, Q and ET: read and release one annual shapefile at a time.
pqe_parts = []
grid01_geometry = None
for year in range(2013, 2024):
    annual = gpd.read_file(find_shapefile(
        RAW_LOCAL["hydroclimate_zip"], f"{year}_data.shp"
    ))
    if annual.crs is None:
        annual = annual.set_crs("EPSG:4326")
    elif str(annual.crs) != "EPSG:4326":
        annual = annual.to_crs("EPSG:4326")
    if grid01_geometry is None:
        grid01_geometry = annual[["grid", annual.geometry.name]].drop_duplicates("grid")
        if grid01_geometry.geometry.name != "geometry":
            grid01_geometry = grid01_geometry.rename_geometry("geometry")
    pqe_parts.append(reshape_p_q_et_year(annual, year))
    del annual
    gc.collect()
prec_temp_runoff_df = pd.concat(pqe_parts, ignore_index=True)
del pqe_parts

# Alternative runoff.
qnew_parts = []
for year in range(2013, 2024):
    annual = gpd.read_file(find_shapefile(
        RAW_LOCAL["new_runoff_zip"], f"R_{year}.shp"
    ))
    qnew_parts.append(reshape_named_months_year(annual, year, "Q_new"))
    del annual
new_runoff_df = pd.concat(qnew_parts, ignore_index=True)
del qnew_parts

# Land-surface temperature.
temp_parts = []
for year in range(2013, 2024):
    annual = gpd.read_file(find_shapefile(
        RAW_LOCAL["land_surface_temperature_zip"], f"LST_{year}.shp"
    ))
    temp_parts.append(reshape_named_months_year(annual, year, "temp"))
    del annual
land_surface_temperature_df = pd.concat(temp_parts, ignore_index=True)
del temp_parts
gc.collect()

# Soil moisture remains a compact long table; geometry is attached once from the grid.
df_sm_raw = pd.read_csv(RAW_LOCAL["soil_moisture_sheet"])
df_sm = df_sm_raw.melt(id_vars=["grid"], var_name="date", value_name="SM")
df_sm["date"] = pd.to_datetime(df_sm["date"], errors="coerce")
sm_gdf = df_sm.merge(gdf_grid[["grid", "geometry"]], on="grid", how="left")
sm_gdf = gpd.GeoDataFrame(sm_gdf, geometry="geometry", crs=gdf_grid.crs)
del df_sm_raw, df_sm

# Use centimetres consistently for every hydrological depth variable.
# The raw products are converted once, immediately after loading.
HYDROLOGICAL_DEPTH_UNIT = 'cm'
RAW_DEPTH_TO_CM = 0.1

for column in ['P', 'Q', 'ET']:
    prec_temp_runoff_df[column] = (
        pd.to_numeric(
            prec_temp_runoff_df[column],
            errors='coerce',
        )
        * RAW_DEPTH_TO_CM
    )

new_runoff_df['Q_new'] = (
    pd.to_numeric(
        new_runoff_df['Q_new'],
        errors='coerce',
    )
    * RAW_DEPTH_TO_CM
)

sm_gdf['SM'] = (
    pd.to_numeric(
        sm_gdf['SM'],
        errors='coerce',
    )
    * RAW_DEPTH_TO_CM
)

print("P-Q-ET monthly rows:", len(prec_temp_runoff_df))
print("Alternative-runoff rows:", len(new_runoff_df))
print("Temperature rows:", len(land_surface_temperature_df))
print("Soil-moisture rows:", len(sm_gdf))
memory_report("after sequential dynamic loading")


### 2.4 Monthly 0.1° predictor table

Hydroclimatic variables are merged by grid cell and month, followed by geometry attachment.

In [ ]:
for frame in (prec_temp_runoff_df, new_runoff_df, land_surface_temperature_df):
    frame["grid"] = frame["grid"].astype(float).astype(str)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")

static_attrs = pd.DataFrame(
    static_feature_gdf.drop(columns=static_feature_gdf.geometry.name, errors="ignore")
)
static_attrs["grid"] = static_attrs["grid"].astype(float).astype(str)
grid01_geometry["grid"] = grid01_geometry["grid"].astype(float).astype(str)

feature_01_df = (
    prec_temp_runoff_df
    .merge(
        new_runoff_df.drop(columns=["year", "month"], errors="ignore"),
        on=["grid", "date"],
        how="left",
    )
    .merge(
        land_surface_temperature_df.drop(columns=["year", "month"], errors="ignore"),
        on=["grid", "date"],
        how="left",
    )
    .merge(static_attrs, on="grid", how="left")
)
feature_01_gdf = feature_01_df.merge(grid01_geometry, on="grid", how="left")
feature_01_gdf = gpd.GeoDataFrame(
    feature_01_gdf, geometry="geometry", crs=grid01_geometry.crs
)

# Release large merge inputs.
del (
    prec_temp_runoff_df,
    new_runoff_df,
    land_surface_temperature_df,
    feature_01_df,
    static_attrs,
)
gc.collect()

print("Merged 0.1° monthly predictor table:", feature_01_gdf.shape)
memory_report("after 0.1° predictor merge")


### 2.5 GRACE date normalization and product merge

In [ ]:
def norm_dates(fl, original):
    fl = fl.copy()
    original = original.copy()
    fl["gr_fl_date"] = fl["date"]
    original["gr_or_date"] = original["date"]
    fl["date"] = pd.to_datetime(fl["date"], format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m")
    original["date"] = pd.to_datetime(original["date"], format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m")
    return fl, original


def make_cell_keys(df, lon_col="lon", lat_col="lat", res=0.25, round_to=5):
    out = df.copy()
    lon = pd.to_numeric(out[lon_col], errors="coerce").to_numpy(dtype=float)
    lat = pd.to_numeric(out[lat_col], errors="coerce").to_numpy(dtype=float)
    eps = 1e-12
    out["lon0"] = np.round(np.floor((lon + eps) / res) * res, round_to)
    out["lat0"] = np.round(np.floor((lat + eps) / res) * res, round_to)
    return out


grace_fl_attrs, grace_or_attrs = norm_dates(grace_fl_gdf, grace_or_gdf)
grace_fl_attrs = make_cell_keys(grace_fl_attrs).drop_duplicates(["lon0", "lat0", "date"])
grace_or_attrs = make_cell_keys(grace_or_attrs).drop_duplicates(["lon0", "lat0", "date"])
grace_or_attrs = grace_or_attrs.rename(columns={"geometry": "geometry_grace"})

grace_joined_gdf = grace_fl_attrs.merge(
    grace_or_attrs,
    on=["lon0", "lat0", "date"],
    how="outer",
    suffixes=("_fl", "_or"),
)[[
    "date", "geometry", "geometry_grace", "lon0", "lat0",
    "grace_fl", "gr_fl_date", "grace_or", "gr_or_date",
]]

# Use the filled-product cell polygon only when the original-product geometry is absent.
grace_joined_gdf["geometry_grace"] = grace_joined_gdf["geometry_grace"].where(
    grace_joined_gdf["geometry_grace"].notna(), grace_joined_gdf["geometry"]
)
grace_joined_gdf["cell_025_key"] = (
    grace_joined_gdf["lon0"].round(5).astype(str)
    + "_" + grace_joined_gdf["lat0"].round(5).astype(str)
)

print("Combined GRACE monthly rows:", len(grace_joined_gdf))

### 2.6 Static attributes for 0.25° GRACE cells

Static overlays are calculated for the 119 unique 0.25° cells and joined to the monthly GRACE records.

In [ ]:
# Compute static overlays once per unique GRACE cell.
grace_cells = (
    grace_joined_gdf[["cell_025_key", "geometry_grace"]]
    .dropna(subset=["geometry_grace"])
    .drop_duplicates("cell_025_key")
)
grace_cells = gpd.GeoDataFrame(
    grace_cells,
    geometry="geometry_grace",
    crs=getattr(grace_fl_gdf, "crs", "EPSG:4326"),
)

static_overlay = static_feature_gdf
if static_overlay.crs != grace_cells.crs:
    static_overlay = static_overlay.to_crs(grace_cells.crs)

# Rectangular grid cells should already be valid; buffer only invalid geometries.
invalid_static = ~static_overlay.geometry.is_valid
if invalid_static.any():
    static_overlay = static_overlay.copy()
    static_overlay.loc[invalid_static, "geometry"] = static_overlay.loc[invalid_static].buffer(0)

intersections = gpd.overlay(
    static_overlay,
    grace_cells,
    how="intersection",
    keep_geom_type=False,
).to_crs("ESRI:54009")
intersections["piece_area"] = intersections.geometry.area

continuous_cols = ["lc_percent", "lith_percent", "elevation"]
continuous_parts = []
area_total = intersections.groupby("cell_025_key", observed=True)["piece_area"].sum()
for column in continuous_cols:
    weighted = (
        pd.to_numeric(intersections[column], errors="coerce")
        * intersections["piece_area"]
    )
    numerator = weighted.groupby(intersections["cell_025_key"], observed=True).sum()
    continuous_parts.append((numerator / area_total).rename(f"{column}_awm"))
continuous_aggs = pd.concat(continuous_parts, axis=1).reset_index()

categorical_cols = ["lc_class", "lc_class_geom", "lith_class"]
category_parts = []
for column in categorical_cols:
    coverage = (
        intersections.groupby(["cell_025_key", column], dropna=False, observed=True)["piece_area"]
        .sum()
        .reset_index()
        .sort_values(["cell_025_key", "piece_area"], ascending=[True, False])
        .drop_duplicates("cell_025_key")
        [["cell_025_key", column]]
        .rename(columns={column: f"{column}_majority"})
    )
    category_parts.append(coverage)

static_025 = continuous_aggs
for part in category_parts:
    static_025 = static_025.merge(part, on="cell_025_key", how="outer")

agg_out = grace_joined_gdf.merge(static_025, on="cell_025_key", how="left")
agg_out = gpd.GeoDataFrame(
    agg_out.drop(columns=["geometry"], errors="ignore"),
    geometry="geometry_grace",
    crs=grace_cells.crs,
)

print("Unique GRACE cells overlaid:", len(grace_cells))
print("Monthly GRACE + static rows:", len(agg_out))

# Release the expensive overlay products immediately.
del intersections, static_025, continuous_aggs, continuous_parts, category_parts, grace_cells
gc.collect()
memory_report("after unique-cell static overlay")

gdf_ml_025 = agg_out.rename_geometry("geometry")
gdf_ml_025["date"] = pd.to_datetime(gdf_ml_025["date"], errors="coerce")
print("Imputation input shape:", gdf_ml_025.shape)


### 2.7 Feature-table summary

The following inventory describes the model-ready datasets available before imputation.

In [ ]:
def _safe_date_extent(df, date_col="date"):
    if date_col not in df.columns:
        return pd.NaT, pd.NaT
    dates = pd.to_datetime(df[date_col], errors="coerce")
    return dates.min(), dates.max()


def _feature_summary_row(name, df, resolution, id_columns=None, date_col="date"):
    start, end = _safe_date_extent(df, date_col=date_col)
    n_cells = (
        int(df[id_columns].drop_duplicates().shape[0])
        if id_columns and all(col in df.columns for col in id_columns)
        else pd.NA
    )
    return {
        "table": name,
        "resolution": resolution,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "spatial_units": n_cells,
        "start_date": start,
        "end_date": end,
    }

feature_engineering_summary = pd.DataFrame([
    _feature_summary_row("Static feature grid", static_feature_gdf, "~0.1°", ["grid"], "__none__"),
    _feature_summary_row("Monthly predictor grid", feature_01_gdf, "0.1°", ["grid"]),
    _feature_summary_row("GRACE + static imputation input", gdf_ml_025, "0.25°", ["lon0", "lat0"]),
])
display(feature_engineering_summary)

# The static overlay source and reference grid are no longer needed.
del static_feature_gdf, gdf_grid, agg_out
gc.collect()


## 3. GRACE imputation and evaluation

The imputation stage standardizes observation dates, constructs a continuous monthly panel, creates GRACE lags, evaluates chronological and pseudo-gap holdouts, fills missing months recursively, and passes the completed record to downscaling.

### 3.1 Canonical monthly timestamps

In [ ]:
def canonical_midmonth(ts: pd.Timestamp):
    if pd.isna(ts):
        return pd.NaT
    day = 14 if ts.days_in_month == 28 else (16 if ts.days_in_month == 31 else 15)
    return pd.Timestamp(ts.year, ts.month, day)


def interpolate_grace_to_midmonth(group: pd.DataFrame) -> pd.DataFrame:
    group = group.dropna(subset=["gr_or_date"]).sort_values("gr_or_date")
    if group.empty or group["grace_or"].isna().all():
        return group.iloc[0:0]

    group = group.set_index("gr_or_date")
    periods = group.index.to_period("M").unique().sort_values()
    mid_dates = pd.DatetimeIndex([
        canonical_midmonth(period.to_timestamp("M")) for period in periods
    ])
    tmp = group.reindex(group.index.union(mid_dates).sort_values())
    tmp["grace_or"] = tmp["grace_or"].interpolate(method="time")
    other = [column for column in tmp.columns if column != "grace_or"]
    tmp[other] = tmp[other].ffill().bfill()
    out = tmp.loc[mid_dates].copy()
    out["gr_or_date"] = out.index
    out["date"] = out.index
    out["is_midmonth_grace_or"] = True
    return out.reset_index(drop=True)


gdf_mid = gdf_ml_025.copy()
gdf_mid["location_id"] = (
    gdf_mid["lon0"].round(3).astype(str)
    + "_" + gdf_mid["lat0"].round(3).astype(str)
)
gdf_mid["gr_or_date"] = pd.to_datetime(gdf_mid["gr_or_date"], errors="coerce").dt.normalize()

midmonth_parts = [
    interpolate_grace_to_midmonth(group)
    for _, group in gdf_mid.groupby("location_id", sort=False)
]
gdf_ml_025 = gpd.GeoDataFrame(
    pd.concat(midmonth_parts, ignore_index=True),
    geometry="geometry",
    crs=gdf_mid.crs,
)
del midmonth_parts, gdf_mid
gc.collect()

print("After mid-month adjustment:", gdf_ml_025.shape)


### 3.2 Continuous monthly panel and lag features

For each cell, one observation is retained per canonical month. Missing months are inserted and GRACE lags of one to three months are created. Static and hydroclimatic predictors are reserved for downscaling.

In [ ]:
def build_panel_from_raw(gdf_ml_025: gpd.GeoDataFrame):
    needed = [
        'date',
        'gr_or_date',
        'lon0',
        'lat0',
        'grace_fl',
        'grace_or',
        'geometry',
    ]
    df = gdf_ml_025[[column for column in needed if column in gdf_ml_025.columns]].copy()
    df["gr_or_date"] = pd.to_datetime(df["gr_or_date"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["cell_id"] = (
        df["lon0"].round(3).astype(str)
        + "_" + df["lat0"].round(3).astype(str)
    )
    df["obs_date_raw"] = df["gr_or_date"].fillna(df["date"])
    df["month_key"] = df["obs_date_raw"].dt.to_period("M").astype(str)
    df["mid_month_date"] = df["obs_date_raw"].apply(canonical_midmonth)
    df["days_from_mid"] = (
        df["obs_date_raw"] - df["mid_month_date"]
    ).abs().dt.days

    statics = (
        df[
            [
                'cell_id',
                'lon0',
                'lat0',
                'geometry',
            ]
        ]
        .drop_duplicates('cell_id')
    )

    # Geometry is not repeated through the panel construction.
    df = df.drop(columns=["geometry"], errors="ignore")
    closest = (
        df.sort_values(["cell_id", "month_key", "days_from_mid"])
        .groupby(["cell_id", "month_key"], as_index=False)
        .first()
    )

    panel_parts = []
    fill_cols = [
        'lon0',
        'lat0',
        'grace_fl',
    ]
    for cell_id, sub in closest.groupby("cell_id", sort=False):
        sub = sub.sort_values("mid_month_date")
        if sub["mid_month_date"].isna().all():
            continue
        months = pd.period_range(
            sub["mid_month_date"].min().to_period("M"),
            sub["mid_month_date"].max().to_period("M"),
            freq="M",
        )
        base = pd.DataFrame({"cell_id": cell_id, "month_key": months.astype(str)})
        merged = base.merge(sub, on=["cell_id", "month_key"], how="left")
        merged["mid_month_date"] = pd.to_datetime(
            merged["month_key"] + "-01"
        ).apply(canonical_midmonth)
        for column in fill_cols:
            if column in merged.columns:
                merged[column] = merged[column].ffill().bfill()
        panel_parts.append(merged)

    panel = pd.concat(panel_parts, ignore_index=True)
    panel = panel.sort_values(["cell_id", "mid_month_date"]).reset_index(drop=True)
    panel["grace_or_obs"] = panel["grace_or"]
    grouped = panel.groupby("cell_id", sort=False)["grace_or"]
    panel["grace_or_lag1"] = grouped.shift(1)
    panel["grace_or_lag2"] = grouped.shift(2)
    panel["grace_or_lag3"] = grouped.shift(3)
    panel["month"] = panel["mid_month_date"].dt.month.astype("int8")
    return panel, statics


### 3.3 Random Forest imputation

A global Random Forest is fitted to observed rows with complete lag histories. Missing values are reconstructed sequentially so each prediction can supply later lag values. The chronological test uses the final 20% of each cell series. Tree dispersion and paired-bootstrap intervals quantify model and sampling uncertainty.

In [ ]:
def train_rf_and_impute(panel):
    feature_cols_num = [
        'grace_or_lag1',
        'grace_or_lag2',
        'grace_or_lag3',
        'month',
    ]
    feature_cols_cat = []
    feature_cols = feature_cols_num

    supervised_mask = (
        panel['grace_or'].notna()
        & panel['grace_or_lag1'].notna()
        & panel['grace_or_lag2'].notna()
        & panel['grace_or_lag3'].notna()
    )
    train_X = panel.loc[supervised_mask, feature_cols]
    train_y = panel.loc[supervised_mask, 'grace_or']

    preprocess = ColumnTransformer([
        ('num', 'passthrough', feature_cols_num),
    ])
    fill_model = Pipeline([
        ('prep', preprocess),
        ('rf', RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])
    fill_model.fit(train_X, train_y)

    panel = panel.reset_index(drop=True)
    panel['imputed_flag'] = False
    panel['grace_or_imputed_std'] = np.nan
    panel['grace_or_imputed_lower'] = np.nan
    panel['grace_or_imputed_upper'] = np.nan

    # Recursive filling is performed cell by cell. The uncertainty stored here
    # is the conditional tree-ensemble dispersion at each prediction step.
    for _, indices in panel.groupby('cell_id', sort=False).groups.items():
        ordered = (
            panel.loc[list(indices)]
            .sort_values('mid_month_date')
            .index
        )
        history = []

        for index in ordered:
            value = panel.at[index, 'grace_or']
            if pd.notna(value):
                history.append(float(value))
                continue
            if len(history) < 3:
                continue

            l1, l2, l3 = history[-1], history[-2], history[-3]
            panel.at[index, 'grace_or_lag1'] = l1
            panel.at[index, 'grace_or_lag2'] = l2
            panel.at[index, 'grace_or_lag3'] = l3
            panel.at[index, 'month'] = panel.at[index, 'mid_month_date'].month

            (
                prediction,
                prediction_std,
                prediction_lower,
                prediction_upper,
            ) = predict_rf_distribution(
                fill_model,
                panel.loc[[index], feature_cols],
            )

            panel.at[index, 'grace_or'] = float(prediction[0])
            panel.at[index, 'grace_or_imputed_std'] = float(prediction_std[0])
            panel.at[index, 'grace_or_imputed_lower'] = float(prediction_lower[0])
            panel.at[index, 'grace_or_imputed_upper'] = float(prediction_upper[0])
            panel.at[index, 'imputed_flag'] = True
            history.append(float(prediction[0]))

    panel['grace_or_imputed'] = panel['grace_or']

    score_mask = (
        panel['grace_or_obs'].notna()
        & panel['grace_or_lag1'].notna()
        & panel['grace_or_lag2'].notna()
        & panel['grace_or_lag3'].notna()
    )
    supervised = (
        panel.loc[score_mask]
        .sort_values('mid_month_date')
        .reset_index(drop=True)
    )
    cut = int(len(supervised) * 0.8)
    train_part = supervised.iloc[:cut]
    test_part = supervised.iloc[cut:]

    preprocess_eval = ColumnTransformer([
        ('num', 'passthrough', feature_cols_num),
    ])
    model_eval = Pipeline([
        ('prep', preprocess_eval),
        ('rf', RandomForestRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])
    model_eval.fit(
        train_part[feature_cols],
        train_part['grace_or_obs'],
    )

    (
        y_pred,
        y_pred_std,
        y_pred_lower,
        y_pred_upper,
    ) = predict_rf_distribution(
        model_eval,
        test_part[feature_cols],
    )

    imputation_test_predictions = test_part[
        ['cell_id', 'mid_month_date', 'grace_or_obs']
    ].copy()
    imputation_test_predictions = imputation_test_predictions.rename(
        columns={'grace_or_obs': 'y_true'}
    )
    imputation_test_predictions['y_pred'] = y_pred
    imputation_test_predictions['y_pred_std'] = y_pred_std
    imputation_test_predictions['y_pred_lower'] = y_pred_lower
    imputation_test_predictions['y_pred_upper'] = y_pred_upper
    imputation_test_predictions['interval_covered'] = (
        imputation_test_predictions['y_true']
        .between(
            imputation_test_predictions['y_pred_lower'],
            imputation_test_predictions['y_pred_upper'],
        )
    )

    metric_uncertainty = bootstrap_regression_metrics(
        test_part['grace_or_obs'],
        y_pred,
        random_state=RANDOM_STATE,
    )

    eval_metrics = {
        'R2': metric_uncertainty['r2']['estimate'],
        'R2_ci_lower': metric_uncertainty['r2']['ci_lower'],
        'R2_ci_upper': metric_uncertainty['r2']['ci_upper'],
        'MAE': metric_uncertainty['mae']['estimate'],
        'MAE_ci_lower': metric_uncertainty['mae']['ci_lower'],
        'MAE_ci_upper': metric_uncertainty['mae']['ci_upper'],
        'RMSE': metric_uncertainty['rmse']['estimate'],
        'RMSE_ci_lower': metric_uncertainty['rmse']['ci_lower'],
        'RMSE_ci_upper': metric_uncertainty['rmse']['ci_upper'],
        'mean_prediction_std': float(np.nanmean(y_pred_std)),
        'median_prediction_std': float(np.nanmedian(y_pred_std)),
        'mean_interval_width': float(np.nanmean(y_pred_upper - y_pred_lower)),
        'interval_coverage_95': float(
            imputation_test_predictions['interval_covered'].mean()
        ),
        'test_start_date': test_part['mid_month_date'].min(),
        'test_end_date': test_part['mid_month_date'].max(),
        'n_train_rows': int(len(train_part)),
        'n_test_rows': int(len(test_part)),
        'n_train_cells': int(train_part['cell_id'].nunique()),
        'n_test_cells': int(test_part['cell_id'].nunique()),
    }

    del fill_model, train_X, train_y
    gc.collect()

    return (
        panel,
        model_eval,
        preprocess_eval,
        feature_cols_num,
        feature_cols_cat,
        eval_metrics['test_start_date'],
        eval_metrics['test_end_date'],
        eval_metrics,
        imputation_test_predictions,
    )


### 3.4 Execute imputation and restore geometry

In [ ]:
def attach_geometry(panel, statics, crs):
    panel_final = panel.merge(statics, on="cell_id", how="left", suffixes=("", "_static"))
    geometry_col = "geometry_static" if "geometry_static" in panel_final.columns else "geometry"
    return gpd.GeoDataFrame(panel_final, geometry=geometry_col, crs=crs)


def get_feature_names(preprocessor, numeric_cols, cat_cols):
    names = list(numeric_cols)
    for name, transformer, columns in preprocessor.transformers_:
        if name == "cat" and hasattr(transformer, "get_feature_names_out"):
            names.extend(transformer.get_feature_names_out(columns).tolist())
    return names

panel, statics = build_panel_from_raw(gdf_ml_025)
(
    panel,
    model_eval,
    preprocess_eval,
    feature_cols_num,
    feature_cols_cat,
    test_start_date,
    test_end_date,
    eval_metrics,
    imputation_test_predictions,
) = train_rf_and_impute(panel)
panel_gdf = attach_geometry(panel, statics, gdf_ml_025.crs)

# Release source tables after geometry attachment to reduce peak memory.
del statics
gc.collect()
print("GRACE imputation completed.")
memory_report("after imputation")


### 3.5 Imputation results

Performance metrics use paired-bootstrap 95% intervals. Prediction intervals are based on Random Forest tree dispersion and exclude GRACE measurement uncertainty.

In [ ]:
imputation_performance_table = pd.DataFrame([{
    'model': 'Random forest',
    'validation': 'Chronological 80/20 holdout',
    'train_rows': eval_metrics['n_train_rows'],
    'test_rows': eval_metrics['n_test_rows'],
    'train_cells': eval_metrics['n_train_cells'],
    'test_cells': eval_metrics['n_test_cells'],
    'test_start': eval_metrics['test_start_date'],
    'test_end': eval_metrics['test_end_date'],
    'R2': eval_metrics['R2'],
    'R2_95_CI_lower': eval_metrics['R2_ci_lower'],
    'R2_95_CI_upper': eval_metrics['R2_ci_upper'],
    'MAE_cm': eval_metrics['MAE'],
    'MAE_95_CI_lower_cm': eval_metrics['MAE_ci_lower'],
    'MAE_95_CI_upper_cm': eval_metrics['MAE_ci_upper'],
    'RMSE_cm': eval_metrics['RMSE'],
    'RMSE_95_CI_lower_cm': eval_metrics['RMSE_ci_lower'],
    'RMSE_95_CI_upper_cm': eval_metrics['RMSE_ci_upper'],
}])

imputation_uncertainty_table = pd.DataFrame([{
    'uncertainty_type': 'Random Forest tree-ensemble dispersion',
    'mean_prediction_std_cm': eval_metrics['mean_prediction_std'],
    'median_prediction_std_cm': eval_metrics['median_prediction_std'],
    'mean_95_interval_width_cm': eval_metrics['mean_interval_width'],
    'empirical_95_interval_coverage': eval_metrics['interval_coverage_95'],
}])

n_total = int(len(panel))
n_observed = int(panel['grace_or_obs'].notna().sum())
n_original_gaps = int(panel['grace_or_obs'].isna().sum())
n_imputed = int(panel['imputed_flag'].fillna(False).sum())
n_remaining = int(panel['grace_or_imputed'].isna().sum())
imputation_rate = (
    100.0 * n_imputed / n_original_gaps
    if n_original_gaps
    else np.nan
)

imputation_coverage_table = pd.DataFrame([{
    'panel_rows': n_total,
    'spatial_cells': int(panel['cell_id'].nunique()),
    'observed_rows': n_observed,
    'original_missing_rows': n_original_gaps,
    'imputed_rows': n_imputed,
    'remaining_missing_rows': n_remaining,
    'filled_gap_percent': imputation_rate,
}])

panel_year = panel.assign(
    year=pd.to_datetime(panel['mid_month_date']).dt.year,
    interval_width=(
        panel['grace_or_imputed_upper']
        - panel['grace_or_imputed_lower']
    ),
)
imputation_by_year_table = (
    panel_year
    .groupby('year', as_index=False)
    .agg(
        rows=('cell_id', 'size'),
        observed=(
            'grace_or_obs',
            lambda values: int(values.notna().sum()),
        ),
        imputed=(
            'imputed_flag',
            lambda values: int(values.fillna(False).sum()),
        ),
        remaining_missing=(
            'grace_or_imputed',
            lambda values: int(values.isna().sum()),
        ),
        mean_imputation_std_cm=('grace_or_imputed_std', 'mean'),
        mean_95_interval_width_cm=('interval_width', 'mean'),
    )
)

with pd.option_context('display.float_format', '{:.4f}'.format):
    display_if_requested(imputation_performance_table)
    display_if_requested(imputation_uncertainty_table)
    display_if_requested(imputation_coverage_table)
    display_if_requested(imputation_by_year_table)


### 3.6 Rolling-origin pseudo-gap experiments

Observed intervals of 1, 2, 3, and 11 months are withheld from a shared, temporally stratified sample of up to 100 cell-window starts. Each model is trained only on observations preceding the gap and reconstructs the interval recursively from three observed lead-in months.

The same start windows are used for every gap length. A persistence benchmark is evaluated on the identical samples. Cluster bootstrap resampling of complete cell-window events preserves dependence among recursively reconstructed months.

In [ ]:
def find_pseudo_gap_candidates(
    panel,
    gap_length,
    minimum_training_years=5,
):
    """
    Identify observed consecutive windows with three observed lead-in months.

    Candidate windows begin only after a minimum historical training period.
    """
    source = (
        panel[
            [
                'cell_id',
                'mid_month_date',
                'grace_or_obs',
            ]
        ]
        .copy()
        .sort_values(
            [
                'cell_id',
                'mid_month_date',
            ]
        )
    )
    source['mid_month_date'] = pd.to_datetime(
        source['mid_month_date'],
        errors='coerce',
    )

    minimum_start_date = (
        source['mid_month_date'].min()
        + pd.DateOffset(years=minimum_training_years)
    )
    candidate_rows = []

    for cell_id, group in source.groupby(
        'cell_id',
        sort=False,
    ):
        group = (
            group
            .sort_values('mid_month_date')
            .reset_index(drop=True)
        )
        month_number = (
            group['mid_month_date'].dt.year * 12
            + group['mid_month_date'].dt.month
        ).to_numpy()
        observed = group[
            'grace_or_obs'
        ].notna().to_numpy()

        for start_position in range(
            3,
            len(group) - gap_length + 1,
        ):
            start_date = group.loc[
                start_position,
                'mid_month_date',
            ]
            if start_date < minimum_start_date:
                continue

            context_start = start_position - 3
            context_end = start_position + gap_length

            dates_are_consecutive = np.all(
                np.diff(
                    month_number[
                        context_start:context_end
                    ]
                )
                == 1
            )
            values_are_observed = np.all(
                observed[
                    context_start:context_end
                ]
            )

            if not (
                dates_are_consecutive
                and values_are_observed
            ):
                continue

            candidate_rows.append({
                'cell_id': cell_id,
                'gap_length_months': int(
                    gap_length
                ),
                'start_date': start_date,
                'end_date': group.loc[
                    start_position
                    + gap_length
                    - 1,
                    'mid_month_date',
                ],
            })

    return pd.DataFrame(candidate_rows)


def add_pseudo_gap_selection_strata(
    candidates,
    temporal_strata=PSEUDO_GAP_TEMPORAL_STRATA,
):
    """
    Divide candidate starts into temporal-period and calendar-quarter strata.
    """
    result = candidates.copy()
    result['start_date'] = pd.to_datetime(
        result['start_date'],
        errors='coerce',
    )
    month_index = (
        result['start_date'].dt.year * 12
        + result['start_date'].dt.month
    ).astype(float)

    minimum_month = float(
        month_index.min()
    )
    maximum_month = float(
        month_index.max()
    )
    month_span = max(
        maximum_month - minimum_month,
        1.0,
    )

    temporal_index = np.floor(
        temporal_strata
        * (month_index - minimum_month)
        / (month_span + 1.0)
    ).astype(int)
    temporal_index = np.clip(
        temporal_index,
        0,
        temporal_strata - 1,
    )

    result['selection_period'] = (
        temporal_index + 1
    ).astype('int8')
    result['calendar_quarter'] = (
        result['start_date']
        .dt.quarter
        .astype('int8')
    )
    result['selection_stratum'] = (
        'T'
        + result['selection_period'].astype(str)
        + '_Q'
        + result['calendar_quarter'].astype(str)
    )

    return result


def select_shared_pseudo_gap_windows(
    candidates,
    maximum_windows=PSEUDO_GAP_WINDOWS_PER_LENGTH,
    random_state=RANDOM_STATE,
):
    """
    Select one window per cell while balancing record period and quarter.

    Selection is based on candidates valid for the longest requested gap.
    The same cell-window starts are subsequently used for every gap length.
    """
    if candidates.empty:
        raise ValueError(
            'No valid observed windows were found for the longest '
            'requested pseudo-gap.'
        )

    candidates = add_pseudo_gap_selection_strata(
        candidates
    )
    rng = np.random.default_rng(
        random_state + 4200
    )

    cell_ids = candidates[
        'cell_id'
    ].drop_duplicates().to_numpy()
    rng.shuffle(cell_ids)

    stratum_counts = {}
    selected_rows = []

    for cell_id in cell_ids:
        cell_candidates = (
            candidates.loc[
                candidates['cell_id'].eq(cell_id)
            ]
            .reset_index(drop=True)
        )
        available_strata = cell_candidates[
            'selection_stratum'
        ].drop_duplicates().tolist()

        minimum_count = min(
            stratum_counts.get(
                stratum,
                0,
            )
            for stratum in available_strata
        )
        preferred_strata = [
            stratum
            for stratum in available_strata
            if stratum_counts.get(
                stratum,
                0,
            ) == minimum_count
        ]
        chosen_stratum = preferred_strata[
            int(
                rng.integers(
                    0,
                    len(preferred_strata),
                )
            )
        ]

        eligible = (
            cell_candidates.loc[
                cell_candidates[
                    'selection_stratum'
                ].eq(chosen_stratum)
            ]
            .reset_index(drop=True)
        )
        chosen_row = eligible.iloc[
            int(
                rng.integers(
                    0,
                    len(eligible),
                )
            )
        ].to_dict()

        selected_rows.append(
            chosen_row
        )
        stratum_counts[
            chosen_stratum
        ] = (
            stratum_counts.get(
                chosen_stratum,
                0,
            )
            + 1
        )

    selected = pd.DataFrame(
        selected_rows
    )

    # Balanced round-robin reduction if more cells are available than needed.
    if len(selected) > maximum_windows:
        selected = (
            selected
            .assign(
                _random_order=rng.random(
                    len(selected)
                )
            )
            .sort_values(
                [
                    'selection_stratum',
                    '_random_order',
                ]
            )
            .reset_index(drop=True)
        )

        queues = {
            stratum: group.index.tolist()
            for stratum, group in selected.groupby(
                'selection_stratum',
                sort=True,
            )
        }
        retained_indices = []

        while (
            len(retained_indices) < maximum_windows
            and any(queues.values())
        ):
            active_strata = [
                stratum
                for stratum, queue in queues.items()
                if queue
            ]
            rng.shuffle(active_strata)

            for stratum in active_strata:
                if len(retained_indices) >= maximum_windows:
                    break
                retained_indices.append(
                    queues[stratum].pop(0)
                )

        selected = (
            selected.loc[
                retained_indices
            ]
            .drop(columns='_random_order')
            .copy()
        )

    selected = (
        selected
        .sort_values(
            [
                'start_date',
                'cell_id',
            ]
        )
        .reset_index(drop=True)
    )
    selected.insert(
        0,
        'shared_window_id',
        [
            f'SW{index + 1:03d}'
            for index in range(len(selected))
        ],
    )

    if len(selected) < maximum_windows:
        print(
            'Pseudo-gap selection used all available unique cells: '
            f'{len(selected):,} windows requested up to '
            f'{maximum_windows:,}.'
        )

    return selected


def build_gap_specific_windows(
    shared_windows,
    candidates,
    gap_length,
):
    """Attach the exact end date for one gap length to shared window starts."""
    candidate_lookup = (
        candidates[
            [
                'cell_id',
                'start_date',
                'end_date',
            ]
        ]
        .drop_duplicates(
            [
                'cell_id',
                'start_date',
            ]
        )
    )

    selected = (
        shared_windows[
            [
                'shared_window_id',
                'cell_id',
                'start_date',
                'selection_period',
                'calendar_quarter',
                'selection_stratum',
            ]
        ]
        .merge(
            candidate_lookup,
            on=[
                'cell_id',
                'start_date',
            ],
            how='inner',
            validate='one_to_one',
        )
    )

    if len(selected) != len(shared_windows):
        raise ValueError(
            f'The shared pseudo-gap sample is not fully valid for '
            f'the {gap_length}-month experiment.'
        )

    selected['gap_length_months'] = int(
        gap_length
    )
    selected.insert(
        1,
        'gap_id',
        (
            'L'
            + f'{gap_length:02d}'
            + '_'
            + selected['shared_window_id']
        ),
    )

    return (
        selected
        .sort_values(
            [
                'start_date',
                'cell_id',
            ]
        )
        .reset_index(drop=True)
    )


def make_imputation_pipeline(
    n_estimators=PSEUDO_GAP_RF_TREES,
    random_state=RANDOM_STATE,
):
    return Pipeline([
        (
            'prep',
            ColumnTransformer([
                (
                    'num',
                    'passthrough',
                    [
                        'grace_or_lag1',
                        'grace_or_lag2',
                        'grace_or_lag3',
                        'month',
                    ],
                ),
            ]),
        ),
        (
            'rf',
            RandomForestRegressor(
                n_estimators=n_estimators,
                random_state=random_state,
                n_jobs=N_JOBS,
            ),
        ),
    ])


def prepare_causal_imputation_table(panel):
    """
    Build lag features only from the observed GRACE record.

    Filtering this table to dates before a pseudo-gap start produces a strictly
    causal training set because every lag is derived from an earlier month.
    """
    source = (
        panel[
            [
                'cell_id',
                'mid_month_date',
                'grace_or_obs',
            ]
        ]
        .copy()
        .sort_values(
            [
                'cell_id',
                'mid_month_date',
            ]
        )
        .reset_index(drop=True)
    )
    source['mid_month_date'] = pd.to_datetime(
        source['mid_month_date'],
        errors='coerce',
    )
    source['month'] = (
        source['mid_month_date']
        .dt.month
        .astype('int8')
    )

    grouped = source.groupby(
        'cell_id',
        sort=False,
    )['grace_or_obs']
    source['grace_or_lag1'] = grouped.shift(1)
    source['grace_or_lag2'] = grouped.shift(2)
    source['grace_or_lag3'] = grouped.shift(3)

    return source


def cluster_bootstrap_regression_metrics(
    frame,
    prediction_column,
    cluster_column='shared_window_id',
    observed_column='observed_cm',
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    random_state=RANDOM_STATE,
):
    """
    Return metrics and uncertainty after resampling complete gap windows.

    All recursively related months belonging to a sampled window are retained
    together, so long-gap observations are not treated as independent rows.
    """
    data = (
        frame[
            [
                cluster_column,
                observed_column,
                prediction_column,
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if data.empty:
        return {
            metric: {
                'estimate': np.nan,
                'std': np.nan,
                'ci_lower': np.nan,
                'ci_upper': np.nan,
            }
            for metric in [
                'r2',
                'mae',
                'rmse',
            ]
        }

    observed = data[
        observed_column
    ].to_numpy(dtype=float)
    predicted = data[
        prediction_column
    ].to_numpy(dtype=float)

    point = {
        'r2': (
            np.nan
            if len(observed) < 2
            or np.allclose(
                observed,
                observed[0],
            )
            else float(
                r2_score(
                    observed,
                    predicted,
                )
            )
        ),
        'mae': float(
            mean_absolute_error(
                observed,
                predicted,
            )
        ),
        'rmse': float(
            np.sqrt(
                mean_squared_error(
                    observed,
                    predicted,
                )
            )
        ),
    }

    grouped_arrays = {
        cluster_id: group[
            [
                observed_column,
                prediction_column,
            ]
        ].to_numpy(dtype=float)
        for cluster_id, group in data.groupby(
            cluster_column,
            sort=False,
        )
    }
    cluster_ids = np.asarray(
        list(grouped_arrays),
        dtype=object,
    )

    if len(cluster_ids) < 2:
        return {
            metric: {
                'estimate': estimate,
                'std': np.nan,
                'ci_lower': np.nan,
                'ci_upper': np.nan,
            }
            for metric, estimate in point.items()
        }

    rng = np.random.default_rng(
        random_state
    )
    bootstrap_values = {
        'r2': [],
        'mae': [],
        'rmse': [],
    }

    for _ in range(n_bootstrap):
        sampled_clusters = rng.choice(
            cluster_ids,
            size=len(cluster_ids),
            replace=True,
        )
        sampled_values = np.vstack([
            grouped_arrays[cluster_id]
            for cluster_id in sampled_clusters
        ])
        true_sample = sampled_values[:, 0]
        prediction_sample = sampled_values[:, 1]

        if not np.allclose(
            true_sample,
            true_sample[0],
        ):
            bootstrap_values['r2'].append(
                r2_score(
                    true_sample,
                    prediction_sample,
                )
            )
        bootstrap_values['mae'].append(
            mean_absolute_error(
                true_sample,
                prediction_sample,
            )
        )
        bootstrap_values['rmse'].append(
            np.sqrt(
                mean_squared_error(
                    true_sample,
                    prediction_sample,
                )
            )
        )

    output = {}
    for metric, estimate in point.items():
        values = np.asarray(
            bootstrap_values[metric],
            dtype=float,
        )
        output[metric] = {
            'estimate': estimate,
            'std': (
                float(
                    np.std(
                        values,
                        ddof=1,
                    )
                )
                if len(values) > 1
                else np.nan
            ),
            'ci_lower': (
                float(
                    np.nanquantile(
                        values,
                        0.025,
                    )
                )
                if len(values)
                else np.nan
            ),
            'ci_upper': (
                float(
                    np.nanquantile(
                        values,
                        0.975,
                    )
                )
                if len(values)
                else np.nan
            ),
        }

    return output


def run_shared_pseudo_gap_experiments(
    panel,
    gap_lengths=PSEUDO_GAP_LENGTHS,
    maximum_windows=PSEUDO_GAP_WINDOWS_PER_LENGTH,
):
    """
    Run all gap lengths on the same stratified cell-window sample.

    One recursive trajectory is generated to the maximum requested horizon.
    Its first months are reused for shorter horizons, ensuring paired tests.
    """
    gap_lengths = sorted(
        {
            int(gap_length)
            for gap_length in gap_lengths
        }
    )
    maximum_gap_length = max(
        gap_lengths
    )

    candidate_tables = {
        gap_length: find_pseudo_gap_candidates(
            panel,
            gap_length,
        )
        for gap_length in gap_lengths
    }

    shared_windows = select_shared_pseudo_gap_windows(
        candidate_tables[
            maximum_gap_length
        ],
        maximum_windows=maximum_windows,
    )

    windows_by_length = {
        gap_length: build_gap_specific_windows(
            shared_windows,
            candidate_tables[
                gap_length
            ],
            gap_length,
        )
        for gap_length in gap_lengths
    }

    selected_windows_all = pd.concat(
        [
            windows_by_length[
                gap_length
            ]
            for gap_length in gap_lengths
        ],
        ignore_index=True,
    )

    causal_table = prepare_causal_imputation_table(
        panel
    )
    feature_columns = [
        'grace_or_lag1',
        'grace_or_lag2',
        'grace_or_lag3',
        'month',
    ]

    prediction_rows = []
    training_rows_by_start = []

    # The model is fitted once for each unique rolling-origin cutoff.
    for start_date, start_windows in shared_windows.groupby(
        'start_date',
        sort=True,
    ):
        training_mask = (
            causal_table['mid_month_date'].lt(start_date)
            & causal_table['grace_or_obs'].notna()
            & causal_table[
                feature_columns
            ].notna().all(axis=1)
        )

        training_rows = int(
            training_mask.sum()
        )
        if training_rows < 100:
            raise ValueError(
                f'Only {training_rows} causal training rows are available '
                f'before {start_date:%Y-%m-%d}.'
            )

        pseudo_gap_model = make_imputation_pipeline(
            n_estimators=PSEUDO_GAP_RF_TREES,
            random_state=(
                RANDOM_STATE
                + 1000
                + int(start_date.year) * 13
                + int(start_date.month)
            ),
        )
        pseudo_gap_model.fit(
            causal_table.loc[
                training_mask,
                feature_columns,
            ],
            causal_table.loc[
                training_mask,
                'grace_or_obs',
            ],
        )

        training_rows_by_start.append({
            'start_date': start_date,
            'training_rows': training_rows,
            'training_last_date': causal_table.loc[
                training_mask,
                'mid_month_date',
            ].max(),
            'future_training_rows': int(
                causal_table.loc[
                    training_mask,
                    'mid_month_date',
                ].ge(start_date).sum()
            ),
            'shared_windows_at_cutoff': len(
                start_windows
            ),
        })

        for window in start_windows.itertuples(
            index=False
        ):
            cell_rows = (
                causal_table.loc[
                    causal_table['cell_id'].eq(
                        window.cell_id
                    )
                ]
                .sort_values(
                    'mid_month_date'
                )
                .reset_index(drop=True)
            )
            start_matches = np.flatnonzero(
                cell_rows[
                    'mid_month_date'
                ].to_numpy()
                == np.datetime64(
                    window.start_date
                )
            )
            if len(start_matches) != 1:
                raise ValueError(
                    f'Could not resolve the start of '
                    f'{window.shared_window_id}.'
                )

            start_position = int(
                start_matches[0]
            )
            history = (
                cell_rows.iloc[
                    start_position - 3:start_position
                ]['grace_or_obs']
                .astype(float)
                .tolist()
            )

            if (
                len(history) != 3
                or not np.all(
                    np.isfinite(history)
                )
            ):
                raise ValueError(
                    f'Pseudo-gap {window.shared_window_id} lacks '
                    'three valid preceding observations.'
                )

            persistence_value = float(
                history[-1]
            )

            for gap_position in range(
                1,
                maximum_gap_length + 1,
            ):
                row_position = (
                    start_position
                    + gap_position
                    - 1
                )
                row = cell_rows.iloc[
                    row_position
                ]

                prediction_features = pd.DataFrame([{
                    'grace_or_lag1': history[-1],
                    'grace_or_lag2': history[-2],
                    'grace_or_lag3': history[-3],
                    'month': int(
                        row['month']
                    ),
                }])

                (
                    prediction,
                    prediction_std,
                    prediction_lower,
                    prediction_upper,
                ) = predict_rf_distribution(
                    pseudo_gap_model,
                    prediction_features,
                )

                predicted_value = float(
                    prediction[0]
                )
                history.append(
                    predicted_value
                )

                true_value = float(
                    row['grace_or_obs']
                )
                lower_value = float(
                    prediction_lower[0]
                )
                upper_value = float(
                    prediction_upper[0]
                )

                common_prediction = {
                    'shared_window_id': (
                        window.shared_window_id
                    ),
                    'gap_position': int(
                        gap_position
                    ),
                    'cell_id': window.cell_id,
                    'date': row[
                        'mid_month_date'
                    ],
                    'start_date': window.start_date,
                    'selection_period': int(
                        window.selection_period
                    ),
                    'calendar_quarter': int(
                        window.calendar_quarter
                    ),
                    'selection_stratum': (
                        window.selection_stratum
                    ),
                    'training_cutoff_date': start_date,
                    'training_rows': training_rows,
                    'future_training_rows': 0,
                    'observed_cm': true_value,
                    'reconstructed_cm': predicted_value,
                    'persistence_cm': persistence_value,
                    'prediction_std_cm': float(
                        prediction_std[0]
                    ),
                    'prediction_lower_cm': lower_value,
                    'prediction_upper_cm': upper_value,
                    'interval_covered': (
                        lower_value
                        <= true_value
                        <= upper_value
                    ),
                }

                for gap_length in gap_lengths:
                    if gap_position > gap_length:
                        continue

                    prediction_rows.append({
                        **common_prediction,
                        'gap_id': (
                            f'L{gap_length:02d}_'
                            f'{window.shared_window_id}'
                        ),
                        'gap_length_months': int(
                            gap_length
                        ),
                    })

        del pseudo_gap_model
        gc.collect()

    predictions = pd.DataFrame(
        prediction_rows
    )
    base_training_diagnostics = pd.DataFrame(
        training_rows_by_start
    )
    training_diagnostics = pd.concat(
        [
            base_training_diagnostics.assign(
                gap_length_months=gap_length
            )
            for gap_length in gap_lengths
        ],
        ignore_index=True,
    )

    summary_rows = []

    for gap_length in gap_lengths:
        gap_predictions = (
            predictions.loc[
                predictions[
                    'gap_length_months'
                ].eq(gap_length)
            ]
            .copy()
        )
        selected_windows = windows_by_length[
            gap_length
        ]

        rf_metrics = cluster_bootstrap_regression_metrics(
            gap_predictions,
            prediction_column='reconstructed_cm',
            random_state=(
                RANDOM_STATE
                + 2000
                + gap_length
            ),
        )
        persistence_metrics = (
            cluster_bootstrap_regression_metrics(
                gap_predictions,
                prediction_column='persistence_cm',
                random_state=(
                    RANDOM_STATE
                    + 2500
                    + gap_length
                ),
            )
        )

        rf_mae = rf_metrics[
            'mae'
        ]['estimate']
        persistence_mae = persistence_metrics[
            'mae'
        ]['estimate']
        rf_rmse = rf_metrics[
            'rmse'
        ]['estimate']
        persistence_rmse = persistence_metrics[
            'rmse'
        ]['estimate']

        window_training_rows = (
            gap_predictions[
                [
                    'shared_window_id',
                    'training_rows',
                ]
            ]
            .drop_duplicates(
                'shared_window_id'
            )
        )

        summary_rows.append({
            'validation': (
                'Strict rolling-origin recursive pseudo-gap'
            ),
            'leakage_control': (
                'Only dates before each shared gap start are used '
                'for model fitting'
            ),
            'sample_design': (
                'Shared stratified cell-window starts; one window per cell'
            ),
            'gap_length_months': int(
                gap_length
            ),
            'selected_windows': len(
                selected_windows
            ),
            'unique_cells': selected_windows[
                'cell_id'
            ].nunique(),
            'unique_training_cutoffs': selected_windows[
                'start_date'
            ].nunique(),
            'withheld_observations': len(
                gap_predictions
            ),
            'minimum_training_rows': window_training_rows[
                'training_rows'
            ].min(),
            'mean_training_rows': window_training_rows[
                'training_rows'
            ].mean(),
            'maximum_training_rows': window_training_rows[
                'training_rows'
            ].max(),
            'future_training_rows': gap_predictions[
                'future_training_rows'
            ].sum(),
            'R2': rf_metrics[
                'r2'
            ]['estimate'],
            'R2_bootstrap_std': rf_metrics[
                'r2'
            ]['std'],
            'R2_95_CI_lower': rf_metrics[
                'r2'
            ]['ci_lower'],
            'R2_95_CI_upper': rf_metrics[
                'r2'
            ]['ci_upper'],
            'MAE_cm': rf_mae,
            'MAE_bootstrap_std_cm': rf_metrics[
                'mae'
            ]['std'],
            'MAE_95_CI_lower_cm': rf_metrics[
                'mae'
            ]['ci_lower'],
            'MAE_95_CI_upper_cm': rf_metrics[
                'mae'
            ]['ci_upper'],
            'RMSE_cm': rf_rmse,
            'RMSE_bootstrap_std_cm': rf_metrics[
                'rmse'
            ]['std'],
            'RMSE_95_CI_lower_cm': rf_metrics[
                'rmse'
            ]['ci_lower'],
            'RMSE_95_CI_upper_cm': rf_metrics[
                'rmse'
            ]['ci_upper'],
            'persistence_R2': persistence_metrics[
                'r2'
            ]['estimate'],
            'persistence_R2_bootstrap_std': persistence_metrics[
                'r2'
            ]['std'],
            'persistence_R2_95_CI_lower': persistence_metrics[
                'r2'
            ]['ci_lower'],
            'persistence_R2_95_CI_upper': persistence_metrics[
                'r2'
            ]['ci_upper'],
            'persistence_MAE_cm': persistence_mae,
            'persistence_MAE_bootstrap_std_cm': persistence_metrics[
                'mae'
            ]['std'],
            'persistence_MAE_95_CI_lower_cm': persistence_metrics[
                'mae'
            ]['ci_lower'],
            'persistence_MAE_95_CI_upper_cm': persistence_metrics[
                'mae'
            ]['ci_upper'],
            'persistence_RMSE_cm': persistence_rmse,
            'persistence_RMSE_bootstrap_std_cm': persistence_metrics[
                'rmse'
            ]['std'],
            'persistence_RMSE_95_CI_lower_cm': persistence_metrics[
                'rmse'
            ]['ci_lower'],
            'persistence_RMSE_95_CI_upper_cm': persistence_metrics[
                'rmse'
            ]['ci_upper'],
            'MAE_skill_vs_persistence': (
                1.0 - rf_mae / persistence_mae
                if persistence_mae > 0
                else np.nan
            ),
            'RMSE_skill_vs_persistence': (
                1.0 - rf_rmse / persistence_rmse
                if persistence_rmse > 0
                else np.nan
            ),
            'mean_prediction_std_cm': gap_predictions[
                'prediction_std_cm'
            ].mean(),
            'mean_95_interval_width_cm': (
                gap_predictions[
                    'prediction_upper_cm'
                ]
                - gap_predictions[
                    'prediction_lower_cm'
                ]
            ).mean(),
            'empirical_95_interval_coverage': gap_predictions[
                'interval_covered'
            ].mean(),
            'bootstrap_unit': (
                'Complete shared cell-window event'
            ),
        })

    results_table = (
        pd.DataFrame(
            summary_rows
        )
        .sort_values(
            'gap_length_months'
        )
        .reset_index(drop=True)
    )

    selection_diagnostics = (
        shared_windows
        .groupby(
            [
                'selection_period',
                'calendar_quarter',
                'selection_stratum',
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            selected_windows=(
                'shared_window_id',
                'nunique',
            ),
            unique_cells=(
                'cell_id',
                'nunique',
            ),
            unique_start_dates=(
                'start_date',
                'nunique',
            ),
            earliest_start=(
                'start_date',
                'min',
            ),
            latest_start=(
                'start_date',
                'max',
            ),
        )
        .sort_values(
            [
                'selection_period',
                'calendar_quarter',
            ]
        )
        .reset_index(drop=True)
    )

    del causal_table
    gc.collect()

    return (
        selected_windows_all,
        predictions,
        results_table,
        training_diagnostics,
        selection_diagnostics,
        shared_windows,
    )


(
    pseudo_gap_windows_df,
    pseudo_gap_predictions_df,
    pseudo_gap_results_table,
    pseudo_gap_training_diagnostics_df,
    pseudo_gap_selection_diagnostics_df,
    pseudo_gap_shared_windows_df,
) = run_shared_pseudo_gap_experiments(
    panel,
)

position_rows = []

for (
    gap_length,
    gap_position,
), group in pseudo_gap_predictions_df.groupby(
    [
        'gap_length_months',
        'gap_position',
    ],
    observed=True,
):
    rf_metrics = cluster_bootstrap_regression_metrics(
        group,
        prediction_column='reconstructed_cm',
        n_bootstrap=min(
            200,
            BOOTSTRAP_ITERATIONS,
        ),
        random_state=(
            RANDOM_STATE
            + 3000
            + int(gap_length) * 20
            + int(gap_position)
        ),
    )
    persistence_metrics = cluster_bootstrap_regression_metrics(
        group,
        prediction_column='persistence_cm',
        n_bootstrap=min(
            200,
            BOOTSTRAP_ITERATIONS,
        ),
        random_state=(
            RANDOM_STATE
            + 3500
            + int(gap_length) * 20
            + int(gap_position)
        ),
    )

    position_rows.append({
        'gap_length_months': int(
            gap_length
        ),
        'gap_position': int(
            gap_position
        ),
        'samples': len(group),
        'distinct_windows': group[
            'shared_window_id'
        ].nunique(),
        'R2': rf_metrics[
            'r2'
        ]['estimate'],
        'MAE_cm': rf_metrics[
            'mae'
        ]['estimate'],
        'MAE_95_CI_lower_cm': rf_metrics[
            'mae'
        ]['ci_lower'],
        'MAE_95_CI_upper_cm': rf_metrics[
            'mae'
        ]['ci_upper'],
        'RMSE_cm': rf_metrics[
            'rmse'
        ]['estimate'],
        'persistence_R2': persistence_metrics[
            'r2'
        ]['estimate'],
        'persistence_MAE_cm': persistence_metrics[
            'mae'
        ]['estimate'],
        'persistence_RMSE_cm': persistence_metrics[
            'rmse'
        ]['estimate'],
        'MAE_skill_vs_persistence': (
            1.0
            - rf_metrics['mae']['estimate']
            / persistence_metrics['mae']['estimate']
            if persistence_metrics[
                'mae'
            ]['estimate'] > 0
            else np.nan
        ),
        'mean_prediction_std_cm': group[
            'prediction_std_cm'
        ].mean(),
        'empirical_95_interval_coverage': group[
            'interval_covered'
        ].mean(),
    })

pseudo_gap_position_table = pd.DataFrame(
    position_rows
).sort_values(
    [
        'gap_length_months',
        'gap_position',
    ]
).reset_index(drop=True)

imputation_validation_comparison_table = pd.concat([
    pd.DataFrame([{
        'validation': (
            'Chronological 80/20 holdout'
        ),
        'leakage_control': (
            'Final 20% withheld from model fitting'
        ),
        'sample_design': (
            'Chronological holdout within every cell'
        ),
        'gap_length_months': np.nan,
        'selected_windows': np.nan,
        'unique_cells': (
            eval_metrics['n_test_cells']
        ),
        'unique_training_cutoffs': np.nan,
        'withheld_observations': (
            eval_metrics['n_test_rows']
        ),
        'minimum_training_rows': (
            eval_metrics['n_train_rows']
        ),
        'mean_training_rows': (
            eval_metrics['n_train_rows']
        ),
        'maximum_training_rows': (
            eval_metrics['n_train_rows']
        ),
        'future_training_rows': 0,
        'R2': eval_metrics['R2'],
        'R2_bootstrap_std': np.nan,
        'R2_95_CI_lower': (
            eval_metrics[
                'R2_ci_lower'
            ]
        ),
        'R2_95_CI_upper': (
            eval_metrics[
                'R2_ci_upper'
            ]
        ),
        'MAE_cm': eval_metrics['MAE'],
        'MAE_bootstrap_std_cm': np.nan,
        'MAE_95_CI_lower_cm': (
            eval_metrics[
                'MAE_ci_lower'
            ]
        ),
        'MAE_95_CI_upper_cm': (
            eval_metrics[
                'MAE_ci_upper'
            ]
        ),
        'RMSE_cm': eval_metrics['RMSE'],
        'RMSE_bootstrap_std_cm': np.nan,
        'RMSE_95_CI_lower_cm': (
            eval_metrics[
                'RMSE_ci_lower'
            ]
        ),
        'RMSE_95_CI_upper_cm': (
            eval_metrics[
                'RMSE_ci_upper'
            ]
        ),
        'persistence_R2': np.nan,
        'persistence_MAE_cm': np.nan,
        'persistence_RMSE_cm': np.nan,
        'MAE_skill_vs_persistence': np.nan,
        'RMSE_skill_vs_persistence': np.nan,
        'mean_prediction_std_cm': (
            eval_metrics[
                'mean_prediction_std'
            ]
        ),
        'mean_95_interval_width_cm': (
            eval_metrics[
                'mean_interval_width'
            ]
        ),
        'empirical_95_interval_coverage': (
            eval_metrics[
                'interval_coverage_95'
            ]
        ),
        'bootstrap_unit': (
            'Individual withheld cell-month row'
        ),
    }]),
    pseudo_gap_results_table,
], ignore_index=True, sort=False)

display(
    pseudo_gap_results_table.round(4)
)
display(
    pseudo_gap_position_table.round(4)
)
display(
    pseudo_gap_selection_diagnostics_df
)
display(
    pseudo_gap_training_diagnostics_df.round(4)
)
display(
    imputation_validation_comparison_table.round(4)
)


### 3.7 Imputation handoff

The completed GRACE record and tree-ensemble uncertainty fields are passed directly to the downscaling stage.

In [ ]:
geometry_col = panel_gdf.geometry.name
export_cols = [
    'month_key',
    'lon0',
    'lat0',
    geometry_col,
    'month',
    'imputed_flag',
    'grace_or_imputed',
    'grace_or_imputed_std',
    'grace_or_imputed_lower',
    'grace_or_imputed_upper',
]

df_gr_imputed = panel_gdf.loc[:, export_cols].copy()
df_gr_imputed['month_key'] = pd.to_datetime(
    df_gr_imputed['month_key'],
    format='%Y-%m',
    errors='coerce',
)
df_gr_imputed['year'] = (
    df_gr_imputed['month_key']
    .dt.year
    .astype('Int64')
)
df_gr_imputed = df_gr_imputed[
    df_gr_imputed['year'].between(2013, 2023)
].copy()
df_gr_imputed = df_gr_imputed.rename(columns={
    geometry_col: 'geometry',
    'month_key': 'date',
})

print(
    'Imputed GRACE records prepared for downscaling:',
    df_gr_imputed.shape,
)


def safe_group_r2(group):
    y_true = group['y_true'].to_numpy(dtype=float)
    y_pred = group['y_pred'].to_numpy(dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2 or np.allclose(y_true, y_true[0]):
        return np.nan
    return r2_score(y_true, y_pred)


def safe_interval_coverage(group):
    valid = (
        group['y_true'].notna()
        & group['y_pred_lower'].notna()
        & group['y_pred_upper'].notna()
    )
    if not valid.any():
        return np.nan
    return group.loc[valid, 'y_true'].between(
        group.loc[valid, 'y_pred_lower'],
        group.loc[valid, 'y_pred_upper'],
    ).mean()


imputation_cell_metrics = (
    imputation_test_predictions
    .groupby('cell_id', observed=True)
    .apply(
        lambda group: pd.Series({
            'n_test': len(group),
            'r2': safe_group_r2(group),
            'mae': mean_absolute_error(
                group['y_true'],
                group['y_pred'],
            ),
            'mean_prediction_std': group['y_pred_std'].mean(),
            'mean_interval_width': (
                group['y_pred_upper']
                - group['y_pred_lower']
            ).mean(),
            'interval_coverage': safe_interval_coverage(group),
        }),
        include_groups=False,
    )
    .reset_index()
)

imputation_geometry = (
    panel_gdf[
        ['cell_id', panel_gdf.geometry.name]
    ]
    .drop_duplicates('cell_id')
)

imputation_cell_metrics_gdf = (
    imputation_geometry
    .merge(
        imputation_cell_metrics,
        on='cell_id',
        how='left',
    )
)
imputation_cell_metrics_gdf = gpd.GeoDataFrame(
    imputation_cell_metrics_gdf,
    geometry=panel_gdf.geometry.name,
    crs=panel_gdf.crs,
)

del imputation_geometry, imputation_cell_metrics
gc.collect()
memory_report('after preparing imputation diagnostics')


## 4. Spatial downscaling and mass-conserving postprocessing

### 4.1 Model-ready 0.25° and 0.1° tables

Hydroclimatic, soil-moisture, physiographic, and GRACE inputs are assembled in memory. A temporary 0.05° bridge transfers variables between the 0.1° and 0.25° grids and is released after use.

In [ ]:
def _snap_geometry(series, grid_size=0.01):
    """Snap polygon coordinates to the 0.01° precision used in the study."""
    try:
        from shapely import set_precision
        return gpd.GeoSeries([set_precision(geometry, grid_size) if geometry is not None else None for geometry in series], index=series.index, crs=getattr(series, 'crs', None))
    except Exception:
        from shapely.ops import transform

        def round_geometry(geometry):
            if geometry is None:
                return None

            def round_coordinates(x, y, z=None):
                rounded_x = np.round(x, 2)
                rounded_y = np.round(y, 2)
                if z is None:
                    return (rounded_x, rounded_y)
                return (rounded_x, rounded_y, np.round(z, 2))
            return transform(round_coordinates, geometry)
        return gpd.GeoSeries(series.apply(round_geometry), index=series.index, crs=getattr(series, 'crs', None))

def _add_lower_left_keys(frame, geometry_col, resolution=0.25, longitude_col='lon0', latitude_col='lat0'):
    """Derive stable lower-left grid coordinates from polygon bounds."""
    frame = frame.copy()
    geometry = gpd.GeoSeries(frame[geometry_col], crs=getattr(frame, 'crs', None))
    bounds = geometry.bounds
    epsilon = 1e-12
    frame[longitude_col] = np.round(np.floor((bounds['minx'].to_numpy(dtype=float) + epsilon) / resolution) * resolution, 5)
    frame[latitude_col] = np.round(np.floor((bounds['miny'].to_numpy(dtype=float) + epsilon) / resolution) * resolution, 5)
    return frame

def _mode_or_na(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if len(mode) else pd.NA

def _aggregate_piece_table(frame, group_columns, geometry_column, numeric_columns, categorical_columns, crs):
    """Aggregate equal-area 0.05° pieces to one target grid resolution."""
    available_numeric = [column for column in numeric_columns if column in frame.columns]
    available_categorical = [column for column in categorical_columns if column in frame.columns]
    aggregation = {column: 'mean' for column in available_numeric}
    aggregation.update({column: _mode_or_na for column in available_categorical})
    aggregation[geometry_column] = 'first'
    aggregated = frame.dropna(subset=[geometry_column]).groupby(group_columns, as_index=False, observed=True).agg(aggregation).sort_values(group_columns).reset_index(drop=True)
    return gpd.GeoDataFrame(aggregated, geometry=geometry_column, crs=crs)
feature_01_work = feature_01_gdf.copy()
if feature_01_work.geometry.name != 'geometry_01':
    feature_01_work = feature_01_work.rename_geometry('geometry_01')
feature_01_work['grid'] = feature_01_work['grid'].astype(float).astype(str)
feature_01_work['date'] = to_month_start(feature_01_work['date'])
feature_01_work['geometry_01'] = _snap_geometry(gpd.GeoSeries(feature_01_work['geometry_01'], crs=feature_01_work.crs))
dynamic_feature_025_gdf = sm_gdf.copy()
if dynamic_feature_025_gdf.geometry.name != 'geometry_0.25':
    dynamic_feature_025_gdf = dynamic_feature_025_gdf.rename_geometry('geometry_0.25')
dynamic_feature_025_gdf['date'] = to_month_start(dynamic_feature_025_gdf['date'])
dynamic_feature_025_gdf['geometry_0.25'] = _snap_geometry(gpd.GeoSeries(dynamic_feature_025_gdf['geometry_0.25'], crs=dynamic_feature_025_gdf.crs))
dynamic_feature_025_gdf = _add_lower_left_keys(dynamic_feature_025_gdf, geometry_col='geometry_0.25', resolution=0.25)
grace_for_downscaling = df_gr_imputed[['date', 'lon0', 'lat0', 'imputed_flag', TARGET_COL, 'grace_or_imputed_std', 'grace_or_imputed_lower', 'grace_or_imputed_upper']].copy()
grace_for_downscaling['date'] = to_month_start(grace_for_downscaling['date'])
grace_for_downscaling['lon0'] = pd.to_numeric(grace_for_downscaling['lon0'], errors='coerce').round(5)
grace_for_downscaling['lat0'] = pd.to_numeric(grace_for_downscaling['lat0'], errors='coerce').round(5)
grace_for_downscaling = grace_for_downscaling.drop_duplicates(['date', 'lon0', 'lat0'])
dynamic_feature_025_gdf = dynamic_feature_025_gdf.merge(grace_for_downscaling, on=['date', 'lon0', 'lat0'], how='left')
dynamic_feature_025_gdf['parent_025_key'] = dynamic_feature_025_gdf['lon0'].map(lambda value: f'{value:.5f}') + '_' + dynamic_feature_025_gdf['lat0'].map(lambda value: f'{value:.5f}')
dynamic_feature_025_gdf = gpd.GeoDataFrame(dynamic_feature_025_gdf, geometry='geometry_0.25', crs=sm_gdf.crs)
grid_01_cells = feature_01_work[['grid', 'geometry_01']].dropna(subset=['geometry_01']).drop_duplicates('grid').reset_index(drop=True)
bounds = gpd.GeoSeries(grid_01_cells['geometry_01'], crs=feature_01_work.crs).bounds
cell_index = np.repeat(np.arange(len(grid_01_cells)), 4)
quarter_index = np.tile(np.arange(4), len(grid_01_cells))
minimum_x = bounds['minx'].to_numpy()[cell_index]
minimum_y = bounds['miny'].to_numpy()[cell_index]
maximum_x = bounds['maxx'].to_numpy()[cell_index]
maximum_y = bounds['maxy'].to_numpy()[cell_index]
middle_x = (minimum_x + maximum_x) / 2.0
middle_y = (minimum_y + maximum_y) / 2.0
piece_minimum_x = np.where(quarter_index % 2 == 0, minimum_x, middle_x)
piece_maximum_x = np.where(quarter_index % 2 == 0, middle_x, maximum_x)
piece_minimum_y = np.where(quarter_index < 2, minimum_y, middle_y)
piece_maximum_y = np.where(quarter_index < 2, middle_y, maximum_y)
piece_geometry = [box(x_min, y_min, x_max, y_max) for x_min, y_min, x_max, y_max in zip(piece_minimum_x, piece_minimum_y, piece_maximum_x, piece_maximum_y)]
piece_grid = gpd.GeoDataFrame({'grid': grid_01_cells['grid'].to_numpy()[cell_index], 'piece_number': quarter_index.astype('int8'), 'geometry_01': grid_01_cells['geometry_01'].to_numpy()[cell_index]}, geometry=gpd.GeoSeries(piece_geometry, crs=feature_01_work.crs), crs=feature_01_work.crs).rename_geometry('geometry_005')
piece_grid['geometry_005'] = _snap_geometry(gpd.GeoSeries(piece_grid['geometry_005'], crs=piece_grid.crs))
parent_cells = dynamic_feature_025_gdf[['parent_025_key', 'geometry_0.25']].dropna(subset=['geometry_0.25']).drop_duplicates('parent_025_key').reset_index(drop=True)
parent_cells = gpd.GeoDataFrame(parent_cells, geometry='geometry_0.25', crs=dynamic_feature_025_gdf.crs)
if piece_grid.crs != parent_cells.crs:
    piece_grid = piece_grid.to_crs(parent_cells.crs)
piece_parent_map = gpd.sjoin(piece_grid, parent_cells, how='left', predicate='within')
piece_parent_map['geometry_0.25'] = piece_parent_map['index_right'].map(parent_cells['geometry_0.25'])
piece_parent_map = piece_parent_map.sort_values(['grid', 'piece_number', 'index_right'], na_position='last').drop_duplicates(['grid', 'piece_number'], keep='first').drop(columns=['index_right'], errors='ignore').reset_index(drop=True)
piece_parent_map['geometry_0.25'] = gpd.GeoSeries(piece_parent_map['geometry_0.25'], crs=parent_cells.crs)
feature_01_attributes = pd.DataFrame(feature_01_work.drop(columns=['geometry_01'], errors='ignore'))
piece_attributes = pd.DataFrame(piece_parent_map[['grid', 'piece_number', 'geometry_005', 'geometry_01', 'parent_025_key', 'geometry_0.25']])
feature_piece_table = feature_01_attributes.merge(piece_attributes, on='grid', how='left')
parent_dynamic_columns = ['date', 'parent_025_key', 'SM', 'imputed_flag', TARGET_COL, 'grace_or_imputed_std', 'grace_or_imputed_lower', 'grace_or_imputed_upper']
parent_dynamic_attributes = pd.DataFrame(dynamic_feature_025_gdf.drop(columns=['geometry_0.25'], errors='ignore'))[[column for column in parent_dynamic_columns if column in dynamic_feature_025_gdf.columns]].drop_duplicates(['date', 'parent_025_key'])
feature_piece_table = feature_piece_table.merge(parent_dynamic_attributes, on=['date', 'parent_025_key'], how='left')
feature_piece_table = gpd.GeoDataFrame(feature_piece_table, geometry='geometry_005', crs=piece_parent_map.crs)
numeric_targets = ['ET', 'P', 'Q', 'Q_new', 'temp', 'lc_percent', 'lith_percent', 'elevation', 'SM', TARGET_COL, 'grace_or_imputed_std', 'grace_or_imputed_lower', 'grace_or_imputed_upper']
categorical_targets = ['lc_class', 'lc_class_geom', 'lith_class', 'imputed_flag']
for column in numeric_targets:
    if column in feature_piece_table.columns:
        feature_piece_table[column] = pd.to_numeric(feature_piece_table[column], errors='coerce')
gdf_ml_01 = _aggregate_piece_table(feature_piece_table, group_columns=['date', 'grid'], geometry_column='geometry_01', numeric_columns=numeric_targets, categorical_columns=categorical_targets, crs=feature_piece_table.crs).rename(columns={'grid': 'cell_01_key'})
gdf_ml_025 = _aggregate_piece_table(feature_piece_table, group_columns=['date', 'parent_025_key'], geometry_column='geometry_0.25', numeric_columns=numeric_targets, categorical_columns=categorical_targets, crs=feature_piece_table.crs).rename(columns={'parent_025_key': 'cell_025_key'})
gdf_ml_01 = gdf_ml_01.loc[gdf_ml_01[TARGET_COL].notna()].reset_index(drop=True)
gdf_ml_025 = gdf_ml_025.loc[gdf_ml_025[TARGET_COL].notna()].reset_index(drop=True)
downscaling_input_summary = pd.DataFrame([{'resolution': '0.25°', 'source': 'constructed in memory from raw sources', 'rows': len(gdf_ml_025), 'columns': gdf_ml_025.shape[1], 'crs': str(gdf_ml_025.crs)}, {'resolution': '0.1°', 'source': 'constructed in memory from raw sources', 'rows': len(gdf_ml_01), 'columns': gdf_ml_01.shape[1], 'crs': str(gdf_ml_01.crs)}])
display(downscaling_input_summary)
for _name in ['feature_01_work', 'feature_01_attributes', 'feature_piece_table', 'piece_attributes', 'piece_grid', 'piece_parent_map', 'parent_cells', 'parent_dynamic_attributes', 'dynamic_feature_025_gdf', 'grace_for_downscaling', 'grid_01_cells', 'feature_01_gdf', 'sm_gdf', 'df_gr_imputed', 'grace_fl_gdf', 'grace_or_gdf', 'grace_fl_attrs', 'grace_or_attrs', 'grace_joined_gdf', 'agg_out']:
    if _name in globals():
        del globals()[_name]
gc.collect()
memory_report('after constructing downscaling feature tables from raw sources')


### 4.2 Spatial identifiers and temporal predictors

Polygon geometries are repaired, stable cell identifiers are assigned, and lagged and rolling predictors are calculated independently within each grid cell.

In [ ]:
# Working copies for the two spatial resolutions
gdf025 = gdf_ml_025.copy()
gdf01 = gdf_ml_01.copy()

# Ensure the intended active geometries
if not isinstance(gdf025, gpd.GeoDataFrame):
    gdf025 = gpd.GeoDataFrame(gdf025, geometry="geometry_0.25", crs="EPSG:4326")
else:
    gdf025 = gdf025.set_geometry("geometry_0.25")

if not isinstance(gdf01, gpd.GeoDataFrame):
    gdf01 = gpd.GeoDataFrame(gdf01, geometry="geometry_01", crs="EPSG:4326")
else:
    gdf01 = gdf01.set_geometry("geometry_01")

if gdf025.crs is None:
    gdf025 = gdf025.set_crs("EPSG:4326")
if gdf01.crs is None:
    gdf01 = gdf01.set_crs("EPSG:4326")
gdf01 = gdf01.to_crs(gdf025.crs)

# Repair invalid polygon geometries before spatial processing
gdf025["geometry_0.25"] = gdf025["geometry_0.25"].buffer(0)
gdf01["geometry_01"] = gdf01["geometry_01"].buffer(0)

gdf025["date"] = pd.to_datetime(gdf025["date"])
gdf01["date"] = pd.to_datetime(gdf01["date"])

gdf025["cell025_id"] = gdf025.groupby("geometry_0.25").ngroup()
gdf01["cell01_id"] = gdf01.groupby("geometry_01").ngroup()

gdf025["lon_centroid"] = gdf025["geometry_0.25"].centroid.x
gdf025["lat_centroid"] = gdf025["geometry_0.25"].centroid.y
gdf01["lon_centroid"] = gdf01["geometry_01"].centroid.x
gdf01["lat_centroid"] = gdf01["geometry_01"].centroid.y

# The download aliases are no longer needed.
del gdf_ml_025, gdf_ml_01
gc.collect()

downscaling_table_summary = pd.DataFrame([
    {
        "table": "0.25° canonical",
        "rows": len(gdf025),
        "dates": gdf025["date"].nunique(),
        "cells": gdf025["cell025_id"].nunique(),
    },
    {
        "table": "0.1° canonical",
        "rows": len(gdf01),
        "dates": gdf01["date"].nunique(),
        "cells": gdf01["cell01_id"].nunique(),
    },
])
display(downscaling_table_summary)


In [ ]:
lag_base_vars = ["ET", "P", "Q", "temp", "SM"]
lags = [1, 2, 3]
roll_win = 3


def add_lag_roll_features(df, id_col, base_vars, lags, roll_win, date_col="date"):
    """Create chronological lagged and rolling predictors within each grid cell."""
    df = df.sort_values([id_col, date_col]).copy()
    variables = [column for column in base_vars if column in df.columns]
    group = df.groupby(id_col, group_keys=False)
    new_columns = []

    for variable in variables:
        for lag in lags:
            name = f"{variable}_lag{lag}"
            df[name] = group[variable].shift(lag)
            new_columns.append(name)

        roll_name = f"{variable}_roll{roll_win}"
        df[roll_name] = (
            group[variable]
            .rolling(roll_win, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )
        new_columns.append(roll_name)

    return df, new_columns


gdf025, new_cols_025 = add_lag_roll_features(
    gdf025, "cell025_id", lag_base_vars, lags, roll_win
)
gdf01, new_cols_01 = add_lag_roll_features(
    gdf01, "cell01_id", lag_base_vars, lags, roll_win
)

static_num_features = [
    'elevation',
]
static_cat_features = [
    'lc_class',
    'lith_class',
]

required_static_features = static_num_features + static_cat_features
missing_static_025 = [
    column
    for column in required_static_features
    if column not in gdf025.columns
]
missing_static_01 = [
    column
    for column in required_static_features
    if column not in gdf01.columns
]

if missing_static_025 or missing_static_01:
    raise KeyError(
        'Static downscaling predictors are missing. '
        f'0.25° missing: {missing_static_025}; '
        f'0.1° missing: {missing_static_01}.'
    )

num_features = [
    'ET',
    'P',
    'Q',
    'temp',
    'SM',
    'elevation',
    'ET_lag1',
    'ET_lag2',
    'ET_lag3',
    'ET_roll3',
    'P_lag1',
    'P_lag2',
    'P_lag3',
    'P_roll3',
    'Q_lag1',
    'Q_lag2',
    'Q_lag3',
    'Q_roll3',
    'SM_lag1',
    'SM_lag2',
    'SM_lag3',
    'SM_roll3',
    'temp_lag1',
    'temp_lag2',
    'temp_lag3',
    'temp_roll3',
]
cat_features = [
    'lc_class',
    'lith_class',
]
feature_cols = num_features + cat_features

print('Downscaling numerical predictors:', num_features)
print('Downscaling categorical predictors:', cat_features)
print('Total downscaling predictors before one-hot encoding:', len(feature_cols))
memory_report('after lag and rolling feature construction')


### 4.3 Model selection and independent evaluation

1. Structural Random Forest parameters are selected by grouped cross-validation on balanced pre-2022 observations.
2. The selected parameters are fixed and evaluated in ten strict tests that jointly withhold 2022–2023 and every two-block combination from five spatial blocks.
3. A 600-tree final model is fitted to all balanced pre-2022 observations and evaluated on the complete 2022–2023 temporal holdout.

Candidate models are ranked by pooled out-of-fold R², with pooled MAE and fold-level R² variability used as tie-breakers.

In [ ]:
def make_rf_pipeline(
    num_features,
    cat_features,
    n_estimators=FINAL_RF_TREES,
    random_state=RANDOM_STATE,
    rf_params=None,
):
    transformers = []

    if num_features:
        transformers.append(
            (
                'num',
                'passthrough',
                list(num_features),
            )
        )

    if cat_features:
        transformers.append(
            (
                'cat',
                OneHotEncoder(
                    handle_unknown='ignore',
                ),
                list(cat_features),
            )
        )

    model_parameters = {
        'n_estimators': n_estimators,
        'random_state': random_state,
        'n_jobs': N_JOBS,
        'max_features': 'sqrt',
        'min_samples_leaf': 1,
        'max_depth': None,
    }

    if rf_params is not None:
        model_parameters.update(
            rf_params
        )

    return Pipeline([
        (
            'prep',
            ColumnTransformer(
                transformers=transformers,
            ),
        ),
        (
            'model',
            RandomForestRegressor(
                **model_parameters,
            ),
        ),
    ])


def prepare_balanced_spatiotemporal_data(
    gdf,
    target_col,
    num_features,
    cat_features,
    random_state=RANDOM_STATE,
    date_col='date',
    lon_col='lon_centroid',
    lat_col='lat_centroid',
    n_blocks=N_SPATIAL_BLOCKS,
    balance_reference_year=BALANCE_REFERENCE_YEAR,
):
    rng = np.random.RandomState(
        random_state
    )

    feature_cols_local = (
        list(num_features)
        + list(cat_features)
    )

    # Complete-case filtering
    n_input_rows = int(
        len(gdf)
    )

    model_df = gdf.copy()

    model_df[date_col] = pd.to_datetime(
        model_df[date_col],
        errors='coerce',
    )

    model_df = model_df.dropna(
        subset=[
            target_col,
            lon_col,
            lat_col,
        ]
        + feature_cols_local
    )

    if model_df.empty:
        raise ValueError(
            'No model rows remain after removing missing values.'
        )

    n_complete_case_rows = int(
        len(model_df)
    )

    n_missing_rows_removed = (
        n_input_rows
        - n_complete_case_rows
    )

    # Aggregate repeated date-GRACE observation units
    model_df['grace_rounded'] = (
        model_df[target_col]
        .round(2)
    )

    aggregation_group_columns = [
        date_col,
        'grace_rounded',
    ]

    aggregation_group_sizes = (
        model_df
        .groupby(
            aggregation_group_columns,
            observed=True,
            dropna=False,
        )
        .size()
        .rename('records_in_unit')
        .reset_index()
    )

    aggregation = {
        target_col: 'mean',
        lon_col: 'mean',
        lat_col: 'mean',
    }

    aggregation.update({
        column: 'mean'
        for column in num_features
    })

    aggregation.update({
        column: 'first'
        for column in cat_features
    })

    aggregated_df = (
        model_df
        .groupby(
            aggregation_group_columns,
            as_index=False,
            observed=True,
            dropna=False,
        )
        .agg(aggregation)
    )

    n_after_aggregation = int(
        len(aggregated_df)
    )

    n_removed_by_aggregation = (
        n_complete_case_rows
        - n_after_aggregation
    )

    aggregation_reduction_percent = (
        100.0
        * n_removed_by_aggregation
        / n_complete_case_rows
        if n_complete_case_rows > 0
        else np.nan
    )

    repeated_units = int(
        aggregation_group_sizes[
            'records_in_unit'
        ]
        .gt(1)
        .sum()
    )

    rows_in_repeated_units = int(
        aggregation_group_sizes.loc[
            aggregation_group_sizes[
                'records_in_unit'
            ].gt(1),
            'records_in_unit',
        ]
        .sum()
    )

    manuscript_sentence = (
        'The aggregation reduced the dataset from '
        f'{n_complete_case_rows:,} to '
        f'{n_after_aggregation:,} records, corresponding to a '
        f'reduction of {aggregation_reduction_percent:.2f}%.'
    )

    aggregation_summary_table = pd.DataFrame([{
        'input_records': n_input_rows,
        'records_after_complete_case_filtering': (
            n_complete_case_rows
        ),
        'records_removed_due_to_missing_values': (
            n_missing_rows_removed
        ),
        'records_before_aggregation': (
            n_complete_case_rows
        ),
        'records_after_aggregation': (
            n_after_aggregation
        ),
        'records_removed_by_aggregation': (
            n_removed_by_aggregation
        ),
        'aggregation_reduction_percent': (
            aggregation_reduction_percent
        ),
        'unique_date_GRACE_units': (
            n_after_aggregation
        ),
        'units_with_multiple_records': (
            repeated_units
        ),
        'records_within_repeated_units': (
            rows_in_repeated_units
        ),
        'mean_records_per_aggregation_unit': float(
            aggregation_group_sizes[
                'records_in_unit'
            ].mean()
        ),
        'median_records_per_aggregation_unit': float(
            aggregation_group_sizes[
                'records_in_unit'
            ].median()
        ),
        'maximum_records_per_aggregation_unit': int(
            aggregation_group_sizes[
                'records_in_unit'
            ].max()
        ),
        'manuscript_sentence': manuscript_sentence,
    }])

    aggregation_group_size_distribution = (
        aggregation_group_sizes[
            'records_in_unit'
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            'original_records_per_aggregation_unit'
        )
        .reset_index(
            name='number_of_aggregation_units'
        )
    )

    print(
        '\nAggregation diagnostics'
    )
    print(
        '-' * 70
    )
    print(
        manuscript_sentence
    )
    print(
        'Aggregation units containing repeated records: '
        f'{repeated_units:,}'
    )
    print(
        'Maximum original records represented by one aggregation '
        f'unit: '
        f'{aggregation_group_sizes["records_in_unit"].max():,}'
    )

    display(
        aggregation_summary_table
        .drop(
            columns='manuscript_sentence'
        )
        .round(2)
    )

    display(
        aggregation_group_size_distribution
    )

    # Spatial-block assignment and balancing
    aggregated_df['year'] = (
        aggregated_df[date_col]
        .dt.year
    )

    aggregated_df['month'] = (
        aggregated_df[date_col]
        .dt.month
    )

    coordinates = aggregated_df[
        [
            lon_col,
            lat_col,
        ]
    ].to_numpy()

    kmeans = KMeans(
        n_clusters=n_blocks,
        random_state=random_state,
        n_init='auto',
    )

    aggregated_df['spatial_block'] = (
        kmeans.fit_predict(
            coordinates
        )
    )

    reference_counts = (
        aggregated_df.loc[
            aggregated_df['year']
            == balance_reference_year
        ]
        .groupby(
            [
                'month',
                'spatial_block',
            ],
            observed=True,
        )
        .size()
        .to_dict()
    )

    selected_indices = []

    for (
        _,
        month,
        block,
    ), index in aggregated_df.groupby(
        [
            'year',
            'month',
            'spatial_block',
        ],
        observed=True,
    ).groups.items():
        index = np.asarray(
            list(index)
        )

        n_reference = reference_counts.get(
            (
                month,
                block,
            ),
            0,
        )

        if (
            n_reference <= 0
            or len(index) <= n_reference
        ):
            selected_indices.append(
                index
            )
        else:
            selected_indices.append(
                rng.choice(
                    index,
                    size=n_reference,
                    replace=False,
                )
            )

    if not selected_indices:
        raise ValueError(
            'No rows were selected during spatiotemporal balancing.'
        )

    balanced_df = (
        aggregated_df
        .loc[
            np.concatenate(
                selected_indices
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    balanced_df['sample_id'] = np.arange(
        len(balanced_df),
        dtype=np.int64,
    )

    spatial_block_summary = (
        balanced_df
        .groupby(
            'spatial_block',
            as_index=False,
            observed=True,
        )
        .agg(
            mean_longitude=(
                lon_col,
                'mean',
            ),
            mean_latitude=(
                lat_col,
                'mean',
            ),
            sample_count=(
                lon_col,
                'size',
            ),
        )
    )

    return (
        balanced_df,
        spatial_block_summary,
        aggregation_summary_table,
        aggregation_group_size_distribution,
    )


def select_rf_parameters_by_grouped_cv(
    selection_df,
    feature_cols,
    target_col,
    parameter_candidates,
    n_splits=CV_SELECTION_FOLDS,
):
    """
    Select structural RF parameters using grouped CV on
    pre-holdout data only.

    Complete year-spatial-block units are kept together.
    Candidate ranking is based on pooled out-of-fold R2,
    with pooled MAE and fold-level R2 variability as
    tie-breakers.
    """
    selection_df = (
        selection_df
        .copy()
        .reset_index(drop=True)
    )

    selection_df['cv_group'] = (
        selection_df['year'].astype(str)
        + '_'
        + selection_df[
            'spatial_block'
        ].astype(str)
    )

    unique_groups = (
        selection_df[
            'cv_group'
        ]
        .nunique()
    )

    effective_splits = min(
        int(n_splits),
        int(unique_groups),
    )

    if effective_splits < 2:
        raise ValueError(
            'At least two year-spatial-block groups are '
            'required for CV selection.'
        )

    grouped_cv = GroupKFold(
        n_splits=effective_splits,
    )

    split_indices = list(
        grouped_cv.split(
            selection_df[
                feature_cols
            ],
            selection_df[
                target_col
            ],
            groups=selection_df[
                'cv_group'
            ],
        )
    )

    candidate_rows = []
    fold_rows = []
    candidate_predictions = {}
    candidate_prediction_std = {}

    for candidate_id, parameters in enumerate(
        parameter_candidates,
        start=1,
    ):
        out_of_fold_prediction = np.full(
            len(selection_df),
            np.nan,
            dtype=float,
        )

        out_of_fold_std = np.full(
            len(selection_df),
            np.nan,
            dtype=float,
        )

        for fold_number, (
            calibration_index,
            validation_index,
        ) in enumerate(
            split_indices,
            start=1,
        ):
            calibration_df = (
                selection_df.iloc[
                    calibration_index
                ]
            )

            validation_df = (
                selection_df.iloc[
                    validation_index
                ]
            )

            candidate_model = make_rf_pipeline(
                num_features,
                cat_features,
                n_estimators=CV_RF_TREES,
                random_state=(
                    RANDOM_STATE
                    + candidate_id * 100
                    + fold_number
                ),
                rf_params=parameters,
            )

            candidate_model.fit(
                calibration_df[
                    feature_cols
                ],
                calibration_df[
                    target_col
                ],
            )

            (
                prediction,
                prediction_std,
                _,
                _,
            ) = predict_rf_distribution(
                candidate_model,
                validation_df[
                    feature_cols
                ],
            )

            out_of_fold_prediction[
                validation_index
            ] = prediction

            out_of_fold_std[
                validation_index
            ] = prediction_std

            validation_values = (
                validation_df[
                    target_col
                ]
                .to_numpy(dtype=float)
            )

            fold_r2 = (
                r2_score(
                    validation_values,
                    prediction,
                )
                if (
                    len(
                        validation_values
                    ) >= 2
                    and not np.allclose(
                        validation_values,
                        validation_values[0],
                    )
                )
                else np.nan
            )

            fold_rows.append({
                'candidate_id': candidate_id,
                'fold': fold_number,
                'calibration_samples': (
                    len(calibration_df)
                ),
                'validation_samples': (
                    len(validation_df)
                ),
                'validation_groups': ', '.join(
                    sorted(
                        validation_df[
                            'cv_group'
                        ].unique()
                    )
                ),
                'r2': fold_r2,
                'mae_cm': mean_absolute_error(
                    validation_values,
                    prediction,
                ),
                'rmse_cm': np.sqrt(
                    mean_squared_error(
                        validation_values,
                        prediction,
                    )
                ),
            })

            del candidate_model
            gc.collect()

        candidate_predictions[
            candidate_id
        ] = out_of_fold_prediction

        candidate_prediction_std[
            candidate_id
        ] = out_of_fold_std

        valid_oof = np.isfinite(
            out_of_fold_prediction
        )

        observed_oof = (
            selection_df.loc[
                valid_oof,
                target_col,
            ]
            .to_numpy(dtype=float)
        )

        predicted_oof = (
            out_of_fold_prediction[
                valid_oof
            ]
        )

        candidate_fold_table = pd.DataFrame(
            [
                row
                for row in fold_rows
                if row[
                    'candidate_id'
                ] == candidate_id
            ]
        )

        candidate_rows.append({
            'candidate_id': candidate_id,
            'max_features': parameters[
                'max_features'
            ],
            'min_samples_leaf': parameters[
                'min_samples_leaf'
            ],
            'max_depth': (
                'None'
                if parameters[
                    'max_depth'
                ] is None
                else parameters[
                    'max_depth'
                ]
            ),
            'cv_folds': effective_splits,
            'selection_samples': (
                len(selection_df)
            ),
            'pooled_oof_r2': r2_score(
                observed_oof,
                predicted_oof,
            ),
            'pooled_oof_mae_cm': (
                mean_absolute_error(
                    observed_oof,
                    predicted_oof,
                )
            ),
            'pooled_oof_rmse_cm': np.sqrt(
                mean_squared_error(
                    observed_oof,
                    predicted_oof,
                )
            ),
            'mean_fold_r2': (
                candidate_fold_table[
                    'r2'
                ].mean()
            ),
            'std_fold_r2': (
                candidate_fold_table[
                    'r2'
                ].std(ddof=1)
            ),
            'mean_fold_mae_cm': (
                candidate_fold_table[
                    'mae_cm'
                ].mean()
            ),
            'std_fold_mae_cm': (
                candidate_fold_table[
                    'mae_cm'
                ].std(ddof=1)
            ),
        })

    candidate_results = (
        pd.DataFrame(
            candidate_rows
        )
        .sort_values(
            [
                'pooled_oof_r2',
                'pooled_oof_mae_cm',
                'std_fold_r2',
            ],
            ascending=[
                False,
                True,
                True,
            ],
            na_position='last',
        )
        .reset_index(drop=True)
    )

    candidate_results.insert(
        0,
        'selection_rank',
        np.arange(
            1,
            len(candidate_results) + 1,
        ),
    )

    fold_results = pd.DataFrame(
        fold_rows
    )

    selected_candidate_id = int(
        candidate_results.iloc[0][
            'candidate_id'
        ]
    )

    selected_parameters = dict(
        parameter_candidates[
            selected_candidate_id - 1
        ]
    )

    selected_prediction = (
        candidate_predictions[
            selected_candidate_id
        ]
    )

    selected_prediction_std = (
        candidate_prediction_std[
            selected_candidate_id
        ]
    )

    selected_valid = np.isfinite(
        selected_prediction
    )

    selected_observed = (
        selection_df.loc[
            selected_valid,
            target_col,
        ]
        .to_numpy(dtype=float)
    )

    selected_prediction = (
        selected_prediction[
            selected_valid
        ]
    )

    selected_prediction_std = (
        selected_prediction_std[
            selected_valid
        ]
    )

    selected_lower = (
        selected_prediction
        - UNCERTAINTY_Z
        * selected_prediction_std
    )

    selected_upper = (
        selected_prediction
        + UNCERTAINTY_Z
        * selected_prediction_std
    )

    selected_metric_uncertainty = (
        bootstrap_regression_metrics(
            selected_observed,
            selected_prediction,
            random_state=(
                RANDOM_STATE + 4000
            ),
        )
    )

    selected_cv_predictions_df = (
        selection_df.loc[
            selected_valid,
            [
                'sample_id',
                'date',
                'year',
                'month',
                'spatial_block',
                target_col,
                'cv_group',
            ],
        ]
        .copy()
        .rename(columns={
            target_col: 'observed',
        })
        .reset_index(drop=True)
    )

    selected_cv_predictions_df[
        'predicted'
    ] = selected_prediction

    selected_cv_predictions_df[
        'prediction_std'
    ] = selected_prediction_std

    selected_cv_predictions_df[
        'prediction_lower'
    ] = selected_lower

    selected_cv_predictions_df[
        'prediction_upper'
    ] = selected_upper

    selected_cv_predictions_df[
        'interval_covered'
    ] = (
        selected_cv_predictions_df[
            'observed'
        ]
        .between(
            selected_cv_predictions_df[
                'prediction_lower'
            ],
            selected_cv_predictions_df[
                'prediction_upper'
            ],
        )
    )

    selected_parameters_table = pd.DataFrame([{
        'selection_data': (
            'Balanced pre-2022 observations only'
        ),
        'selection_method': (
            f'{effective_splits}-fold GroupKFold by '
            'year-spatial-block'
        ),
        'selection_criterion': (
            'Highest pooled OOF R2; lower MAE and R2 '
            'variability as tie-breakers'
        ),
        'selected_candidate_id': (
            selected_candidate_id
        ),
        'max_features': (
            selected_parameters[
                'max_features'
            ]
        ),
        'min_samples_leaf': (
            selected_parameters[
                'min_samples_leaf'
            ]
        ),
        'max_depth': (
            'None'
            if selected_parameters[
                'max_depth'
            ] is None
            else selected_parameters[
                'max_depth'
            ]
        ),
        'final_n_estimators': (
            FINAL_RF_TREES
        ),
        'selection_pooled_oof_r2': (
            selected_metric_uncertainty[
                'r2'
            ]['estimate']
        ),
        'selection_r2_95_CI_lower': (
            selected_metric_uncertainty[
                'r2'
            ]['ci_lower']
        ),
        'selection_r2_95_CI_upper': (
            selected_metric_uncertainty[
                'r2'
            ]['ci_upper']
        ),
        'selection_pooled_oof_mae_cm': (
            selected_metric_uncertainty[
                'mae'
            ]['estimate']
        ),
        'selection_mae_95_CI_lower_cm': (
            selected_metric_uncertainty[
                'mae'
            ]['ci_lower']
        ),
        'selection_mae_95_CI_upper_cm': (
            selected_metric_uncertainty[
                'mae'
            ]['ci_upper']
        ),
        'selection_pooled_oof_rmse_cm': (
            selected_metric_uncertainty[
                'rmse'
            ]['estimate']
        ),
        'selection_rmse_95_CI_lower_cm': (
            selected_metric_uncertainty[
                'rmse'
            ]['ci_lower']
        ),
        'selection_rmse_95_CI_upper_cm': (
            selected_metric_uncertainty[
                'rmse'
            ]['ci_upper']
        ),
        'selection_mean_prediction_std_cm': float(
            np.nanmean(
                selected_prediction_std
            )
        ),
        'selection_95_interval_coverage': float(
            selected_cv_predictions_df[
                'interval_covered'
            ].mean()
        ),
    }])

    return (
        selected_parameters,
        selected_parameters_table,
        candidate_results,
        fold_results,
        selected_cv_predictions_df,
    )


def split_repeated_outer_holdout(
    balanced_df,
    holdout_blocks,
    holdout_years=HOLDOUT_YEARS,
):
    holdout_blocks = {
        int(value)
        for value in holdout_blocks
    }

    holdout_years = {
        int(value)
        for value in holdout_years
    }

    holdout_mask = (
        balanced_df[
            'year'
        ].isin(
            holdout_years
        )
        & balanced_df[
            'spatial_block'
        ].isin(
            holdout_blocks
        )
    )

    training_mask = (
        ~balanced_df[
            'year'
        ].isin(
            holdout_years
        )
        & ~balanced_df[
            'spatial_block'
        ].isin(
            holdout_blocks
        )
    )

    return (
        balanced_df.loc[
            training_mask
        ].copy(),
        balanced_df.loc[
            holdout_mask
        ].copy(),
    )


def evaluate_repeated_outer_holdouts(
    balanced_df,
    feature_cols,
    target_col,
    block_combinations,
    rf_params,
):
    performance_rows = []
    prediction_frames = []

    for run_number, holdout_blocks in enumerate(
        block_combinations,
        start=1,
    ):
        (
            training_run,
            holdout_run,
        ) = split_repeated_outer_holdout(
            balanced_df,
            holdout_blocks,
        )

        if (
            training_run.empty
            or len(holdout_run) < 2
        ):
            continue

        run_model = make_rf_pipeline(
            num_features,
            cat_features,
            n_estimators=(
                OUTER_TEST_RF_TREES
            ),
            random_state=(
                RANDOM_STATE
                + run_number
            ),
            rf_params=rf_params,
        )

        run_model.fit(
            training_run[
                feature_cols
            ],
            training_run[
                target_col
            ],
        )

        (
            holdout_prediction,
            holdout_std,
            holdout_lower,
            holdout_upper,
        ) = predict_rf_distribution(
            run_model,
            holdout_run[
                feature_cols
            ],
        )

        metric_uncertainty = (
            bootstrap_regression_metrics(
                holdout_run[
                    target_col
                ],
                holdout_prediction,
                random_state=(
                    RANDOM_STATE
                    + 1000
                    + run_number
                ),
            )
        )

        coverage = np.mean(
            (
                holdout_run[
                    target_col
                ].to_numpy()
                >= holdout_lower
            )
            & (
                holdout_run[
                    target_col
                ].to_numpy()
                <= holdout_upper
            )
        )

        block_label = '+'.join(
            str(value)
            for value in holdout_blocks
        )

        performance_rows.append({
            'outer_test': (
                run_number
            ),
            'holdout_blocks': (
                block_label
            ),
            'holdout_years': ', '.join(
                str(value)
                for value in HOLDOUT_YEARS
            ),
            'training_samples': (
                len(training_run)
            ),
            'holdout_samples': (
                len(holdout_run)
            ),
            'holdout_r2': (
                metric_uncertainty[
                    'r2'
                ]['estimate']
            ),
            'holdout_r2_ci_lower': (
                metric_uncertainty[
                    'r2'
                ]['ci_lower']
            ),
            'holdout_r2_ci_upper': (
                metric_uncertainty[
                    'r2'
                ]['ci_upper']
            ),
            'holdout_mae_cm': (
                metric_uncertainty[
                    'mae'
                ]['estimate']
            ),
            'holdout_mae_ci_lower_cm': (
                metric_uncertainty[
                    'mae'
                ]['ci_lower']
            ),
            'holdout_mae_ci_upper_cm': (
                metric_uncertainty[
                    'mae'
                ]['ci_upper']
            ),
            'holdout_rmse_cm': (
                metric_uncertainty[
                    'rmse'
                ]['estimate']
            ),
            'holdout_rmse_ci_lower_cm': (
                metric_uncertainty[
                    'rmse'
                ]['ci_lower']
            ),
            'holdout_rmse_ci_upper_cm': (
                metric_uncertainty[
                    'rmse'
                ]['ci_upper']
            ),
            'mean_prediction_std_cm': float(
                np.nanmean(
                    holdout_std
                )
            ),
            'mean_95_interval_width_cm': float(
                np.nanmean(
                    holdout_upper
                    - holdout_lower
                )
            ),
            'empirical_95_interval_coverage': float(
                coverage
            ),
        })

        prediction_frame = (
            holdout_run[
                [
                    'sample_id',
                    'date',
                    'year',
                    'month',
                    'spatial_block',
                    target_col,
                ]
            ]
            .copy()
        )

        prediction_frame[
            'outer_test'
        ] = run_number

        prediction_frame[
            'holdout_blocks'
        ] = block_label

        prediction_frame[
            'observed'
        ] = prediction_frame[
            target_col
        ]

        prediction_frame[
            'predicted'
        ] = holdout_prediction

        prediction_frame[
            'prediction_std'
        ] = holdout_std

        prediction_frame[
            'prediction_lower'
        ] = holdout_lower

        prediction_frame[
            'prediction_upper'
        ] = holdout_upper

        prediction_frame[
            'interval_covered'
        ] = (
            prediction_frame[
                'observed'
            ]
            .between(
                prediction_frame[
                    'prediction_lower'
                ],
                prediction_frame[
                    'prediction_upper'
                ],
            )
        )

        prediction_frames.append(
            prediction_frame
        )

        del run_model
        del training_run
        del holdout_run
        gc.collect()

    performance_df = pd.DataFrame(
        performance_rows
    )

    if not prediction_frames:
        raise ValueError(
            'No valid repeated outer-test predictions were generated.'
        )

    predictions_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    pooled_predictions = (
        predictions_df
        .groupby(
            'sample_id',
            observed=True,
        )
        .agg(
            date=(
                'date',
                'first',
            ),
            year=(
                'year',
                'first',
            ),
            month=(
                'month',
                'first',
            ),
            spatial_block=(
                'spatial_block',
                'first',
            ),
            observed=(
                'observed',
                'first',
            ),
            predicted=(
                'predicted',
                'mean',
            ),
            repeated_models=(
                'outer_test',
                'nunique',
            ),
            mean_within_model_variance=(
                'prediction_std',
                lambda values: np.mean(
                    np.square(
                        values
                    )
                ),
            ),
            between_model_variance=(
                'predicted',
                lambda values: (
                    np.var(
                        values,
                        ddof=1,
                    )
                    if len(values) > 1
                    else 0.0
                ),
            ),
        )
        .reset_index()
    )

    pooled_predictions[
        'prediction_std'
    ] = np.sqrt(
        pooled_predictions[
            'mean_within_model_variance'
        ]
        + pooled_predictions[
            'between_model_variance'
        ]
    )

    pooled_predictions[
        'prediction_lower'
    ] = (
        pooled_predictions[
            'predicted'
        ]
        - UNCERTAINTY_Z
        * pooled_predictions[
            'prediction_std'
        ]
    )

    pooled_predictions[
        'prediction_upper'
    ] = (
        pooled_predictions[
            'predicted'
        ]
        + UNCERTAINTY_Z
        * pooled_predictions[
            'prediction_std'
        ]
    )

    pooled_predictions[
        'interval_covered'
    ] = (
        pooled_predictions[
            'observed'
        ]
        .between(
            pooled_predictions[
                'prediction_lower'
            ],
            pooled_predictions[
                'prediction_upper'
            ],
        )
    )

    return (
        performance_df,
        predictions_df,
        pooled_predictions,
    )


def performance_row_from_predictions(
    design,
    role,
    period,
    spatial_scope,
    predictions,
    random_state,
    interval_definition,
):
    metric_uncertainty = (
        bootstrap_regression_metrics(
            predictions[
                'observed'
            ],
            predictions[
                'predicted'
            ],
            random_state=(
                random_state
            ),
        )
    )

    return {
        'validation_design': design,
        'role': role,
        'period': period,
        'spatial_scope': spatial_scope,
        'samples': len(
            predictions
        ),
        'r2': (
            metric_uncertainty[
                'r2'
            ]['estimate']
        ),
        'r2_95_CI_lower': (
            metric_uncertainty[
                'r2'
            ]['ci_lower']
        ),
        'r2_95_CI_upper': (
            metric_uncertainty[
                'r2'
            ]['ci_upper']
        ),
        'mae_cm': (
            metric_uncertainty[
                'mae'
            ]['estimate']
        ),
        'mae_95_CI_lower_cm': (
            metric_uncertainty[
                'mae'
            ]['ci_lower']
        ),
        'mae_95_CI_upper_cm': (
            metric_uncertainty[
                'mae'
            ]['ci_upper']
        ),
        'rmse_cm': (
            metric_uncertainty[
                'rmse'
            ]['estimate']
        ),
        'rmse_95_CI_lower_cm': (
            metric_uncertainty[
                'rmse'
            ]['ci_lower']
        ),
        'rmse_95_CI_upper_cm': (
            metric_uncertainty[
                'rmse'
            ]['ci_upper']
        ),
        'mean_prediction_std_cm': (
            predictions[
                'prediction_std'
            ].mean()
        ),
        'empirical_95_interval_coverage': (
            predictions[
                'interval_covered'
            ].mean()
        ),
        'interval_definition': (
            interval_definition
        ),
    }


(
    balanced_model_df,
    spatial_block_summary,
    aggregation_summary_table,
    aggregation_group_size_distribution,
) = prepare_balanced_spatiotemporal_data(
    gdf025,
    TARGET_COL,
    num_features,
    cat_features,
)

aggregation_manuscript_sentence = (
    aggregation_summary_table.loc[
        0,
        'manuscript_sentence',
    ]
)

print(
    '\nManuscript-ready aggregation statement:'
)
print(
    aggregation_manuscript_sentence
)

aggregation_summary_table.to_csv(
    OUTPUT_DIR
    / 'downscaling_aggregation_summary.csv',
    index=False,
)

aggregation_group_size_distribution.to_csv(
    OUTPUT_DIR
    / 'downscaling_aggregation_group_size_distribution.csv',
    index=False,
)

# Parameter selection never sees the fixed
# 2022-2023 temporal holdout.
cv_selection_df = (
    balanced_model_df.loc[
        ~balanced_model_df[
            'year'
        ].isin(
            HOLDOUT_YEARS
        )
    ]
    .copy()
)

(
    selected_rf_parameters,
    selected_rf_parameters_table,
    cv_parameter_results_df,
    cv_parameter_fold_results_df,
    selected_cv_predictions_df,
) = select_rf_parameters_by_grouped_cv(
    cv_selection_df,
    feature_cols,
    TARGET_COL,
    RF_PARAMETER_CANDIDATES,
)

# All ten spatial combinations evaluate the same frozen
# CV-selected parameters.
(
    outer_test_results_df,
    outer_test_predictions_df,
    pooled_outer_predictions_df,
) = evaluate_repeated_outer_holdouts(
    balanced_model_df,
    feature_cols,
    TARGET_COL,
    OUTER_BLOCK_COMBINATIONS,
    rf_params=(
        selected_rf_parameters
    ),
)

outer_test_summary_table = (
    summarize_repeated_metrics(
        outer_test_results_df,
        [
            'holdout_r2',
            'holdout_mae_cm',
            'holdout_rmse_cm',
            'mean_prediction_std_cm',
            'mean_95_interval_width_cm',
            'empirical_95_interval_coverage',
        ],
    )
)

block_holdout_frequency = pd.Series(
    [
        block
        for combination
        in OUTER_BLOCK_COMBINATIONS
        for block
        in combination
    ]
).value_counts()

spatial_block_summary[
    'outer_holdout_repetitions'
] = (
    spatial_block_summary[
        'spatial_block'
    ]
    .map(
        block_holdout_frequency
    )
    .fillna(0)
    .astype(int)
)

# Final model: CV-selected parameters and all pre-2022
# spatial blocks.
training_df = (
    cv_selection_df.copy()
)

holdout_df = (
    balanced_model_df.loc[
        balanced_model_df[
            'year'
        ].isin(
            HOLDOUT_YEARS
        )
    ]
    .copy()
)

downscaling_model = make_rf_pipeline(
    num_features,
    cat_features,
    n_estimators=FINAL_RF_TREES,
    random_state=RANDOM_STATE,
    rf_params=(
        selected_rf_parameters
    ),
)

downscaling_model.fit(
    training_df[
        feature_cols
    ],
    training_df[
        TARGET_COL
    ],
)

(
    training_predictions,
    training_prediction_std,
    training_prediction_lower,
    training_prediction_upper,
) = predict_rf_distribution(
    downscaling_model,
    training_df[
        feature_cols
    ],
)

(
    holdout_predictions,
    holdout_prediction_std,
    holdout_prediction_lower,
    holdout_prediction_upper,
) = predict_rf_distribution(
    downscaling_model,
    holdout_df[
        feature_cols
    ],
)

temporal_metric_uncertainty = (
    bootstrap_regression_metrics(
        holdout_df[
            TARGET_COL
        ],
        holdout_predictions,
        random_state=(
            RANDOM_STATE + 2000
        ),
    )
)

selected_cv_row = (
    cv_parameter_results_df
    .loc[
        cv_parameter_results_df[
            'selection_rank'
        ].eq(1)
    ]
    .iloc[0]
)

downscaling_performance_df = pd.DataFrame([{
    'model': (
        'Random Forest'
    ),
    'parameter_selection': (
        'Grouped CV on balanced pre-2022 '
        'year-spatial-block groups'
    ),
    'validation': (
        'Complete 2022-2023 temporal holdout'
    ),
    'selected_max_features': (
        selected_rf_parameters[
            'max_features'
        ]
    ),
    'selected_min_samples_leaf': (
        selected_rf_parameters[
            'min_samples_leaf'
        ]
    ),
    'selected_max_depth': (
        'None'
        if selected_rf_parameters[
            'max_depth'
        ] is None
        else selected_rf_parameters[
            'max_depth'
        ]
    ),
    'selection_cv_pooled_r2': (
        selected_cv_row[
            'pooled_oof_r2'
        ]
    ),
    'selection_cv_pooled_mae_cm': (
        selected_cv_row[
            'pooled_oof_mae_cm'
        ]
    ),
    'selection_cv_pooled_rmse_cm': (
        selected_cv_row[
            'pooled_oof_rmse_cm'
        ]
    ),
    'holdout_r2': (
        temporal_metric_uncertainty[
            'r2'
        ]['estimate']
    ),
    'holdout_r2_ci_lower': (
        temporal_metric_uncertainty[
            'r2'
        ]['ci_lower']
    ),
    'holdout_r2_ci_upper': (
        temporal_metric_uncertainty[
            'r2'
        ]['ci_upper']
    ),
    'holdout_mae_cm': (
        temporal_metric_uncertainty[
            'mae'
        ]['estimate']
    ),
    'holdout_mae_ci_lower_cm': (
        temporal_metric_uncertainty[
            'mae'
        ]['ci_lower']
    ),
    'holdout_mae_ci_upper_cm': (
        temporal_metric_uncertainty[
            'mae'
        ]['ci_upper']
    ),
    'holdout_rmse_cm': (
        temporal_metric_uncertainty[
            'rmse'
        ]['estimate']
    ),
    'holdout_rmse_ci_lower_cm': (
        temporal_metric_uncertainty[
            'rmse'
        ]['ci_lower']
    ),
    'holdout_rmse_ci_upper_cm': (
        temporal_metric_uncertainty[
            'rmse'
        ]['ci_upper']
    ),
    'training_r2': r2_score(
        training_df[
            TARGET_COL
        ],
        training_predictions,
    ),
    'training_mae_cm': (
        mean_absolute_error(
            training_df[
                TARGET_COL
            ],
            training_predictions,
        )
    ),
    'balanced_samples': (
        len(balanced_model_df)
    ),
    'training_samples': (
        len(training_df)
    ),
    'holdout_samples': (
        len(holdout_df)
    ),
    'mean_holdout_prediction_std_cm': float(
        np.nanmean(
            holdout_prediction_std
        )
    ),
    'mean_holdout_95_interval_width_cm': float(
        np.nanmean(
            holdout_prediction_upper
            - holdout_prediction_lower
        )
    ),
    'holdout_95_interval_coverage': float(
        np.mean(
            holdout_df[
                TARGET_COL
            ]
            .between(
                holdout_prediction_lower,
                holdout_prediction_upper,
            )
        )
    ),
}])

prediction_comparison_df = pd.DataFrame({
    'subset': (
        ['training']
        * len(training_df)
        + ['holdout']
        * len(holdout_df)
    ),
    'observed': np.concatenate([
        training_df[
            TARGET_COL
        ].to_numpy(),
        holdout_df[
            TARGET_COL
        ].to_numpy(),
    ]),
    'predicted': np.concatenate([
        training_predictions,
        holdout_predictions,
    ]),
    'prediction_std': np.concatenate([
        training_prediction_std,
        holdout_prediction_std,
    ]),
    'prediction_lower': np.concatenate([
        training_prediction_lower,
        holdout_prediction_lower,
    ]),
    'prediction_upper': np.concatenate([
        training_prediction_upper,
        holdout_prediction_upper,
    ]),
})

final_temporal_predictions_df = (
    holdout_df[
        [
            'sample_id',
            'date',
            'year',
            'month',
            'spatial_block',
            TARGET_COL,
        ]
    ]
    .copy()
    .rename(columns={
        TARGET_COL: 'observed',
    })
)

final_temporal_predictions_df[
    'predicted'
] = holdout_predictions

final_temporal_predictions_df[
    'prediction_std'
] = holdout_prediction_std

final_temporal_predictions_df[
    'prediction_lower'
] = holdout_prediction_lower

final_temporal_predictions_df[
    'prediction_upper'
] = holdout_prediction_upper

final_temporal_predictions_df[
    'interval_covered'
] = (
    final_temporal_predictions_df[
        'observed'
    ]
    .between(
        final_temporal_predictions_df[
            'prediction_lower'
        ],
        final_temporal_predictions_df[
            'prediction_upper'
        ],
    )
)

outer_cv_comparison_table = pd.DataFrame([
    performance_row_from_predictions(
        design=(
            'Grouped CV out-of-fold'
        ),
        role=(
            'Hyperparameter selection estimate'
        ),
        period=(
            'Pre-2022'
        ),
        spatial_scope=(
            'Year-spatial-block groups withheld '
            'within pre-2022 data'
        ),
        predictions=(
            selected_cv_predictions_df
        ),
        random_state=(
            RANDOM_STATE + 4100
        ),
        interval_definition=(
            'Paired-bootstrap metric CI; '
            'fold-specific RF dispersion'
        ),
    ),
    performance_row_from_predictions(
        design=(
            'Repeated outer tests, pooled'
        ),
        role=(
            'Untouched spatiotemporal evaluation'
        ),
        period=(
            '2022-2023'
        ),
        spatial_scope=(
            'All ten two-block holdout combinations'
        ),
        predictions=(
            pooled_outer_predictions_df
        ),
        random_state=(
            RANDOM_STATE + 4200
        ),
        interval_definition=(
            'Paired-bootstrap metric CI; within- plus '
            'between-outer-model dispersion'
        ),
    ),
    performance_row_from_predictions(
        design=(
            'Complete temporal holdout'
        ),
        role=(
            'Final model evaluation'
        ),
        period=(
            '2022-2023'
        ),
        spatial_scope=(
            'All spatial blocks'
        ),
        predictions=(
            final_temporal_predictions_df
        ),
        random_state=(
            RANDOM_STATE + 4300
        ),
        interval_definition=(
            'Paired-bootstrap metric CI; '
            'final RF tree dispersion'
        ),
    ),
])

display(
    aggregation_summary_table.round(4)
)

display(
    aggregation_group_size_distribution
)

display(
    selected_rf_parameters_table
)

display(
    cv_parameter_results_df.round(4)
)

display(
    outer_cv_comparison_table.round(4)
)

display(
    downscaling_performance_df.round(4)
)

display(
    outer_test_results_df.round(4)
)

display(
    outer_test_summary_table.round(4)
)

display(
    spatial_block_summary.round(4)
)

memory_report(
    'after CV selection, repeated outer tests, '
    'and final model fitting'
)


### 4.4 Selected parameters and validation designs

Parameter-selection results, fold metrics, strict outer-test metrics, and the complete temporal holdout are reported separately.

In [ ]:
downscaling_results_table = downscaling_performance_df.rename(columns={
    'model': 'Model',
    'parameter_selection': 'Parameter selection',
    'validation': 'Validation',
    'selected_max_features': 'Selected max_features',
    'selected_min_samples_leaf': 'Selected min_samples_leaf',
    'selected_max_depth': 'Selected max_depth',
    'selection_cv_pooled_r2': 'Selection CV pooled R²',
    'selection_cv_pooled_mae_cm': 'Selection CV pooled MAE (cm)',
    'selection_cv_pooled_rmse_cm': 'Selection CV pooled RMSE (cm)',
    'holdout_r2': 'Holdout R²',
    'holdout_r2_ci_lower': 'R² 95% CI lower',
    'holdout_r2_ci_upper': 'R² 95% CI upper',
    'holdout_mae_cm': 'Holdout MAE (cm)',
    'holdout_mae_ci_lower_cm': 'MAE 95% CI lower (cm)',
    'holdout_mae_ci_upper_cm': 'MAE 95% CI upper (cm)',
    'holdout_rmse_cm': 'Holdout RMSE (cm)',
    'holdout_rmse_ci_lower_cm': 'RMSE 95% CI lower (cm)',
    'holdout_rmse_ci_upper_cm': 'RMSE 95% CI upper (cm)',
    'training_r2': 'Training R²',
    'training_mae_cm': 'Training MAE (cm)',
    'balanced_samples': 'Balanced samples',
    'training_samples': 'Training samples',
    'holdout_samples': 'Holdout samples',
    'mean_holdout_prediction_std_cm': 'Mean prediction SD (cm)',
    'mean_holdout_95_interval_width_cm': 'Mean 95% interval width (cm)',
    'holdout_95_interval_coverage': 'Empirical 95% coverage',
})

display_if_requested(selected_rf_parameters_table)
display_if_requested(cv_parameter_results_df.round(4))
display_if_requested(outer_cv_comparison_table.round(4))
display_if_requested(downscaling_results_table.round(4))
display_if_requested(outer_test_results_df.round(4))
display_if_requested(outer_test_summary_table.round(4))
display_if_requested(spatial_block_summary.round(4))


### 4.5 Year–block diagnostic cross-validation

Each pre-2022 year–spatial-block unit is withheld once using fixed selected parameters. Fold-level metrics, bootstrap intervals, ensemble dispersion, and interval coverage diagnose spatiotemporal stability.

In [ ]:
def compute_year_block_cv(
    balanced_df,
    feature_cols,
    target_col,
    rf_params,
):
    rows = []

    grouped_indices = balanced_df.groupby(
        [
            'year',
            'spatial_block',
        ]
    ).groups

    for fold_number, (
        (
            year,
            block,
        ),
        indices,
    ) in enumerate(
        sorted(grouped_indices.items()),
        start=1,
    ):
        validation_df = balanced_df.loc[
            indices
        ]
        if len(validation_df) < 5:
            continue

        calibration_df = balanced_df.drop(
            index=indices
        )

        diagnostic_model = make_rf_pipeline(
            num_features,
            cat_features,
            n_estimators=CV_RF_TREES,
            random_state=(
                RANDOM_STATE + fold_number
            ),
            rf_params=rf_params,
        )
        diagnostic_model.fit(
            calibration_df[feature_cols],
            calibration_df[target_col],
        )
        (
            prediction,
            prediction_std,
            prediction_lower,
            prediction_upper,
        ) = predict_rf_distribution(
            diagnostic_model,
            validation_df[feature_cols],
        )

        metric_uncertainty = (
            bootstrap_regression_metrics(
                validation_df[target_col],
                prediction,
                n_bootstrap=min(
                    150,
                    BOOTSTRAP_ITERATIONS,
                ),
                random_state=(
                    RANDOM_STATE
                    + 3000
                    + fold_number
                ),
            )
        )

        rows.append({
            'year': int(year),
            'spatial_block': int(block),
            'validation_samples': int(
                len(validation_df)
            ),
            'r2_cv': metric_uncertainty[
                'r2'
            ]['estimate'],
            'r2_ci_lower': metric_uncertainty[
                'r2'
            ]['ci_lower'],
            'r2_ci_upper': metric_uncertainty[
                'r2'
            ]['ci_upper'],
            'mae_cv': metric_uncertainty[
                'mae'
            ]['estimate'],
            'mae_ci_lower_cm': metric_uncertainty[
                'mae'
            ]['ci_lower'],
            'mae_ci_upper_cm': metric_uncertainty[
                'mae'
            ]['ci_upper'],
            'rmse_cv': metric_uncertainty[
                'rmse'
            ]['estimate'],
            'rmse_ci_lower_cm': metric_uncertainty[
                'rmse'
            ]['ci_lower'],
            'rmse_ci_upper_cm': metric_uncertainty[
                'rmse'
            ]['ci_upper'],
            'mean_prediction_std_cm': float(
                np.nanmean(
                    prediction_std
                )
            ),
            'mean_95_interval_width_cm': float(
                np.nanmean(
                    prediction_upper
                    - prediction_lower
                )
            ),
            'empirical_95_interval_coverage': float(
                np.mean(
                    validation_df[
                        target_col
                    ].between(
                        prediction_lower,
                        prediction_upper,
                    )
                )
            ),
        })

        del diagnostic_model
        gc.collect()

    return pd.DataFrame(rows)


spatiotemporal_cv_df = compute_year_block_cv(
    cv_selection_df,
    feature_cols,
    TARGET_COL,
    rf_params=selected_rf_parameters,
)

cv_summary_table = (
    spatiotemporal_cv_df[
        [
            'r2_cv',
            'mae_cv',
            'rmse_cv',
            'mean_prediction_std_cm',
            'mean_95_interval_width_cm',
            'empirical_95_interval_coverage',
        ]
    ]
    .agg(
        [
            'mean',
            'std',
            'min',
            'median',
            'max',
        ]
    )
    .rename(columns={
        'r2_cv': 'R²',
        'mae_cv': 'MAE (cm)',
        'rmse_cv': 'RMSE (cm)',
        'mean_prediction_std_cm': (
            'Prediction SD (cm)'
        ),
        'mean_95_interval_width_cm': (
            '95% interval width (cm)'
        ),
        'empirical_95_interval_coverage': (
            '95% coverage'
        ),
    })
)

outer_cv_fold_distribution_table = pd.DataFrame([
    {
        'validation_design': (
            'Diagnostic leave-one-year–block-out CV'
        ),
        'period': 'Pre-2022',
        'number_of_folds_or_tests': len(
            spatiotemporal_cv_df
        ),
        'mean_r2': spatiotemporal_cv_df[
            'r2_cv'
        ].mean(),
        'std_r2': spatiotemporal_cv_df[
            'r2_cv'
        ].std(ddof=1),
        'median_r2': spatiotemporal_cv_df[
            'r2_cv'
        ].median(),
        'mean_mae_cm': spatiotemporal_cv_df[
            'mae_cv'
        ].mean(),
        'std_mae_cm': spatiotemporal_cv_df[
            'mae_cv'
        ].std(ddof=1),
        'median_mae_cm': spatiotemporal_cv_df[
            'mae_cv'
        ].median(),
        'mean_rmse_cm': spatiotemporal_cv_df[
            'rmse_cv'
        ].mean(),
        'std_rmse_cm': spatiotemporal_cv_df[
            'rmse_cv'
        ].std(ddof=1),
        'median_rmse_cm': spatiotemporal_cv_df[
            'rmse_cv'
        ].median(),
        'mean_coverage': spatiotemporal_cv_df[
            'empirical_95_interval_coverage'
        ].mean(),
    },
    {
        'validation_design': (
            'Repeated two-block outer tests'
        ),
        'period': '2022–2023',
        'number_of_folds_or_tests': len(
            outer_test_results_df
        ),
        'mean_r2': outer_test_results_df[
            'holdout_r2'
        ].mean(),
        'std_r2': outer_test_results_df[
            'holdout_r2'
        ].std(ddof=1),
        'median_r2': outer_test_results_df[
            'holdout_r2'
        ].median(),
        'mean_mae_cm': outer_test_results_df[
            'holdout_mae_cm'
        ].mean(),
        'std_mae_cm': outer_test_results_df[
            'holdout_mae_cm'
        ].std(ddof=1),
        'median_mae_cm': outer_test_results_df[
            'holdout_mae_cm'
        ].median(),
        'mean_rmse_cm': outer_test_results_df[
            'holdout_rmse_cm'
        ].mean(),
        'std_rmse_cm': outer_test_results_df[
            'holdout_rmse_cm'
        ].std(ddof=1),
        'median_rmse_cm': outer_test_results_df[
            'holdout_rmse_cm'
        ].median(),
        'mean_coverage': outer_test_results_df[
            'empirical_95_interval_coverage'
        ].mean(),
    },
])

display_if_requested(cv_summary_table.round(3))
display_if_requested(outer_cv_fold_distribution_table.round(4))

spatiotemporal_cv_df.to_csv(
    OUTPUT_DIR
    / 'spatiotemporal_cross_validation.csv',
    index=False,
)


### 4.6 Mass-conserving 0.25°→0.1° redistribution

The correction is applied by month and conservation block. Raw prediction variance is combined in quadrature with imputed-GRACE anchoring variance for the corrected field. Predictor, weighting-field, and GRACE measurement uncertainties are not included.

In [ ]:
# Mass-conserving 0.25°→0.1° redistribution
def downscale_grace(gdf025, gdf01, model_pipeline, num_features, cat_features, target_col='grace_or_imputed', date_col='date', bias_method='uniform', weight_col='Q_roll3', mass_blocks=None):
    """
    GRACE downscaling from 0.25° to 0.1° with mass-conserving bias correction,
    operating per date.

    - NaN-safe: predictions are done only where all feature columns are non-NaN.
    - mass_blocks:
        * None or <=1: conserve mass per 0.25° cell.
        * int > 1    : subdivide the 0.25° domain into `mass_blocks` spatial
                       clusters using KMeans, and conserve mass per cluster.
    - IMPORTANT: target GRACE mass is computed using the *effective* area,
      i.e. only where 0.1° cells exist (adjusts for area loss).
    """
    gdf_025_pred = gdf025.copy()
    gdf_01_pred = gdf01.copy()
    gdf_025_pred['row_025'] = np.arange(len(gdf_025_pred), dtype=np.int64)
    gdf_01_pred['row_01'] = np.arange(len(gdf_01_pred), dtype=np.int64)
    feat_cols = list(num_features) + list(cat_features)
    X_025_all = gdf_025_pred[feat_cols]
    y_025_true = gdf_025_pred[target_col].values
    mask_feat_025 = X_025_all.notna().all(axis=1)
    y_025_pred_raw = np.full(len(gdf_025_pred), np.nan, dtype=float)
    y_025_pred_std = np.full(len(gdf_025_pred), np.nan, dtype=float)
    y_025_pred_lower = np.full(len(gdf_025_pred), np.nan, dtype=float)
    y_025_pred_upper = np.full(len(gdf_025_pred), np.nan, dtype=float)
    if mask_feat_025.any():
        prediction_025, prediction_std_025, prediction_lower_025, prediction_upper_025 = predict_rf_distribution(model_pipeline, X_025_all.loc[mask_feat_025])
        valid_index = mask_feat_025.to_numpy()
        y_025_pred_raw[valid_index] = prediction_025
        y_025_pred_std[valid_index] = prediction_std_025
        y_025_pred_lower[valid_index] = prediction_lower_025
        y_025_pred_upper[valid_index] = prediction_upper_025
    gdf_025_pred['y_true'] = y_025_true
    gdf_025_pred['y_pred_raw'] = y_025_pred_raw
    gdf_025_pred['y_pred_raw_std'] = y_025_pred_std
    gdf_025_pred['y_pred_raw_lower'] = y_025_pred_lower
    gdf_025_pred['y_pred_raw_upper'] = y_025_pred_upper
    gdf_025_pred['y_bias'] = gdf_025_pred['y_true'] - gdf_025_pred['y_pred_raw']
    X_01_all = gdf_01_pred[feat_cols]
    y_01_true = gdf_01_pred[target_col] if target_col in gdf_01_pred.columns else None
    mask_feat_01 = X_01_all.notna().all(axis=1)
    y_01_pred_raw = np.full(len(gdf_01_pred), np.nan, dtype=float)
    y_01_pred_std = np.full(len(gdf_01_pred), np.nan, dtype=float)
    y_01_pred_lower = np.full(len(gdf_01_pred), np.nan, dtype=float)
    y_01_pred_upper = np.full(len(gdf_01_pred), np.nan, dtype=float)
    if mask_feat_01.any():
        prediction_01, prediction_std_01, prediction_lower_01, prediction_upper_01 = predict_rf_distribution(model_pipeline, X_01_all.loc[mask_feat_01])
        valid_index = mask_feat_01.to_numpy()
        y_01_pred_raw[valid_index] = prediction_01
        y_01_pred_std[valid_index] = prediction_std_01
        y_01_pred_lower[valid_index] = prediction_lower_01
        y_01_pred_upper[valid_index] = prediction_upper_01
    gdf_01_pred['y_pred_raw'] = y_01_pred_raw
    gdf_01_pred['y_pred_raw_std'] = y_01_pred_std
    gdf_01_pred['y_pred_raw_lower'] = y_01_pred_lower
    gdf_01_pred['y_pred_raw_upper'] = y_01_pred_upper
    crs_025 = getattr(gdf025, 'crs', None)
    crs_01 = getattr(gdf01, 'crs', None)
    wkb_025 = gpd.GeoSeries(gdf_025_pred['geometry_0.25'], crs=crs_025).to_wkb()
    codes_025, uniques_025 = pd.factorize(wkb_025)
    gdf_025_pred['cell_025'] = codes_025
    wkb_01 = gpd.GeoSeries(gdf_01_pred['geometry_01'], crs=crs_01).to_wkb()
    codes_01, uniques_01 = pd.factorize(wkb_01)
    gdf_01_pred['cell_01'] = codes_01
    cells_025 = gpd.GeoDataFrame({'cell_025': np.arange(len(uniques_025), dtype=np.int64)}, geometry=gpd.GeoSeries.from_wkb(uniques_025, crs=crs_025), crs=crs_025)
    cells_01 = gpd.GeoDataFrame({'cell_01': np.arange(len(uniques_01), dtype=np.int64)}, geometry=gpd.GeoSeries.from_wkb(uniques_01, crs=crs_01), crs=crs_01)
    if cells_025.crs is not None and (not cells_025.crs.is_projected):
        try:
            proj_crs = cells_025.estimate_utm_crs()
        except Exception:
            proj_crs = None
        if proj_crs is None:
            proj_crs = 'EPSG:6933'
        cells_025_proj = cells_025.to_crs(proj_crs)
        cells_01_proj = cells_01.to_crs(proj_crs)
    else:
        cells_025_proj = cells_025
        cells_01_proj = cells_01
    cells_025['area_025'] = cells_025_proj.geometry.area.to_numpy()
    cells_01['area_01'] = cells_01_proj.geometry.area.to_numpy()
    area_025_map = cells_025.set_index('cell_025')['area_025']
    area_01_map = cells_01.set_index('cell_01')['area_01']
    gdf_025_pred['area_025'] = gdf_025_pred['cell_025'].map(area_025_map)
    gdf_01_pred['area_01'] = gdf_01_pred['cell_01'].map(area_01_map)
    cells_01_pts = cells_01_proj.copy()
    cells_01_pts.geometry = cells_01_pts.geometry.centroid
    cells_025_for_join = cells_025_proj[['cell_025', 'geometry']].copy()
    mapping = gpd.sjoin(cells_01_pts, cells_025_for_join, how='left', predicate='within')[['cell_01', 'cell_025']].rename(columns={'cell_025': 'parent_025'})
    gdf_01_pred = gdf_01_pred.merge(mapping, on='cell_01', how='left')
    if mass_blocks is not None and mass_blocks > 1:
        centroids = cells_025_proj.geometry.centroid
        coords = np.column_stack([centroids.x.values, centroids.y.values])
        km = KMeans(n_clusters=mass_blocks, n_init=10, random_state=0)
        labels = km.fit_predict(coords)
        cells_025['mass_block_id'] = labels
        block_map = cells_025.set_index('cell_025')['mass_block_id']
        gdf_025_pred['mass_block_id'] = gdf_025_pred['cell_025'].map(block_map)
        parent_block_map = gdf_025_pred[['cell_025', 'mass_block_id']].drop_duplicates('cell_025').set_index('cell_025')['mass_block_id']
        gdf_01_pred['mass_block_id'] = gdf_01_pred['parent_025'].map(parent_block_map)
    else:
        mass_blocks = None
    gdf_01_pred['y_pred_final_uniform'] = np.nan
    gdf_01_pred['y_pred_final_prop'] = np.nan
    gdf_01_pred['y_pred_final'] = np.nan
    all_dates = gdf_025_pred[date_col].unique()
    for d in all_dates:
        mask_p = gdf_025_pred[date_col] == d
        mask_c = gdf_01_pred[date_col] == d
        if not mask_p.any() or not mask_c.any():
            continue
        child_area = gdf_01_pred.loc[mask_c, ['parent_025', 'area_01']].dropna(subset=['parent_025', 'area_01']).groupby('parent_025', observed=True)['area_01'].sum()
        if mass_blocks is None:
            parent_d = gdf_025_pred.loc[mask_p, ['cell_025', 'y_true']].copy()
            parent_d['area_eff'] = parent_d['cell_025'].map(child_area).fillna(0.0)
            parent_d['mass_025'] = parent_d['y_true'] * parent_d['area_eff']
            parent_d = parent_d.set_index('cell_025')
            child_d = gdf_01_pred.loc[mask_c, ['row_01', 'parent_025', 'area_01', 'y_pred_raw']].copy()
            child_d = child_d.rename(columns={'parent_025': 'group_id'})
            group_key = 'group_id'
        else:
            parent_cells = gdf_025_pred.loc[mask_p, ['cell_025', 'mass_block_id', 'y_true']].copy()
            parent_cells['area_eff'] = parent_cells['cell_025'].map(child_area).fillna(0.0)
            parent_cells['mass_eff'] = parent_cells['y_true'] * parent_cells['area_eff']
            parent_d = parent_cells.groupby('mass_block_id', observed=True).agg(area_025=('area_eff', 'sum'), mass_025=('mass_eff', 'sum'))
            child_d = gdf_01_pred.loc[mask_c, ['row_01', 'mass_block_id', 'area_01', 'y_pred_raw']].copy()
            child_d = child_d.rename(columns={'mass_block_id': 'group_id'})
            group_key = 'group_id'
        child_d = child_d[child_d[group_key].notna() & child_d['y_pred_raw'].notna() & child_d['area_01'].notna()]
        if child_d.empty:
            continue
        child_d['raw_mass_child'] = child_d['y_pred_raw'] * child_d['area_01']
        grp = child_d.groupby(group_key, observed=True)
        sum_raw_mass = grp['raw_mass_child'].sum()
        sum_area_child = grp['area_01'].sum()
        parent_d['sum_raw_mass'] = sum_raw_mass
        parent_d['sum_area_child'] = sum_area_child
        parent_d[['sum_raw_mass', 'sum_area_child']] = parent_d[['sum_raw_mass', 'sum_area_child']].fillna(0.0)
        parent_d['target_mass'] = parent_d['mass_025']
        parent_d['delta_mass'] = parent_d['target_mass'] - parent_d['sum_raw_mass']
        parent_d['bias_uniform_group'] = 0.0
        nz = parent_d['sum_area_child'] > 0
        parent_d.loc[nz, 'bias_uniform_group'] = parent_d.loc[nz, 'delta_mass'] / parent_d.loc[nz, 'sum_area_child']
        bias_uniform_map_d = parent_d['bias_uniform_group'].to_dict()
        child_d['bias_uniform'] = child_d[group_key].map(bias_uniform_map_d)
        child_d['bias_uniform'] = child_d['bias_uniform'].fillna(0.0)
        child_d['y_pred_final_uniform'] = child_d['y_pred_raw'] + child_d['bias_uniform']
        if bias_method == 'proportional':
            if weight_col not in gdf_01_pred.columns:
                raise ValueError(f"bias_method='proportional' requires '{weight_col}' in gdf01.")
            child_d = child_d.merge(gdf_01_pred.loc[mask_c, ['row_01', weight_col]], on='row_01', how='left')
            w = child_d[weight_col].to_numpy()
            w_clean = np.where(np.isfinite(w) & (w > 0), w, 0.0)
            child_d['w_clean'] = w_clean
            child_d['w_area'] = child_d['w_clean'] * child_d['area_01']
            sum_w_area = child_d.groupby(group_key, observed=True)['w_area'].sum()
            parent_d['sum_w_area'] = sum_w_area.fillna(0.0)
            parent_d['factor_prop'] = np.where(parent_d['sum_w_area'] > 0, parent_d['delta_mass'] / parent_d['sum_w_area'], np.nan)
            factor_prop_map_d = parent_d['factor_prop'].to_dict()
            child_d['factor_prop'] = child_d[group_key].map(factor_prop_map_d)
            child_d['bias_prop'] = child_d['factor_prop'] * child_d['w_clean']
            child_d['bias_prop'] = child_d['bias_prop'].where(np.isfinite(child_d['bias_prop']), child_d['bias_uniform'])
            child_d['y_pred_final_prop'] = child_d['y_pred_raw'] + child_d['bias_prop']
        idx_rows = child_d['row_01'].values
        gdf_01_pred.loc[idx_rows, 'y_pred_final_uniform'] = child_d['y_pred_final_uniform'].values
        if bias_method == 'proportional':
            gdf_01_pred.loc[idx_rows, 'y_pred_final_prop'] = child_d['y_pred_final_prop'].values
    if bias_method == 'uniform':
        gdf_01_pred['y_pred_final'] = gdf_01_pred['y_pred_final_uniform']
    elif bias_method == 'proportional':
        gdf_01_pred['y_pred_final'] = gdf_01_pred['y_pred_final_prop']
    else:
        raise ValueError("bias_method must be 'uniform' or 'proportional'")
    anchor_std = pd.to_numeric(gdf_01_pred.get('grace_or_imputed_std', pd.Series(np.nan, index=gdf_01_pred.index)), errors='coerce').fillna(0.0).to_numpy(dtype=float)
    raw_std = pd.to_numeric(gdf_01_pred['y_pred_raw_std'], errors='coerce').to_numpy(dtype=float)
    gdf_01_pred['y_pred_final_std'] = np.sqrt(np.square(raw_std) + np.square(anchor_std))
    gdf_01_pred['y_pred_final_lower'] = gdf_01_pred['y_pred_final'] - UNCERTAINTY_Z * gdf_01_pred['y_pred_final_std']
    gdf_01_pred['y_pred_final_upper'] = gdf_01_pred['y_pred_final'] + UNCERTAINTY_Z * gdf_01_pred['y_pred_final_std']
    metrics = []

    def append_metric_set(scale, step, y_true_values, y_pred_values, std_values=None):
        uncertainty = bootstrap_regression_metrics(y_true_values, y_pred_values, random_state=RANDOM_STATE)
        for metric_name in ('r2', 'mae', 'rmse'):
            metrics.append({'scale': scale, 'step': step, 'metric': metric_name, 'value': uncertainty[metric_name]['estimate'], 'ci_lower': uncertainty[metric_name]['ci_lower'], 'ci_upper': uncertainty[metric_name]['ci_upper']})
        if std_values is not None:
            std_values = np.asarray(std_values, dtype=float)
            metrics.append({'scale': scale, 'step': step, 'metric': 'mean_prediction_std', 'value': float(np.nanmean(std_values)), 'ci_lower': np.nan, 'ci_upper': np.nan})
            metrics.append({'scale': scale, 'step': step, 'metric': 'mean_95_interval_width', 'value': float(np.nanmean(2 * UNCERTAINTY_Z * std_values)), 'ci_lower': np.nan, 'ci_upper': np.nan})
    mask_eval_025 = np.isfinite(y_025_true) & np.isfinite(y_025_pred_raw)
    if mask_eval_025.sum() >= 2:
        append_metric_set('0.25', 'raw_pred', y_025_true[mask_eval_025], y_025_pred_raw[mask_eval_025], y_025_pred_std[mask_eval_025])
    if y_01_true is not None:
        mask_truth_01 = gdf_01_pred[target_col].notna()
        mask_eval_raw_01 = mask_truth_01 & gdf_01_pred['y_pred_raw'].notna()
        if mask_eval_raw_01.sum() >= 2:
            append_metric_set('0.1', 'raw_pred', gdf_01_pred.loc[mask_eval_raw_01, target_col].to_numpy(), gdf_01_pred.loc[mask_eval_raw_01, 'y_pred_raw'].to_numpy(), gdf_01_pred.loc[mask_eval_raw_01, 'y_pred_raw_std'].to_numpy())
        mask_eval_final_01 = mask_truth_01 & gdf_01_pred['y_pred_final'].notna()
        if mask_eval_final_01.sum() >= 2:
            append_metric_set('0.1', f'final_{bias_method}', gdf_01_pred.loc[mask_eval_final_01, target_col].to_numpy(), gdf_01_pred.loc[mask_eval_final_01, 'y_pred_final'].to_numpy(), gdf_01_pred.loc[mask_eval_final_01, 'y_pred_final_std'].to_numpy())
            coverage = gdf_01_pred.loc[mask_eval_final_01, target_col].between(gdf_01_pred.loc[mask_eval_final_01, 'y_pred_final_lower'], gdf_01_pred.loc[mask_eval_final_01, 'y_pred_final_upper']).mean()
            metrics.append({'scale': '0.1', 'step': f'final_{bias_method}', 'metric': 'empirical_95_interval_coverage', 'value': float(coverage), 'ci_lower': np.nan, 'ci_upper': np.nan})
    metrics_df = pd.DataFrame(metrics)
    return (gdf_025_pred, gdf_01_pred, metrics_df)


In [ ]:
gdf_025_pred, gdf_01_pred, metrics_df = downscale_grace(
    gdf025=gdf025,
    gdf01=gdf01,
    model_pipeline=downscaling_model,
    num_features=num_features,
    cat_features=cat_features,
    target_col=TARGET_COL,
    date_col="date",
    bias_method=BIAS_METHOD,
    weight_col=BIAS_WEIGHT_COL,
    mass_blocks=MASS_BLOCKS,
)
print("Downscaling and mass correction complete.")
memory_report("after downscaling")

# Preserve the complete 0.1° predictor table required by the
# mass-weight sensitivity analysis before the lagged source table is released.
_publication_feature_01_columns = [
    'date',
    'cell_01_key',
    *feature_cols,
]
publication_feature_01 = (
    gdf01[
        [
            column
            for column in _publication_feature_01_columns
            if column in gdf01.columns
        ]
    ]
    .drop_duplicates(['date', 'cell_01_key'])
    .copy()
)

missing_publication_feature_01 = [
    column
    for column in feature_cols
    if column not in publication_feature_01.columns
]
if missing_publication_feature_01:
    raise KeyError(
        'The preserved 0.1° feature table is incomplete: '
        f'{missing_publication_feature_01}'
    )

# Keep only the columns required by manuscript Figures 2 and 10.
_publication_cols = [
    'date',
    'cell_025_key',
    'geometry_0.25',
    'ET',
    'P',
    'Q',
    'temp',
    'SM',
    TARGET_COL,
    'grace_or_imputed_std',
    'grace_or_imputed_lower',
    'grace_or_imputed_upper',
]
publication_feature_025 = gdf025[[
    column for column in _publication_cols if column in gdf025.columns
]].copy()
downscaling_crs = gdf025.crs

# Predictions are materialized; release the lagged source tables.
del gdf01, gdf025, _publication_cols, _publication_feature_01_columns
gc.collect()
memory_report("after releasing downscaling feature tables")


### 4.7 Downscaling and uncertainty outputs

The tables summarize predictive performance, ensemble dispersion, interval coverage, and block-scale mass closure.

In [ ]:
downscaling_metrics_table = metrics_df.copy()


def compute_mass_balance_diagnostics(parent_df, child_df):
    child = child_df.loc[
        child_df['parent_025'].notna()
        & child_df['area_01'].notna()
    ].copy()
    effective_area = (
        child
        .groupby(['date', 'parent_025'], observed=True)['area_01']
        .sum()
        .rename('effective_area')
        .reset_index()
    )
    parent = parent_df.merge(
        effective_area,
        left_on=['date', 'cell_025'],
        right_on=['date', 'parent_025'],
        how='left',
    )
    parent['target_mass'] = (
        parent['y_true']
        * parent['effective_area'].fillna(0.0)
    )
    parent_mass = parent.groupby(
        ['date', 'mass_block_id'],
        observed=True,
    )['target_mass'].sum()

    child['corrected_mass'] = (
        child['y_pred_final'] * child['area_01']
    )
    child_mass = child.groupby(
        ['date', 'mass_block_id'],
        observed=True,
    )['corrected_mass'].sum()

    check = pd.concat(
        [parent_mass, child_mass],
        axis=1,
    ).dropna().reset_index()
    check['mass_error'] = (
        check['corrected_mass']
        - check['target_mass']
    )
    check['absolute_mass_error'] = check['mass_error'].abs()
    check['relative_error_percent'] = np.where(
        check['target_mass'].abs() > 0,
        100
        * check['absolute_mass_error']
        / check['target_mass'].abs(),
        np.nan,
    )

    return (
        check
        .groupby('date', as_index=False)
        .agg(
            groups_checked=('mass_block_id', 'size'),
            mean_absolute_mass_error=('absolute_mass_error', 'mean'),
            maximum_absolute_mass_error=('absolute_mass_error', 'max'),
            mean_relative_error_percent=('relative_error_percent', 'mean'),
            maximum_relative_error_percent=('relative_error_percent', 'max'),
        )
    )


mass_balance_by_date_table = compute_mass_balance_diagnostics(
    gdf_025_pred,
    gdf_01_pred,
)
mass_balance_summary_table = pd.DataFrame([{
    'dates_checked': int(len(mass_balance_by_date_table)),
    'groups_checked': int(
        mass_balance_by_date_table['groups_checked'].sum()
    ),
    'mean_absolute_mass_error': (
        mass_balance_by_date_table[
            'mean_absolute_mass_error'
        ].mean()
    ),
    'maximum_absolute_mass_error': (
        mass_balance_by_date_table[
            'maximum_absolute_mass_error'
        ].max()
    ),
    'mean_relative_error_percent': (
        mass_balance_by_date_table[
            'mean_relative_error_percent'
        ].mean()
    ),
    'maximum_relative_error_percent': (
        mass_balance_by_date_table[
            'maximum_relative_error_percent'
        ].max()
    ),
}])

final_prediction_summary_table = pd.DataFrame([
    {
        'resolution': '0.25°',
        'rows': len(gdf_025_pred),
        'dates': gdf_025_pred['date'].nunique(),
        'spatial_cells': gdf_025_pred['cell_025'].nunique(),
        'raw_predictions': gdf_025_pred['y_pred_raw'].notna().sum(),
        'final_predictions': gdf_025_pred['y_pred_raw'].notna().sum(),
        'mean_prediction_std_cm': gdf_025_pred['y_pred_raw_std'].mean(),
        'mean_95_interval_width_cm': (
            gdf_025_pred['y_pred_raw_upper']
            - gdf_025_pred['y_pred_raw_lower']
        ).mean(),
    },
    {
        'resolution': '0.1°',
        'rows': len(gdf_01_pred),
        'dates': gdf_01_pred['date'].nunique(),
        'spatial_cells': gdf_01_pred['cell_01'].nunique(),
        'raw_predictions': gdf_01_pred['y_pred_raw'].notna().sum(),
        'final_predictions': gdf_01_pred['y_pred_final'].notna().sum(),
        'mean_prediction_std_cm': gdf_01_pred['y_pred_final_std'].mean(),
        'mean_95_interval_width_cm': (
            gdf_01_pred['y_pred_final_upper']
            - gdf_01_pred['y_pred_final_lower']
        ).mean(),
    },
])

uncertainty_scope_table = pd.DataFrame([
    {
        'stage': 'GRACE imputation',
        'uncertainty_represented': (
            'Random Forest tree dispersion and bootstrap metric confidence intervals'
        ),
        'not_represented': (
            'GRACE measurement uncertainty and recursive error propagation'
        ),
    },
    {
        'stage': 'Pseudo-gap imputation experiments',
        'uncertainty_represented': (
            'Conditional tree dispersion and bootstrap confidence intervals '
            'for artificially withheld observed intervals'
        ),
        'not_represented': (
            'Full propagation of uncertainty through recursively predicted lags'
        ),
    },
    {
        'stage': 'Grouped CV parameter selection',
        'uncertainty_represented': (
            'Out-of-fold tree dispersion and bootstrap confidence intervals'
        ),
        'not_represented': (
            'Uncertainty associated with the finite candidate parameter grid'
        ),
    },
    {
        'stage': 'Repeated outer tests',
        'uncertainty_represented': (
            'Tree dispersion, bootstrap confidence intervals, and spatial-split sensitivity'
        ),
        'not_represented': (
            'Hydroclimatic forcing-data uncertainty'
        ),
    },
    {
        'stage': 'Final 0.1° product',
        'uncertainty_represented': (
            'RF dispersion combined with imputed-GRACE anchor uncertainty'
        ),
        'not_represented': (
            'Runoff-weight uncertainty and original GRACE measurement uncertainty'
        ),
    },
    {
        'stage': 'GRACE-SeDA and WGHM external comparison',
        'uncertainty_represented': (
            'Bootstrap confidence intervals, spatial standard errors, '
            'and uncertainty propagated from the present downscaled product'
        ),
        'not_represented': (
            'Product-specific uncertainty for GRACE-SeDA and WGHM, '
            'which is not provided in the supplied tables'
        ),
    },
])

display_if_requested(downscaling_metrics_table.round(4))
display_if_requested(mass_balance_summary_table)
display_if_requested(final_prediction_summary_table.round(4))
display_if_requested(uncertainty_scope_table)

_keep_025 = [
    'date',
    'cell_025_key',
    'cell_025',
    'mass_block_id',
    TARGET_COL,
    'grace_or_imputed_std',
    'grace_or_imputed_lower',
    'grace_or_imputed_upper',
    'y_true',
    'y_pred_raw',
    'y_pred_raw_std',
    'y_pred_raw_lower',
    'y_pred_raw_upper',
    'y_bias',
    'area_025',
    'geometry_0.25',
]
_keep_01 = [
    'date',
    'cell_01_key',
    'cell_01',
    'parent_025_key',
    'parent_025',
    'cell_025',
    'mass_block_id',
    'P',
    'ET',
    'Q',
    TARGET_COL,
    'grace_or_imputed_std',
    'grace_or_imputed_lower',
    'grace_or_imputed_upper',
    'area_01',
    'y_pred_raw',
    'y_pred_raw_std',
    'y_pred_raw_lower',
    'y_pred_raw_upper',
    'y_pred_final_uniform',
    'y_pred_final_prop',
    'y_pred_final',
    'y_pred_final_std',
    'y_pred_final_lower',
    'y_pred_final_upper',
    'geometry_01',
]
gdf_025_pred = gpd.GeoDataFrame(
    gdf_025_pred[
        [column for column in _keep_025 if column in gdf_025_pred.columns]
    ].copy(),
    geometry='geometry_0.25',
    crs=getattr(gdf_025_pred, 'crs', downscaling_crs),
)
gdf_01_pred = gpd.GeoDataFrame(
    gdf_01_pred[
        [column for column in _keep_01 if column in gdf_01_pred.columns]
    ].copy(),
    geometry='geometry_01',
    crs=getattr(gdf_01_pred, 'crs', downscaling_crs),
)
del _keep_025, _keep_01
gc.collect()
memory_report('after compacting prediction frames')


### 4.8 Validation summary

Chronological imputation, each pseudo-gap length, grouped cross-validation, strict repeated outer testing, and the complete temporal holdout are reported as separate estimates. Bootstrap uncertainty follows the sampling unit of each validation design.

In [ ]:
# Consolidated imputation and downscaling validation summary
def paired_bootstrap_metric_std(
    y_true,
    y_pred,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    random_state=RANDOM_STATE,
):
    """
    Estimate the standard deviation of R² and MAE by paired bootstrap.

    The same resampled indices are applied to observed and predicted values.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2:
        return {
            'r2_std': np.nan,
            'mae_std_cm': np.nan,
        }

    rng = np.random.default_rng(random_state)
    bootstrap_r2 = []
    bootstrap_mae = []

    for _ in range(n_bootstrap):
        sample_index = rng.integers(
            0,
            len(y_true),
            size=len(y_true),
        )
        observed_sample = y_true[sample_index]
        predicted_sample = y_pred[sample_index]

        if not np.allclose(
            observed_sample,
            observed_sample[0],
        ):
            bootstrap_r2.append(
                r2_score(
                    observed_sample,
                    predicted_sample,
                )
            )

        bootstrap_mae.append(
            mean_absolute_error(
                observed_sample,
                predicted_sample,
            )
        )

    return {
        'r2_std': (
            float(
                np.std(
                    bootstrap_r2,
                    ddof=1,
                )
            )
            if len(bootstrap_r2) > 1
            else np.nan
        ),
        'mae_std_cm': (
            float(
                np.std(
                    bootstrap_mae,
                    ddof=1,
                )
            )
            if len(bootstrap_mae) > 1
            else np.nan
        ),
    }


def format_estimate_sd(
    estimate,
    std_value,
    decimals=2,
    suffix='',
):
    """Format an estimate and bootstrap/replicate standard deviation."""
    if not np.isfinite(estimate):
        return 'NA'

    if np.isfinite(std_value):
        return (
            f'{estimate:.{decimals}f} ± '
            f'{std_value:.{decimals}f}{suffix}'
        )

    return f'{estimate:.{decimals}f}{suffix}'


def format_estimate_ci(
    estimate,
    lower,
    upper,
    decimals=2,
    suffix='',
):
    """Format an estimate with its 95% confidence interval."""
    if not np.isfinite(estimate):
        return 'NA'

    if np.isfinite(lower) and np.isfinite(upper):
        return (
            f'{estimate:.{decimals}f} '
            f'[{lower:.{decimals}f}–{upper:.{decimals}f}]'
            f'{suffix}'
        )

    return f'{estimate:.{decimals}f}{suffix}'


summary_rows = []

# Imputation: chronological 80/20 temporal holdout
temporal_imputation_std = paired_bootstrap_metric_std(
    imputation_test_predictions['y_true'],
    imputation_test_predictions['y_pred'],
    random_state=RANDOM_STATE + 7000,
)

summary_rows.append({
    'Step': 'Imputation',
    'Validation design': 'Chronological temporal holdout',
    'ML algorithm': 'RF',
    'R² estimate and uncertainty': format_estimate_sd(
        eval_metrics['R2'],
        temporal_imputation_std['r2_std'],
    ),
    'MAE estimate and uncertainty': format_estimate_sd(
        eval_metrics['MAE'],
        temporal_imputation_std['mae_std_cm'],
        suffix=' cm',
    ),
    'Train set size': (
        f"{eval_metrics['n_train_rows']:,} "
        f"({eval_metrics['n_train_cells']:,} unique cells)"
    ),
    'Test set size': (
        f"{eval_metrics['n_test_rows']:,} "
        f"({eval_metrics['n_test_cells']:,} unique cells)"
    ),
    'Separation strategy and interpretation': (
        'Last 20% of every cell time series; '
        f"test window {pd.Timestamp(eval_metrics['test_start_date']):%Y-%m} "
        f"to {pd.Timestamp(eval_metrics['test_end_date']):%Y-%m}. "
        'Uncertainty is reported as paired-bootstrap SD.'
    ),
})

# Imputation: strict rolling-origin recursive pseudo-gaps
for gap_length in sorted(PSEUDO_GAP_LENGTHS):
    gap_summary = (
        pseudo_gap_results_table.loc[
            pseudo_gap_results_table[
                'gap_length_months'
            ].eq(gap_length)
        ]
        .iloc[0]
    )

    mean_training_rows = float(
        gap_summary['mean_training_rows']
    )
    minimum_training_rows = int(
        gap_summary['minimum_training_rows']
    )
    maximum_training_rows = int(
        gap_summary['maximum_training_rows']
    )
    selected_windows = int(
        gap_summary['selected_windows']
    )
    unique_cells = int(
        gap_summary['unique_cells']
    )
    withheld_observations = int(
        gap_summary['withheld_observations']
    )

    summary_rows.append({
        'Step': 'Imputation',
        'Validation design': (
            f'{gap_length}-month recursive pseudo-gap'
        ),
        'ML algorithm': 'RF',
        'R² estimate and uncertainty': format_estimate_sd(
            gap_summary['R2'],
            gap_summary['R2_bootstrap_std'],
        ),
        'MAE estimate and uncertainty': format_estimate_sd(
            gap_summary['MAE_cm'],
            gap_summary['MAE_bootstrap_std_cm'],
            suffix=' cm',
        ),
        'Train set size': (
            f'{mean_training_rows:,.0f} mean rows '
            f'({minimum_training_rows:,}–'
            f'{maximum_training_rows:,})'
        ),
        'Test set size': (
            f'{withheld_observations:,} monthly predictions '
            f'from {selected_windows:,} distinct windows '
            f'({unique_cells:,} unique cells)'
        ),
        'Separation strategy and interpretation': (
            'Strict rolling-origin reconstruction using the same '
            'stratified cell-window starts for every gap length; '
            'one window per cell and only dates preceding the gap '
            'start used for training. Uncertainty is reported as '
            'complete-window cluster-bootstrap SD.'
        ),
    })

# Downscaling: three validation designs reported separately
selected_candidate_id = int(
    cv_parameter_results_df.loc[
        cv_parameter_results_df[
            'selection_rank'
        ].eq(1),
        'candidate_id',
    ].iloc[0]
)

selected_cv_fold_summary = (
    cv_parameter_fold_results_df.loc[
        cv_parameter_fold_results_df[
            'candidate_id'
        ].eq(selected_candidate_id),
        [
            'fold',
            'calibration_samples',
            'validation_samples',
            'r2',
            'mae_cm',
        ],
    ]
    .copy()
)

strict_outer_summary = (
    outer_test_results_df[
        [
            'outer_test',
            'training_samples',
            'holdout_samples',
            'holdout_r2',
            'holdout_mae_cm',
        ]
    ]
    .copy()
)

comparison_by_design = (
    outer_cv_comparison_table
    .set_index('validation_design')
)

grouped_cv_metrics = comparison_by_design.loc[
    'Grouped CV out-of-fold'
]
strict_outer_metrics = comparison_by_design.loc[
    'Repeated outer tests, pooled'
]
temporal_holdout_metrics = comparison_by_design.loc[
    'Complete temporal holdout'
]

summary_rows.append({
    'Step': 'Downscaling',
    'Validation design': 'Grouped-CV pooled out-of-fold',
    'ML algorithm': 'RF',
    'R² estimate and uncertainty': format_estimate_ci(
        grouped_cv_metrics['r2'],
        grouped_cv_metrics['r2_95_CI_lower'],
        grouped_cv_metrics['r2_95_CI_upper'],
    ),
    'MAE estimate and uncertainty': format_estimate_ci(
        grouped_cv_metrics['mae_cm'],
        grouped_cv_metrics['mae_95_CI_lower_cm'],
        grouped_cv_metrics['mae_95_CI_upper_cm'],
        suffix=' cm',
    ),
    'Train set size': (
        f"{selected_cv_fold_summary['calibration_samples'].mean():,.0f} "
        f"± {selected_cv_fold_summary['calibration_samples'].std(ddof=1):,.0f} "
        'rows per fold'
    ),
    'Test set size': (
        f'{len(selected_cv_predictions_df):,} pooled OOF predictions; '
        f"{selected_cv_fold_summary['validation_samples'].mean():,.0f} "
        f"± {selected_cv_fold_summary['validation_samples'].std(ddof=1):,.0f} "
        'rows per fold'
    ),
    'Separation strategy and interpretation': (
        'Five-fold GroupKFold on balanced pre-2022 '
        'year–spatial-block groups. This estimate supports '
        'hyperparameter selection and is not the principal '
        'generalization estimate. Uncertainty is a paired-bootstrap '
        '95% confidence interval.'
    ),
})

summary_rows.append({
    'Step': 'Downscaling',
    'Validation design': (
        'Repeated strict outer tests, pooled (principal)'
    ),
    'ML algorithm': 'RF',
    'R² estimate and uncertainty': format_estimate_ci(
        strict_outer_metrics['r2'],
        strict_outer_metrics['r2_95_CI_lower'],
        strict_outer_metrics['r2_95_CI_upper'],
    ),
    'MAE estimate and uncertainty': format_estimate_ci(
        strict_outer_metrics['mae_cm'],
        strict_outer_metrics['mae_95_CI_lower_cm'],
        strict_outer_metrics['mae_95_CI_upper_cm'],
        suffix=' cm',
    ),
    'Train set size': (
        f"{strict_outer_summary['training_samples'].mean():,.0f} "
        f"± {strict_outer_summary['training_samples'].std(ddof=1):,.0f} "
        'rows per outer model'
    ),
    'Test set size': (
        f'{len(pooled_outer_predictions_df):,} unique pooled predictions '
        f'across {len(strict_outer_summary)} tests; '
        f"{strict_outer_summary['holdout_samples'].mean():,.0f} "
        f"± {strict_outer_summary['holdout_samples'].std(ddof=1):,.0f} "
        'rows per test'
    ),
    'Separation strategy and interpretation': (
        'The complete 2022–2023 period and every combination of '
        'two validation blocks were withheld simultaneously. '
        'Predictions were pooled by sample across the ten strict '
        'outer tests. This is the principal estimate of '
        'spatiotemporal generalization. Uncertainty is a '
        'paired-bootstrap 95% confidence interval.'
    ),
})

summary_rows.append({
    'Step': 'Downscaling',
    'Validation design': (
        'Complete 2022–2023 temporal holdout of the final model'
    ),
    'ML algorithm': 'RF',
    'R² estimate and uncertainty': format_estimate_ci(
        temporal_holdout_metrics['r2'],
        temporal_holdout_metrics['r2_95_CI_lower'],
        temporal_holdout_metrics['r2_95_CI_upper'],
    ),
    'MAE estimate and uncertainty': format_estimate_ci(
        temporal_holdout_metrics['mae_cm'],
        temporal_holdout_metrics['mae_95_CI_lower_cm'],
        temporal_holdout_metrics['mae_95_CI_upper_cm'],
        suffix=' cm',
    ),
    'Train set size': (
        f'{len(training_df):,} pre-2022 rows'
    ),
    'Test set size': (
        f'{len(holdout_df):,} rows from 2022–2023'
    ),
    'Separation strategy and interpretation': (
        'The 600-tree final model was trained on all balanced '
        'pre-2022 samples and evaluated across all spatial blocks '
        'during 2022–2023. This estimate isolates temporal transfer '
        'of the final fitted model. Uncertainty is a paired-bootstrap '
        '95% confidence interval.'
    ),
})

model_validation_summary_table = pd.DataFrame(
    summary_rows,
    columns=[
        'Step',
        'Validation design',
        'ML algorithm',
        'R² estimate and uncertainty',
        'MAE estimate and uncertainty',
        'Train set size',
        'Test set size',
        'Separation strategy and interpretation',
    ],
)

display_if_requested(
    model_validation_summary_table.style.set_properties(
        **{
            'text-align': 'left',
            'white-space': 'normal',
        }
    )
)


## 5. Manuscript and supplementary analytical tables

This stage computes process-consistency, uncertainty, water-balance, and benchmark tables used by the publication outputs.

In [ ]:
# Independent validation: monthly storage change versus P - ET - R
def build_water_balance_change_data(
    frame,
    cell_column,
    geometry_column,
    value_column,
    value_std_column,
    product_label,
    crs,
):
    required_columns = [
        'date',
        cell_column,
        geometry_column,
        'P',
        'ET',
        'Q',
        value_column,
    ]
    missing = [
        column
        for column in required_columns
        if column not in frame.columns
    ]
    if missing:
        raise KeyError(
            f'Missing variables for {product_label} validation: {missing}'
        )

    data = frame[required_columns + [
        column
        for column in [value_std_column]
        if column in frame.columns
    ]].copy()
    data['date'] = pd.to_datetime(data['date'], errors='coerce')

    for column in ['P', 'ET', 'Q', value_column]:
        data[column] = pd.to_numeric(
            data[column],
            errors='coerce',
        )

    if value_std_column in data.columns:
        data['storage_std_cm'] = pd.to_numeric(
            data[value_std_column],
            errors='coerce',
        ).fillna(0.0)
    else:
        data['storage_std_cm'] = 0.0

    data = data.sort_values(
        [cell_column, 'date']
    ).reset_index(drop=True)

    data['water_balance_dTWS_cm'] = (
        data['P'] - data['ET'] - data['Q']
    )
    data['storage_dTWS_cm'] = (
        data
        .groupby(cell_column, observed=True)[value_column]
        .diff()
    )

    previous_std = (
        data['storage_std_cm']
        .groupby(data[cell_column], observed=True)
        .shift(1)
        .fillna(0.0)
    )
    data['storage_dTWS_std_cm'] = np.sqrt(
        np.square(data['storage_std_cm'])
        + np.square(previous_std)
    )

    data = data.dropna(
        subset=[
            'storage_dTWS_cm',
            'water_balance_dTWS_cm',
        ]
    ).copy()
    data['product'] = product_label

    domain_rows = []
    for date, group in data.groupby(
        'date',
        observed=True,
    ):
        n_cells = len(group)
        storage_mean = group['storage_dTWS_cm'].mean()
        balance_mean = group['water_balance_dTWS_cm'].mean()

        storage_model_se = (
            np.sqrt(
                np.nansum(
                    np.square(
                        group['storage_dTWS_std_cm']
                    )
                )
            )
            / n_cells
            if n_cells
            else np.nan
        )
        balance_spatial_se = (
            group['water_balance_dTWS_cm'].std(ddof=1)
            / np.sqrt(n_cells)
            if n_cells > 1
            else np.nan
        )

        domain_rows.append({
            'product': product_label,
            'date': date,
            'n_cells': n_cells,
            'storage_dTWS_cm': storage_mean,
            'storage_model_se_cm': storage_model_se,
            'storage_lower_cm': (
                storage_mean
                - UNCERTAINTY_Z * storage_model_se
            ),
            'storage_upper_cm': (
                storage_mean
                + UNCERTAINTY_Z * storage_model_se
            ),
            'water_balance_dTWS_cm': balance_mean,
            'water_balance_spatial_se_cm': balance_spatial_se,
            'water_balance_lower_cm': (
                balance_mean
                - UNCERTAINTY_Z * balance_spatial_se
            ),
            'water_balance_upper_cm': (
                balance_mean
                + UNCERTAINTY_Z * balance_spatial_se
            ),
        })

    domain_table = (
        pd.DataFrame(domain_rows)
        .sort_values('date')
        .reset_index(drop=True)
    )

    product_seed_offset = (
        sum(
            (character_index + 1) * ord(character)
            for character_index, character in enumerate(product_label)
        )
        % 10000
    )

    metric_uncertainty = bootstrap_regression_metrics(
        domain_table['storage_dTWS_cm'],
        domain_table['water_balance_dTWS_cm'],
        random_state=(
            RANDOM_STATE
            + 4000
            + product_seed_offset
        ),
    )
    pearson_r = (
        float(
            np.corrcoef(
                domain_table['storage_dTWS_cm'],
                domain_table['water_balance_dTWS_cm'],
            )[0, 1]
        )
        if len(domain_table) >= 2
        else np.nan
    )
    mean_bias = float(
        (
            domain_table['water_balance_dTWS_cm']
            - domain_table['storage_dTWS_cm']
        ).mean()
    )

    metrics = pd.DataFrame([{
        'product': product_label,
        'comparison': 'Domain-mean monthly ΔTWS versus P - ET - R',
        'months': len(domain_table),
        'pearson_r': pearson_r,
        'R2': metric_uncertainty['r2']['estimate'],
        'R2_95_CI_lower': metric_uncertainty['r2']['ci_lower'],
        'R2_95_CI_upper': metric_uncertainty['r2']['ci_upper'],
        'MAE_cm': metric_uncertainty['mae']['estimate'],
        'MAE_95_CI_lower_cm': metric_uncertainty['mae']['ci_lower'],
        'MAE_95_CI_upper_cm': metric_uncertainty['mae']['ci_upper'],
        'RMSE_cm': metric_uncertainty['rmse']['estimate'],
        'RMSE_95_CI_lower_cm': metric_uncertainty['rmse']['ci_lower'],
        'RMSE_95_CI_upper_cm': metric_uncertainty['rmse']['ci_upper'],
        'mean_bias_P_minus_ET_minus_R_minus_storage_cm': mean_bias,
    }])

    cell_rows = []
    for cell_id, group in data.groupby(
        cell_column,
        observed=True,
    ):
        if len(group) < 6:
            continue

        storage_values = group[
            'storage_dTWS_cm'
        ].to_numpy(dtype=float)
        balance_values = group[
            'water_balance_dTWS_cm'
        ].to_numpy(dtype=float)

        correlation = (
            float(
                np.corrcoef(
                    storage_values,
                    balance_values,
                )[0, 1]
            )
            if (
                np.std(storage_values) > 0
                and np.std(balance_values) > 0
            )
            else np.nan
        )

        cell_rows.append({
            cell_column: cell_id,
            'product': product_label,
            'n_months': len(group),
            'pearson_r': correlation,
            'mae_cm': mean_absolute_error(
                storage_values,
                balance_values,
            ),
            'bias_cm': float(
                np.mean(
                    balance_values - storage_values
                )
            ),
            'mean_storage_model_std_cm': group[
                'storage_dTWS_std_cm'
            ].mean(),
        })

    cell_metrics = pd.DataFrame(cell_rows)
    geometry_lookup = (
        frame[
            [cell_column, geometry_column]
        ]
        .drop_duplicates(cell_column)
    )
    cell_metrics_gdf = geometry_lookup.merge(
        cell_metrics,
        on=cell_column,
        how='left',
    )
    cell_metrics_gdf = gpd.GeoDataFrame(
        cell_metrics_gdf,
        geometry=geometry_column,
        crs=crs,
    )

    return data, domain_table, metrics, cell_metrics, cell_metrics_gdf


(
    coarse_water_balance_data,
    coarse_water_balance_domain_table,
    coarse_water_balance_metrics,
    coarse_water_balance_cell_metrics,
    coarse_water_balance_cell_metrics_gdf,
) = build_water_balance_change_data(
    publication_feature_025,
    cell_column='cell_025_key',
    geometry_column='geometry_0.25',
    value_column=TARGET_COL,
    value_std_column='grace_or_imputed_std',
    product_label='GRACE / imputed 0.25° TWS',
    crs=getattr(publication_feature_025, 'crs', None),
)

(
    downscaled_water_balance_data,
    downscaled_water_balance_domain_table,
    downscaled_water_balance_metrics,
    water_balance_cell_metrics,
    water_balance_cell_metrics_gdf,
) = build_water_balance_change_data(
    gdf_01_pred,
    cell_column='cell_01',
    geometry_column='geometry_01',
    value_column='y_pred_final',
    value_std_column='y_pred_final_std',
    product_label='Final downscaled 0.1° TWS',
    crs=getattr(gdf_01_pred, 'crs', None),
)

water_balance_validation_metrics = pd.concat(
    [
        coarse_water_balance_metrics,
        downscaled_water_balance_metrics,
    ],
    ignore_index=True,
)
water_balance_domain_table = pd.concat(
    [
        coarse_water_balance_domain_table,
        downscaled_water_balance_domain_table,
    ],
    ignore_index=True,
)

display_if_requested(water_balance_validation_metrics.round(4))
display_if_requested(
    water_balance_cell_metrics[
        [
            'n_months',
            'pearson_r',
            'mae_cm',
            'bias_cm',
            'mean_storage_model_std_cm',
        ]
    ]
    .describe()
    .round(3)
)


## 6. External benchmarks

The downscaled product is aligned with GRACE-SeDA, WGHM, and the monthly water balance on common dates and spatial support. Interpolation checks are retained as optional diagnostics.

In [ ]:
# External GRACE-SeDA and WGHM benchmark analysis
EXTERNAL_TWS_METADATA = {
    'GRACE-SeDA': {
        'interpolated_01_key': 'grace_seda_interpolated_01_sheet',
        'native_05_key': 'grace_seda_original_05_sheet',
        'native_resolution_deg': 0.5,
        'interpolated_resolution_deg': 0.1,
        'epoch_type': 'irregular GRACE/GRACE-FO observation epochs',
        'provided_period': '2002-04 to 2022-12',
        'original_reference_period': '2004.0–2009.999',
        'unit': 'cm EWH',
    },
    'WGHM': {
        'interpolated_01_key': 'wghm_interpolated_01_sheet',
        'native_05_key': 'wghm_original_05_sheet',
        'native_resolution_deg': 0.5,
        'interpolated_resolution_deg': 0.1,
        'epoch_type': 'regular monthly epochs',
        'provided_period': '2000-01 to 2019-12',
        'original_reference_period': '2000-01 to 2019-12 mean removed',
        'unit': 'cm EWH',
    },
}


def load_external_tws_long(
    csv_path,
    product,
    resolution_deg,
):
    """Read a supplied wide TWS sheet and return a compact long table."""
    wide = pd.read_csv(csv_path)

    required = ['Lat', 'Lon']
    missing = [
        column
        for column in required
        if column not in wide.columns
    ]
    if missing:
        raise KeyError(
            f'{product} {resolution_deg}° table is missing: {missing}'
        )

    date_columns = [
        column
        for column in wide.columns
        if str(column).startswith('Date_')
    ]
    if not date_columns:
        raise ValueError(
            f'No Date_* columns were found for {product} {resolution_deg}°.'
        )

    long = wide.melt(
        id_vars=[
            column
            for column in ['Nr', 'Lat', 'Lon']
            if column in wide.columns
        ],
        value_vars=date_columns,
        var_name='date_label',
        value_name='tws_cm',
    )

    date_token = (
        long['date_label']
        .astype(str)
        .str.replace('Date_', '', regex=False)
    )
    separator_count = date_token.str.count('_')

    long['epoch_date'] = pd.NaT
    monthly_mask = separator_count.eq(1)
    irregular_mask = separator_count.eq(2)

    long.loc[monthly_mask, 'epoch_date'] = pd.to_datetime(
        date_token.loc[monthly_mask],
        format='%Y_%m',
        errors='coerce',
    )
    long.loc[irregular_mask, 'epoch_date'] = pd.to_datetime(
        date_token.loc[irregular_mask],
        format='%Y_%m_%d',
        errors='coerce',
    )

    long['date'] = to_month_start(long['epoch_date'])

    # GRACE-SeDA can contain more than one observation epoch in the same
    # calendar month. The present product contains one value per month, so
    # select the observation closest to the canonical model-ready mid-month
    # date rather than averaging distinct GRACE epochs or treating them as
    # erroneous duplicates.
    month_days = long['date'].dt.days_in_month
    canonical_day = np.where(
        long['date'].dt.month.eq(2),
        14,
        np.where(
            month_days.eq(31),
            16,
            15,
        ),
    )
    long['canonical_mid_month_date'] = (
        long['date']
        + pd.to_timedelta(
            canonical_day - 1,
            unit='D',
        )
    )
    long['epoch_distance_days'] = (
        long['epoch_date']
        - long['canonical_mid_month_date']
    ).abs().dt.days
    long['lat'] = pd.to_numeric(
        long['Lat'],
        errors='coerce',
    ).round(5)
    long['lon'] = pd.to_numeric(
        long['Lon'],
        errors='coerce',
    ).round(5)
    long['tws_cm'] = pd.to_numeric(
        long['tws_cm'],
        errors='coerce',
    )
    long['product'] = product
    long['resolution_deg'] = float(resolution_deg)
    long['cell_key'] = (
        long['lon'].map(lambda value: f'{value:.5f}')
        + '_'
        + long['lat'].map(lambda value: f'{value:.5f}')
    )

    long = (
        long[
            [
                'product',
                'resolution_deg',
                'cell_key',
                'lat',
                'lon',
                'epoch_date',
                'date',
                'canonical_mid_month_date',
                'epoch_distance_days',
                'tws_cm',
            ]
        ]
        .dropna(
            subset=[
                'lat',
                'lon',
                'date',
                'tws_cm',
            ]
        )
        .sort_values(
            [
                'cell_key',
                'date',
                'epoch_distance_days',
                'epoch_date',
            ]
        )
        .reset_index(drop=True)
    )

    duplicate_mask = long.duplicated(
        [
            'cell_key',
            'date',
        ],
        keep=False,
    )

    if duplicate_mask.any():
        duplicate_months = (
            long.loc[
                duplicate_mask,
                [
                    'cell_key',
                    'date',
                ],
            ]
            .drop_duplicates()
            .shape[0]
        )

        long = (
            long
            .drop_duplicates(
                [
                    'cell_key',
                    'date',
                ],
                keep='first',
            )
            .reset_index(drop=True)
        )

        print_if_requested(
            f'{product} {resolution_deg}°: selected the epoch closest '
            f'to canonical mid-month for {duplicate_months:,} '
            'cell-month combinations containing multiple epochs.'
        )

    long = long.drop(
        columns=[
            'canonical_mid_month_date',
            'epoch_distance_days',
        ],
        errors='ignore',
    )

    return long


external_tables = {}
external_inventory_rows = []

for product, metadata in EXTERNAL_TWS_METADATA.items():
    for support_name, source_key, resolution in [
        (
            'interpolated 0.1°',
            metadata['interpolated_01_key'],
            metadata['interpolated_resolution_deg'],
        ),
        (
            'native 0.5°',
            metadata['native_05_key'],
            metadata['native_resolution_deg'],
        ),
    ]:
        table = load_external_tws_long(
            RAW_LOCAL[source_key],
            product=product,
            resolution_deg=resolution,
        )
        external_tables[(product, support_name)] = table

        external_inventory_rows.append({
            'product': product,
            'support': support_name,
            'source_key': source_key,
            'source_url': RAW_SOURCE_LINKS[source_key],
            'unit': metadata['unit'],
            'epoch_type': metadata['epoch_type'],
            'original_reference_period': (
                metadata['original_reference_period']
            ),
            'rows_long': len(table),
            'spatial_cells': table['cell_key'].nunique(),
            'epochs': table['date'].nunique(),
            'first_epoch': table['epoch_date'].min(),
            'last_epoch': table['epoch_date'].max(),
            'latitude_min': table['lat'].min(),
            'latitude_max': table['lat'].max(),
            'longitude_min': table['lon'].min(),
            'longitude_max': table['lon'].max(),
        })

external_tws_inventory = pd.DataFrame(
    external_inventory_rows
)

grace_seda_01 = external_tables[
    ('GRACE-SeDA', 'interpolated 0.1°')
]
grace_seda_05 = external_tables[
    ('GRACE-SeDA', 'native 0.5°')
]
wghm_01 = external_tables[
    ('WGHM', 'interpolated 0.1°')
]
wghm_05 = external_tables[
    ('WGHM', 'native 0.5°')
]

display_if_requested(external_tws_inventory)


def add_cell_centres_from_geometry(
    frame,
    geometry_column,
):
    """Derive stable cell-centre coordinates without geographic-CRS centroids."""
    frame = frame.copy()
    bounds = gpd.GeoSeries(
        frame[geometry_column],
        crs=getattr(frame, 'crs', None),
    ).bounds
    frame['lon'] = (
        (bounds['minx'] + bounds['maxx']) / 2.0
    ).round(5)
    frame['lat'] = (
        (bounds['miny'] + bounds['maxy']) / 2.0
    ).round(5)
    frame['cell_key'] = (
        frame['lon'].map(lambda value: f'{value:.5f}')
        + '_'
        + frame['lat'].map(lambda value: f'{value:.5f}')
    )
    return frame


our_01 = add_cell_centres_from_geometry(
    gdf_01_pred[
        [
            'date',
            'cell_01',
            'P',
            'ET',
            'Q',
            'area_01',
            'y_pred_final',
            'y_pred_final_std',
            'geometry_01',
        ]
    ].copy(),
    geometry_column='geometry_01',
)
our_01['date'] = to_month_start(our_01['date'])
our_01 = our_01.rename(columns={
    'y_pred_final': 'our_tws_cm',
    'y_pred_final_std': 'our_tws_std_cm',
})
our_01 = (
    our_01
    .dropna(
        subset=[
            'date',
            'cell_key',
            'our_tws_cm',
        ]
    )
    .drop_duplicates(
        [
            'date',
            'cell_key',
        ]
    )
    .reset_index(drop=True)
)

our_geometry_lookup = (
    our_01[
        [
            'cell_key',
            'cell_01',
            'geometry_01',
        ]
    ]
    .drop_duplicates('cell_key')
)


def comparison_bootstrap(
    frame,
    true_column,
    predicted_column,
    group_column=None,
    random_state=RANDOM_STATE,
):
    """
    Bootstrap correlation, R², MAE, and RMSE.

    When group_column is supplied, complete temporal groups are resampled so
    that all spatial cells from the same month remain together.
    """
    data = frame[
        [
            true_column,
            predicted_column,
        ]
        + (
            [group_column]
            if group_column is not None
            else []
        )
    ].dropna().reset_index(drop=True)

    y_true = data[true_column].to_numpy(dtype=float)
    y_pred = data[predicted_column].to_numpy(dtype=float)

    def compute_metrics(true_values, predicted_values):
        if len(true_values) < 2:
            return {
                'pearson_r': np.nan,
                'r2': np.nan,
                'mae_cm': np.nan,
                'rmse_cm': np.nan,
                'bias_cm': np.nan,
            }

        correlation = (
            float(
                np.corrcoef(
                    true_values,
                    predicted_values,
                )[0, 1]
            )
            if (
                np.std(true_values) > 0
                and np.std(predicted_values) > 0
            )
            else np.nan
        )

        return {
            'pearson_r': correlation,
            'r2': (
                float(r2_score(true_values, predicted_values))
                if np.std(true_values) > 0
                else np.nan
            ),
            'mae_cm': float(
                mean_absolute_error(
                    true_values,
                    predicted_values,
                )
            ),
            'rmse_cm': float(
                np.sqrt(
                    mean_squared_error(
                        true_values,
                        predicted_values,
                    )
                )
            ),
            'bias_cm': float(
                np.mean(
                    predicted_values - true_values
                )
            ),
        }

    point = compute_metrics(
        y_true,
        y_pred,
    )

    if len(data) < 2:
        return {
            metric: {
                'estimate': value,
                'ci_lower': np.nan,
                'ci_upper': np.nan,
            }
            for metric, value in point.items()
        }

    if group_column is None:
        groups = [
            np.asarray([index], dtype=int)
            for index in range(len(data))
        ]
    else:
        groups = [
            group.index.to_numpy(dtype=int)
            for _, group in data.groupby(
                group_column,
                observed=True,
                sort=True,
            )
        ]

    rng = np.random.default_rng(random_state)
    sampled_values = {
        metric: []
        for metric in point
    }

    for _ in range(BOOTSTRAP_ITERATIONS):
        selected_groups = rng.integers(
            0,
            len(groups),
            size=len(groups),
        )
        selected_indices = np.concatenate([
            groups[group_index]
            for group_index in selected_groups
        ])

        sample_metrics = compute_metrics(
            data.loc[
                selected_indices,
                true_column,
            ].to_numpy(dtype=float),
            data.loc[
                selected_indices,
                predicted_column,
            ].to_numpy(dtype=float),
        )

        for metric, value in sample_metrics.items():
            sampled_values[metric].append(value)

    result = {}
    for metric, estimate in point.items():
        samples = np.asarray(
            sampled_values[metric],
            dtype=float,
        )
        samples = samples[np.isfinite(samples)]

        result[metric] = {
            'estimate': estimate,
            'ci_lower': (
                float(np.quantile(samples, 0.025))
                if samples.size
                else np.nan
            ),
            'ci_upper': (
                float(np.quantile(samples, 0.975))
                if samples.size
                else np.nan
            ),
        }

    return result


def build_pairwise_external_comparison(
    our_table,
    external_table,
    external_product,
    support_label,
):
    """
    Match, re-centre, and compare an external product with the downscaled product.
    """
    matched = (
        our_table[
            [
                'date',
                'cell_key',
                'lat',
                'lon',
                'our_tws_cm',
                'our_tws_std_cm',
            ]
        ]
        .merge(
            external_table[
                [
                    'date',
                    'epoch_date',
                    'cell_key',
                    'tws_cm',
                ]
            ].rename(columns={
                'tws_cm': 'external_tws_cm',
            }),
            on=[
                'date',
                'cell_key',
            ],
            how='inner',
        )
        .dropna(
            subset=[
                'our_tws_cm',
                'external_tws_cm',
            ]
        )
        .sort_values(
            [
                'cell_key',
                'date',
            ]
        )
        .reset_index(drop=True)
    )

    if matched.empty:
        raise ValueError(
            f'No matched observations were found for {external_product}.'
        )

    matched['our_centered_cm'] = (
        matched['our_tws_cm']
        - matched.groupby(
            'cell_key',
            observed=True,
        )['our_tws_cm'].transform('mean')
    )
    matched['external_centered_cm'] = (
        matched['external_tws_cm']
        - matched.groupby(
            'cell_key',
            observed=True,
        )['external_tws_cm'].transform('mean')
    )
    matched['difference_our_minus_external_cm'] = (
        matched['our_centered_cm']
        - matched['external_centered_cm']
    )
    matched['external_product'] = external_product
    matched['support'] = support_label

    domain_rows = []
    for date, group in matched.groupby(
        'date',
        observed=True,
        sort=True,
    ):
        n_cells = len(group)
        our_mean = group['our_centered_cm'].mean()
        external_mean = group[
            'external_centered_cm'
        ].mean()

        our_model_se = (
            np.sqrt(
                np.nansum(
                    np.square(
                        group['our_tws_std_cm']
                    )
                )
            )
            / n_cells
            if n_cells
            else np.nan
        )
        external_spatial_se = (
            group['external_centered_cm'].std(ddof=1)
            / np.sqrt(n_cells)
            if n_cells > 1
            else np.nan
        )

        domain_rows.append({
            'external_product': external_product,
            'support': support_label,
            'date': date,
            'epoch_date': group['epoch_date'].iloc[0],
            'n_cells': n_cells,
            'our_centered_cm': our_mean,
            'our_model_se_cm': our_model_se,
            'our_lower_cm': (
                our_mean
                - UNCERTAINTY_Z * our_model_se
            ),
            'our_upper_cm': (
                our_mean
                + UNCERTAINTY_Z * our_model_se
            ),
            'external_centered_cm': external_mean,
            'external_spatial_se_cm': external_spatial_se,
            'external_lower_cm': (
                external_mean
                - UNCERTAINTY_Z * external_spatial_se
            ),
            'external_upper_cm': (
                external_mean
                + UNCERTAINTY_Z * external_spatial_se
            ),
        })

    domain_table = pd.DataFrame(
        domain_rows
    ).sort_values('date').reset_index(drop=True)

    domain_bootstrap = comparison_bootstrap(
        domain_table,
        true_column='external_centered_cm',
        predicted_column='our_centered_cm',
        random_state=(
            RANDOM_STATE
            + 5000
            + sum(ord(character) for character in external_product)
        ),
    )
    pooled_bootstrap = comparison_bootstrap(
        matched,
        true_column='external_centered_cm',
        predicted_column='our_centered_cm',
        group_column='date',
        random_state=(
            RANDOM_STATE
            + 6000
            + sum(ord(character) for character in external_product)
        ),
    )

    metric_rows = []
    for comparison_scale, bootstrap_result, sample_size in [
        (
            'domain monthly mean',
            domain_bootstrap,
            len(domain_table),
        ),
        (
            'pooled cell-month',
            pooled_bootstrap,
            len(matched),
        ),
    ]:
        row = {
            'external_product': external_product,
            'support': support_label,
            'comparison_scale': comparison_scale,
            'common_start': matched['date'].min(),
            'common_end': matched['date'].max(),
            'matched_months': matched['date'].nunique(),
            'matched_cells': matched['cell_key'].nunique(),
            'sample_size': sample_size,
            'anomaly_recentering': (
                'Per-cell mean removed over exact matched dates'
            ),
        }

        for metric, values in bootstrap_result.items():
            row[metric] = values['estimate']
            row[f'{metric}_95_CI_lower'] = values['ci_lower']
            row[f'{metric}_95_CI_upper'] = values['ci_upper']

        metric_rows.append(row)

    metrics = pd.DataFrame(metric_rows)

    cell_rows = []
    for cell_key, group in matched.groupby(
        'cell_key',
        observed=True,
    ):
        if len(group) < 6:
            continue

        external_values = group[
            'external_centered_cm'
        ].to_numpy(dtype=float)
        our_values = group[
            'our_centered_cm'
        ].to_numpy(dtype=float)
        residuals = our_values - external_values
        absolute_errors = np.abs(residuals)
        n_values = len(group)

        correlation = (
            float(
                np.corrcoef(
                    external_values,
                    our_values,
                )[0, 1]
            )
            if (
                np.std(external_values) > 0
                and np.std(our_values) > 0
            )
            else np.nan
        )

        if (
            n_values > 3
            and np.isfinite(correlation)
            and abs(correlation) < 1
        ):
            fisher_z = np.arctanh(correlation)
            fisher_se = 1.0 / np.sqrt(n_values - 3)
            correlation_lower = np.tanh(
                fisher_z - UNCERTAINTY_Z * fisher_se
            )
            correlation_upper = np.tanh(
                fisher_z + UNCERTAINTY_Z * fisher_se
            )
        else:
            correlation_lower = np.nan
            correlation_upper = np.nan

        mae = float(np.mean(absolute_errors))
        mae_se = (
            float(
                np.std(
                    absolute_errors,
                    ddof=1,
                )
                / np.sqrt(n_values)
            )
            if n_values > 1
            else np.nan
        )
        bias = float(np.mean(residuals))
        bias_se = (
            float(
                np.std(
                    residuals,
                    ddof=1,
                )
                / np.sqrt(n_values)
            )
            if n_values > 1
            else np.nan
        )

        cell_rows.append({
            'external_product': external_product,
            'support': support_label,
            'cell_key': cell_key,
            'lat': group['lat'].iloc[0],
            'lon': group['lon'].iloc[0],
            'n_months': n_values,
            'pearson_r': correlation,
            'pearson_r_95_CI_lower': correlation_lower,
            'pearson_r_95_CI_upper': correlation_upper,
            'mae_cm': mae,
            'mae_95_CI_lower_cm': max(
                0.0,
                mae - UNCERTAINTY_Z * mae_se,
            ),
            'mae_95_CI_upper_cm': (
                mae + UNCERTAINTY_Z * mae_se
            ),
            'rmse_cm': float(
                np.sqrt(
                    np.mean(
                        np.square(residuals)
                    )
                )
            ),
            'bias_our_minus_external_cm': bias,
            'bias_95_CI_lower_cm': (
                bias - UNCERTAINTY_Z * bias_se
            ),
            'bias_95_CI_upper_cm': (
                bias + UNCERTAINTY_Z * bias_se
            ),
            'mean_our_prediction_std_cm': group[
                'our_tws_std_cm'
            ].mean(),
        })

    cell_metrics = pd.DataFrame(cell_rows)
    cell_metrics_gdf = our_geometry_lookup.merge(
        cell_metrics,
        on='cell_key',
        how='inner',
    )
    cell_metrics_gdf = gpd.GeoDataFrame(
        cell_metrics_gdf,
        geometry='geometry_01',
        crs=getattr(gdf_01_pred, 'crs', None),
    )

    return (
        matched,
        domain_table,
        metrics,
        cell_metrics,
        cell_metrics_gdf,
    )


pairwise_outputs = {}
for product, external_table in [
    ('GRACE-SeDA', grace_seda_01),
    ('WGHM', wghm_01),
]:
    pairwise_outputs[product] = (
        build_pairwise_external_comparison(
            our_01,
            external_table,
            external_product=product,
            support_label='interpolated 0.1°',
        )
    )

external_pairwise_matches = pd.concat(
    [
        output[0]
        for output in pairwise_outputs.values()
    ],
    ignore_index=True,
)
external_pairwise_domain_table = pd.concat(
    [
        output[1]
        for output in pairwise_outputs.values()
    ],
    ignore_index=True,
)
external_pairwise_metrics = pd.concat(
    [
        output[2]
        for output in pairwise_outputs.values()
    ],
    ignore_index=True,
)
external_pairwise_cell_metrics = pd.concat(
    [
        output[3]
        for output in pairwise_outputs.values()
    ],
    ignore_index=True,
)
external_pairwise_cell_metrics_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            output[4]
            for output in pairwise_outputs.values()
        ],
        ignore_index=True,
    ),
    geometry='geometry_01',
    crs=getattr(gdf_01_pred, 'crs', None),
)

display_if_requested(external_pairwise_metrics.round(4))


def add_parent_05_coordinates(
    frame,
    lon_column='lon',
    lat_column='lat',
):
    frame = frame.copy()
    epsilon = 1e-12

    frame['parent_05_lon'] = (
        np.floor(
            (
                pd.to_numeric(
                    frame[lon_column],
                    errors='coerce',
                )
                + epsilon
            )
            / 0.5
        )
        * 0.5
        + 0.25
    ).round(5)
    frame['parent_05_lat'] = (
        np.floor(
            (
                pd.to_numeric(
                    frame[lat_column],
                    errors='coerce',
                )
                + epsilon
            )
            / 0.5
        )
        * 0.5
        + 0.25
    ).round(5)
    frame['parent_05_key'] = (
        frame['parent_05_lon']
        .map(lambda value: f'{value:.5f}')
        + '_'
        + frame['parent_05_lat']
        .map(lambda value: f'{value:.5f}')
    )
    return frame


def aggregate_our_to_native_05(our_table):
    source = add_parent_05_coordinates(
        our_table
    )
    rows = []

    for (
        date,
        parent_key,
        parent_lon,
        parent_lat,
    ), group in source.groupby(
        [
            'date',
            'parent_05_key',
            'parent_05_lon',
            'parent_05_lat',
        ],
        observed=True,
    ):
        weights = pd.to_numeric(
            group['area_01'],
            errors='coerce',
        ).to_numpy(dtype=float)
        values = group[
            'our_tws_cm'
        ].to_numpy(dtype=float)
        standard_deviations = group[
            'our_tws_std_cm'
        ].fillna(0.0).to_numpy(dtype=float)

        valid = (
            np.isfinite(weights)
            & np.isfinite(values)
            & (weights > 0)
        )

        if not valid.any():
            continue

        weights = weights[valid]
        values = values[valid]
        standard_deviations = standard_deviations[valid]
        normalized_weights = weights / weights.sum()

        rows.append({
            'date': date,
            'cell_key': parent_key,
            'lon': parent_lon,
            'lat': parent_lat,
            'our_tws_cm': float(
                np.sum(
                    normalized_weights * values
                )
            ),
            'our_tws_std_cm': float(
                np.sqrt(
                    np.sum(
                        np.square(
                            normalized_weights
                            * standard_deviations
                        )
                    )
                )
            ),
            'n_subcells': int(valid.sum()),
            'covered_area_km2': float(
                np.sum(weights)
            ),
        })

    return pd.DataFrame(rows)


our_05 = aggregate_our_to_native_05(
    our_01
)

native_pairwise_outputs = {}
for product, external_native in [
    ('GRACE-SeDA', grace_seda_05),
    ('WGHM', wghm_05),
]:
    native_pairwise_outputs[product] = (
        build_pairwise_external_comparison(
            our_05,
            external_native,
            external_product=product,
            support_label='native 0.5°',
        )
    )

external_native_pairwise_domain_table = pd.concat(
    [
        output[1]
        for output in native_pairwise_outputs.values()
    ],
    ignore_index=True,
)
external_native_pairwise_metrics = pd.concat(
    [
        output[2]
        for output in native_pairwise_outputs.values()
    ],
    ignore_index=True,
)


def build_interpolation_qc(
    interpolated_01,
    native_05,
    product,
):
    interpolated = add_parent_05_coordinates(
        interpolated_01
    )

    aggregated = (
        interpolated
        .groupby(
            [
                'date',
                'parent_05_key',
                'parent_05_lon',
                'parent_05_lat',
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            interpolated_01_mean_cm=(
                'tws_cm',
                'mean',
            ),
            n_interpolated_subcells=(
                'cell_key',
                'nunique',
            ),
        )
        .rename(columns={
            'parent_05_key': 'cell_key',
            'parent_05_lon': 'lon',
            'parent_05_lat': 'lat',
        })
    )

    native = native_05[
        [
            'date',
            'cell_key',
            'tws_cm',
        ]
    ].rename(columns={
        'tws_cm': 'native_05_cm',
    })

    matched = (
        aggregated
        .merge(
            native,
            on=[
                'date',
                'cell_key',
            ],
            how='inner',
        )
        .dropna(
            subset=[
                'interpolated_01_mean_cm',
                'native_05_cm',
            ]
        )
    )
    matched['product'] = product
    matched['interpolation_difference_cm'] = (
        matched['interpolated_01_mean_cm']
        - matched['native_05_cm']
    )

    domain_table = (
        matched
        .groupby(
            'date',
            as_index=False,
            observed=True,
        )
        .agg(
            native_05_cm=('native_05_cm', 'mean'),
            interpolated_01_mean_cm=(
                'interpolated_01_mean_cm',
                'mean',
            ),
            mean_subcells=(
                'n_interpolated_subcells',
                'mean',
            ),
        )
    )

    bootstrap = comparison_bootstrap(
        domain_table,
        true_column='native_05_cm',
        predicted_column='interpolated_01_mean_cm',
        random_state=(
            RANDOM_STATE
            + 7000
            + sum(ord(character) for character in product)
        ),
    )

    row = {
        'product': product,
        'comparison': (
            'Mean interpolated 0.1° values aggregated to 0.5° '
            'versus supplied native 0.5° values'
        ),
        'matched_months': matched['date'].nunique(),
        'matched_native_cells': matched['cell_key'].nunique(),
        'matched_cell_months': len(matched),
        'mean_number_of_01_subcells': (
            matched['n_interpolated_subcells'].mean()
        ),
    }
    for metric, values in bootstrap.items():
        row[metric] = values['estimate']
        row[f'{metric}_95_CI_lower'] = values['ci_lower']
        row[f'{metric}_95_CI_upper'] = values['ci_upper']

    return matched, domain_table, pd.DataFrame([row])


interpolation_qc_outputs = {}
for product, interpolated, native in [
    ('GRACE-SeDA', grace_seda_01, grace_seda_05),
    ('WGHM', wghm_01, wghm_05),
]:
    interpolation_qc_outputs[product] = build_interpolation_qc(
        interpolated,
        native,
        product,
    )

external_interpolation_qc_matches = pd.concat(
    [
        output[0]
        for output in interpolation_qc_outputs.values()
    ],
    ignore_index=True,
)
external_interpolation_qc_domain = pd.concat(
    [
        output[1].assign(product=product)
        for product, output in interpolation_qc_outputs.items()
    ],
    ignore_index=True,
)
external_interpolation_qc_metrics = pd.concat(
    [
        output[2]
        for output in interpolation_qc_outputs.values()
    ],
    ignore_index=True,
)

display_if_requested(external_native_pairwise_metrics.round(4))
display_if_requested(external_interpolation_qc_metrics.round(4))


def build_subgrid_variability(
    table,
    value_column,
    product,
):
    source = add_parent_05_coordinates(
        table
    )

    variability = (
        source
        .groupby(
            [
                'date',
                'parent_05_key',
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            subgrid_standard_deviation_cm=(
                value_column,
                'std',
            ),
            n_subcells=(
                'cell_key',
                'nunique',
            ),
        )
    )
    variability = variability.loc[
        variability['n_subcells'] >= 4
    ].copy()
    variability['product'] = product

    values = variability[
        'subgrid_standard_deviation_cm'
    ].dropna()

    summary = pd.DataFrame([{
        'product': product,
        'parent_months': len(values),
        'mean_subgrid_standard_deviation_cm': values.mean(),
        'standard_deviation_of_subgrid_sd_cm': values.std(ddof=1),
        'median_subgrid_standard_deviation_cm': values.median(),
        'p05_subgrid_standard_deviation_cm': values.quantile(0.05),
        'p95_subgrid_standard_deviation_cm': values.quantile(0.95),
    }])

    return variability, summary


subgrid_outputs = []
for table, value_column, product in [
    (our_01, 'our_tws_cm', 'Present downscaled product'),
    (grace_seda_01, 'tws_cm', 'GRACE-SeDA interpolated'),
    (wghm_01, 'tws_cm', 'WGHM interpolated'),
]:
    subgrid_outputs.append(
        build_subgrid_variability(
            table,
            value_column,
            product,
        )
    )

external_subgrid_variability = pd.concat(
    [
        output[0]
        for output in subgrid_outputs
    ],
    ignore_index=True,
)
external_subgrid_variability_summary = pd.concat(
    [
        output[1]
        for output in subgrid_outputs
    ],
    ignore_index=True,
)

display_if_requested(
    external_subgrid_variability_summary.round(4)
)


def build_common_process_consistency(
    our_table,
    seda_table,
    wghm_table,
):
    common = (
        our_table[
            [
                'date',
                'cell_key',
                'P',
                'ET',
                'Q',
                'our_tws_cm',
                'our_tws_std_cm',
            ]
        ]
        .merge(
            seda_table[
                [
                    'date',
                    'cell_key',
                    'tws_cm',
                ]
            ].rename(columns={
                'tws_cm': 'seda_tws_cm',
            }),
            on=[
                'date',
                'cell_key',
            ],
            how='inner',
        )
        .merge(
            wghm_table[
                [
                    'date',
                    'cell_key',
                    'tws_cm',
                ]
            ].rename(columns={
                'tws_cm': 'wghm_tws_cm',
            }),
            on=[
                'date',
                'cell_key',
            ],
            how='inner',
        )
    )

    common = common.loc[
        common['date'].between(
            pd.Timestamp('2013-01-01'),
            pd.Timestamp('2019-12-01'),
        )
    ].copy()
    common = common.sort_values(
        [
            'cell_key',
            'date',
        ]
    ).reset_index(drop=True)

    month_number = (
        common['date'].dt.year * 12
        + common['date'].dt.month
    )
    previous_month_number = (
        month_number
        .groupby(
            common['cell_key'],
            observed=True,
        )
        .shift(1)
    )
    common['month_gap'] = (
        month_number - previous_month_number
    )

    for column in [
        'our_tws_cm',
        'seda_tws_cm',
        'wghm_tws_cm',
    ]:
        common[f'd_{column}'] = (
            common
            .groupby(
                'cell_key',
                observed=True,
            )[column]
            .diff()
        )

    previous_our_std = (
        common['our_tws_std_cm']
        .groupby(
            common['cell_key'],
            observed=True,
        )
        .shift(1)
        .fillna(0.0)
    )
    common['d_our_tws_std_cm'] = np.sqrt(
        np.square(common['our_tws_std_cm'])
        + np.square(previous_our_std)
    )
    common['water_balance_dTWS_cm'] = (
        common['P']
        - common['ET']
        - common['Q']
    )

    common = common.loc[
        common['month_gap'].eq(1)
    ].dropna(
        subset=[
            'd_our_tws_cm',
            'd_seda_tws_cm',
            'd_wghm_tws_cm',
            'water_balance_dTWS_cm',
        ]
    ).copy()

    product_columns = {
        'Present downscaled product': (
            'd_our_tws_cm',
            'd_our_tws_std_cm',
        ),
        'GRACE-SeDA interpolated': (
            'd_seda_tws_cm',
            None,
        ),
        'WGHM interpolated': (
            'd_wghm_tws_cm',
            None,
        ),
    }

    domain_tables = []
    metric_rows = []

    for product, (
        storage_column,
        uncertainty_column,
    ) in product_columns.items():
        rows = []

        for date, group in common.groupby(
            'date',
            observed=True,
            sort=True,
        ):
            n_cells = len(group)
            storage_mean = group[storage_column].mean()
            balance_mean = group[
                'water_balance_dTWS_cm'
            ].mean()

            if uncertainty_column is not None:
                storage_se = (
                    np.sqrt(
                        np.nansum(
                            np.square(
                                group[uncertainty_column]
                            )
                        )
                    )
                    / n_cells
                )
                uncertainty_type = 'propagated model standard error'
            else:
                storage_se = (
                    group[storage_column].std(ddof=1)
                    / np.sqrt(n_cells)
                    if n_cells > 1
                    else np.nan
                )
                uncertainty_type = 'spatial standard error only'

            balance_se = (
                group['water_balance_dTWS_cm'].std(ddof=1)
                / np.sqrt(n_cells)
                if n_cells > 1
                else np.nan
            )

            rows.append({
                'product': product,
                'date': date,
                'n_cells': n_cells,
                'storage_dTWS_cm': storage_mean,
                'storage_standard_error_cm': storage_se,
                'storage_uncertainty_type': uncertainty_type,
                'water_balance_dTWS_cm': balance_mean,
                'water_balance_spatial_se_cm': balance_se,
            })

        domain = pd.DataFrame(rows)
        domain_tables.append(domain)

        bootstrap = comparison_bootstrap(
            domain,
            true_column='water_balance_dTWS_cm',
            predicted_column='storage_dTWS_cm',
            random_state=(
                RANDOM_STATE
                + 8000
                + sum(ord(character) for character in product)
            ),
        )

        metric_row = {
            'product': product,
            'comparison': (
                'Domain-mean monthly ΔTWS versus P - ET - R '
                'on common consecutive observation months'
            ),
            'common_start': common['date'].min(),
            'common_end': common['date'].max(),
            'months': len(domain),
            'cells': common['cell_key'].nunique(),
            'fair_comparison_subset': (
                'Same 2013–2019 cells and consecutive GRACE-SeDA months'
            ),
        }
        for metric, values in bootstrap.items():
            metric_row[metric] = values['estimate']
            metric_row[
                f'{metric}_95_CI_lower'
            ] = values['ci_lower']
            metric_row[
                f'{metric}_95_CI_upper'
            ] = values['ci_upper']

        metric_rows.append(metric_row)

    return (
        common,
        pd.concat(
            domain_tables,
            ignore_index=True,
        ),
        pd.DataFrame(metric_rows),
    )


(
    external_common_process_data,
    external_common_process_domain_table,
    external_common_process_metrics,
) = build_common_process_consistency(
    our_01,
    grace_seda_01,
    wghm_01,
)

display_if_requested(external_common_process_metrics.round(4))


# Direct dTWS comparison on exactly the same dates and 0.1° cells
def build_common_dTWS_domain_table(common):
    rows = []

    for date, group in common.groupby(
        'date',
        observed=True,
        sort=True,
    ):
        n_cells = len(group)

        present_mean = group[
            'd_our_tws_cm'
        ].mean()
        seda_mean = group[
            'd_seda_tws_cm'
        ].mean()
        wghm_mean = group[
            'd_wghm_tws_cm'
        ].mean()
        balance_mean = group[
            'water_balance_dTWS_cm'
        ].mean()

        present_se = (
            np.sqrt(
                np.nansum(
                    np.square(
                        group[
                            'd_our_tws_std_cm'
                        ]
                    )
                )
            )
            / n_cells
            if n_cells
            else np.nan
        )
        seda_se = (
            group[
                'd_seda_tws_cm'
            ].std(ddof=1)
            / np.sqrt(n_cells)
            if n_cells > 1
            else np.nan
        )
        wghm_se = (
            group[
                'd_wghm_tws_cm'
            ].std(ddof=1)
            / np.sqrt(n_cells)
            if n_cells > 1
            else np.nan
        )
        balance_se = (
            group[
                'water_balance_dTWS_cm'
            ].std(ddof=1)
            / np.sqrt(n_cells)
            if n_cells > 1
            else np.nan
        )

        rows.append({
            'date': date,
            'n_cells': n_cells,
            'present_dTWS_cm': present_mean,
            'present_standard_error_cm': present_se,
            'present_lower_cm': (
                present_mean
                - UNCERTAINTY_Z * present_se
            ),
            'present_upper_cm': (
                present_mean
                + UNCERTAINTY_Z * present_se
            ),
            'grace_seda_dTWS_cm': seda_mean,
            'grace_seda_spatial_se_cm': seda_se,
            'grace_seda_lower_cm': (
                seda_mean
                - UNCERTAINTY_Z * seda_se
            ),
            'grace_seda_upper_cm': (
                seda_mean
                + UNCERTAINTY_Z * seda_se
            ),
            'wghm_dTWS_cm': wghm_mean,
            'wghm_spatial_se_cm': wghm_se,
            'wghm_lower_cm': (
                wghm_mean
                - UNCERTAINTY_Z * wghm_se
            ),
            'wghm_upper_cm': (
                wghm_mean
                + UNCERTAINTY_Z * wghm_se
            ),
            'water_balance_dTWS_cm': balance_mean,
            'water_balance_spatial_se_cm': balance_se,
            'water_balance_lower_cm': (
                balance_mean
                - UNCERTAINTY_Z * balance_se
            ),
            'water_balance_upper_cm': (
                balance_mean
                + UNCERTAINTY_Z * balance_se
            ),
        })

    return pd.DataFrame(rows)


def build_common_dTWS_pairwise_metrics(
    common_cell_table,
    common_domain_table,
):
    series_columns = {
        'Present downscaled product': 'd_our_tws_cm',
        'GRACE-SeDA': 'd_seda_tws_cm',
        'WGHM': 'd_wghm_tws_cm',
        'P − ET − R': 'water_balance_dTWS_cm',
    }
    domain_columns = {
        'Present downscaled product': 'present_dTWS_cm',
        'GRACE-SeDA': 'grace_seda_dTWS_cm',
        'WGHM': 'wghm_dTWS_cm',
        'P − ET − R': 'water_balance_dTWS_cm',
    }

    pair_labels = [
        (
            'GRACE-SeDA',
            'Present downscaled product',
        ),
        (
            'WGHM',
            'Present downscaled product',
        ),
        (
            'P − ET − R',
            'Present downscaled product',
        ),
        (
            'WGHM',
            'GRACE-SeDA',
        ),
        (
            'P − ET − R',
            'GRACE-SeDA',
        ),
        (
            'P − ET − R',
            'WGHM',
        ),
    ]

    rows = []

    for reference, comparison in pair_labels:
        seed_offset = sum(
            ord(character)
            for character in (
                reference + comparison
            )
        )

        for scale, table, reference_column, comparison_column, group_column in [
            (
                'domain monthly mean',
                common_domain_table,
                domain_columns[reference],
                domain_columns[comparison],
                None,
            ),
            (
                'pooled cell-month',
                common_cell_table,
                series_columns[reference],
                series_columns[comparison],
                'date',
            ),
        ]:
            metrics = comparison_bootstrap(
                table,
                true_column=reference_column,
                predicted_column=comparison_column,
                group_column=group_column,
                random_state=(
                    RANDOM_STATE
                    + 9000
                    + seed_offset
                    + (
                        100
                        if scale == 'pooled cell-month'
                        else 0
                    )
                ),
            )

            row = {
                'reference': reference,
                'comparison_product': comparison,
                'comparison_label': (
                    f'{comparison} versus {reference}'
                ),
                'comparison_scale': scale,
                'common_start': common_cell_table[
                    'date'
                ].min(),
                'common_end': common_cell_table[
                    'date'
                ].max(),
                'months': common_cell_table[
                    'date'
                ].nunique(),
                'cells': common_cell_table[
                    'cell_key'
                ].nunique(),
                'sample_size': len(table),
                'fair_comparison_subset': (
                    'Same 2013–2019 0.1° cells and consecutive '
                    'GRACE-SeDA observation months'
                ),
            }

            for metric, values in metrics.items():
                row[metric] = values[
                    'estimate'
                ]
                row[
                    f'{metric}_95_CI_lower'
                ] = values[
                    'ci_lower'
                ]
                row[
                    f'{metric}_95_CI_upper'
                ] = values[
                    'ci_upper'
                ]

            rows.append(row)

    return pd.DataFrame(rows)


external_common_dTWS_domain_table = (
    build_common_dTWS_domain_table(
        external_common_process_data
    )
)
external_common_dTWS_pairwise_metrics = (
    build_common_dTWS_pairwise_metrics(
        external_common_process_data,
        external_common_dTWS_domain_table,
    )
)
external_downscaling_vs_benchmarks_table = (
    external_common_dTWS_pairwise_metrics.loc[
        external_common_dTWS_pairwise_metrics[
            'comparison_product'
        ].eq('Present downscaled product')
    ]
    .sort_values(
        [
            'comparison_scale',
            'reference',
        ]
    )
    .reset_index(drop=True)
)

display_if_requested(
    external_downscaling_vs_benchmarks_table.round(4)
)
display_if_requested(
    external_common_dTWS_pairwise_metrics.round(4)
)


external_comparison_interpretation = pd.DataFrame([
    {
        'analysis': 'Pairwise 0.1° agreement',
        'what_it_tests': (
            'Similarity of re-centred anomalies on the common target grid'
        ),
        'possible_improvement_indicator': (
            'Higher correlation and lower MAE/RMSE indicate closer agreement'
        ),
        'important_limitation': (
            'Agreement with another product is not independent proof of accuracy'
        ),
    },
    {
        'analysis': 'Native 0.5° agreement',
        'what_it_tests': (
            'Similarity after aggregating the downscaled product to the original support'
        ),
        'possible_improvement_indicator': (
            'Consistency across both 0.1° and 0.5° comparisons'
        ),
        'important_limitation': (
            'Basin-mask coverage may be incomplete inside edge 0.5° cells'
        ),
    },
    {
        'analysis': 'Interpolation quality control',
        'what_it_tests': (
            'Distortion introduced by linear interpolation from 0.5° to 0.1°'
        ),
        'possible_improvement_indicator': (
            'Quantifies how much of the external native signal is retained'
        ),
        'important_limitation': (
            'Interpolation cannot create independent fine-scale information'
        ),
    },
    {
        'analysis': 'Common water-balance consistency',
        'what_it_tests': (
            'Agreement of monthly storage changes with P - ET - R'
        ),
        'possible_improvement_indicator': (
            'Higher correlation and lower errors indicate stronger process consistency'
        ),
        'important_limitation': (
            'P, ET, and R are predictors in the present downscaling model'
        ),
    },
    {
        'analysis': 'Within-0.5° sub-grid variability',
        'what_it_tests': (
            'Amount of spatial detail represented at 0.1°'
        ),
        'possible_improvement_indicator': (
            'More resolved variability than linear interpolation may indicate information gain'
        ),
        'important_limitation': (
            'Greater spatial variance is not automatically more accurate'
        ),
    },
])

display_if_requested(external_comparison_interpretation)


# Prepare a common 2013–2019 example-map table.
external_three_product_common = (
    our_01[
        [
            'date',
            'cell_key',
            'geometry_01',
            'our_tws_cm',
            'our_tws_std_cm',
        ]
    ]
    .merge(
        grace_seda_01[
            [
                'date',
                'cell_key',
                'tws_cm',
            ]
        ].rename(columns={
            'tws_cm': 'seda_tws_cm',
        }),
        on=[
            'date',
            'cell_key',
        ],
        how='inner',
    )
    .merge(
        wghm_01[
            [
                'date',
                'cell_key',
                'tws_cm',
            ]
        ].rename(columns={
            'tws_cm': 'wghm_tws_cm',
        }),
        on=[
            'date',
            'cell_key',
        ],
        how='inner',
    )
)
external_three_product_common = (
    external_three_product_common.loc[
        external_three_product_common['date'].between(
            pd.Timestamp('2013-01-01'),
            pd.Timestamp('2019-12-01'),
        )
    ]
    .copy()
)

for column in [
    'our_tws_cm',
    'seda_tws_cm',
    'wghm_tws_cm',
]:
    external_three_product_common[
        f'{column}_centered'
    ] = (
        external_three_product_common[column]
        - external_three_product_common.groupby(
            'cell_key',
            observed=True,
        )[column].transform('mean')
    )

date_coverage = (
    external_three_product_common
    .groupby(
        'date',
        observed=True,
    )
    .size()
)
maximum_date_coverage = date_coverage.max()
candidate_dates = date_coverage.loc[
    date_coverage >= 0.90 * maximum_date_coverage
].index

target_example_date = pd.Timestamp('2017-06-01')
external_example_date = min(
    candidate_dates,
    key=lambda date: abs(date - target_example_date),
)

external_example_map_gdf = gpd.GeoDataFrame(
    external_three_product_common.loc[
        external_three_product_common['date']
        == external_example_date
    ].copy(),
    geometry='geometry_01',
    crs=getattr(gdf_01_pred, 'crs', None),
)

print_if_requested('External benchmark example date:', external_example_date.date())


## 7. Export analytical products and validation tables

In [ ]:
gdf_025_pred = gpd.GeoDataFrame(
    gdf_025_pred,
    geometry='geometry_0.25',
    crs=downscaling_crs,
)
gdf_01_pred = gpd.GeoDataFrame(
    gdf_01_pred,
    geometry='geometry_01',
    crs=getattr(gdf_01_pred, 'crs', downscaling_crs),
)

if EXPORT_GEOPARQUET:
    if EXPORT_FULL_FEATURE_OUTPUTS:
        export_025 = gdf_025_pred
        export_01 = gdf_01_pred
    else:
        cols_025 = [
            'date',
            'cell_025_key',
            'cell_025',
            'mass_block_id',
            TARGET_COL,
            'grace_or_imputed_std',
            'grace_or_imputed_lower',
            'grace_or_imputed_upper',
            'y_true',
            'y_pred_raw',
            'y_pred_raw_std',
            'y_pred_raw_lower',
            'y_pred_raw_upper',
            'y_bias',
            'geometry_0.25',
        ]
        cols_01 = [
            'date',
            'cell_01_key',
            'cell_01',
            'parent_025_key',
            'parent_025',
            'cell_025',
            'mass_block_id',
            'P',
            'ET',
            'Q',
            TARGET_COL,
            'grace_or_imputed_std',
            'grace_or_imputed_lower',
            'grace_or_imputed_upper',
            'area_01',
            'y_pred_raw',
            'y_pred_raw_std',
            'y_pred_raw_lower',
            'y_pred_raw_upper',
            'y_pred_final_uniform',
            'y_pred_final_prop',
            'y_pred_final',
            'y_pred_final_std',
            'y_pred_final_lower',
            'y_pred_final_upper',
            'geometry_01',
        ]

        export_025 = gdf_025_pred[
            [
                column
                for column in cols_025
                if column in gdf_025_pred.columns
            ]
        ].copy()
        export_01 = gdf_01_pred[
            [
                column
                for column in cols_01
                if column in gdf_01_pred.columns
            ]
        ].copy()

    export_025.to_parquet(
        OUTPUT_DIR / 'gdf_025_pred.geoparquet',
        index=False,
        compression='zstd',
    )
    export_01.to_parquet(
        OUTPUT_DIR / 'gdf_01_pred.geoparquet',
        index=False,
        compression='zstd',
    )

    water_balance_cell_metrics_gdf.to_parquet(
        OUTPUT_DIR / 'water_balance_cell_validation.geoparquet',
        index=False,
        compression='zstd',
    )
    external_pairwise_cell_metrics_gdf.to_parquet(
        OUTPUT_DIR / 'external_tws_cell_comparison.geoparquet',
        index=False,
        compression='zstd',
    )

    del export_025, export_01
    gc.collect()

result_tables = [
    (raw_data_inventory, 'raw_data_inventory.csv'),
    (feature_engineering_summary, 'feature_engineering_summary.csv'),
    (imputation_performance_table, 'imputation_performance.csv'),
    (imputation_uncertainty_table, 'imputation_uncertainty.csv'),
    (imputation_coverage_table, 'imputation_coverage.csv'),
    (imputation_by_year_table, 'imputation_by_year.csv'),
    (
        pseudo_gap_results_table,
        'imputation_pseudo_gap_summary.csv',
    ),
    (
        pseudo_gap_selection_diagnostics_df,
        'imputation_pseudo_gap_selection_balance.csv',
    ),
    (
        pseudo_gap_shared_windows_df,
        'imputation_pseudo_gap_shared_windows.csv',
    ),
    (
        pseudo_gap_position_table,
        'imputation_pseudo_gap_by_position.csv',
    ),
    (
        pseudo_gap_training_diagnostics_df,
        'imputation_pseudo_gap_training_diagnostics.csv',
    ),
    (
        imputation_validation_comparison_table,
        'imputation_validation_comparison.csv',
    ),
    (
        model_validation_summary_table,
        'model_validation_summary.csv',
    ),
    (
        selected_rf_parameters_table,
        'downscaling_selected_cv_parameters.csv',
    ),
    (
        cv_parameter_results_df,
        'downscaling_cv_parameter_candidates.csv',
    ),
    (
        cv_parameter_fold_results_df,
        'downscaling_cv_parameter_folds.csv',
    ),
    (
        outer_cv_comparison_table,
        'downscaling_outer_cv_comparison.csv',
    ),
    (
        outer_cv_fold_distribution_table,
        'downscaling_outer_cv_fold_distribution.csv',
    ),
    (downscaling_results_table, 'downscaling_final_temporal_holdout.csv'),
    (outer_test_results_df, 'downscaling_repeated_outer_tests.csv'),
    (outer_test_summary_table, 'downscaling_repeated_outer_summary.csv'),
    (downscaling_table_summary, 'downscaling_input_summary.csv'),
    (spatial_block_summary, 'spatial_block_summary.csv'),
    (cv_summary_table, 'spatiotemporal_cv_summary.csv'),
    (downscaling_metrics_table, 'downscaling_metrics_with_uncertainty.csv'),
    (mass_balance_by_date_table, 'mass_balance_by_date.csv'),
    (mass_balance_summary_table, 'mass_balance_summary.csv'),
    (final_prediction_summary_table, 'final_prediction_summary.csv'),
    (uncertainty_scope_table, 'uncertainty_scope.csv'),
    (
        water_balance_validation_metrics,
        'water_balance_validation_metrics.csv',
    ),
    (
        water_balance_domain_table,
        'water_balance_monthly_validation.csv',
    ),
    (
        coarse_water_balance_cell_metrics,
        'water_balance_coarse_cell_validation.csv',
    ),
    (
        water_balance_cell_metrics,
        'water_balance_downscaled_cell_validation.csv',
    ),
    (
        external_tws_inventory,
        'external_tws_inventory.csv',
    ),
    (
        external_pairwise_metrics,
        'external_tws_pairwise_metrics.csv',
    ),
    (
        external_pairwise_domain_table,
        'external_tws_pairwise_domain_timeseries.csv',
    ),
    (
        external_pairwise_cell_metrics,
        'external_tws_pairwise_cell_metrics.csv',
    ),
    (
        external_native_pairwise_metrics,
        'external_tws_native_05_metrics.csv',
    ),
    (
        external_native_pairwise_domain_table,
        'external_tws_native_05_domain_timeseries.csv',
    ),
    (
        external_interpolation_qc_metrics,
        'external_tws_interpolation_qc_metrics.csv',
    ),
    (
        external_interpolation_qc_domain,
        'external_tws_interpolation_qc_timeseries.csv',
    ),
    (
        external_subgrid_variability_summary,
        'external_tws_subgrid_variability_summary.csv',
    ),
    (
        external_common_process_metrics,
        'external_tws_common_process_metrics.csv',
    ),
    (
        external_common_process_domain_table,
        'external_tws_common_process_timeseries.csv',
    ),
    (
        external_common_dTWS_domain_table,
        'external_tws_common_dTWS_domain_timeseries.csv',
    ),
    (
        external_common_dTWS_pairwise_metrics,
        'external_tws_common_dTWS_pairwise_metrics.csv',
    ),
    (
        external_downscaling_vs_benchmarks_table,
        'downscaling_dTWS_vs_GRACE_SeDA_WGHM_water_balance.csv',
    ),
    (
        external_comparison_interpretation,
        'external_tws_interpretation_guide.csv',
    ),
]

for table, filename in result_tables:
    table.to_csv(
        OUTPUT_DIR / filename,
        index=True if table.index.name is not None else False,
    )

spatiotemporal_cv_df.to_csv(
    OUTPUT_DIR / 'spatiotemporal_cross_validation.csv',
    index=False,
)
outer_test_predictions_df.to_csv(
    OUTPUT_DIR / 'downscaling_repeated_outer_predictions.csv',
    index=False,
)
pooled_outer_predictions_df.to_csv(
    OUTPUT_DIR / 'downscaling_pooled_outer_predictions.csv',
    index=False,
)
imputation_test_predictions.to_csv(
    OUTPUT_DIR / 'imputation_holdout_predictions_with_uncertainty.csv',
    index=False,
)
pseudo_gap_windows_df.to_csv(
    OUTPUT_DIR / 'imputation_pseudo_gap_windows.csv',
    index=False,
)
pseudo_gap_predictions_df.to_csv(
    OUTPUT_DIR / 'imputation_pseudo_gap_predictions.csv',
    index=False,
)
selected_cv_predictions_df.to_csv(
    OUTPUT_DIR / 'downscaling_selected_cv_oof_predictions.csv',
    index=False,
)
final_temporal_predictions_df.to_csv(
    OUTPUT_DIR / 'downscaling_final_temporal_predictions.csv',
    index=False,
)

print('Final outputs written to:', OUTPUT_DIR)

# Release feature-assembly objects before figure generation.
for name in [
    "balanced_model_df",
    "agg_out",
    "dynamic_feature_025_gdf",
    "df_gr_imputed",
    "gdf025",
    "gdf01",
]:
    if name in globals():
        del globals()[name]

gc.collect()
memory_report("before publication figures")


# Manuscript figures and tables

Figure 1 is assembled separately. Figures 2–20 and Tables 1–4 are generated below in manuscript order.

## Figure 2 — Latitude-resolved hydroclimatic time series

In [ ]:
if RUN_PUBLICATION_FIGURES:
    # Figure 2: latitude-resolved hydroclimatic time series

    DATE_COL = "date"
    GEOM_COL = "geometry_0.25"
    LAT_BIN_DEG = None
    FONT_SIZE = 14

    figure_2_specs = [
        ("ET", "ET, cm"),
        ("P", "P, cm"),
        ("Q", "R, cm"),
        ("temp", "T, °C"),
        ("SM", "SM, cm"),
        ("grace_or_imputed", "GRACE TWS, cm"),
    ]

    required = [DATE_COL, GEOM_COL] + [column for column, _ in figure_2_specs]
    missing = [column for column in required if column not in publication_feature_025.columns]
    if missing:
        raise KeyError(f"Missing variables required for Figure 2: {missing}")

    figure_2_df = publication_feature_025[required].copy()
    figure_2_df[DATE_COL] = pd.to_datetime(figure_2_df[DATE_COL], errors="coerce")

    # Use polygon centroids to assign each 0.25° cell to a latitude band.
    geometry_series = gpd.GeoSeries(figure_2_df[GEOM_COL], crs=getattr(publication_feature_025, "crs", None))
    figure_2_df["latitude"] = geometry_series.centroid.y.to_numpy()

    if LAT_BIN_DEG is not None:
        figure_2_df["latitude_band"] = (
            np.round(figure_2_df["latitude"] / LAT_BIN_DEG) * LAT_BIN_DEG
        )
    else:
        figure_2_df["latitude_band"] = figure_2_df["latitude"].round(5)

    latitudes = np.sort(figure_2_df["latitude_band"].dropna().unique())
    if latitudes.size == 0:
        raise ValueError("No valid latitude bands were found for Figure 2.")

    lat_norm = Normalize(vmin=float(latitudes.min()), vmax=float(latitudes.max()))
    lat_cmap = plt.get_cmap("coolwarm_r")  # southern latitudes red, northern latitudes blue

    fig, axes = plt.subplots(6, 1, figsize=(12, 16), sharex=True)
    panel_labels = list("abcdef")

    for ax, (column, ylabel), panel_tag in zip(axes, figure_2_specs, panel_labels):
        latitude_series = (
            figure_2_df.groupby([DATE_COL, "latitude_band"], as_index=False)[column]
            .mean()
            .dropna(subset=[column])
        )

        for latitude, group in latitude_series.groupby("latitude_band", sort=True):
            group = group.sort_values(DATE_COL)
            ax.plot(
                group[DATE_COL],
                group[column],
                linewidth=0.9,
                alpha=0.75,
                color=lat_cmap(lat_norm(latitude)),
            )

        # Linear trend of the domain-mean series.
        domain_mean = (
            latitude_series.groupby(DATE_COL, as_index=False)[column]
            .mean()
            .dropna(subset=[column])
            .sort_values(DATE_COL)
        )
        if len(domain_mean) >= 2:
            x = domain_mean[DATE_COL].map(pd.Timestamp.toordinal).to_numpy(dtype=float)
            y = domain_mean[column].to_numpy(dtype=float)
            slope, intercept = np.polyfit(x, y, 1)
            ax.plot(
                domain_mean[DATE_COL],
                intercept + slope * x,
                linestyle="--",
                linewidth=1.4,
                color="black",
            )

        ax.set_ylabel(ylabel, fontsize=FONT_SIZE)
        ax.tick_params(axis="both", labelsize=FONT_SIZE - 1)
        ax.grid(True, linestyle="--", alpha=0.25)
        ax.text(
            0.005,
            0.95,
            f"{panel_tag})",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=FONT_SIZE,
            fontweight="bold",
        )

    axes[-1].set_xlabel("Date", fontsize=FONT_SIZE)

    axes[-1].set_xlabel('Date', fontsize=FONT_SIZE)

    # Horizontal latitude colorbar below all subplots.
    sm = ScalarMappable(
        norm=lat_norm,
        cmap=lat_cmap,
    )
    sm.set_array([])

    # Reserve additional space below the subplots for the colorbar.
    fig.subplots_adjust(
        left=0.10,
        right=0.98,
        top=0.99,
        bottom=0.10,
        hspace=0.08,
    )

    # [left, bottom, width, height] in figure coordinates.
    colorbar_axis = fig.add_axes([
        0.22,
        0.035,
        0.56,
        0.015,
    ])

    cbar = fig.colorbar(
        sm,
        cax=colorbar_axis,
        orientation='horizontal',
    )

    cbar.set_label(
        'Latitude',
        fontsize=FONT_SIZE,
        labelpad=5,
    )

    cbar.ax.tick_params(
        axis='x',
        labelsize=FONT_SIZE - 1,
    )

    fig.savefig(
        MANUSCRIPT_DIR / 'fig_2_latitude_resolved_timeseries.jpeg',
        dpi=EXPORT_DPI,
        bbox_inches='tight',
    )

    if DISPLAY_FIGURES:
        plt.show()

    plt.close(fig)


## Table 1 — Datasets

In [ ]:
# Table 1. Datasets used in the study
table_1 = pd.DataFrame([
    ['Total water storage / GRACE', 'Satellite observations', '0.25°', 'Global', 'CSR RL06.3'],
    ['Precipitation / E-OBS', 'Gridded observations', '0.1°', 'Europe', 'Copernicus Climate Change Service'],
    ['Land-surface temperature / ERA5-Land', 'Reanalysis', '0.1°', 'Global', 'Copernicus Climate Change Service'],
    ['Evapotranspiration / SSEBop', 'Satellite-model product', '0.01°', 'Global', 'USGS'],
    ['River runoff / monitoring databases', 'Ground-based observations', 'Gauge/catchment', 'Poland and Ukraine', 'IMGW and Ukrainian Hydrometeorological Institute'],
    ['Elevation / SRTM', 'Satellite-derived DEM', '~90 m', 'Global', 'USGS / NASA'],
    ['Land cover / CORINE', 'Mapped land cover', '1:100,000', 'Europe', 'European Environment Agency'],
    ['Lithology / geological maps', 'Mapped geology', '1:50,000', 'Poland, Ukraine and Belarus', 'National geological institutions'],
], columns=['Parameter / source', 'Data type', 'Native spatial resolution', 'Spatial coverage', 'Provider'])

display(table_1)
table_1.to_csv(MANUSCRIPT_DIR / 'table_1_datasets.csv', index=False)


## Table 2 — Hydroclimatic summary for 2013–2023

In [ ]:
# Table 2. Statistical summary for the common downscaling period
# Hydrological variables are reported in centimetres (cm)

stats = publication_feature_025.copy()
stats['date'] = pd.to_datetime(stats['date'], errors='coerce')

stats = stats.loc[
    stats['date'].between('2013-01-01', '2023-12-31')
].copy()

table_2_variables = {
    'Evapotranspiration (cm)': ('ET', 1.0),
    'Precipitation (cm)': ('P', 1.0),
    'River runoff (cm)': ('Q', 1.0),
    'Surface temperature (°C)': ('temp', 1.0),
    'Soil moisture (cm)': ('SM', 1.0),
    'GRACE TWS (cm)': (TARGET_COL, 1.0),
}

quantiles = [0.05, 0.25, 0.50, 0.75, 0.95]

rows = []

for label, (column, scale) in table_2_variables.items():
    if column not in stats.columns:
        raise KeyError(
            f'Column {column!r} required for Table 2 was not found.'
        )

    values = (
        pd.to_numeric(stats[column], errors='coerce')
        .dropna()
        .mul(scale)
    )

    if values.empty:
        print(f'Warning: no valid values found for {label}.')
        continue

    q = values.quantile(quantiles)

    rows.append({
        'Variable': label,
        'Count': values.count(),
        'Mean': values.mean(),
        'Std': values.std(ddof=1),
        'Min': values.min(),
        'P05': q.loc[0.05],
        'P25': q.loc[0.25],
        'P50': q.loc[0.50],
        'P75': q.loc[0.75],
        'P95': q.loc[0.95],
        'Max': values.max(),
    })

table_2 = pd.DataFrame(rows)

display(table_2.round(2))

table_2.to_csv(
    MANUSCRIPT_DIR / 'table_2_statistical_summary_cm.csv',
    index=False,
)


## Figure 3 — Imputation and downscaling design

In [ ]:
if RUN_PUBLICATION_FIGURES:
    def draw_flow_panel(axis, steps, label, title):
        axis.set_axis_off()
        y = np.linspace(0.90, 0.10, len(steps))
        for index, (position, text) in enumerate(zip(y, steps)):
            axis.text(
                0.5, position, text, transform=axis.transAxes,
                ha='center', va='center', fontsize=10.5,
                bbox={'boxstyle': 'round,pad=0.45', 'facecolor': 'white', 'edgecolor': 'black'},
            )
            if index < len(steps) - 1:
                axis.annotate(
                    '', xy=(0.5, y[index + 1] + 0.045), xytext=(0.5, position - 0.045),
                    xycoords=axis.transAxes, arrowprops={'arrowstyle': '->', 'linewidth': 1.2},
                )
        axis.set_title(title, fontsize=13, fontweight='bold')
        add_panel_label(axis, label, x=0.01, y=0.98)

    imputation_steps = [
        'Observed GRACE TWS (2002–2024)',
        'Canonical mid-month alignment',
        'GRACE lags 1–3 months + calendar month',
        'Chronological holdout and strict pseudo-gaps',
        'Recursive Random Forest reconstruction',
        'Completed observed–imputed GRACE record',
    ]
    downscaling_steps = [
        'Completed 0.25° GRACE target',
        '0.25° hydroclimatic and physiographic predictors',
        'Grouped CV + strict spatiotemporal outer tests',
        'Global Random Forest fitted at 0.25°',
        'Prediction on the 0.1° predictor grid',
        'Runoff-weighted block mass conservation',
        'Final 0.1° TWS product',
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 8), constrained_layout=True)
    draw_flow_panel(axes[0], imputation_steps, 'a)', 'GRACE gap filling')
    draw_flow_panel(axes[1], downscaling_steps, 'b)', 'GRACE spatial downscaling')
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_3_imputation_and_downscaling_workflow.jpeg')


## Figure 4 — Downscaling workflow

In [ ]:
if RUN_PUBLICATION_FIGURES:
    from matplotlib.patches import FancyArrowPatch

    def temporal_mean_maps(g025, g01, date_col='date', target_col=TARGET_COL):
        coarse = as_geodataframe(g025, 'geometry_0.25')
        fine = as_geodataframe(g01, 'geometry_01', coarse.crs)
        coarse[date_col] = to_month_start(coarse[date_col])
        fine[date_col] = to_month_start(fine[date_col])

        common = pd.DatetimeIndex(coarse[date_col].dropna().unique()).intersection(
            pd.DatetimeIndex(fine[date_col].dropna().unique())
        )
        coarse = coarse.loc[coarse[date_col].isin(common)]
        fine = fine.loc[fine[date_col].isin(common)]

        coarse_mean = coarse.groupby(['cell_025', 'mass_block_id'], as_index=False, observed=True).agg(
            mean_target=(target_col, 'mean'),
            mean_coarse=('y_pred_raw', 'mean'),
            geometry_025=('geometry_0.25', 'first'),
        )
        fine_mean = fine.groupby(['cell_01', 'parent_025', 'mass_block_id'], as_index=False, observed=True).agg(
            mean_raw=('y_pred_raw', 'mean'),
            mean_final=('y_pred_final', 'mean'),
            geometry_01=('geometry_01', 'first'),
        )
        return (
            as_geodataframe(coarse_mean, 'geometry_025', coarse.crs),
            as_geodataframe(fine_mean, 'geometry_01', fine.crs),
            common,
        )

    coarse_mean, fine_mean, figure_4_dates = temporal_mean_maps(gdf_025_pred, gdf_01_pred)
    coarse_grid = as_geodataframe(
        coarse_mean[['cell_025', 'mass_block_id', 'geometry_025']].drop_duplicates('cell_025'),
        'geometry_025', coarse_mean.crs,
    )
    fine_grid = as_geodataframe(
        fine_mean[['cell_01', 'mass_block_id', 'geometry_01']].drop_duplicates('cell_01'),
        'geometry_01', fine_mean.crs,
    )

    projected = coarse_grid.to_crs(coarse_grid.estimate_utm_crs() or 'EPSG:6933')
    blocks_projected = projected.dissolve('mass_block_id', as_index=False)
    blocks = blocks_projected.to_crs(coarse_grid.crs)
    centroids = gpd.GeoSeries(blocks_projected.geometry.centroid, crs=blocks_projected.crs).to_crs(coarse_grid.crs)

    values = np.concatenate([
        coarse_mean['mean_target'].dropna().to_numpy(),
        coarse_mean['mean_coarse'].dropna().to_numpy(),
        fine_mean['mean_raw'].dropna().to_numpy(),
        fine_mean['mean_final'].dropna().to_numpy(),
    ])
    norm = Normalize(*np.nanquantile(values, [0.02, 0.98]))

    panels = [
        (coarse_mean, 'mean_target', 'Observed or imputed GRACE TWS\nat 0.25° resolution', 'coarse'),
        (coarse_mean, 'mean_coarse', 'Random Forest representation\nat 0.25° resolution', 'coarse'),
        (fine_mean, 'mean_raw', 'Raw Random Forest prediction\nat 0.1° resolution', 'fine'),
        (fine_mean, 'mean_final', 'Mass-conserving TWS product\nat 0.1° resolution', 'fine'),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(18, 5.8))
    fig.subplots_adjust(left=0.02, right=0.92, bottom=0.14, top=0.82, wspace=0.14)

    for index, (axis, (frame, column, title, resolution)) in enumerate(zip(axes, panels)):
        frame.plot(column=column, ax=axis, cmap='viridis', norm=norm, edgecolor='none')
        (coarse_grid if resolution == 'coarse' else fine_grid).boundary.plot(
            ax=axis, color='0.45', linewidth=0.40 if resolution == 'coarse' else 0.16
        )
        if resolution == 'fine':
            blocks.boundary.plot(ax=axis, color='black', linewidth=0.85)
        if index == 3:
            for block_id, point in zip(blocks['mass_block_id'], centroids):
                axis.text(point.x, point.y, str(block_id), fontsize=8, ha='center', va='center',
                          bbox={'facecolor': 'white', 'alpha': 0.68, 'edgecolor': 'none', 'pad': 1.2})
        axis.set_title(title, fontsize=13)
        axis.set_axis_off()
        add_panel_label(axis, f'{chr(97 + index)})', x=0.01, y=0.98)

    process_labels = ['Model fitting', 'Transfer to the\n0.1° predictor grid', 'Block-wise mass\nconservation correction']
    for index, label in enumerate(process_labels):
        left, right = axes[index].get_position(), axes[index + 1].get_position()
        start = (left.x1 + 0.005, (left.y0 + left.y1) / 2)
        end = (right.x0 - 0.005, (right.y0 + right.y1) / 2)
        fig.add_artist(FancyArrowPatch(start, end, transform=fig.transFigure, arrowstyle='-|>',
                                       mutation_scale=15, linewidth=1.3, color='black'))
        fig.text((start[0] + end[0]) / 2, start[1] + 0.055, label,
                 fontsize=10, ha='center', va='bottom')

    sm = ScalarMappable(norm=norm, cmap='viridis')
    sm.set_array([])
    cax = fig.add_axes([0.945, 0.20, 0.015, 0.60])
    cbar = fig.colorbar(sm, cax=cax, extend='both')
    cbar.set_label('Temporal-mean GRACE TWS, cm', rotation=270, labelpad=18, fontsize=13)
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_4_downscaling_workflow_schematic.jpg')


## Figure 5 — Pearson correlation matrix

In [ ]:
if RUN_PUBLICATION_FIGURES:
    correlation_source = publication_feature_025[['P', 'ET', 'Q', 'temp', 'SM', TARGET_COL]].copy()
    correlation_source.columns = [
        'Precipitation (cm)', 'ET (cm)', 'Runoff (cm)',
        'Temperature (°C)', 'SM (cm)', 'GRACE TWS (cm)',
    ]
    correlation_matrix = correlation_source.apply(pd.to_numeric, errors='coerce').corr()

    fig, axis = plt.subplots(figsize=(10, 8))
    image = axis.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    ticks = np.arange(len(correlation_matrix))
    axis.set_xticks(ticks, correlation_matrix.columns, rotation=45, ha='right')
    axis.set_yticks(ticks, correlation_matrix.index)
    for row in range(len(ticks)):
        for column in range(len(ticks)):
            value = correlation_matrix.iloc[row, column]
            axis.text(column, row, f'{value:.2f}', ha='center', va='center',
                      color='white' if abs(value) >= 0.60 else 'black')
    axis.set_title('Pearson correlation matrix')
    cbar = fig.colorbar(image, ax=axis)
    cbar.set_label('Pearson correlation')
    fig.tight_layout()
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_5_pearson_correlation_matrix.jpeg')


## Figure 6 — TWS relationships with hydroclimatic predictors

In [ ]:
if RUN_PUBLICATION_FIGURES:
    hexbin_source = publication_feature_025[['P', 'ET', 'Q', 'temp', 'SM', TARGET_COL]].copy()
    panel_specs = [
        ('P', 'Precipitation (cm)'), ('ET', 'ET (cm)'), ('Q', 'Runoff (cm)'),
        ('temp', 'Temperature (°C)'), ('SM', 'SM (cm)'),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(13, 15), constrained_layout=True)
    axes = axes.ravel()
    for index, (column, label) in enumerate(panel_specs):
        axis = axes[index]
        data = hexbin_source[[column, TARGET_COL]].apply(pd.to_numeric, errors='coerce').dropna()
        limits = data[column].quantile([0.01, 0.99]).to_numpy()
        data = data.loc[data[column].between(*limits)]
        hexbin = axis.hexbin(data[column], data[TARGET_COL], gridsize=40, bins='log', mincnt=1, cmap='viridis')
        correlation = data[column].corr(data[TARGET_COL])
        axis.text(0.04, 0.96, f'r = {correlation:.2f}\nN = {len(data):,}', transform=axis.transAxes,
                  va='top', bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'})
        axis.set_xlabel(label)
        axis.set_ylabel('GRACE TWS (cm)')
        add_panel_label(axis, f'{chr(97 + index)})', x=-0.10, y=1.02)
        cbar = fig.colorbar(hexbin, ax=axis)
        cbar.set_label('Logarithmic point density')
    axes[-1].set_axis_off()
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_6_tws_hydroclimatic_hexbin.jpeg')


## Figure 7 — Imputation SHAP analysis

In [ ]:
import shap


def transformed_feature_names(
    fitted_preprocessor,
    numeric_columns,
    categorical_columns,
    rename_map=None,
):
    names = list(numeric_columns)

    if categorical_columns:
        categorical_transformer = fitted_preprocessor.named_transformers_["cat"]
        names.extend(
            categorical_transformer.get_feature_names_out(
                categorical_columns
            ).tolist()
        )

    if rename_map:
        names = [rename_map.get(name, name) for name in names]

    return names


imputation_feature_labels = {
    'grace_or_lag1': 'GRACE lag −1 month',
    'grace_or_lag2': 'GRACE lag −2 months',
    'grace_or_lag3': 'GRACE lag −3 months',
    'month': 'Month (seasonality)',
}

imputation_supervised = panel.loc[
    panel["grace_or_obs"].notna()
    & panel["grace_or_lag1"].notna()
    & panel["grace_or_lag2"].notna()
    & panel["grace_or_lag3"].notna()
].sort_values("mid_month_date")

imputation_cut = int(len(imputation_supervised) * 0.8)
imputation_holdout = imputation_supervised.iloc[imputation_cut:]

imputation_X = imputation_holdout[
    feature_cols_num + feature_cols_cat
]
if len(imputation_X) > SHAP_SAMPLE_SIZE:
    imputation_X = imputation_X.sample(
        SHAP_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )

imputation_preprocessor = model_eval.named_steps["prep"]
imputation_rf = model_eval.named_steps["rf"]

imputation_X_transformed = imputation_preprocessor.transform(
    imputation_X
)
if hasattr(imputation_X_transformed, "toarray"):
    imputation_X_transformed = imputation_X_transformed.toarray()

imputation_names = transformed_feature_names(
    imputation_preprocessor,
    feature_cols_num,
    feature_cols_cat,
    rename_map=imputation_feature_labels,
)

imputation_explainer = shap.TreeExplainer(imputation_rf)
imputation_shap_values = imputation_explainer.shap_values(
    imputation_X_transformed,
    check_additivity=False,
)

plt.close("all")
shap.summary_plot(
    imputation_shap_values,
    imputation_X_transformed,
    feature_names=imputation_names,
    max_display=12,
    show=False,
)

figure_7 = plt.gcf()
figure_7.set_size_inches(10, 5.5)
figure_7.tight_layout()
figure_7_path = MANUSCRIPT_DIR / 'fig_7_imputation_shap.jpeg'
figure_7.savefig(
    figure_7_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
)
if DISPLAY_FIGURES:
    plt.show()
plt.close(figure_7)

del imputation_X_transformed, imputation_shap_values, imputation_explainer, imputation_X
plt.close('all')
gc.collect()


## Figure 8 — Downscaling SHAP analysis

In [ ]:
downscaling_feature_labels = {
    "num__Q": "Runoff (R)",
    "num__Q_roll3": "R rolling mean, 3 months",
    "num__Q_lag1": "R lag −1 month",
    "num__Q_lag2": "R lag −2 months",
    "num__Q_lag3": "R lag −3 months",
    "num__SM": "Soil moisture (SM)",
    "num__SM_roll3": "SM rolling mean, 3 months",
    "num__SM_lag1": "SM lag −1 month",
    "num__SM_lag2": "SM lag −2 months",
    "num__SM_lag3": "SM lag −3 months",
    "num__P": "Precipitation (P)",
    "num__P_roll3": "P rolling mean, 3 months",
    "num__P_lag1": "P lag −1 month",
    "num__P_lag2": "P lag −2 months",
    "num__P_lag3": "P lag −3 months",
    "num__ET": "Evapotranspiration (ET)",
    "num__ET_roll3": "ET rolling mean, 3 months",
    "num__ET_lag1": "ET lag −1 month",
    "num__ET_lag2": "ET lag −2 months",
    "num__ET_lag3": "ET lag −3 months",
    "num__temp": "Surface temperature (T)",
    "num__temp_roll3": "T rolling mean, 3 months",
    "num__temp_lag1": "T lag −1 month",
    "num__temp_lag2": "T lag −2 months",
    "num__temp_lag3": "T lag −3 months",
    'num__elevation': 'Elevation',
}


def format_downscaling_feature_name(name):
    name = str(name)

    if name in downscaling_feature_labels:
        return downscaling_feature_labels[name]

    land_cover_prefix = 'cat__lc_class_'
    if name.startswith(land_cover_prefix):
        category = name.removeprefix(land_cover_prefix)
        category = category.removesuffix('.0')
        return f'Land cover: Type {category}'

    lithology_prefix = 'cat__lith_class_'
    if name.startswith(lithology_prefix):
        category = name.removeprefix(lithology_prefix)
        category = category.removesuffix('.0')
        return f'Lithology: Class {category}'

    return name

downscaling_X = training_df[feature_cols]
if len(downscaling_X) > SHAP_SAMPLE_SIZE:
    downscaling_X = downscaling_X.sample(
        SHAP_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )

downscaling_preprocessor = downscaling_model.named_steps["prep"]
downscaling_rf = downscaling_model.named_steps["model"]

downscaling_X_transformed = downscaling_preprocessor.transform(
    downscaling_X
)
if hasattr(downscaling_X_transformed, "toarray"):
    downscaling_X_transformed = downscaling_X_transformed.toarray()

raw_downscaling_names = (
    downscaling_preprocessor.get_feature_names_out()
)
downscaling_names = [
    format_downscaling_feature_name(name)
    for name in raw_downscaling_names
]

downscaling_explainer = shap.TreeExplainer(downscaling_rf)
downscaling_shap_values = downscaling_explainer.shap_values(
    downscaling_X_transformed,
    check_additivity=False,
)

plt.close("all")
shap.summary_plot(
    downscaling_shap_values,
    downscaling_X_transformed,
    feature_names=downscaling_names,
    max_display=len(downscaling_names),
    show=False,
)

figure_8 = plt.gcf()
figure_8.set_size_inches(
    10,
    max(8, 0.28 * len(downscaling_names)),
)
figure_8.tight_layout()
figure_8_path = MANUSCRIPT_DIR / 'fig_8_downscaling_shap.jpeg'
figure_8.savefig(
    figure_8_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
)
if DISPLAY_FIGURES:
    plt.show()
plt.close(figure_8)

del downscaling_X_transformed, downscaling_shap_values, downscaling_explainer, downscaling_X
plt.close('all')
gc.collect()


## Table 3 — Model validation

Grouped out-of-fold cross-validation is the parameter-selection estimate, repeated strict outer testing is the principal spatiotemporal generalization estimate, and the complete 2022–2023 holdout measures temporal transfer of the final model.

In [ ]:
# Table 3. Imputation and downscaling validation results
display(
    model_validation_summary_table.style.set_properties(
        **{
            'text-align': 'left',
            'white-space': 'normal',
        }
    )
)
model_validation_summary_table.to_csv(
    MANUSCRIPT_DIR / 'table_3_model_validation.csv',
    index=False,
)


## Figure 9 — Observed and predicted GRACE TWS

In [ ]:
if RUN_PUBLICATION_FIGURES:
    y_imputation_true = (
        imputation_test_predictions['y_true']
        .to_numpy(dtype=float)
    )
    y_imputation_pred = (
        imputation_test_predictions['y_pred']
        .to_numpy(dtype=float)
    )
    imputation_lower = (
        imputation_test_predictions['y_pred_lower']
        .to_numpy(dtype=float)
    )
    imputation_upper = (
        imputation_test_predictions['y_pred_upper']
        .to_numpy(dtype=float)
    )

    downscaling_test = pooled_outer_predictions_df[
        [
            'observed',
            'predicted',
            'prediction_lower',
            'prediction_upper',
        ]
    ].dropna()
    y_downscaling_true = (
        downscaling_test['observed']
        .to_numpy(dtype=float)
    )
    y_downscaling_pred = (
        downscaling_test['predicted']
        .to_numpy(dtype=float)
    )
    downscaling_lower = (
        downscaling_test['prediction_lower']
        .to_numpy(dtype=float)
    )
    downscaling_upper = (
        downscaling_test['prediction_upper']
        .to_numpy(dtype=float)
    )

    def _panel_metrics(y_true, y_pred, random_state):
        uncertainty = bootstrap_regression_metrics(
            y_true,
            y_pred,
            random_state=random_state,
        )
        return {
            'R²': uncertainty['r2']['estimate'],
            'R² CI lower': uncertainty['r2']['ci_lower'],
            'R² CI upper': uncertainty['r2']['ci_upper'],
            'MAE': uncertainty['mae']['estimate'],
            'MAE CI lower': uncertainty['mae']['ci_lower'],
            'MAE CI upper': uncertainty['mae']['ci_upper'],
            'Pearson r': (
                float(np.corrcoef(y_true, y_pred)[0, 1])
                if (
                    len(y_true) >= 2
                    and np.std(y_true) > 0
                    and np.std(y_pred) > 0
                )
                else np.nan
            ),
            'N': len(y_true),
        }

    figure_9_metrics = pd.DataFrame([
        {
            'stage': 'Imputation',
            **_panel_metrics(
                y_imputation_true,
                y_imputation_pred,
                RANDOM_STATE + 5000,
            ),
        },
        {
            'stage': 'Repeated downscaling outer tests',
            **_panel_metrics(
                y_downscaling_true,
                y_downscaling_pred,
                RANDOM_STATE + 5001,
            ),
        },
    ])
    display(figure_9_metrics.round(3))

    def _draw_observed_predicted(
        ax,
        y_true,
        y_pred,
        lower,
        upper,
        title,
        panel_label,
    ):
        ax.scatter(
            y_true,
            y_pred,
            alpha=0.45,
            s=18,
        )

        # Draw uncertainty bars for a reproducible subset to avoid clutter.
        if len(y_true) > 220:
            rng = np.random.default_rng(RANDOM_STATE)
            sample_index = np.sort(
                rng.choice(
                    len(y_true),
                    size=220,
                    replace=False,
                )
            )
        else:
            sample_index = np.arange(len(y_true))

        lower_error = (
            y_pred[sample_index]
            - lower[sample_index]
        )
        upper_error = (
            upper[sample_index]
            - y_pred[sample_index]
        )
        ax.errorbar(
            y_true[sample_index],
            y_pred[sample_index],
            yerr=np.vstack([
                lower_error,
                upper_error,
            ]),
            fmt='none',
            alpha=0.18,
            linewidth=0.5,
        )

        minimum = float(
            min(
                np.nanmin(y_true),
                np.nanmin(y_pred),
            )
        )
        maximum = float(
            max(
                np.nanmax(y_true),
                np.nanmax(y_pred),
            )
        )
        ax.plot(
            [minimum, maximum],
            [minimum, maximum],
            linestyle='--',
        )
        ax.set_xlabel(
            'Observed GRACE TWS, cm',
            fontsize=14,
        )
        ax.set_ylabel(
            'Predicted GRACE TWS, cm',
            fontsize=14,
        )
        ax.set_title(title, fontsize=14)
        ax.tick_params(axis='both', labelsize=13)
        ax.grid(True, alpha=0.35)
        ax.text(
            -0.10,
            1.02,
            panel_label,
            transform=ax.transAxes,
            fontsize=16,
            fontweight='bold',
            va='top',
        )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 6),
    )
    _draw_observed_predicted(
        axes[0],
        y_imputation_true,
        y_imputation_pred,
        imputation_lower,
        imputation_upper,
        'Imputation ML',
        'a)',
    )
    _draw_observed_predicted(
        axes[1],
        y_downscaling_true,
        y_downscaling_pred,
        downscaling_lower,
        downscaling_upper,
        'Repeated downscaling outer tests',
        'b)',
    )
    fig.tight_layout()
    fig.savefig(
        MANUSCRIPT_DIR / 'fig_9_observed_vs_predicted.jpeg',
        dpi=EXPORT_DPI,
        bbox_inches='tight',
    )
    if DISPLAY_FIGURES:
        plt.show()

    for (
        filename,
        y_true,
        y_pred,
        lower,
        upper,
        title,
        panel_label,
    ) in [
        (
            OUTPUT_DIR
            / 'fig_9a_imputation_observed_vs_predicted.jpeg',
            y_imputation_true,
            y_imputation_pred,
            imputation_lower,
            imputation_upper,
            'Imputation ML',
            'a)',
        ),
        (
            OUTPUT_DIR
            / 'fig_9b_downscaling_observed_vs_predicted.jpeg',
            y_downscaling_true,
            y_downscaling_pred,
            downscaling_lower,
            downscaling_upper,
            'Repeated downscaling outer tests',
            'b)',
        ),
    ]:
        panel_fig, panel_ax = plt.subplots(
            figsize=(7, 7)
        )
        _draw_observed_predicted(
            panel_ax,
            y_true,
            y_pred,
            lower,
            upper,
            title,
            panel_label,
        )
        panel_fig.tight_layout()
        panel_fig.savefig(
            filename,
            dpi=EXPORT_DPI,
            bbox_inches='tight',
        )
        plt.close(panel_fig)

    plt.close('all')


## Figure 10 — Spatial imputation performance

In [ ]:
if RUN_PUBLICATION_FIGURES:
    plot_gdf = imputation_cell_metrics_gdf

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5),
        constrained_layout=True,
    )

    # Panel a: R²
    plot_gdf.plot(
        column='r2',
        ax=axes[0],
        cmap='inferno',
        vmin=0,
        vmax=1,
        edgecolor='none',
        linewidth=0,
        missing_kwds={
            'color': 'lightgrey',
        },
        legend=True,
        legend_kwds={
            'label': 'R²',
            'shrink': 0.85,
            'fraction': 0.035,
            'aspect': 40,
            'ticks': [0, 0.25, 0.50, 0.75, 1.00],
            'format': '%.2f',
        },
    )

    # Panel b: MAE
    plot_gdf.plot(
        column='mae',
        ax=axes[1],
        cmap='inferno_r',
        edgecolor='none',
        linewidth=0,
        missing_kwds={
            'color': 'lightgrey',
        },
        legend=True,
        legend_kwds={
            'label': 'MAE, cm',
            'shrink': 0.85,
            'fraction': 0.035,
            'aspect': 40,
            'extend': 'max',
        },
    )

    for ax, label in zip(
        axes,
        ('a)', 'b)'),
    ):
        ax.set_axis_off()

        ax.text(
            -0.06,
            1.02,
            label,
            transform=ax.transAxes,
            fontsize=17,
            fontweight='bold',
            ha='left',
            va='top',
        )

    # Adjust colorbar label and tick font sizes.
    colorbar_axes = [
        ax
        for ax in fig.axes
        if ax not in axes
    ]

    for colorbar_ax in colorbar_axes:
        colorbar_ax.tick_params(
            labelsize=14,
        )

        colorbar_ax.set_ylabel(
            colorbar_ax.get_ylabel(),
            fontsize=16,
            labelpad=8,
            rotation=90,
        )

    fig.savefig(
        OUTPUT_DIR
        / 'fig_10_imputation_holdout_spatial_skill.jpeg',
        dpi=EXPORT_DPI,
        bbox_inches='tight',
    )

    if DISPLAY_FIGURES:
        plt.show()

    plt.close(fig)
    gc.collect()


## Figure 11 — Spatiotemporal validation diagnostics

In [ ]:
figure_11_data = spatiotemporal_cv_df.sort_values(
    ["spatial_block", "year"]
).reset_index(drop=True)

pivot_r2 = figure_11_data.pivot(
    index="spatial_block",
    columns="year",
    values="r2_cv",
)
pivot_mae = figure_11_data.pivot(
    index="spatial_block",
    columns="year",
    values="mae_cv",
)

figure_11, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharex=True,
)

years = pivot_r2.columns.to_numpy()
x_ticks = np.arange(len(years))

r2_image = axes[0].imshow(
    pivot_r2.values,
    aspect="auto",
    origin="lower",
    cmap="inferno",
)
axes[0].set_yticks(np.arange(pivot_r2.shape[0]))
axes[0].set_yticklabels(pivot_r2.index, fontsize=15)
axes[0].set_xticks(x_ticks)
axes[0].set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=14,
)
axes[0].set_xlabel("Year", fontsize=16)
axes[0].set_ylabel("Spatial block", fontsize=16)
axes[0].text(
    -0.10,
    1.02,
    "a)",
    transform=axes[0].transAxes,
    fontsize=19,
    fontweight="bold",
)
r2_colorbar = figure_11.colorbar(
    r2_image,
    ax=axes[0],
    fraction=0.046,
    pad=0.02,
)
r2_colorbar.set_label("R²", fontsize=15)
r2_colorbar.ax.tick_params(labelsize=13)

mae_image = axes[1].imshow(
    pivot_mae.values,
    aspect="auto",
    origin="lower",
    cmap="inferno_r",
)
axes[1].set_yticks(np.arange(pivot_mae.shape[0]))
axes[1].set_yticklabels(pivot_mae.index, fontsize=15)
axes[1].set_xticks(x_ticks)
axes[1].set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=14,
)
axes[1].set_xlabel("Year", fontsize=16)
axes[1].set_ylabel("Spatial block", fontsize=16)
axes[1].text(
    -0.10,
    1.02,
    "b)",
    transform=axes[1].transAxes,
    fontsize=19,
    fontweight="bold",
)
mae_colorbar = figure_11.colorbar(
    mae_image,
    ax=axes[1],
    fraction=0.046,
    pad=0.02,
)
mae_colorbar.set_label("MAE, cm", fontsize=15)
mae_colorbar.ax.tick_params(labelsize=13)

figure_11.tight_layout()
figure_11_path = MANUSCRIPT_DIR / 'fig_11_spatiotemporal_cv.jpeg'
figure_11.savefig(
    figure_11_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
)
if DISPLAY_FIGURES:
    plt.show()
plt.close(figure_11)

del figure_11_data, pivot_r2, pivot_mae
gc.collect()


## Figure 12 — Spatial GRACE TWS trends

In [ ]:
if RUN_PUBLICATION_FIGURES:
    def build_slope_maps_by_cell(
        panel_gdf,
        geometry_col="geometry",
        date_col="mid_month_date",
        value_col="grace_or_imputed",
        periods=None,
        min_points=2,


        target_crs="EPSG:4326",          # WGS-84
        assume_crs_if_missing="EPSG:4326",  # if input CRS missing, assign this (no reprojection)
    ):
        if periods is None:
            periods = {
                "slope_2002_2024": ("2002-01-01", "2024-12-31"),
                "slope_2013_2023": ("2013-01-01", "2023-12-31"),
            }

        # Ensure GeoDataFrame + geometry
        if not isinstance(panel_gdf, gpd.GeoDataFrame):
            gdf = gpd.GeoDataFrame(panel_gdf.copy(), geometry=geometry_col)
        else:
            gdf = panel_gdf.copy()
            if gdf.geometry.name != geometry_col:
                gdf = gdf.set_geometry(geometry_col)

        gdf[date_col] = pd.to_datetime(gdf[date_col])

        # --- CRS handling (same fix as before) ---
        if gdf.crs is None and assume_crs_if_missing is not None:
            # ASSIGN CRS only (no coordinate changes)
            gdf = gdf.set_crs(assume_crs_if_missing, allow_override=True)

        if target_crs is not None and gdf.crs is not None and str(gdf.crs) != str(target_crs):
            # Reproject if different
            gdf = gdf.to_crs(target_crs)

        # Stable cell identifier derived from geometry WKB in the analysis CRS.
        wkb = gpd.GeoSeries(gdf[geometry_col], crs=gdf.crs).to_wkb()
        codes, uniques = pd.factorize(wkb)
        gdf["__cell_id"] = codes.astype(np.int64)

        cells = gpd.GeoDataFrame(
            {"__cell_id": np.arange(len(uniques), dtype=np.int64)},
            geometry=gpd.GeoSeries.from_wkb(uniques, crs=gdf.crs),
            crs=gdf.crs,
        )

        def slope_per_year(df_cell):
            y = df_cell[value_col].to_numpy(dtype=float)
            t = pd.to_datetime(df_cell[date_col]).to_numpy(dtype="datetime64[ns]")

            m = np.isfinite(y) & pd.notna(t)
            if m.sum() < min_points:
                return np.nan

            t = t[m].astype("datetime64[ns]")
            y = y[m]

            t_years = t.astype("datetime64[s]").astype(np.int64) / (365.25 * 24 * 3600.0)
            slope, _ = np.polyfit(t_years, y, 1)
            return float(slope)

        slope_maps = {}
        for name, (start, end) in periods.items():
            start = pd.Timestamp(start)
            end = pd.Timestamp(end)

            sub = gdf.loc[
                (gdf[date_col] >= start) & (gdf[date_col] <= end),
                ["__cell_id", date_col, value_col],
            ].copy()

            slopes = (
                sub.groupby("__cell_id", observed=True)
                   .apply(slope_per_year)
                   .rename(name)
                   .reset_index()
            )

            out = cells.merge(slopes, on="__cell_id", how="left")
            slope_maps[name] = gpd.GeoDataFrame(out, geometry="geometry", crs=cells.crs)

        return slope_maps

    def plot_two_slope_maps_shared_colorbar(
        slope_maps,
        key_a="slope_2002_2024",
        key_b="slope_2013_2023",
        cmap="magma",
        focus="decline",
        clip_quantiles=(0.02, 0.98),
        vmin=None,
        vmax=None,
        cbar_label="Trend (GRACE TWS / year)",
        font_scale=1.2,
        fig_size=(12, 5),
        panel_labels=("a)", "b)"),
        panel_xy=(-0.06, 1.02),
        edgecolor="none",
        linewidth=0.0,
        missing_color="lightgrey",
        save_path=None,
        dpi=EXPORT_DPI,


        geographic_aspect=True,        # latitude-aware aspect in lon/lat
        aspect_exaggeration=1.0,       # set e.g. 1.1 to stretch vertically a bit more
    ):
        base_fs = 14 * float(font_scale)

        gdf_a = slope_maps[key_a]
        gdf_b = slope_maps[key_b]

        vals = pd.concat([gdf_a[key_a], gdf_b[key_b]], axis=0).astype(float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            raise ValueError("No finite slope values found to set shared color scale.")

        if vmin is None or vmax is None:
            qlo, qhi = clip_quantiles
            lo = float(np.nanquantile(vals, qlo))
            hi = float(np.nanquantile(vals, qhi))

            if focus == "decline":
                vmin_auto = min(lo, 0.0)
                vmax_auto = 0.0
            else:
                vmin_auto = lo
                vmax_auto = hi

            if vmin is None:
                vmin = vmin_auto
            if vmax is None:
                vmax = vmax_auto

        plt.close("all")
        fig, axes = plt.subplots(1, 2, figsize=fig_size, constrained_layout=True)

        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])

        # Apply latitude-aware aspect scaling for geographic coordinates.
        aspect_val = "auto"
        if geographic_aspect:
            # If data are lon/lat, CRS is usually EPSG:4326. We’ll apply the standard cos(lat) correction.
            try:
                all_gdf = pd.concat([gdf_a[["geometry"]], gdf_b[["geometry"]]], axis=0)
                mean_lat = float(gpd.GeoSeries(all_gdf["geometry"]).centroid.y.mean())
                aspect_val = (aspect_exaggeration / max(np.cos(np.deg2rad(mean_lat)), 1e-6))
            except Exception:
                aspect_val = "auto"

        def _plot(ax, gdf, col):
            gdf.plot(
                column=col,
                ax=ax,
                cmap=cmap,
                norm=norm,
                legend=False,
                missing_kwds={"color": missing_color},
                edgecolor=edgecolor,
                linewidth=linewidth,
            )
            ax.set_axis_off()
            ax.set_aspect(aspect_val)

        _plot(axes[0], gdf_a, key_a)
        _plot(axes[1], gdf_b, key_b)

        for ax, lab in zip(axes, panel_labels):
            ax.text(
                panel_xy[0], panel_xy[1],
                lab,
                transform=ax.transAxes,
                fontsize=17 * float(font_scale),
                fontweight="bold",
                ha="left", va="top",
            )

        cbar = fig.colorbar(sm, ax=axes, fraction=0.035, pad=0.02)
        cbar.set_label(cbar_label, fontsize=base_fs)
        cbar.ax.tick_params(labelsize=base_fs * 0.85)

        if save_path is not None:
            fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

        if DISPLAY_FIGURES:
            plt.show()
        return fig, axes, (vmin, vmax)

    slope_maps = build_slope_maps_by_cell(
        panel_gdf=panel_gdf,
        geometry_col="geometry",
        date_col="mid_month_date",
        value_col="grace_or_imputed",
        periods={
            "slope_2002_2024": ("2002-01-01", "2024-12-31"),
            "slope_2013_2023": ("2013-01-01", "2023-12-31"),
        },
        min_points=2,
        target_crs="EPSG:4326",
        assume_crs_if_missing="EPSG:4326",
    )

    fig, axes, (vmin, vmax) = plot_two_slope_maps_shared_colorbar(
        slope_maps,
        key_a="slope_2002_2024",
        key_b="slope_2013_2023",
        cmap="magma",
        focus="decline",
        clip_quantiles=(0.02, 0.98),
        cbar_label="Trend (TWS cm / year)",
        font_scale=1.3,
        fig_size=(12, 5),
        save_path=MANUSCRIPT_DIR / 'fig_12_tws_trends_2002_2024_2013_2023.jpg',
        dpi=EXPORT_DPI,

        geographic_aspect=True,
        aspect_exaggeration=1.0,  # try 1.1 if you want a bit more vertical stretch
    )

    plt.close("all")


## Figures 13–14 and Table 4 — Weighting sensitivity and mass conservation

In [ ]:
if RUN_PUBLICATION_FIGURES:
    sensitivity_schemes = [('Runoff', 'Temporal-mean runoff weight R', 'River runoff, cm'), ('Soil moisture', 'Temporal-mean soil-moisture weight SM', 'Soil moisture, cm'), ('Precipitation', 'Temporal-mean precipitation weight P', 'Precipitation, cm'), ('Uniform area', 'Uniform-area weight', 'Uniform weight'), ('All-feature SHAP composite', 'Temporal-mean all-feature SHAP composite', 'SHAP-weighted score')]
    SHAP_COMPOSITE_UNIFORM_BLEND = 0.1

    def sensitivity_positive(values):
        values = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
        return np.where(np.isfinite(values) & (values > 0), values, 0.0)

    def sensitivity_grouped_minmax(frame, values, group_columns=('date', 'mass_block_id')):
        """Normalize one transformed feature within each month–block."""
        working = frame.loc[:, list(group_columns)].copy()
        working['_value'] = np.asarray(values, dtype=float)
        working['_value'] = working['_value'].replace([np.inf, -np.inf], np.nan)
        minimum = working.groupby(list(group_columns), observed=True)['_value'].transform('min')
        maximum = working.groupby(list(group_columns), observed=True)['_value'].transform('max')
        value_range = maximum - minimum
        normalized = pd.Series(0.0, index=working.index, dtype=float)
        variable = value_range.gt(0)
        normalized.loc[variable] = (working.loc[variable, '_value'] - minimum.loc[variable]) / value_range.loc[variable]
        return normalized.to_numpy(dtype=float)

    def sensitivity_robust_norm(values):
        finite = np.asarray(values, dtype=float)
        finite = finite[np.isfinite(finite)]
        if finite.size == 0:
            return Normalize(vmin=0.0, vmax=1.0)
        lower = float(np.nanquantile(finite, 0.02))
        upper = float(np.nanquantile(finite, 0.98))
        if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
            lower = float(np.nanmin(finite))
            upper = float(np.nanmax(finite))
        if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
            lower = 0.0
            upper = 1.0
        return Normalize(vmin=lower, vmax=upper)

    def sensitivity_feature_label(name):
        name = str(name)
        label_map = {'num__Q': 'Runoff (R)', 'num__Q_roll3': 'R rolling mean, 3 months', 'num__Q_lag1': 'R lag −1 month', 'num__Q_lag2': 'R lag −2 months', 'num__Q_lag3': 'R lag −3 months', 'num__SM': 'Soil moisture (SM)', 'num__SM_roll3': 'SM rolling mean, 3 months', 'num__SM_lag1': 'SM lag −1 month', 'num__SM_lag2': 'SM lag −2 months', 'num__SM_lag3': 'SM lag −3 months', 'num__P': 'Precipitation (P)', 'num__P_roll3': 'P rolling mean, 3 months', 'num__P_lag1': 'P lag −1 month', 'num__P_lag2': 'P lag −2 months', 'num__P_lag3': 'P lag −3 months', 'num__ET': 'Evapotranspiration (ET)', 'num__ET_roll3': 'ET rolling mean, 3 months', 'num__ET_lag1': 'ET lag −1 month', 'num__ET_lag2': 'ET lag −2 months', 'num__ET_lag3': 'ET lag −3 months', 'num__temp': 'Surface temperature (T)', 'num__temp_roll3': 'T rolling mean, 3 months', 'num__temp_lag1': 'T lag −1 month', 'num__temp_lag2': 'T lag −2 months', 'num__temp_lag3': 'T lag −3 months', 'num__elevation': 'Elevation'}
        if name in label_map:
            return label_map[name]
        if name.startswith('cat__lc_class_'):
            category = name.removeprefix('cat__lc_class_').removesuffix('.0')
            return f'Land cover: Type {category}'
        if name.startswith('cat__lith_class_'):
            category = name.removeprefix('cat__lith_class_').removesuffix('.0')
            return f'Lithology: Class {category}'
        return name
    required_prediction_columns = ['date', 'cell_01_key', 'cell_01', 'parent_025', 'mass_block_id', 'area_01', 'y_pred_raw', 'geometry_01']
    required_parent_columns = ['date', 'cell_025', 'mass_block_id', 'y_true', 'geometry_0.25']
    missing_prediction_columns = [column for column in required_prediction_columns if column not in gdf_01_pred.columns]
    missing_parent_columns = [column for column in required_parent_columns if column not in gdf_025_pred.columns]
    if missing_prediction_columns:
        raise KeyError(f'Figure 13 is missing compact 0.1° prediction columns: {missing_prediction_columns}')
    if missing_parent_columns:
        raise KeyError(f'Figure 13 is missing 0.25° prediction columns: {missing_parent_columns}')
    if 'publication_feature_01' not in globals():
        raise NameError('publication_feature_01 was not created. Rerun the feature-preparation downscaling-application cell once before running Figure 13.')
    missing_preserved_features = [column for column in feature_cols if column not in publication_feature_01.columns]
    if missing_preserved_features:
        raise KeyError(f'The preserved fine-grid feature table is incomplete: {missing_preserved_features}')
    compact_fine = gdf_01_pred[required_prediction_columns].copy()
    compact_fine['date'] = to_month_start(compact_fine['date'])
    fine_feature_lookup = publication_feature_01[['date', 'cell_01_key', *feature_cols]].copy()
    fine_feature_lookup['date'] = to_month_start(fine_feature_lookup['date'])
    fine_feature_lookup = fine_feature_lookup.drop_duplicates(['date', 'cell_01_key'])
    fine = compact_fine.merge(fine_feature_lookup, on=['date', 'cell_01_key'], how='left', validate='many_to_one')
    fine = gpd.GeoDataFrame(fine, geometry='geometry_01', crs=getattr(gdf_01_pred, 'crs', None))
    parent = gpd.GeoDataFrame(gdf_025_pred[required_parent_columns].copy(), geometry='geometry_0.25', crs=getattr(gdf_025_pred, 'crs', None))
    parent['date'] = to_month_start(parent['date'])
    missing_after_merge = [column for column in feature_cols if fine[column].notna().sum() == 0]
    if missing_after_merge:
        raise KeyError(f'Fine-grid predictors could not be restored after the merge: {missing_after_merge}')
    fine = fine.dropna(subset=['date', 'cell_01', 'parent_025', 'mass_block_id', 'area_01', 'y_pred_raw', 'geometry_01', *feature_cols]).copy()
    parent = parent.dropna(subset=['date', 'cell_025', 'mass_block_id', 'y_true', 'geometry_0.25']).drop_duplicates(['date', 'cell_025']).copy()
    if fine.empty or parent.empty:
        raise ValueError('No complete observations remain for Figure 13.')
    sensitivity_preprocessor = downscaling_model.named_steps['prep']
    sensitivity_rf = downscaling_model.named_steps['model']
    sensitivity_shap_X = training_df[feature_cols]
    if len(sensitivity_shap_X) > SHAP_SAMPLE_SIZE:
        sensitivity_shap_X = sensitivity_shap_X.sample(SHAP_SAMPLE_SIZE, random_state=RANDOM_STATE)
    sensitivity_shap_X_transformed = sensitivity_preprocessor.transform(sensitivity_shap_X)
    if hasattr(sensitivity_shap_X_transformed, 'toarray'):
        sensitivity_shap_X_transformed = sensitivity_shap_X_transformed.toarray()
    sensitivity_transformed_names = np.asarray(sensitivity_preprocessor.get_feature_names_out(), dtype=object)
    sensitivity_shap_values = shap.TreeExplainer(sensitivity_rf).shap_values(sensitivity_shap_X_transformed, check_additivity=False)
    sensitivity_shap_values = np.asarray(sensitivity_shap_values, dtype=float)
    if sensitivity_shap_values.ndim != 2:
        raise ValueError(f'Unexpected SHAP array dimensions for the downscaling model: {sensitivity_shap_values.shape}')
    sensitivity_mean_abs_shap = np.nanmean(np.abs(sensitivity_shap_values), axis=0)
    if len(sensitivity_mean_abs_shap) != len(sensitivity_transformed_names):
        raise ValueError('SHAP importance and transformed-feature counts differ.')
    if not np.isfinite(sensitivity_mean_abs_shap).any() or np.nansum(sensitivity_mean_abs_shap) <= 0:
        sensitivity_mean_abs_shap = np.ones(len(sensitivity_transformed_names), dtype=float)
    sensitivity_shap_shares = sensitivity_mean_abs_shap / np.nansum(sensitivity_mean_abs_shap)
    figure_13_shap_feature_weights = pd.DataFrame({'transformed_feature': sensitivity_transformed_names, 'feature_label': [sensitivity_feature_label(name) for name in sensitivity_transformed_names], 'mean_absolute_SHAP': sensitivity_mean_abs_shap, 'normalized_SHAP_share': sensitivity_shap_shares}).sort_values('normalized_SHAP_share', ascending=False).reset_index(drop=True)
    figure_13_shap_feature_weights.to_csv(DIAGNOSTIC_DIR / 'diagnostic_all_feature_SHAP_composite_weights.csv', index=False)
    fine_transformed = sensitivity_preprocessor.transform(fine[feature_cols])
    if hasattr(fine_transformed, 'toarray'):
        fine_transformed = fine_transformed.toarray()
    fine_transformed = np.asarray(fine_transformed, dtype=float)
    if fine_transformed.shape[1] != len(sensitivity_transformed_names):
        raise ValueError('The transformed fine-grid matrix does not match the SHAP feature-importance vector.')
    shap_composite = np.zeros(len(fine), dtype=float)
    for feature_index, feature_share in enumerate(sensitivity_shap_shares):
        if not np.isfinite(feature_share) or feature_share <= 0:
            continue
        shap_composite += feature_share * sensitivity_grouped_minmax(fine, fine_transformed[:, feature_index])
    shap_composite = SHAP_COMPOSITE_UNIFORM_BLEND + (1.0 - SHAP_COMPOSITE_UNIFORM_BLEND) * shap_composite
    child_area = fine.groupby(['date', 'parent_025'], as_index=False, observed=True).agg(effective_area_m2=('area_01', 'sum'))
    parent_mass = parent.merge(child_area, left_on=['date', 'cell_025'], right_on=['date', 'parent_025'], how='inner')
    parent_mass['target_mass_cm_m2'] = parent_mass['y_true'] * parent_mass['effective_area_m2']
    target_mass = parent_mass.groupby(['date', 'mass_block_id'], as_index=False, observed=True).agg(target_mass_cm_m2=('target_mass_cm_m2', 'sum'))
    raw_mass = fine.assign(raw_mass_cm_m2=lambda frame: frame['y_pred_raw'] * frame['area_01']).groupby(['date', 'mass_block_id'], as_index=False, observed=True).agg(raw_mass_cm_m2=('raw_mass_cm_m2', 'sum'), available_area_m2=('area_01', 'sum'))
    block_balance = target_mass.merge(raw_mass, on=['date', 'mass_block_id'], how='inner')
    block_balance['residual_mass_cm_m2'] = block_balance['target_mass_cm_m2'] - block_balance['raw_mass_cm_m2']
    fine = fine.merge(block_balance[['date', 'mass_block_id', 'target_mass_cm_m2', 'raw_mass_cm_m2', 'available_area_m2', 'residual_mass_cm_m2']], on=['date', 'mass_block_id'], how='inner')
    fine = gpd.GeoDataFrame(fine, geometry='geometry_01', crs=getattr(gdf_01_pred, 'crs', None))
    scheme_weights = {'Runoff': sensitivity_positive(fine['Q']), 'Soil moisture': sensitivity_positive(fine['SM']), 'Precipitation': sensitivity_positive(fine['P']), 'Uniform area': np.ones(len(fine), dtype=float), 'All-feature SHAP composite': shap_composite}
    sensitivity_results = []
    sensitivity_summary_rows = []
    sensitivity_closure_rows = []
    runoff_reference = None
    for scheme_name, weight_title, weight_label in sensitivity_schemes:
        scheme = fine.copy()
        scheme['weight'] = scheme_weights[scheme_name]
        scheme['weighted_area'] = scheme['weight'] * scheme['area_01']
        scheme['sum_weighted_area'] = scheme.groupby(['date', 'mass_block_id'], observed=True)['weighted_area'].transform('sum')
        fallback = ~np.isfinite(scheme['sum_weighted_area']) | scheme['sum_weighted_area'].le(0)
        proportional = np.divide((scheme['residual_mass_cm_m2'] * scheme['weight']).to_numpy(dtype=float), scheme['sum_weighted_area'].to_numpy(dtype=float), out=np.zeros(len(scheme), dtype=float), where=scheme['sum_weighted_area'].to_numpy(dtype=float) > 0)
        uniform = np.divide(scheme['residual_mass_cm_m2'].to_numpy(dtype=float), scheme['available_area_m2'].to_numpy(dtype=float), out=np.zeros(len(scheme), dtype=float), where=scheme['available_area_m2'].to_numpy(dtype=float) > 0)
        scheme['correction_cm'] = np.where(fallback, uniform, proportional)
        scheme['final_TWS_cm'] = scheme['y_pred_raw'] + scheme['correction_cm']
        scheme['final_mass_cm_m2'] = scheme['final_TWS_cm'] * scheme['area_01']
        final_mass = scheme.groupby(['date', 'mass_block_id'], as_index=False, observed=True).agg(final_mass_cm_m2=('final_mass_cm_m2', 'sum'))
        closure = block_balance.merge(final_mass, on=['date', 'mass_block_id'], how='inner')
        closure['target_minus_raw_cm_km2'] = (closure['target_mass_cm_m2'] - closure['raw_mass_cm_m2']) / 1000000.0
        closure['target_minus_final_cm_km2'] = (closure['target_mass_cm_m2'] - closure['final_mass_cm_m2']) / 1000000.0
        closure['weighting_scheme'] = scheme_name
        sensitivity_closure_rows.append(closure)
        current = scheme.set_index(['date', 'cell_01'])['correction_cm']
        if runoff_reference is None:
            runoff_reference = current.copy()
        common = current.index.intersection(runoff_reference.index)
        difference = current.loc[common] - runoff_reference.loc[common]
        fallback_block_months = scheme.loc[fallback, ['date', 'mass_block_id']].drop_duplicates().shape[0]
        sensitivity_summary_rows.append({'weighting_scheme': scheme_name, 'available_months': int(scheme['date'].nunique()), 'evaluated_cell_months': int(len(scheme)), 'mean_correction_cm': float(scheme['correction_cm'].mean()), 'mean_absolute_correction_cm': float(scheme['correction_cm'].abs().mean()), 'spatiotemporal_std_correction_cm': float(scheme['correction_cm'].std(ddof=0)), 'minimum_correction_cm': float(scheme['correction_cm'].min()), 'maximum_correction_cm': float(scheme['correction_cm'].max()), 'mean_absolute_difference_vs_runoff_cm': float(difference.abs().mean()), 'correlation_with_runoff_correction': float(current.loc[common].corr(runoff_reference.loc[common])) if len(common) >= 2 else np.nan, 'uniform_fallback_block_months': int(fallback_block_months), 'maximum_post_correction_residual_cm_km2': float(closure['target_minus_final_cm_km2'].abs().max())})
        mean_map = scheme.groupby('cell_01', as_index=False, observed=True).agg(mean_weight=('weight', 'mean'), mean_correction_cm=('correction_cm', 'mean'), available_months=('date', 'nunique'), geometry_01=('geometry_01', 'first'))
        mean_map = gpd.GeoDataFrame(mean_map, geometry='geometry_01', crs=fine.crs)
        mean_closure = closure.groupby('mass_block_id', as_index=False, observed=True).agg(mean_target_minus_raw_cm_km2=('target_minus_raw_cm_km2', 'mean'), mean_target_minus_final_cm_km2=('target_minus_final_cm_km2', 'mean'), evaluated_months=('date', 'nunique')).sort_values('mass_block_id').reset_index(drop=True)
        sensitivity_results.append({'name': scheme_name, 'weight_title': weight_title, 'weight_label': weight_label, 'map': mean_map, 'closure': mean_closure})
    block_boundaries = gpd.GeoDataFrame(parent[['cell_025', 'mass_block_id', 'geometry_0.25']].drop_duplicates('cell_025'), geometry='geometry_0.25', crs=parent.crs).dissolve(by='mass_block_id').boundary
    if block_boundaries.crs is not None and fine.crs is not None and (block_boundaries.crs != fine.crs):
        block_boundaries = block_boundaries.to_crs(fine.crs)
    correction_values = np.concatenate([result['map']['mean_correction_cm'].dropna().to_numpy(dtype=float) for result in sensitivity_results])
    correction_limit = float(np.nanquantile(np.abs(correction_values), 0.98))
    if not np.isfinite(correction_limit) or correction_limit <= 0:
        correction_limit = float(np.nanmax(np.abs(correction_values)))
    if not np.isfinite(correction_limit) or correction_limit <= 0:
        correction_limit = 1.0
    correction_norm = TwoSlopeNorm(vmin=-correction_limit, vcenter=0.0, vmax=correction_limit)
    figure_13, axes_13 = plt.subplots(len(sensitivity_results), 2, figsize=(12, 22), constrained_layout=True, squeeze=False)
    for row, result in enumerate(sensitivity_results):
        map_data = result['map']
        row_label = f'{chr(97 + row)})'
        weight_axis = axes_13[row, 0]
        correction_axis = axes_13[row, 1]
        weight_values = map_data['mean_weight'].to_numpy(dtype=float)
        if result['name'] == 'Uniform area':
            weight_norm = Normalize(vmin=0.0, vmax=1.0)
        else:
            weight_norm = sensitivity_robust_norm(weight_values)
        map_data.plot(column='mean_weight', ax=weight_axis, cmap='viridis', norm=weight_norm, edgecolor='none', legend=False, missing_kwds={'color': 'lightgrey'})
        block_boundaries.plot(ax=weight_axis, color='black', linewidth=0.45)
        weight_axis.set_title(f"{row_label} {result['weight_title']}", fontsize=13)
        weight_axis.set_axis_off()
        weight_scalar = ScalarMappable(norm=weight_norm, cmap='viridis')
        weight_scalar.set_array([])
        weight_cbar = figure_13.colorbar(weight_scalar, ax=weight_axis, fraction=0.046, pad=0.02)
        weight_cbar.set_label(result['weight_label'])
        map_data.plot(column='mean_correction_cm', ax=correction_axis, cmap='RdBu_r', norm=correction_norm, edgecolor='none', legend=False, missing_kwds={'color': 'lightgrey'})
        block_boundaries.plot(ax=correction_axis, color='black', linewidth=0.45)
        correction_axis.set_title('Temporal-mean bias-aware correction', fontsize=13)
        correction_axis.set_axis_off()
        correction_scalar = ScalarMappable(norm=correction_norm, cmap='RdBu_r')
        correction_scalar.set_array([])
        correction_cbar = figure_13.colorbar(correction_scalar, ax=correction_axis, fraction=0.046, pad=0.02, extend='both')
        correction_cbar.set_label('Mean ΔTWS, cm')
    figure_13.suptitle('Temporal-mean sensitivity of the bias-aware mass-conservation correction', fontsize=16, fontweight='bold')
    figure_13.savefig(MANUSCRIPT_DIR / 'fig_13_mass_weight_sensitivity_5x2.jpg', dpi=EXPORT_DPI, bbox_inches='tight', pad_inches=0.02)
    if DISPLAY_FIGURES:
        plt.show()
    plt.close(figure_13)
    raw_closure = sensitivity_results[0]['closure']
    post_residual_values = np.concatenate([result['closure']['mean_target_minus_final_cm_km2'].to_numpy(dtype=float) for result in sensitivity_results])
    post_residual_values = post_residual_values[np.isfinite(post_residual_values)]
    post_limit = float(np.nanmax(np.abs(post_residual_values))) if post_residual_values.size else 1.0
    if not np.isfinite(post_limit) or post_limit <= 0:
        post_limit = 1.0
    figure_14, axes_14 = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
    axes_14[0].plot(raw_closure['mass_block_id'], raw_closure['mean_target_minus_raw_cm_km2'], marker='o', linewidth=1.5)
    axes_14[0].axhline(0.0, linewidth=0.8)
    axes_14[0].set_title('a) Before mass-conservation correction')
    axes_14[0].set_xlabel('Block ID')
    axes_14[0].set_ylabel('Temporal-mean mass residual, cm × km²')
    axes_14[0].grid(True, alpha=0.3)
    for result in sensitivity_results:
        closure = result['closure']
        axes_14[1].plot(closure['mass_block_id'], closure['mean_target_minus_final_cm_km2'], marker='o', linewidth=1.2, label=result['name'])
    axes_14[1].axhline(0.0, linewidth=0.8)
    axes_14[1].set_ylim(-1.1 * post_limit, 1.1 * post_limit)
    axes_14[1].set_title('b) After correction: all weighting schemes')
    axes_14[1].set_xlabel('Block ID')
    axes_14[1].set_ylabel('Temporal-mean mass residual, cm × km²')
    axes_14[1].grid(True, alpha=0.3)
    axes_14[1].legend(fontsize=9)
    axes_14[1].ticklabel_format(axis='y', style='sci', scilimits=(-2, 2))
    figure_14.savefig(MANUSCRIPT_DIR / 'fig_14_temporal_mean_mass_conservation_per_block.jpg', dpi=EXPORT_DPI, bbox_inches='tight', pad_inches=0.02)
    if DISPLAY_FIGURES:
        plt.show()
    plt.close(figure_14)
    del sensitivity_shap_X_transformed
    del sensitivity_shap_values
    del sensitivity_shap_X
    del fine_transformed
    del fine
    del parent
    gc.collect()
    table_4_sensitivity_summary = pd.DataFrame(sensitivity_summary_rows)
    figure_14_mass_closure_table = pd.concat(sensitivity_closure_rows, ignore_index=True)
    print('\nTable 4 mass-weight sensitivity summary across all available months')
    display(table_4_sensitivity_summary.round(6))
    print('\nAll-feature SHAP shares used in the composite weight')
    display(figure_13_shap_feature_weights.round(6))
    table_4_sensitivity_summary.to_csv(MANUSCRIPT_DIR / 'table_4_mass_weight_sensitivity_summary.csv', index=False)
    figure_14_mass_closure_table.to_csv(DIAGNOSTIC_DIR / 'diagnostic_mass_closure_by_month_block_scheme.csv', index=False)
if 'downscaling_model' in globals():
    del downscaling_model
gc.collect()


## Figure 15 — Mean GRACE TWS and downscaling stages

In [ ]:
if RUN_PUBLICATION_FIGURES:
    def plot_downscaling_maps_2x2_mean(
        gdf_025_pred,
        gdf_01_pred,
        date_col="date",
        target_col=TARGET_COL,
        cmap="viridis",
        figsize=(10, 8),
        show_mass_blocks=True,
        title_fs=13,
        label_fs=14,
        block_label_fs=8,
        panel_label_xy=(0.01, 0.99),         # keep INSIDE axes
        cbar_rect=(0.92, 0.14, 0.02, 0.72),  # [left, bottom, width, height] in figure coords
        cbar_label="Mean GRACE TWS, cm",
        cbar_label_fs=13,
        cbar_tick_fs=11,
    ):
        """
        2x2 methods-style figure using MEANS across time:
          a) mean 0.25° GRACE target
          b) mean 0.25° ML prediction
          c) mean 0.1° ML prediction
          d) mean 0.1° mass-conserving result

        Notes:
        - Aggregates across all dates (mean per cell geometry).
        - Mass-block boundaries/centroids computed in projected CRS then transformed back.
        - Colorbar placed via fig.add_axes (stable).
        - Panel labels kept inside axes (safe bbox_inches='tight').
        """

        # --- GeoDataFrames ---
        g025 = gpd.GeoDataFrame(
            gdf_025_pred.copy(),
            geometry="geometry_0.25",
            crs=getattr(gdf_025_pred, "crs", None),
        )
        g01 = gpd.GeoDataFrame(
            gdf_01_pred.copy(),
            geometry="geometry_01",
            crs=getattr(gdf_01_pred, "crs", None),
        )

        # ensure datetime (not strictly required for mean, but keeps things consistent)
        if date_col in g025.columns:
            g025[date_col] = pd.to_datetime(g025[date_col], errors="coerce")
        if date_col in g01.columns:
            g01[date_col] = pd.to_datetime(g01[date_col], errors="coerce")

        # --- Aggregate to MEAN per cell ---
        # Prefer stable unique cell ids if they exist; fall back to geometry grouping.

        # 0.25°
        group_025 = "cell_025" if "cell_025" in g025.columns else "geometry_0.25"
        g025_mean = (
            g025.groupby(group_025, dropna=False, as_index=False)[[target_col, "y_pred_raw"]]
            .mean(numeric_only=True)
        )
        # attach geometry back
        if group_025 == "cell_025":
            geom_025 = g025[[group_025, "geometry_0.25"]].drop_duplicates(group_025)
            g025_mean = g025_mean.merge(geom_025, on=group_025, how="left")
        else:
            g025_mean["geometry_0.25"] = g025_mean[group_025]
        g025_mean = gpd.GeoDataFrame(g025_mean, geometry="geometry_0.25", crs=g025.crs)

        # 0.1°
        group_01 = "cell_01" if "cell_01" in g01.columns else "geometry_01"
        cols_01 = [c for c in ["y_pred_raw", "y_pred_final"] if c in g01.columns]
        if len(cols_01) < 2:
            missing = {"y_pred_raw", "y_pred_final"} - set(cols_01)
            raise ValueError(f"Missing required columns in gdf_01_pred: {missing}")

        g01_mean = (
            g01.groupby(group_01, dropna=False, as_index=False)[cols_01]
            .mean(numeric_only=True)
        )
        # attach geometry back
        if group_01 == "cell_01":
            geom_01 = g01[[group_01, "geometry_01"]].drop_duplicates(group_01)
            g01_mean = g01_mean.merge(geom_01, on=group_01, how="left")
        else:
            g01_mean["geometry_01"] = g01_mean[group_01]
        g01_mean = gpd.GeoDataFrame(g01_mean, geometry="geometry_01", crs=g01.crs)

        # --- Mass blocks (from unique 0.25° cells, not date-filtered) ---
        mass_blocks_gdf = None
        centroid_xy = None

        if show_mass_blocks and "mass_block_id" in g025.columns:
            block_cells = (
                g025[["mass_block_id", "cell_025", "geometry_0.25"]]
                .dropna(subset=["mass_block_id"])
                .drop_duplicates("cell_025") if "cell_025" in g025.columns
                else g025[["mass_block_id", "geometry_0.25"]].dropna(subset=["mass_block_id"]).drop_duplicates()
            )
            block_cells = gpd.GeoDataFrame(block_cells, geometry="geometry_0.25", crs=g025.crs)

            try:
                proj_crs = block_cells.estimate_utm_crs()
            except Exception:
                proj_crs = "EPSG:6933"

            block_cells_proj = block_cells.to_crs(proj_crs)
            mass_blocks_proj = block_cells_proj.dissolve(by="mass_block_id", as_index=False)

            cent_proj = mass_blocks_proj.geometry.centroid
            cent_plot = gpd.GeoSeries(cent_proj, crs=proj_crs).to_crs(g025.crs)

            mass_blocks_gdf = mass_blocks_proj.to_crs(g025.crs)

            centroid_xy = {
                int(bid): (pt.x, pt.y)
                for bid, pt in zip(mass_blocks_gdf["mass_block_id"].astype(int).values, cent_plot.values)
            }

        # --- Shared color scale across the 4 panels (MEANS) ---
        values = np.concatenate([
            g025_mean[target_col].dropna().to_numpy(),
            g025_mean["y_pred_raw"].dropna().to_numpy(),
            g01_mean["y_pred_raw"].dropna().to_numpy(),
            g01_mean["y_pred_final"].dropna().to_numpy(),
        ])
        if values.size == 0:
            raise ValueError("No finite values found to build color scale.")
        vmin, vmax = float(np.nanmin(values)), float(np.nanmax(values))
        norm = Normalize(vmin=vmin, vmax=vmax)

        panels = [
            (g025_mean, target_col, "Mean 0.25° GRACE TWS"),
            (g025_mean, "y_pred_raw", "Mean 0.25° ML prediction"),
            (g01_mean, "y_pred_raw", "Mean 0.1° ML prediction"),
            (g01_mean, "y_pred_final", "Mean corrected 0.1° TWS product"),
        ]
        labels = ["a)", "b)", "c)", "d)"]

        # --- Figure ---
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.flatten()

        for ax, (df, col, title), lab in zip(axes, panels, labels):
            df.plot(column=col, ax=ax, cmap=cmap, norm=norm, legend=False)
            ax.set_title(title, fontsize=title_fs)
            ax.set_axis_off()

            ax.text(
                panel_label_xy[0], panel_label_xy[1], lab,
                transform=ax.transAxes,
                fontsize=label_fs,
                fontweight="bold",
                va="top",
                ha="left",
                bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1.0),
            )

            if mass_blocks_gdf is not None:
                mass_blocks_gdf.boundary.plot(ax=ax, color="black", linewidth=0.7)
                if centroid_xy is not None:
                    for bid, (x, y) in centroid_xy.items():
                        ax.text(
                            x, y, str(bid),
                            fontsize=block_label_fs,
                            ha="center",
                            va="center",
                            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1.5),
                        )

        fig.subplots_adjust(left=0.04, right=0.88, wspace=0.02, hspace=0.08)

        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cax = fig.add_axes(list(cbar_rect))
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label(cbar_label, fontsize=cbar_label_fs, rotation=270, labelpad=18)
        cbar.ax.tick_params(labelsize=cbar_tick_fs)

        return fig, axes


    # -------------------- RUN + SAVE --------------------
    fig, axes = plot_downscaling_maps_2x2_mean(
        gdf_025_pred,
        gdf_01_pred,
        target_col=TARGET_COL,
        show_mass_blocks=True,
        cbar_rect=(0.85, 0.14, 0.02, 0.72),
        panel_label_xy=(-0.1, 0.99),
    )

    fig.savefig(
        MANUSCRIPT_DIR / 'fig_15_downscaling_maps_mean.jpg',
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.02,
    )

    if DISPLAY_FIGURES:
        plt.show()

    plt.close("all")


## Figure 16 — Mean residual patterns

In [ ]:
if RUN_PUBLICATION_FIGURES:
    def plot_mean_residual_maps_2x2(
        gdf_025_pred,
        gdf_01_pred,
        target_col=TARGET_COL,
        cmap="RdBu_r",
        figsize=(10, 8),
        show_mass_blocks=True,
        title_fs=13,
        label_fs=14,
        block_label_fs=8,
        panel_label_xy=(0.01, 0.99),
        cbar_rect=(0.92, 0.14, 0.02, 0.72),
        cbar_label="Mean residual, cm (prediction − target)",
        cbar_label_fs=13,
        cbar_tick_fs=11,
    ):
        """
        Mean residual maps (mean over time).

        Residual = prediction − target

        Panels (auto-adapting):
          a) 0.25° raw − target
          b) 0.25° (0.1° raw → 0.25°) − target   [ONLY if cell_025 exists in 0.1°]
          c) 0.1° raw − target
          d) 0.1° final − target
        """

        # GeoDataFrames
        g025 = gpd.GeoDataFrame(
            gdf_025_pred.copy(),
            geometry="geometry_0.25",
            crs=getattr(gdf_025_pred, "crs", None),
        )
        g01 = gpd.GeoDataFrame(
            gdf_01_pred.copy(),
            geometry="geometry_01",
            crs=getattr(gdf_01_pred, "crs", None),
        )

        # Mean residual at 0.25°
        group_025 = "cell_025" if "cell_025" in g025.columns else "geometry_0.25"

        g025_mean = (
            g025.groupby(group_025, as_index=False)[[target_col, "y_pred_raw"]]
            .mean(numeric_only=True)
        )

        if group_025 == "cell_025":
            geom_025 = g025[[group_025, "geometry_0.25"]].drop_duplicates(group_025)
            g025_mean = g025_mean.merge(geom_025, on=group_025, how="left")
        else:
            g025_mean["geometry_0.25"] = g025_mean[group_025]

        g025_mean = gpd.GeoDataFrame(
            g025_mean, geometry="geometry_0.25", crs=g025.crs
        )

        g025_mean["res_025_raw"] = g025_mean["y_pred_raw"] - g025_mean[target_col]

        # Mean target per 0.25° (for joining to 0.1°)
        if "cell_025" in g025.columns:
            target_025 = (
                g025.groupby("cell_025", as_index=False)[target_col]
                .mean(numeric_only=True)
            )
        else:
            raise ValueError("To compute 0.1° residuals, gdf_025_pred must contain 'cell_025'.")

        # Join target to 0.1°
        if target_col not in g01.columns:
            if "cell_025" not in g01.columns:
                raise ValueError(
                    "gdf_01_pred must contain either the target column "
                    f"'{target_col}' or 'cell_025' to join the target."
                )
            g01 = g01.merge(target_025, on="cell_025", how="left")

        # Mean residuals at 0.1°
        group_01 = "cell_01" if "cell_01" in g01.columns else "geometry_01"

        g01_mean = (
            g01.groupby(group_01, as_index=False)[
                [target_col, "y_pred_raw", "y_pred_final"]
            ]
            .mean(numeric_only=True)
        )

        if group_01 == "cell_01":
            geom_01 = g01[[group_01, "geometry_01"]].drop_duplicates(group_01)
            g01_mean = g01_mean.merge(geom_01, on=group_01, how="left")
        else:
            g01_mean["geometry_01"] = g01_mean[group_01]

        g01_mean = gpd.GeoDataFrame(
            g01_mean, geometry="geometry_01", crs=g01.crs
        )

        g01_mean["res_01_raw"] = g01_mean["y_pred_raw"] - g01_mean[target_col]
        g01_mean["res_01_final"] = g01_mean["y_pred_final"] - g01_mean[target_col]

        # Optional panel (b): 0.1° raw → 0.25°
        has_panel_b = "cell_025" in g01.columns and "cell_025" in g025_mean.columns

        if has_panel_b:
            agg01_to_025 = (
                g01.groupby("cell_025", as_index=False)[["y_pred_raw"]]
                .mean(numeric_only=True)
                .rename(columns={"y_pred_raw": "y_pred_raw_01_to_025"})
            )
            g025_b = g025_mean.merge(agg01_to_025, on="cell_025", how="left")
            g025_b["res_025_from01_raw"] = (
                g025_b["y_pred_raw_01_to_025"] - g025_b[target_col]
            )

        # Shared symmetric color scale
        vals = [
            g025_mean["res_025_raw"],
            g01_mean["res_01_raw"],
            g01_mean["res_01_final"],
        ]
        if has_panel_b:
            vals.insert(1, g025_b["res_025_from01_raw"])

        vmax = float(np.nanmax(np.abs(np.concatenate([v.dropna().to_numpy() for v in vals]))))
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

        # Panels (dynamic)
        panels = [
            (g025_mean, "res_025_raw", "Mean residual 0.25° (ML)"),
        ]

        if has_panel_b:
            panels.append(
                (g025_b, "res_025_from01_raw", "Mean residual 0.25° (0.1° raw → 0.25° − target)")
            )

        panels.extend([
            (g01_mean, "res_01_raw", "Mean residual 0.1° (ML)"),
            (g01_mean, "res_01_final", "Mean residual 0.1° (after correction)"),
        ])

        labels = ["a)", "b)", "c)", "d)"][: len(panels)]

        # Figure
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.flatten()

        for ax, (df, col, title), lab in zip(axes, panels, labels):
            df.plot(column=col, ax=ax, cmap=cmap, norm=norm, legend=False)
            ax.set_title(title, fontsize=title_fs)
            ax.set_axis_off()

            ax.text(
                panel_label_xy[0], panel_label_xy[1], lab,
                transform=ax.transAxes,
                fontsize=label_fs,
                fontweight="bold",
                va="top",
                ha="left",
                bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1.0),
            )

        for ax in axes[len(panels):]:
            ax.set_axis_off()

        fig.subplots_adjust(left=0.04, right=0.88, wspace=0.02, hspace=0.08)

        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cax = fig.add_axes(list(cbar_rect))
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label(cbar_label, fontsize=cbar_label_fs, rotation=270, labelpad=18)
        cbar.ax.tick_params(labelsize=cbar_tick_fs)

        return fig, axes

    # -------------------- RUN + SAVE --------------------
    fig, axes = plot_mean_residual_maps_2x2(
        gdf_025_pred,
        gdf_01_pred,
        target_col=TARGET_COL,
        show_mass_blocks=True,
        cbar_rect=(0.85, 0.14, 0.02, 0.72),
        panel_label_xy=(-0.1, 0.99),
    )

    fig.savefig(
        MANUSCRIPT_DIR / 'fig_16_downscaling_mean_residuals.jpg',
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        pad_inches=0.02,
    )

    if DISPLAY_FIGURES:
        plt.show()

    plt.close("all")


## Figure 17 — ML and ARIMA GRACE reconstruction

In [ ]:
if RUN_PUBLICATION_FIGURES:
    def build_plot_df_domain(panel):
        columns = [
            'mid_month_date',
            'grace_or_imputed',
            'grace_arima',
            'imputed_flag',
            'grace_or_imputed_std',
        ]
        df = panel[
            [column for column in columns if column in panel.columns]
        ].copy()

        df['mid_month_date'] = pd.to_datetime(
            df['mid_month_date'],
            errors='coerce',
        )
        df['observed_part'] = np.where(
            df['imputed_flag'] == False,
            df['grace_or_imputed'],
            np.nan,
        )
        df['grace_or_imputed_std'] = pd.to_numeric(
            df.get(
                'grace_or_imputed_std',
                pd.Series(np.nan, index=df.index),
            ),
            errors='coerce',
        ).fillna(0.0)
        return df


    def plot_domain_timeseries(
        panel,
        test_start_date,
        test_end_date,
        font_size=12,
        save_path=MANUSCRIPT_DIR / 'fig_17_imputation_vs_arima.jpg',
        dpi=EXPORT_DPI,
    ):
        plt.rcParams.update({
            'font.size': font_size,
            'axes.titlesize': font_size + 4,
            'axes.labelsize': font_size + 2,
            'legend.fontsize': font_size,
            'xtick.labelsize': font_size + 1,
            'ytick.labelsize': font_size + 1,
        })

        arima_start = pd.Timestamp('2013-01-01')
        arima_end = pd.Timestamp('2023-12-31')

        df = build_plot_df_domain(panel)
        df['grace_arima_plot'] = df['grace_arima'].where(
            df['mid_month_date'].between(
                arima_start,
                arima_end,
            )
        )

        domain_rows = []
        for date, group in df.groupby(
            'mid_month_date',
            observed=True,
        ):
            n_cells = len(group)
            imputed_mean = group['grace_or_imputed'].mean()
            model_se = (
                np.sqrt(
                    np.nansum(
                        np.square(
                            group['grace_or_imputed_std']
                        )
                    )
                )
                / n_cells
                if n_cells
                else np.nan
            )
            domain_rows.append({
                'date': date,
                'observed': group['observed_part'].mean(),
                'imputed': imputed_mean,
                'imputed_lower': (
                    imputed_mean
                    - UNCERTAINTY_Z * model_se
                ),
                'imputed_upper': (
                    imputed_mean
                    + UNCERTAINTY_Z * model_se
                ),
                'arima': group['grace_arima_plot'].mean(),
            })

        domain = (
            pd.DataFrame(domain_rows)
            .sort_values('date')
        )

        fig, ax = plt.subplots(figsize=(11, 5))

        ax.fill_between(
            domain['date'],
            domain['imputed_lower'],
            domain['imputed_upper'],
            alpha=0.18,
            label='ML model uncertainty',
        )
        ax.plot(
            domain['date'],
            domain['imputed'],
            linestyle='--',
            linewidth=1.5,
            color='red',
            label='ML Imputed TWS',
        )
        ax.plot(
            domain['date'],
            domain['observed'],
            linestyle='-',
            linewidth=1.5,
            color='tab:green',
            label='Observed TWS',
        )
        ax.plot(
            domain['date'],
            domain['arima'],
            linestyle='-',
            linewidth=1.5,
            color='tab:blue',
            label='ARIMA TWS',
        )

        ax.axvspan(
            test_start_date,
            test_end_date,
            facecolor='none',
            edgecolor='gray',
            hatch='///',
            linewidth=0.0,
            alpha=0.4,
            label='Test Set Holdout',
        )

        ax.set_ylabel('GRACE TWS, cm')
        ax.set_xlabel('Date')
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(ncol=2)

        plt.tight_layout()
        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches='tight',
        )
        if DISPLAY_FIGURES:
            plt.show()
        plt.close(fig)


    arima_raw = load_grace_cached(
        RAW_LOCAL['grace_filled_sheet'],
        'grace_fl',
    )
    arima_gdf = add_grid_polygons(
        arima_raw,
        res=0.25,
    )
    arima_gdf['date_key'] = (
        pd.to_datetime(
            arima_gdf['date'],
            errors='coerce',
        )
        .dt.strftime('%Y-%m')
    )
    arima_gdf = (
        arima_gdf[
            ['geometry', 'date_key', 'grace_fl']
        ]
        .rename(columns={'grace_fl': 'grace_arima'})
        .drop_duplicates(['geometry', 'date_key'])
    )

    timeseries_gdf = panel_gdf.copy()
    timeseries_gdf['date_key'] = (
        pd.to_datetime(
            timeseries_gdf['mid_month_date'],
            errors='coerce',
        )
        .dt.strftime('%Y-%m')
    )
    timeseries_gdf = timeseries_gdf.merge(
        arima_gdf,
        on=['geometry', 'date_key'],
        how='left',
    )

    plot_domain_timeseries(
        timeseries_gdf,
        test_start_date,
        test_end_date,
        font_size=14,
        save_path=(
            OUTPUT_DIR
            / 'fig_17_imputation_vs_arima.jpg'
        ),
        dpi=EXPORT_DPI,
    )

    del arima_raw, arima_gdf, timeseries_gdf
    plt.close('all')
    gc.collect()


## Figure 18 — Water-balance comparison

Basin-scale monthly ΔTWS, uncertainty-aware agreement, and cell-level spatial bias are evaluated against P − ET − R.

In [ ]:
# Figure 18: water-balance comparison
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
FIGURE_TITLE = None
FS_SUPTITLE = 25
FS_TITLE = 24
FS_LABEL = 22
FS_TICK = 20
FS_LEGEND = 19
FS_ANNOT = 19
FS_PANEL = 24
FS_CBAR_LABEL = 18
FS_CBAR_TICK = 17
if RUN_PUBLICATION_FIGURES:
    fig = plt.figure(figsize=(20, 13), constrained_layout=True)
    outer = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.72, 1.28], hspace=0.02)
    ax_ts = fig.add_subplot(outer[0, 0])
    bottom = outer[1].subgridspec(nrows=1, ncols=2, width_ratios=[1.02, 1.0], wspace=0.08)
    ax_sc = fig.add_subplot(bottom[0, 0])
    ax_map = fig.add_subplot(bottom[0, 1])
    if FIGURE_TITLE:
        fig.suptitle(FIGURE_TITLE, fontsize=FS_SUPTITLE, fontweight='bold', y=1.01)
    ax_ts.plot(coarse_water_balance_domain_table['date'], coarse_water_balance_domain_table['storage_dTWS_cm'], linewidth=2.0, label='GRACE 0.25° ΔTWS')
    ax_ts.plot(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['storage_dTWS_cm'], linewidth=2.0, label='Final downscaled 0.1° ΔTWS')
    ax_ts.plot(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['water_balance_dTWS_cm'], linewidth=1.8, label='P − ET − R')
    ax_ts.axhline(0, linewidth=1.0)
    ax_ts.set_title('Monthly ΔTWS Time Series', fontsize=FS_TITLE, pad=10)
    ax_ts.set_xlabel('Date', fontsize=FS_LABEL)
    ax_ts.set_ylabel('Monthly ΔTWS, cm', fontsize=FS_LABEL)
    ax_ts.tick_params(axis='both', labelsize=FS_TICK)
    ax_ts.grid(True, alpha=0.3)
    ax_ts.legend(ncol=3, fontsize=FS_LEGEND, frameon=True, loc='upper right')
    ax_ts.text(-0.045, 1.02, 'a)', transform=ax_ts.transAxes, fontsize=FS_PANEL, fontweight='bold', ha='left', va='bottom')
    x_values = downscaled_water_balance_domain_table['storage_dTWS_cm'].to_numpy(dtype=float)
    y_values = downscaled_water_balance_domain_table['water_balance_dTWS_cm'].to_numpy(dtype=float)
    x_err = UNCERTAINTY_Z * downscaled_water_balance_domain_table['storage_model_se_cm'].to_numpy(dtype=float)
    y_err = UNCERTAINTY_Z * downscaled_water_balance_domain_table['water_balance_spatial_se_cm'].to_numpy(dtype=float)
    valid = np.isfinite(x_values) & np.isfinite(y_values) & np.isfinite(x_err) & np.isfinite(y_err)
    x_values = x_values[valid]
    y_values = y_values[valid]
    x_err = np.maximum(x_err[valid], 0)
    y_err = np.maximum(y_err[valid], 0)
    ax_sc.errorbar(x_values, y_values, xerr=x_err, yerr=y_err, fmt='o', markersize=6.0, alpha=0.5, linewidth=0.7, elinewidth=0.7, capsize=0)
    all_limits = np.concatenate([x_values - x_err, x_values + x_err, y_values - y_err, y_values + y_err])
    lower_bound = float(np.nanmin(all_limits))
    upper_bound = float(np.nanmax(all_limits))
    value_range = upper_bound - lower_bound
    if not np.isfinite(value_range) or value_range <= 0:
        value_range = 1.0
    padding = 0.05 * value_range
    bounds = (lower_bound - padding, upper_bound + padding)
    ax_sc.plot(bounds, bounds, linestyle='--', linewidth=1.5)
    ax_sc.set_xlim(bounds)
    ax_sc.set_ylim(bounds)
    ax_sc.set_aspect('equal', adjustable='box')
    ax_sc.set_title('Monthly Agreement at 0.1°', fontsize=FS_TITLE, pad=10)
    ax_sc.set_xlabel('Present-product monthly ΔTWS, cm', fontsize=FS_LABEL)
    ax_sc.set_ylabel('P − ET − R, cm', fontsize=FS_LABEL)
    ax_sc.tick_params(axis='both', labelsize=FS_TICK)
    ax_sc.grid(True, alpha=0.3)
    metric_row = downscaled_water_balance_metrics.iloc[0]
    annotation = f"r = {metric_row['pearson_r']:.2f}\nMAE = {metric_row['MAE_cm']:.2f} cm\n95% CI: {metric_row['MAE_95_CI_lower_cm']:.2f}–{metric_row['MAE_95_CI_upper_cm']:.2f} cm"
    ax_sc.text(0.04, 0.96, annotation, transform=ax_sc.transAxes, va='top', ha='left', fontsize=FS_ANNOT, bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'})
    ax_sc.text(-0.1, 1.02, 'b)', transform=ax_sc.transAxes, fontsize=FS_PANEL, fontweight='bold', ha='left', va='bottom')
    bias_values = water_balance_cell_metrics_gdf['bias_cm'].dropna().to_numpy(dtype=float)
    bias_limit = np.nanquantile(np.abs(bias_values), 0.98)
    if not np.isfinite(bias_limit) or bias_limit <= 0:
        bias_limit = 1.0
    bias_norm = Normalize(vmin=-bias_limit, vmax=bias_limit)
    water_balance_cell_metrics_gdf.plot(column='bias_cm', ax=ax_map, cmap='RdBu_r', norm=bias_norm, edgecolor='none', legend=False, missing_kwds={'color': 'lightgrey'})
    ax_map.set_title('Spatial Bias', fontsize=FS_TITLE, pad=10)
    ax_map.set_axis_off()
    ax_map.set_anchor('C')
    ax_map.text(-0.06, 1.02, 'c)', transform=ax_map.transAxes, fontsize=FS_PANEL, fontweight='bold', ha='left', va='bottom')
    scalar_mappable = ScalarMappable(norm=bias_norm, cmap='RdBu_r')
    scalar_mappable.set_array([])
    cax = inset_axes(ax_map, width='5%', height='88%', loc='center right', borderpad=1.1)
    colorbar = fig.colorbar(scalar_mappable, cax=cax, orientation='vertical')
    colorbar.set_label('Bias, cm (P − ET − R minus downscaled ΔTWS)', fontsize=FS_CBAR_LABEL, labelpad=12)
    colorbar.ax.tick_params(labelsize=FS_CBAR_TICK, width=1.0, length=4)
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_18_water_balance_validation.jpeg', dpi=600, facecolor='white')


## Figure 19 — GRACE-SeDA and WGHM comparison

Basin-mean re-centred TWS series and matched scatter comparisons are calculated for both reference products.

In [ ]:
# Figure 19: GRACE-SeDA and WGHM comparison
def complete_month_index(table, date_col='date'):
    """Insert missing monthly timestamps so gaps remain visible in line plots."""
    table = table.copy()
    table[date_col] = pd.to_datetime(table[date_col], errors='coerce')
    table = table.dropna(subset=[date_col]).sort_values(date_col).drop_duplicates(date_col)
    if table.empty:
        return table
    months = pd.date_range(table[date_col].min().to_period('M').to_timestamp(), table[date_col].max().to_period('M').to_timestamp(), freq='MS')
    return table.set_index(date_col).reindex(months).rename_axis(date_col).reset_index()
FIGURE_TITLE = None
FS_SUPTITLE = 27
FS_TITLE = 24
FS_LABEL = 22
FS_TICK = 20
FS_LEGEND = 19
FS_ANNOTATION = 19
FS_PANEL = 24
PRODUCTS = ['GRACE-SeDA', 'WGHM']
if RUN_PUBLICATION_FIGURES:
    fig = plt.figure(figsize=(16, 18), constrained_layout=True)
    grid = fig.add_gridspec(nrows=3, ncols=2, height_ratios=[1.0, 1.0, 1.25])
    time_axes = [fig.add_subplot(grid[0, :]), fig.add_subplot(grid[1, :])]
    correlation_axes = [fig.add_subplot(grid[2, 0]), fig.add_subplot(grid[2, 1])]
    if FIGURE_TITLE:
        fig.suptitle(FIGURE_TITLE, fontsize=FS_SUPTITLE, fontweight='bold')
    for axis, product, panel_label in zip(time_axes, PRODUCTS, ['a)', 'b)']):
        table = complete_month_index(external_pairwise_domain_table.loc[external_pairwise_domain_table['external_product'].eq(product)].copy())
        dates = pd.to_datetime(table['date']).to_numpy()
        axis.plot(dates, table['our_centered_cm'], linewidth=1.8, label='Present downscaled product')
        axis.plot(dates, table['external_centered_cm'], linewidth=1.6, label=product)
        axis.axhline(0, linewidth=0.9)
        axis.set_title(f'Present Product versus {product}', fontsize=FS_TITLE, pad=10)
        axis.set_ylabel('Re-centred TWS anomaly, cm', fontsize=FS_LABEL)
        axis.tick_params(axis='both', labelsize=FS_TICK)
        axis.grid(True, alpha=0.3)
        axis.legend(ncol=2, fontsize=FS_LEGEND, loc='upper right', frameon=True)
        add_panel_label(axis, panel_label, x=-0.045, y=1.02, fontsize=FS_PANEL)
    time_axes[-1].set_xlabel('Date', fontsize=FS_LABEL)
    for axis, product, panel_label in zip(correlation_axes, PRODUCTS, ['c)', 'd)']):
        required_columns = ['external_centered_cm', 'our_centered_cm', 'external_spatial_se_cm', 'our_model_se_cm']
        table = external_pairwise_domain_table.loc[external_pairwise_domain_table['external_product'].eq(product)].dropna(subset=['external_centered_cm', 'our_centered_cm']).copy()
        metrics = external_pairwise_metrics.loc[external_pairwise_metrics['external_product'].eq(product) & external_pairwise_metrics['comparison_scale'].eq('domain monthly mean')].iloc[0]
        x_values = table['external_centered_cm'].to_numpy(dtype=float)
        y_values = table['our_centered_cm'].to_numpy(dtype=float)
        x_uncertainty = UNCERTAINTY_Z * table['external_spatial_se_cm'].fillna(0).to_numpy(dtype=float)
        y_uncertainty = UNCERTAINTY_Z * table['our_model_se_cm'].fillna(0).to_numpy(dtype=float)
        x_uncertainty = np.maximum(x_uncertainty, 0)
        y_uncertainty = np.maximum(y_uncertainty, 0)
        axis.errorbar(x_values, y_values, xerr=x_uncertainty, yerr=y_uncertainty, fmt='o', markersize=4.5, alpha=0.55, linewidth=0.6, elinewidth=0.6, capsize=1.5, zorder=2)
        values_with_uncertainty = np.concatenate([x_values - x_uncertainty, x_values + x_uncertainty, y_values - y_uncertainty, y_values + y_uncertainty])
        lower_bound = np.nanmin(values_with_uncertainty)
        upper_bound = np.nanmax(values_with_uncertainty)
        value_range = upper_bound - lower_bound
        if not np.isfinite(value_range) or value_range <= 0:
            value_range = 1.0
        padding = 0.04 * value_range
        bounds = (lower_bound - padding, upper_bound + padding)
        axis.plot(bounds, bounds, '--', linewidth=1.3, zorder=1)
        axis.set_xlim(bounds)
        axis.set_ylim(bounds)
        axis.set_aspect('equal', adjustable='box')
        axis.set_title(product, fontsize=FS_TITLE, pad=10)
        axis.set_xlabel(f'{product} re-centred TWS, cm', fontsize=FS_LABEL)
        axis.set_ylabel('Present-product re-centred TWS, cm', fontsize=FS_LABEL)
        axis.tick_params(axis='both', labelsize=FS_TICK)
        axis.grid(True, alpha=0.3)
        metrics_text = f"r = {metrics['pearson_r']:.2f}\n95% CI: {metrics['pearson_r_95_CI_lower']:.2f}–{metrics['pearson_r_95_CI_upper']:.2f}\nMAE = {metrics['mae_cm']:.2f} cm\nRMSE = {metrics['rmse_cm']:.2f} cm"
        axis.text(0.04, 0.96, metrics_text, transform=axis.transAxes, va='top', ha='left', fontsize=FS_ANNOTATION, bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'}, zorder=3)
        axis.text(0.96, 0.04, 'Error bars: 95% uncertainty', transform=axis.transAxes, va='bottom', ha='right', fontsize=FS_ANNOTATION - 1)
        add_panel_label(axis, panel_label, x=-0.11, y=1.02, fontsize=FS_PANEL)
    save_show_close(fig, MANUSCRIPT_DIR / 'fig_19_external_tws_benchmark_comparison.jpeg', dpi=600, facecolor='white')


## Figure 20 — Spatial benchmark comparison

Re-centred TWS maps are compared for June 2017, November 2018, and November 2019.

In [ ]:
# Figure 20: spatial benchmark comparison
FONT_FAMILY = 'Liberation Serif'
FS_COLUMN_TITLE = 23
FS_DATE = 25
FS_PANEL = 25
FS_CBAR_LABEL = 27
FS_CBAR_TICKS = 25
FIGURE_WIDTH = 20
ROW_HEIGHT = 5.4
PANEL_X = -0.04
PANEL_Y = 1.02
COLORBAR_FRACTION = 0.02
COLORBAR_PAD = 0.018
plt.rcParams.update({'font.family': FONT_FAMILY, 'font.size': 23, 'axes.titlesize': FS_COLUMN_TITLE, 'axes.labelsize': 21, 'xtick.labelsize': 21, 'ytick.labelsize': 21})
raw_geometry = panel_gdf.geometry.name
raw_grace_025 = panel_gdf[['mid_month_date', 'cell_id', 'grace_or_obs', raw_geometry]].copy().rename_geometry('geometry_025').rename(columns={'mid_month_date': 'raw_epoch_date', 'grace_or_obs': 'raw_grace_cm'})
raw_grace_025['raw_epoch_date'] = pd.to_datetime(raw_grace_025['raw_epoch_date'], errors='coerce')
raw_grace_025['date'] = to_month_start(raw_grace_025['raw_epoch_date'])
raw_grace_025 = raw_grace_025.loc[raw_grace_025['date'].between(pd.Timestamp('2013-01-01'), pd.Timestamp('2019-12-01')) & raw_grace_025['raw_grace_cm'].notna()].drop_duplicates(['date', 'cell_id'])
raw_grace_025 = as_geodataframe(raw_grace_025, 'geometry_025', panel_gdf.crs)
raw_grace_025['raw_grace_cm_centered'] = raw_grace_025['raw_grace_cm'] - raw_grace_025.groupby('cell_id', observed=True)['raw_grace_cm'].transform('mean')
figure_20_dates = pd.DatetimeIndex(pd.to_datetime(['2017-06-01', '2018-11-01', '2019-11-01']))
available_fine_dates = pd.DatetimeIndex(external_three_product_common['date'].dropna().unique())
available_raw_dates = pd.DatetimeIndex(raw_grace_025['date'].dropna().unique())
missing_dates = figure_20_dates.difference(available_fine_dates.intersection(available_raw_dates))
if len(missing_dates) > 0:
    raise ValueError('Figure 20 dates are unavailable in all four products: ' + ', '.join((date.strftime('%Y-%m') for date in missing_dates)))
fine_maps = as_geodataframe(external_three_product_common.loc[external_three_product_common['date'].isin(figure_20_dates)], 'geometry_01', getattr(gdf_01_pred, 'crs', None))
raw_maps = as_geodataframe(raw_grace_025.loc[raw_grace_025['date'].isin(figure_20_dates)], 'geometry_025', raw_grace_025.crs)
columns = [('raw_grace_cm_centered', 'Raw GRACE 0.25°'), ('our_tws_cm_centered', 'Present downscaled 0.1°'), ('seda_tws_cm_centered', 'GRACE-SeDA 0.1°'), ('wghm_tws_cm_centered', 'WGHM 0.1°')]
map_values = np.concatenate([raw_maps['raw_grace_cm_centered'].dropna().to_numpy(), *[fine_maps[column].dropna().to_numpy() for column, _ in columns[1:]]])
vmin, vmax = np.nanquantile(map_values, [0.02, 0.98])
norm = Normalize(vmin=vmin, vmax=vmax)
if RUN_PUBLICATION_FIGURES:
    number_of_rows = len(figure_20_dates)
    fig, axes = plt.subplots(number_of_rows, 4, figsize=(FIGURE_WIDTH, ROW_HEIGHT * number_of_rows), constrained_layout=True, squeeze=False)
    panel_index = 0
    for row, date in enumerate(figure_20_dates):
        row_raw = raw_maps.loc[raw_maps['date'].eq(date)]
        row_fine = fine_maps.loc[fine_maps['date'].eq(date)]
        row_raw.plot(column=columns[0][0], ax=axes[row, 0], cmap='viridis', norm=norm, edgecolor='none')
        for column_index, (column, _) in enumerate(columns[1:], start=1):
            row_fine.plot(column=column, ax=axes[row, column_index], cmap='viridis', norm=norm, edgecolor='none')
        for column_index, (_, title) in enumerate(columns):
            axis = axes[row, column_index]
            axis.set_axis_off()
            if row == 0:
                axis.set_title(title, fontsize=FS_COLUMN_TITLE, fontweight='normal', pad=14)
            if column_index == 0:
                axis.text(-0.1, 0.5, f'{date:%Y-%m}', transform=axis.transAxes, rotation=90, va='center', ha='right', fontsize=FS_DATE, fontweight='bold', clip_on=False)
            panel_label = f'{chr(97 + panel_index)})'
            axis.text(PANEL_X, PANEL_Y, panel_label, transform=axis.transAxes, va='bottom', ha='left', fontsize=FS_PANEL, fontweight='bold', clip_on=False)
            panel_index += 1
    scalar_mappable = ScalarMappable(norm=norm, cmap='viridis')
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(scalar_mappable, ax=axes, fraction=COLORBAR_FRACTION, pad=COLORBAR_PAD)
    colorbar.set_label('Re-centred TWS anomaly, cm', fontsize=FS_CBAR_LABEL, labelpad=14)
    colorbar.ax.tick_params(labelsize=FS_CBAR_TICKS, width=1.0, length=5)
    output_path = MANUSCRIPT_DIR / 'fig_20_raw_GRACE_and_model_comparison.jpeg'
    fig.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    if DISPLAY_FIGURES:
        plt.show()
    plt.close(fig)


# Supplementary material

Static supplementary figures, tables, videos, and optional diagnostic outputs are generated below.

## Supplementary Figure S1 — Spatial hydroclimatic statistics

In [ ]:
# Supplementary Figure S1. Spatial distribution of hydroclimatic statistics
if RUN_PUBLICATION_FIGURES:
    source = publication_feature_025.copy()
    source['date'] = pd.to_datetime(source['date'], errors='coerce')
    source = source.loc[source['date'].between('2013-01-01', '2023-12-31')].copy()

    # Hydrological variables are stored and plotted in centimetres.
    variables = [
        ('ET', 'ET', 'cm'),
        ('P', 'P', 'cm'),
        ('Q', 'R', 'cm'),
        ('temp', 'T', '°C'),
        ('SM', 'SM', 'cm'),
        (TARGET_COL, 'GRACE TWS', 'cm'),
    ]

    statistic_specs = [
        ('minimum', 'Temporal minimum', 'coolwarm'),
        ('maximum', 'Temporal maximum', 'coolwarm'),
        ('std', 'Temporal standard deviation', 'viridis'),
        ('amplitude', 'Seasonal amplitude', 'plasma'),
    ]

    geometry = (
        source.groupby('cell_025_key', observed=True)['geometry_0.25']
        .first()
    )

    statistic_maps = {}

    for column, _, _ in variables:
        grouped = source.groupby('cell_025_key', observed=True)[column]

        monthly = (
            source.assign(month=source['date'].dt.month)
            .groupby(['cell_025_key', 'month'], observed=True)[column]
            .mean()
        )

        monthly_by_cell = monthly.groupby('cell_025_key', observed=True)

        statistics = pd.DataFrame({
            'minimum': grouped.min(),
            'maximum': grouped.max(),
            'std': grouped.std(ddof=0),
            'amplitude': monthly_by_cell.max() - monthly_by_cell.min(),
            'geometry_0.25': geometry,
        }).reset_index()

        statistic_maps[column] = as_geodataframe(
            statistics,
            'geometry_0.25',
            getattr(publication_feature_025, 'crs', None),
        )

    def robust_limits(values, lower_q=0.02, upper_q=0.98):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]

        if values.size == 0:
            return 0.0, 1.0

        lower, upper = np.nanquantile(values, [lower_q, upper_q])

        if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
            lower, upper = np.nanmin(values), np.nanmax(values)

        if lower >= upper:
            upper = lower + 1.0

        return float(lower), float(upper)

    fig, axes = plt.subplots(
        6,
        4,
        figsize=(16, 17),
        constrained_layout=True,
    )

    for row, (column, row_label, unit) in enumerate(variables):
        row_data = statistic_maps[column]

        # Common scale for temporal minimum and maximum.
        min_max_values = np.concatenate([
            row_data['minimum'].dropna().to_numpy(dtype=float),
            row_data['maximum'].dropna().to_numpy(dtype=float),
        ])
        min_max_limits = robust_limits(min_max_values)

        # Independent scales for variability statistics.
        statistic_limits = {
            'minimum': min_max_limits,
            'maximum': min_max_limits,
            'std': robust_limits(row_data['std']),
            'amplitude': robust_limits(row_data['amplitude']),
        }

        for col, (statistic, title, cmap) in enumerate(statistic_specs):
            axis = axes[row, col]
            lower, upper = statistic_limits[statistic]

            row_data.plot(
                column=statistic,
                ax=axis,
                cmap=cmap,
                vmin=lower,
                vmax=upper,
                edgecolor='0.75',
                linewidth=0.25,
                legend=True,
                legend_kwds={
                    'label': unit,
                    'shrink': 0.78,
                    'extend': 'both',
                    'format': '%.2f',
                },
            )

            if row == 0:
                axis.set_title(
                    title,
                    fontsize=13,
                    fontweight='bold',
                )

            if col == 0:
                axis.set_ylabel(
                    f'{row_label}\n({unit})',
                    fontsize=11,
                    fontweight='bold',
                )

            axis.set_xticks([])
            axis.set_yticks([])

    fig.suptitle(
        'Spatial distribution of hydroclimatic statistics '
        '(2013-01–2023-12)',
        fontsize=15,
        fontweight='bold',
    )

    save_show_close(
        fig,
        SUPPLEMENT_DIR / 'fig_S1_hydroclimatic_spatial_statistics.jpg',
    )


## Supplementary Table S1 and Figure S2 — GRACE gaps and completed record

In [ ]:
# Supplementary Table S1 and Figure S2
required_columns = ['cell_id', 'mid_month_date', 'grace_or_obs', 'grace_or_imputed']
missing_columns = [column for column in required_columns if column not in panel_gdf.columns]
if missing_columns:
    raise KeyError(f'The completed GRACE panel is missing required columns: {missing_columns}')
panel_supplement = pd.DataFrame(panel_gdf.loc[:, required_columns]).copy()
panel_supplement['mid_month_date'] = pd.to_datetime(panel_supplement['mid_month_date'], errors='coerce')
panel_supplement = panel_supplement.dropna(subset=['cell_id', 'mid_month_date'])
supplementary_monthly_table = panel_supplement.groupby('mid_month_date', as_index=False, observed=True).agg(total_cells=('cell_id', 'nunique'), observed_cells=('grace_or_obs', lambda values: int(values.notna().sum())), missing_cells=('grace_or_obs', lambda values: int(values.isna().sum())), domain_observed_TWS_cm=('grace_or_obs', 'mean'), domain_completed_TWS_cm=('grace_or_imputed', 'mean')).rename(columns={'mid_month_date': 'date'}).sort_values('date').reset_index(drop=True)
supplementary_monthly_table['observed_coverage_percent'] = 100.0 * supplementary_monthly_table['observed_cells'] / supplementary_monthly_table['total_cells']
supplementary_monthly_table['gap_type'] = np.select([supplementary_monthly_table['missing_cells'].eq(0), supplementary_monthly_table['observed_cells'].eq(0)], ['Fully observed month', 'Complete original gap'], default='Partial original gap')
supplementary_original_gap_months = supplementary_monthly_table.loc[supplementary_monthly_table['missing_cells'].gt(0), ['date', 'gap_type', 'total_cells', 'observed_cells', 'missing_cells', 'observed_coverage_percent']].copy().reset_index(drop=True)
print('\nSupplementary Table S1. Original GRACE months containing missing observations')
display(supplementary_original_gap_months.round(2))
supplementary_original_gap_months.to_csv(SUPPLEMENT_DIR / 'table_S1_original_GRACE_gap_months.csv', index=False)

def shade_original_gap_months(axis, gap_months, include_labels=False):
    complete_label_used = False
    partial_label_used = False
    for row in gap_months.itertuples(index=False):
        start = pd.Timestamp(row.date) - pd.Timedelta(days=14)
        end = pd.Timestamp(row.date) + pd.Timedelta(days=14)
        if row.gap_type == 'Complete original gap':
            label = 'Complete original GRACE gap' if include_labels and (not complete_label_used) else None
            axis.axvspan(start, end, facecolor='0.72', edgecolor='none', alpha=0.4, label=label)
            complete_label_used = True
        else:
            label = 'Partial original GRACE gap' if include_labels and (not partial_label_used) else None
            axis.axvspan(start, end, facecolor='0.90', edgecolor='none', alpha=0.55, label=label)
            partial_label_used = True
if RUN_PUBLICATION_FIGURES:
    figure_s2_data = supplementary_monthly_table.sort_values('date').reset_index(drop=True)
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, constrained_layout=True)
    shade_original_gap_months(axes[0], supplementary_original_gap_months, include_labels=True)
    shade_original_gap_months(axes[1], supplementary_original_gap_months, include_labels=False)
    axes[0].plot(figure_s2_data['date'], figure_s2_data['observed_coverage_percent'], linewidth=1.5, label='Observed GRACE spatial coverage')
    axes[0].set_ylim(-3, 103)
    axes[0].set_ylabel('Spatial coverage, %')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=3, loc='lower left')
    axes[0].text(-0.06, 1.02, 'a)', transform=axes[0].transAxes, fontsize=16, fontweight='bold')
    axes[1].plot(figure_s2_data['date'], figure_s2_data['domain_completed_TWS_cm'], linestyle='--', linewidth=1.5, label='Completed observed–imputed TWS')
    axes[1].plot(figure_s2_data['date'], figure_s2_data['domain_observed_TWS_cm'], linewidth=1.5, label='Observed GRACE TWS')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Domain-mean GRACE TWS, cm')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(ncol=2, loc='best')
    axes[1].text(-0.06, 1.02, 'b)', transform=axes[1].transAxes, fontsize=16, fontweight='bold')
    date_locator = mdates.AutoDateLocator(minticks=8, maxticks=14)
    axes[1].xaxis.set_major_locator(date_locator)
    axes[1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(date_locator))
    save_show_close(fig, SUPPLEMENT_DIR / 'fig_S2_GRACE_missingness_and_completed_timeseries.jpeg')
del panel_supplement
gc.collect()


## Supplementary Figure S3 and Table S2 — Representative 0.25° time series

In [ ]:
GRID_RESOLUTION = 0.25
FIGURE_S3_PATH = SUPPLEMENT_DIR / 'fig_S3_representative_025_observed_vs_predicted_timeseries.jpeg'
TABLE_S2_PATH = SUPPLEMENT_DIR / 'table_S2_representative_025_locations.csv'
FIGURE_S3_DATA_PATH = SUPPLEMENT_DIR / 'figure_S3_representative_025_timeseries_data.csv'
FIGURE_WIDTH = 19
FIGURE_HEIGHT = 15
FONT_TITLE = 20
FONT_AXIS_LABEL = 18
FONT_TICK = 15
FONT_LEGEND = 13
FONT_METRICS = 14
FONT_MAP_LABEL = 17
LINEWIDTH_PREDICTED = 2.4
LINEWIDTH_OBSERVED = 2.1
require_columns(
    gdf_025_pred,
    ['date', 'y_pred_raw', 'geometry_0.25'],
    'gdf_025_pred',
)
require_columns(
    panel_gdf,
    ['mid_month_date', 'lon0', 'lat0', 'grace_or_obs'],
    'panel_gdf',
)

def snap_to_025_grid(values, resolution=GRID_RESOLUTION, decimals=5):
    """
    Snap numeric grid coordinates to the nearest 0.25° increment.
    """
    numeric_values = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    snapped_values = np.round(numeric_values / resolution) * resolution
    return np.round(snapped_values, decimals)

def parse_cell_025_key(key_series):
    """
    Parse keys such as:

        21.25_52.25
        21.25000_52.25000

    into numeric longitude and latitude values.
    """
    key_parts = key_series.astype('string').str.strip().str.split('_', n=1, expand=True)
    if key_parts.shape[1] != 2:
        return (pd.Series(np.nan, index=key_series.index, dtype=float), pd.Series(np.nan, index=key_series.index, dtype=float))
    longitude = pd.to_numeric(key_parts.iloc[:, 0], errors='coerce')
    latitude = pd.to_numeric(key_parts.iloc[:, 1], errors='coerce')
    return (longitude, latitude)

def format_cell_025_key(longitude, latitude):
    """
    Construct a consistently formatted 0.25° grid-cell key.
    """
    if pd.isna(longitude) or pd.isna(latitude):
        return pd.NA
    return f'{float(longitude):.5f}_{float(latitude):.5f}'

def add_canonical_025_coordinates(frame, geometry_column):
    """
    Add canonical numeric 0.25° coordinates and a stable cell key.

    Coordinate sources are used in this order:

    1. lon0 and lat0;
    2. existing cell_025_key;
    3. lower-left polygon bounds in EPSG:4326.
    """
    result = frame.copy()
    longitude = pd.Series(np.nan, index=result.index, dtype=float)
    latitude = pd.Series(np.nan, index=result.index, dtype=float)
    if 'lon0' in result.columns and 'lat0' in result.columns:
        longitude = pd.to_numeric(result['lon0'], errors='coerce')
        latitude = pd.to_numeric(result['lat0'], errors='coerce')
    if 'cell_025_key' in result.columns:
        key_longitude, key_latitude = parse_cell_025_key(result['cell_025_key'])
        longitude = longitude.fillna(key_longitude)
        latitude = latitude.fillna(key_latitude)
    missing_coordinates = longitude.isna() | latitude.isna()
    if missing_coordinates.any():
        spatial_frame = gpd.GeoDataFrame(result.copy(), geometry=geometry_column, crs=getattr(frame, 'crs', None))
        if spatial_frame.crs is None:
            raise ValueError('Grid coordinates cannot be derived because the 0.25° geometry has no CRS.')
        if spatial_frame.crs.is_geographic:
            spatial_wgs84 = spatial_frame
        else:
            spatial_wgs84 = spatial_frame.to_crs('EPSG:4326')
        polygon_bounds = spatial_wgs84.geometry.bounds
        longitude = longitude.fillna(polygon_bounds['minx'])
        latitude = latitude.fillna(polygon_bounds['miny'])
    result['grid_lon0'] = snap_to_025_grid(longitude)
    result['grid_lat0'] = snap_to_025_grid(latitude)
    result['cell_025_key'] = [format_cell_025_key(longitude_value, latitude_value) for longitude_value, latitude_value in zip(result['grid_lon0'], result['grid_lat0'])]
    return result

def create_complete_monthly_series(location_series, global_start_date, global_end_date):
    """
    Reindex a location to a complete monthly sequence.

    Missing observations remain NaN. Matplotlib therefore breaks the
    corresponding line instead of joining values across missing months.
    """
    location_series = location_series.sort_values('date').drop_duplicates(subset=['date'], keep='first').copy()
    complete_dates = pd.date_range(start=global_start_date, end=global_end_date, freq='MS')
    retained_values = {}
    for column in ['location_label', 'cell_025_key', 'grid_lon0', 'grid_lat0']:
        if column in location_series.columns:
            available_values = location_series[column].dropna()
            retained_values[column] = available_values.iloc[0] if len(available_values) else pd.NA
    location_series = location_series.set_index('date').reindex(complete_dates).rename_axis('date').reset_index()
    for column, value in retained_values.items():
        location_series[column] = location_series[column].fillna(value)
    return location_series
original_grace_025 = panel_gdf[['mid_month_date', 'lon0', 'lat0', 'grace_or_obs']].copy()
original_grace_025['date'] = to_month_start(original_grace_025['mid_month_date'])
original_grace_025['grid_lon0'] = snap_to_025_grid(original_grace_025['lon0'])
original_grace_025['grid_lat0'] = snap_to_025_grid(original_grace_025['lat0'])
original_grace_025['cell_025_key'] = [format_cell_025_key(longitude, latitude) for longitude, latitude in zip(original_grace_025['grid_lon0'], original_grace_025['grid_lat0'])]
original_grace_025 = original_grace_025.dropna(subset=['date', 'grid_lon0', 'grid_lat0', 'cell_025_key']).sort_values(['grid_lon0', 'grid_lat0', 'date']).drop_duplicates(subset=['date', 'grid_lon0', 'grid_lat0'], keep='first')[['date', 'grid_lon0', 'grid_lat0', 'cell_025_key', 'grace_or_obs']].rename(columns={'grace_or_obs': 'observed_original_GRACE_cm'})
prediction_columns = [column for column in ['date', 'lon0', 'lat0', 'cell_025_key', 'y_pred_raw', 'geometry_0.25'] if column in gdf_025_pred.columns]
predicted_grace_025 = gdf_025_pred[prediction_columns].copy()
predicted_grace_025 = gpd.GeoDataFrame(predicted_grace_025, geometry='geometry_0.25', crs=gdf_025_pred.crs)
predicted_grace_025['date'] = to_month_start(predicted_grace_025['date'])
predicted_grace_025 = add_canonical_025_coordinates(predicted_grace_025, geometry_column='geometry_0.25')
predicted_grace_025 = predicted_grace_025.dropna(subset=['date', 'grid_lon0', 'grid_lat0', 'cell_025_key', 'geometry_0.25']).sort_values(['grid_lon0', 'grid_lat0', 'date']).drop_duplicates(subset=['date', 'grid_lon0', 'grid_lat0'], keep='first')
original_cell_coordinates = set(zip(original_grace_025['grid_lon0'], original_grace_025['grid_lat0']))
predicted_cell_coordinates = set(zip(predicted_grace_025['grid_lon0'], predicted_grace_025['grid_lat0']))
common_cell_coordinates = original_cell_coordinates & predicted_cell_coordinates
original_dates = pd.DatetimeIndex(original_grace_025['date'].dropna().unique())
prediction_dates = pd.DatetimeIndex(predicted_grace_025['date'].dropna().unique())
common_dates = original_dates.intersection(prediction_dates)
print('Original GRACE 0.25° grid cells:', len(original_cell_coordinates))
print('Predicted 0.25° grid cells:', len(predicted_cell_coordinates))
print('Spatially overlapping 0.25° grid cells:', len(common_cell_coordinates))
print('Temporally overlapping months:', len(common_dates))
if len(common_cell_coordinates) < 5:
    raise ValueError(f'Fewer than five spatially overlapping 0.25° grid cells were found after matching canonical numeric grid coordinates. Original cells: {len(original_cell_coordinates)}; predicted cells: {len(predicted_cell_coordinates)}; common cells: {len(common_cell_coordinates)}.')
if len(common_dates) == 0:
    raise ValueError('The original GRACE observations and raw predictions have no temporally overlapping months.')
representative_series_025 = predicted_grace_025.merge(original_grace_025[['date', 'grid_lon0', 'grid_lat0', 'observed_original_GRACE_cm']], on=['date', 'grid_lon0', 'grid_lat0'], how='left', validate='one_to_one')
representative_series_025 = gpd.GeoDataFrame(representative_series_025, geometry='geometry_0.25', crs=predicted_grace_025.crs)
representative_series_025['paired_value'] = representative_series_025['y_pred_raw'].notna() & representative_series_025['observed_original_GRACE_cm'].notna()
cell_availability = representative_series_025.groupby(['grid_lon0', 'grid_lat0', 'cell_025_key'], as_index=False, observed=True).agg(n_predicted_months=('y_pred_raw', 'count'), n_observed_months=('observed_original_GRACE_cm', 'count'), n_paired_months=('paired_value', 'sum'))
cell_geometry = predicted_grace_025[['grid_lon0', 'grid_lat0', 'cell_025_key', 'geometry_0.25']].drop_duplicates(subset=['grid_lon0', 'grid_lat0'], keep='first').copy()
candidate_cells = cell_geometry.merge(cell_availability, on=['grid_lon0', 'grid_lat0', 'cell_025_key'], how='inner', validate='one_to_one')
candidate_cells = gpd.GeoDataFrame(candidate_cells, geometry='geometry_0.25', crs=predicted_grace_025.crs)
candidate_cells = candidate_cells.loc[candidate_cells['n_paired_months'].gt(0)].copy()
print('0.25° cells with paired observed and predicted values:', len(candidate_cells))
if len(candidate_cells) < 5:
    raise ValueError(f'Only {len(candidate_cells)} spatially matched 0.25° grid cells contain paired original GRACE observations and raw model predictions.')
if candidate_cells.crs is None:
    raise ValueError('The 0.25° candidate-cell dataset has no CRS.')
if candidate_cells.crs.is_geographic:
    selection_crs = candidate_cells.estimate_utm_crs() or 'EPSG:3035'
else:
    selection_crs = candidate_cells.crs
candidate_cells_projected = candidate_cells.to_crs(selection_crs)
candidate_centroids_projected = candidate_cells_projected.geometry.centroid
candidate_cells_projected['centroid_x'] = candidate_centroids_projected.x
candidate_cells_projected['centroid_y'] = candidate_centroids_projected.y
centroid_coordinates = candidate_cells_projected[['centroid_x', 'centroid_y']].to_numpy(dtype=float)
location_selector = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=20)
candidate_cells_projected['selection_cluster'] = location_selector.fit_predict(centroid_coordinates)
selected_indices = []
for cluster_id in range(5):
    cluster_table = candidate_cells_projected.loc[candidate_cells_projected['selection_cluster'].eq(cluster_id)].copy()
    cluster_center = location_selector.cluster_centers_[cluster_id]
    cluster_table['distance_to_cluster_center'] = np.sqrt((cluster_table['centroid_x'] - cluster_center[0]) ** 2 + (cluster_table['centroid_y'] - cluster_center[1]) ** 2)
    selected_index = cluster_table.sort_values(['distance_to_cluster_center', 'n_paired_months', 'cell_025_key'], ascending=[True, False, True]).index[0]
    selected_indices.append(selected_index)
selected_cells_projected = candidate_cells_projected.loc[selected_indices].copy()
selected_cells_projected = selected_cells_projected.sort_values(['centroid_y', 'centroid_x'], ascending=[False, True]).reset_index(drop=True)
selected_cells_projected['location_label'] = [str(location_number) for location_number in range(1, 6)]
selected_centroid_points_projected = gpd.GeoDataFrame(selected_cells_projected[['location_label', 'cell_025_key']].copy(), geometry=selected_cells_projected.geometry.centroid, crs=selection_crs)
selected_cells = selected_cells_projected.to_crs(candidate_cells.crs)
selected_centroid_points_plot = selected_centroid_points_projected.to_crs(candidate_cells.crs)
selected_centroid_points_wgs84 = selected_centroid_points_projected.to_crs('EPSG:4326')
selected_cells['longitude'] = selected_centroid_points_wgs84.geometry.x.to_numpy(dtype=float)
selected_cells['latitude'] = selected_centroid_points_wgs84.geometry.y.to_numpy(dtype=float)
global_start_date = predicted_grace_025['date'].min()
global_end_date = predicted_grace_025['date'].max()
location_metric_rows = []
selected_series_parts = []
for location in selected_cells.itertuples(index=False):
    location_series = representative_series_025.loc[representative_series_025['cell_025_key'].eq(location.cell_025_key)].sort_values('date').copy()
    location_series['location_label'] = location.location_label
    location_series = create_complete_monthly_series(location_series, global_start_date=global_start_date, global_end_date=global_end_date)
    paired_rows = location_series.dropna(subset=['observed_original_GRACE_cm', 'y_pred_raw'])
    if len(paired_rows) >= 2 and paired_rows['observed_original_GRACE_cm'].nunique() > 1:
        location_r2 = r2_score(paired_rows['observed_original_GRACE_cm'], paired_rows['y_pred_raw'])
    else:
        location_r2 = np.nan
    if len(paired_rows):
        location_mae = mean_absolute_error(paired_rows['observed_original_GRACE_cm'], paired_rows['y_pred_raw'])
    else:
        location_mae = np.nan
    selected_series_parts.append(location_series)
    location_metric_rows.append({'location_label': location.location_label, 'cell_025_key': location.cell_025_key, 'grid_lon0': location.grid_lon0, 'grid_lat0': location.grid_lat0, 'longitude': location.longitude, 'latitude': location.latitude, 'n_predicted_months': int(location.n_predicted_months), 'n_observed_months': int(location.n_observed_months), 'n_paired_months': int(len(paired_rows)), 'r2': location_r2, 'mae_cm': location_mae})
representative_location_metrics = pd.DataFrame(location_metric_rows)
representative_location_series = pd.concat(selected_series_parts, ignore_index=True)
representative_location_metrics.to_csv(TABLE_S2_PATH, index=False)
representative_location_series[['location_label', 'cell_025_key', 'grid_lon0', 'grid_lat0', 'date', 'observed_original_GRACE_cm', 'y_pred_raw']].to_csv(FIGURE_S3_DATA_PATH, index=False)
print('\nSupplementary Table S2. Representative 0.25° grid cells used in Figure S3')
display(representative_location_metrics.round({'grid_lon0': 3, 'grid_lat0': 3, 'longitude': 3, 'latitude': 3, 'r2': 3, 'mae_cm': 3}))
if RUN_PUBLICATION_FIGURES:
    fig = plt.figure(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT), constrained_layout=True)
    grid_spec = fig.add_gridspec(nrows=3, ncols=2, height_ratios=[1.0, 1.0, 1.0])
    map_axis = fig.add_subplot(grid_spec[0, 0])
    series_axes = [fig.add_subplot(grid_spec[0, 1]), fig.add_subplot(grid_spec[1, 0]), fig.add_subplot(grid_spec[1, 1]), fig.add_subplot(grid_spec[2, 0]), fig.add_subplot(grid_spec[2, 1])]
    candidate_cells.boundary.plot(ax=map_axis, color='0.72', linewidth=0.65)
    selected_cells.plot(ax=map_axis, facecolor='none', edgecolor='black', linewidth=2.0)
    for location, point in zip(selected_cells.itertuples(index=False), selected_centroid_points_plot.geometry):
        map_axis.text(point.x, point.y, location.location_label, ha='center', va='center', fontsize=FONT_MAP_LABEL, fontweight='bold', bbox={'facecolor': 'white', 'edgecolor': 'black', 'boxstyle': 'circle,pad=0.25', 'alpha': 0.95})
    map_axis.set_axis_off()
    selected_values = np.concatenate([representative_location_series['observed_original_GRACE_cm'].dropna().to_numpy(dtype=float), representative_location_series['y_pred_raw'].dropna().to_numpy(dtype=float)])
    if selected_values.size == 0:
        raise ValueError('No finite TWS values are available for Figure S3.')
    y_min = float(np.nanmin(selected_values))
    y_max = float(np.nanmax(selected_values))
    y_range = y_max - y_min
    if not np.isfinite(y_range) or y_range <= 0:
        y_range = 1.0
    y_padding = 0.07 * y_range
    common_y_limits = (y_min - y_padding, y_max + y_padding)
    for axis, location in zip(series_axes, selected_cells.itertuples(index=False)):
        location_series = representative_location_series.loc[representative_location_series['location_label'].eq(location.location_label)].sort_values('date').copy()
        axis.plot(location_series['date'], location_series['y_pred_raw'], linewidth=LINEWIDTH_PREDICTED, label='Raw 0.25° ML prediction', zorder=2)
        axis.plot(location_series['date'], location_series['observed_original_GRACE_cm'], linewidth=LINEWIDTH_OBSERVED, label='Original observed GRACE', zorder=3)
        location_metrics = representative_location_metrics.loc[representative_location_metrics['location_label'].eq(location.location_label)].iloc[0]
        if np.isfinite(location_metrics['r2']):
            r2_text = f"{location_metrics['r2']:.2f}"
        else:
            r2_text = 'NA'
        if np.isfinite(location_metrics['mae_cm']):
            mae_text = f"{location_metrics['mae_cm']:.2f}"
        else:
            mae_text = 'NA'
        metric_text = f"R² = {r2_text}\nMAE = {mae_text} cm\nn = {int(location_metrics['n_paired_months'])}"
        axis.text(0.02, 0.96, metric_text, transform=axis.transAxes, ha='left', va='top', fontsize=FONT_METRICS, bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.86})
        axis.set_title(f'Location {location.location_label}: {location.latitude:.2f}°N, {location.longitude:.2f}°E', fontsize=FONT_TITLE, pad=11)
        axis.set_ylim(common_y_limits)
        axis.set_ylabel('TWS, cm', fontsize=FONT_AXIS_LABEL)
        axis.grid(True, alpha=0.3)
        axis.tick_params(axis='both', labelsize=FONT_TICK)
        date_locator = mdates.AutoDateLocator(minticks=5, maxticks=9)
        axis.xaxis.set_major_locator(date_locator)
        axis.xaxis.set_major_formatter(mdates.ConciseDateFormatter(date_locator))
        axis.legend(loc='upper right', fontsize=FONT_LEGEND, frameon=True, framealpha=0.92)
    series_axes[-2].set_xlabel('Date', fontsize=FONT_AXIS_LABEL)
    series_axes[-1].set_xlabel('Date', fontsize=FONT_AXIS_LABEL)
    save_show_close(fig, FIGURE_S3_PATH)


## Supplementary Videos S1–S2

In [ ]:
if RUN_SUPPLEMENTAL_VIDEOS:
    import imageio.v2 as imageio
    OVERWRITE_VIDEOS = False
    VIDEO_OUTPUT_FPS = max(float(VIDEO_FPS) / 2.0, 0.1)
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)

    def figure_rgb(fig):
        fig.canvas.draw()
        return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

    def dynamic_frame_norm(series_list, quantiles=(0.01, 0.99)):
        """
        Return one robust colour normalization shared by all
        panels within an individual video frame.
        """
        arrays = [pd.to_numeric(series, errors='coerce').to_numpy(dtype=float) for series in series_list if series is not None and len(series)]
        if not arrays:
            return Normalize(vmin=0.0, vmax=1.0)
        values = np.concatenate(arrays)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return Normalize(vmin=0.0, vmax=1.0)
        vmin, vmax = np.nanquantile(values, quantiles)
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            vmin = np.nanmin(values)
            vmax = np.nanmax(values)
        if vmin >= vmax:
            padding = max(abs(float(vmin)) * 0.05, 0.1)
            vmin -= padding
            vmax += padding
        return Normalize(vmin=float(vmin), vmax=float(vmax))

    def padded_extent(*frames, padding=0.02):
        """
        Return common padded map limits for several GeoDataFrames.
        """
        valid_frames = [frame for frame in frames if frame is not None and (not frame.empty)]
        if not valid_frames:
            raise ValueError('No non-empty spatial datasets were supplied.')
        bounds = np.vstack([frame.total_bounds for frame in valid_frames])
        minx, miny = bounds[:, :2].min(axis=0)
        maxx, maxy = bounds[:, 2:].max(axis=0)
        dx = max(maxx - minx, 1e-09)
        dy = max(maxy - miny, 1e-09)
        return ((minx - padding * dx, maxx + padding * dx), (miny - padding * dy, maxy + padding * dy))

    def write_mp4(path, dates, render_frame, fps=VIDEO_OUTPUT_FPS, overwrite=OVERWRITE_VIDEOS):
        path = Path(path)
        if path.exists() and (not overwrite):
            print(f'Video already exists and was not regenerated: {path}')
            return
        dates = pd.DatetimeIndex(dates).dropna().unique().sort_values()
        if dates.empty:
            raise ValueError(f'No dates are available for {path.name}.')
        writer = imageio.get_writer(str(path), fps=fps, codec='libx264', quality=8, pixelformat='yuv420p', macro_block_size=16)
        try:
            for frame_number, date in enumerate(dates, start=1):
                date = pd.Timestamp(date)
                fig = render_frame(date)
                if fig is None:
                    raise RuntimeError(f'Frame rendering returned None for {date:%Y-%m}.')
                writer.append_data(figure_rgb(fig))
                plt.close(fig)
                if frame_number == 1 or frame_number % 12 == 0 or frame_number == len(dates):
                    print(f'{path.name}: {frame_number}/{len(dates)} frames')
        finally:
            writer.close()
        duration = len(dates) / fps
        print(f'Video saved: {path}\nFrames: {len(dates)} | FPS: {fps:.2f} | Duration: {duration:.1f} s')
    video_s1_path = VIDEO_DIR / 'video_S1_monthly_downscaling_sequence.mp4'
    if OVERWRITE_VIDEOS or not video_s1_path.exists():
        require_columns(gdf_025_pred, ['date', 'cell_025', 'mass_block_id', TARGET_COL, 'y_pred_raw', 'geometry_0.25'], 'gdf_025_pred')
        require_columns(gdf_01_pred, ['date', 'cell_01', 'mass_block_id', 'y_pred_raw', 'y_pred_final', 'geometry_01'], 'gdf_01_pred')
        video_coarse = as_geodataframe(gdf_025_pred.copy(), 'geometry_0.25')
        video_fine = as_geodataframe(gdf_01_pred.copy(), 'geometry_01')
        if video_coarse.crs is not None and video_fine.crs is not None and (video_fine.crs != video_coarse.crs):
            video_fine = video_fine.to_crs(video_coarse.crs)
        for frame in [video_coarse, video_fine]:
            frame['date'] = to_month_start(frame['date'])
        video_s1_dates = pd.DatetimeIndex(video_coarse['date'].dropna().unique()).intersection(pd.DatetimeIndex(video_fine['date'].dropna().unique())).sort_values()
        coarse_grid = as_geodataframe(video_coarse[['cell_025', 'mass_block_id', 'geometry_0.25']].drop_duplicates('cell_025').copy(), 'geometry_0.25', video_coarse.crs)
        fine_grid = as_geodataframe(video_fine[['cell_01', 'mass_block_id', 'geometry_01']].drop_duplicates('cell_01').copy(), 'geometry_01', video_fine.crs)
        projected_crs = coarse_grid.estimate_utm_crs() or 'EPSG:6933'
        video_blocks = coarse_grid.to_crs(projected_crs).dissolve(by='mass_block_id', as_index=False).to_crs(coarse_grid.crs)
        xlim_s1, ylim_s1 = padded_extent(coarse_grid, fine_grid, padding=0.02)

        def render_video_s1(date):
            coarse_date = video_coarse.loc[video_coarse['date'].eq(date)].copy()
            fine_date = video_fine.loc[video_fine['date'].eq(date)].copy()
            if coarse_date.empty or fine_date.empty:
                raise ValueError(f'Missing Figure 15 video data for {date:%Y-%m}.')
            panels = [(coarse_date, TARGET_COL, 'GRACE TWS at 0.25°', coarse_grid), (coarse_date, 'y_pred_raw', 'ML prediction at 0.25°', coarse_grid), (fine_date, 'y_pred_raw', 'Raw ML prediction at 0.1°', fine_grid), (fine_date, 'y_pred_final', 'Mass-conserving TWS at 0.1°', fine_grid)]
            frame_norm = dynamic_frame_norm([frame[column] for frame, column, _, _ in panels])
            fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12.8, 7.2), dpi=VIDEO_DPI)
            fig.patch.set_facecolor('white')
            fig.subplots_adjust(left=0.025, right=0.885, bottom=0.035, top=0.895, wspace=0.025, hspace=0.13)
            for panel_index, (axis, panel) in enumerate(zip(axes.ravel(), panels)):
                frame, column, title, grid = panel
                frame.plot(column=column, ax=axis, cmap='viridis', norm=frame_norm, edgecolor='none', legend=False)
                grid.boundary.plot(ax=axis, color='0.45', linewidth=0.28 if panel_index < 2 else 0.1)
                if panel_index >= 2:
                    video_blocks.boundary.plot(ax=axis, color='black', linewidth=0.6)
                axis.set_xlim(xlim_s1)
                axis.set_ylim(ylim_s1)
                axis.set_title(title, fontsize=12, pad=4)
                axis.set_axis_off()
            fig.text(0.025, 0.965, f'{date:%Y-%m}', fontsize=14, fontweight='bold', ha='left', va='top')
            scalar_mappable = ScalarMappable(norm=frame_norm, cmap='viridis')
            scalar_mappable.set_array([])
            colorbar_axis = fig.add_axes([0.91, 0.16, 0.016, 0.68])
            colorbar = fig.colorbar(scalar_mappable, cax=colorbar_axis, extend='both')
            colorbar.set_label('GRACE TWS (cm)', rotation=270, labelpad=14, fontsize=11)
            colorbar.ax.tick_params(labelsize=9)
            return fig
        write_mp4(video_s1_path, video_s1_dates, render_video_s1)
        del (video_coarse, video_fine, coarse_grid, fine_grid, video_blocks)
        gc.collect()
    else:
        print(f'Video S1 already exists: {video_s1_path}')
    video_s2_path = VIDEO_DIR / 'video_S2_monthly_model_comparison.mp4'
    if OVERWRITE_VIDEOS or not video_s2_path.exists():
        require_columns(our_01, ['date', 'cell_key', 'geometry_01', 'our_tws_cm'], 'our_01')
        require_columns(grace_seda_01, ['date', 'cell_key', 'tws_cm'], 'grace_seda_01')
        require_columns(wghm_01, ['date', 'cell_key', 'tws_cm'], 'wghm_01')
        our_video_table = our_01[['date', 'cell_key', 'geometry_01', 'our_tws_cm']].copy()
        if 'our_tws_std_cm' in our_01.columns:
            our_video_table['our_tws_std_cm'] = our_01['our_tws_std_cm'].to_numpy()
        else:
            our_video_table['our_tws_std_cm'] = np.nan
        seda_video_table = grace_seda_01[['date', 'cell_key', 'tws_cm']].copy().rename(columns={'tws_cm': 'seda_tws_cm'})
        wghm_video_table = wghm_01[['date', 'cell_key', 'tws_cm']].copy().rename(columns={'tws_cm': 'wghm_tws_cm'})
        for frame in [our_video_table, seda_video_table, wghm_video_table]:
            frame['date'] = to_month_start(frame['date'])
            frame.dropna(subset=['date', 'cell_key'], inplace=True)
            frame.drop_duplicates(subset=['date', 'cell_key'], inplace=True)
        external_three_product_common = our_video_table.merge(seda_video_table, on=['date', 'cell_key'], how='inner', validate='one_to_one').merge(wghm_video_table, on=['date', 'cell_key'], how='inner', validate='one_to_one')
        external_three_product_common = external_three_product_common.loc[external_three_product_common['date'].between('2013-01-01', '2019-12-01')].copy()
        if external_three_product_common.empty:
            raise ValueError('No common 2013–2019 0.1° observations were found for the present product, GRACE-SeDA, and WGHM.')
        require_columns(panel_gdf, ['mid_month_date', 'cell_id', 'grace_or_obs', panel_gdf.geometry.name], 'panel_gdf')
        raw_geometry_column = panel_gdf.geometry.name
        raw_grace_025 = panel_gdf[['mid_month_date', 'cell_id', 'grace_or_obs', raw_geometry_column]].copy()
        raw_grace_025 = raw_grace_025.rename_geometry('geometry_025').rename(columns={'mid_month_date': 'raw_epoch_date', 'grace_or_obs': 'raw_grace_cm'})
        raw_grace_025['raw_epoch_date'] = pd.to_datetime(raw_grace_025['raw_epoch_date'], errors='coerce')
        raw_grace_025['date'] = to_month_start(raw_grace_025['raw_epoch_date'])
        raw_grace_025 = raw_grace_025.loc[raw_grace_025['date'].between('2013-01-01', '2019-12-01') & raw_grace_025['raw_grace_cm'].notna()].drop_duplicates(subset=['date', 'cell_id']).copy()
        raw_grace_025 = as_geodataframe(raw_grace_025, 'geometry_025', panel_gdf.crs)
        raw_coverage = raw_grace_025.groupby('date', observed=True)['cell_id'].nunique()
        if raw_coverage.empty:
            raise ValueError('No original GRACE observations are available for Video S2.')
        minimum_raw_cells = 0.9 * raw_coverage.max()
        raw_grace_dates = raw_coverage.loc[raw_coverage >= minimum_raw_cells].index
        all_product_map_dates = pd.DatetimeIndex(external_three_product_common['date'].dropna().unique()).intersection(pd.DatetimeIndex(raw_grace_dates)).sort_values()
        if all_product_map_dates.empty:
            raise ValueError('No common dates remain after aligning original GRACE, the present product, GRACE-SeDA, and WGHM.')
        external_four_product_common = external_three_product_common.loc[external_three_product_common['date'].isin(all_product_map_dates)].copy()
        raw_grace_four_product_common = raw_grace_025.loc[raw_grace_025['date'].isin(all_product_map_dates)].copy()
        for column in ['our_tws_cm', 'seda_tws_cm', 'wghm_tws_cm']:
            external_four_product_common[f'{column}_centered'] = external_four_product_common[column] - external_four_product_common.groupby('cell_key', observed=True)[column].transform('mean')
        raw_grace_four_product_common['raw_grace_cm_centered'] = raw_grace_four_product_common['raw_grace_cm'] - raw_grace_four_product_common.groupby('cell_id', observed=True)['raw_grace_cm'].transform('mean')
        fine_comparison = as_geodataframe(external_four_product_common, 'geometry_01', getattr(gdf_01_pred, 'crs', None))
        raw_comparison = as_geodataframe(raw_grace_four_product_common, 'geometry_025', raw_grace_025.crs)
        if fine_comparison.crs is not None and raw_comparison.crs is not None and (raw_comparison.crs != fine_comparison.crs):
            raw_comparison = raw_comparison.to_crs(fine_comparison.crs)
        for frame in [fine_comparison, raw_comparison]:
            frame['date'] = to_month_start(frame['date'])
        video_s2_dates = pd.DatetimeIndex(all_product_map_dates).intersection(pd.DatetimeIndex(raw_comparison['date'].dropna().unique())).intersection(pd.DatetimeIndex(fine_comparison['date'].dropna().unique())).sort_values()
        xlim_s2, ylim_s2 = padded_extent(raw_comparison, fine_comparison, padding=0.02)

        def render_video_s2(date):
            raw_date = raw_comparison.loc[raw_comparison['date'].eq(date)].copy()
            fine_date = fine_comparison.loc[fine_comparison['date'].eq(date)].copy()
            if raw_date.empty or fine_date.empty:
                raise ValueError(f'Missing Figure 20 comparison data for {date:%Y-%m}.')
            panels = [(raw_date, 'raw_grace_cm_centered', 'Raw GRACE (0.25°)'), (fine_date, 'our_tws_cm_centered', 'Present product (0.1°)'), (fine_date, 'seda_tws_cm_centered', 'GRACE-SeDA (0.1°)'), (fine_date, 'wghm_tws_cm_centered', 'WGHM (0.1°)')]
            frame_norm = dynamic_frame_norm([frame[column] for frame, column, _ in panels])
            fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12.8, 7.2), dpi=VIDEO_DPI)
            fig.patch.set_facecolor('white')
            fig.subplots_adjust(left=0.025, right=0.875, bottom=0.04, top=0.895, wspace=0.025, hspace=0.13)
            for axis, (frame, column, title) in zip(axes.ravel(), panels):
                frame.plot(column=column, ax=axis, cmap='viridis', norm=frame_norm, edgecolor='0.55', linewidth=0.1, legend=False)
                axis.set_xlim(xlim_s2)
                axis.set_ylim(ylim_s2)
                axis.set_title(title, fontsize=12, pad=4)
                axis.set_axis_off()
            fig.text(0.025, 0.965, f'{date:%Y-%m}', fontsize=14, fontweight='bold', ha='left', va='top')
            scalar_mappable = ScalarMappable(norm=frame_norm, cmap='viridis')
            scalar_mappable.set_array([])
            colorbar_axis = fig.add_axes([0.9, 0.16, 0.017, 0.68])
            colorbar = fig.colorbar(scalar_mappable, cax=colorbar_axis, extend='both')
            colorbar.set_label('Re-centred TWS anomaly (cm)', rotation=270, labelpad=13, fontsize=10)
            colorbar.ax.tick_params(labelsize=9)
            return fig
        write_mp4(video_s2_path, video_s2_dates, render_video_s2)
        del (our_video_table, seda_video_table, wghm_video_table, external_three_product_common, external_four_product_common, raw_grace_025, raw_grace_four_product_common, fine_comparison, raw_comparison)
        gc.collect()
    else:
        print(f'Video S2 already exists: {video_s2_path}')
else:
    print('Supplementary videos skipped. Set RUN_SUPPLEMENTAL_VIDEOS=True to generate them.')


## Optional diagnostics

In [ ]:
# Optional diagnostic figures
if RUN_DIAGNOSTIC_ANALYTICS:
    repeated = outer_test_results_df.sort_values('outer_test').copy()
    x = np.arange(len(repeated))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    r2_lower_error = np.maximum(repeated['holdout_r2'] - repeated['holdout_r2_ci_lower'], 0)
    r2_upper_error = np.maximum(repeated['holdout_r2_ci_upper'] - repeated['holdout_r2'], 0)
    axes[0].errorbar(x, repeated['holdout_r2'], yerr=np.vstack([r2_lower_error, r2_upper_error]), fmt='o', capsize=4)
    axes[0].axhline(repeated['holdout_r2'].mean(), linestyle='--', linewidth=1.2, label='Mean across splits')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(repeated['holdout_blocks'], rotation=45, ha='right')
    axes[0].set_xlabel('Held-out spatial blocks')
    axes[0].set_ylabel('R²')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    mae_lower_error = np.maximum(repeated['holdout_mae_cm'] - repeated['holdout_mae_ci_lower_cm'], 0)
    mae_upper_error = np.maximum(repeated['holdout_mae_ci_upper_cm'] - repeated['holdout_mae_cm'], 0)
    axes[1].errorbar(x, repeated['holdout_mae_cm'], yerr=np.vstack([mae_lower_error, mae_upper_error]), fmt='o', capsize=4)
    axes[1].axhline(repeated['holdout_mae_cm'].mean(), linestyle='--', linewidth=1.2, label='Mean across splits')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(repeated['holdout_blocks'], rotation=45, ha='right')
    axes[1].set_xlabel('Held-out spatial blocks')
    axes[1].set_ylabel('MAE, cm')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[0].text(-0.1, 1.02, 'a)', transform=axes[0].transAxes, fontsize=16, fontweight='bold')
    axes[1].text(-0.1, 1.02, 'b)', transform=axes[1].transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D1_repeated_outer_tests.jpeg')
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(coarse_water_balance_domain_table['date'], coarse_water_balance_domain_table['storage_lower_cm'], coarse_water_balance_domain_table['storage_upper_cm'], alpha=0.16, label='GRACE / imputation uncertainty')
    ax.plot(coarse_water_balance_domain_table['date'], coarse_water_balance_domain_table['storage_dTWS_cm'], linewidth=1.5, label='GRACE 0.25° ΔTWS')
    ax.fill_between(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['storage_lower_cm'], downscaled_water_balance_domain_table['storage_upper_cm'], alpha=0.15, label='Downscaled-model uncertainty')
    ax.plot(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['storage_dTWS_cm'], linewidth=1.4, label='Final downscaled 0.1° ΔTWS')
    ax.fill_between(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['water_balance_lower_cm'], downscaled_water_balance_domain_table['water_balance_upper_cm'], alpha=0.14, label='Water-balance spatial uncertainty')
    ax.plot(downscaled_water_balance_domain_table['date'], downscaled_water_balance_domain_table['water_balance_dTWS_cm'], linewidth=1.2, label='P − ET − R')
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel('Date')
    ax.set_ylabel('Monthly ΔTWS, cm')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D2_water_balance_dTWS_timeseries.jpeg')
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

    def draw_water_balance_scatter(axis, domain_table, metric_row, title, panel_label):
        x_values = domain_table['storage_dTWS_cm'].to_numpy()
        y_values = domain_table['water_balance_dTWS_cm'].to_numpy()
        axis.errorbar(x_values, y_values, xerr=UNCERTAINTY_Z * domain_table['storage_model_se_cm'].to_numpy(), yerr=UNCERTAINTY_Z * domain_table['water_balance_spatial_se_cm'].to_numpy(), fmt='o', markersize=4, alpha=0.5, linewidth=0.5)
        minimum = float(np.nanmin([x_values, y_values]))
        maximum = float(np.nanmax([x_values, y_values]))
        axis.plot([minimum, maximum], [minimum, maximum], linestyle='--', linewidth=1.2)
        axis.set_xlabel('Storage-product monthly ΔTWS, cm')
        axis.set_ylabel('P − ET − R, cm')
        axis.set_title(title)
        axis.grid(True, alpha=0.3)
        annotation = f"r = {metric_row['pearson_r']:.2f}\nMAE = {metric_row['MAE_cm']:.2f} cm\n95% CI: {metric_row['MAE_95_CI_lower_cm']:.2f}–{metric_row['MAE_95_CI_upper_cm']:.2f} cm"
        axis.text(0.04, 0.96, annotation, transform=axis.transAxes, va='top', bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'})
        axis.text(-0.1, 1.02, panel_label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    draw_water_balance_scatter(axes[0], coarse_water_balance_domain_table, coarse_water_balance_metrics.iloc[0], 'GRACE / imputed 0.25° TWS', 'a)')
    draw_water_balance_scatter(axes[1], downscaled_water_balance_domain_table, downscaled_water_balance_metrics.iloc[0], 'Final downscaled 0.1° TWS', 'b)')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D3_water_balance_dTWS_scatter.jpeg')
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    water_balance_cell_metrics_gdf.plot(column='pearson_r', ax=axes[0], cmap='coolwarm', vmin=-1, vmax=1, edgecolor='none', legend=True, legend_kwds={'label': 'Pearson r', 'shrink': 0.85}, missing_kwds={'color': 'lightgrey'})
    bias_limit = np.nanquantile(np.abs(water_balance_cell_metrics_gdf['bias_cm']), 0.98)
    if not np.isfinite(bias_limit) or bias_limit <= 0:
        bias_limit = 1.0
    water_balance_cell_metrics_gdf.plot(column='bias_cm', ax=axes[1], cmap='RdBu_r', vmin=-bias_limit, vmax=bias_limit, edgecolor='none', legend=True, legend_kwds={'label': 'Bias, cm (P − ET − R minus downscaled ΔTWS)', 'shrink': 0.85}, missing_kwds={'color': 'lightgrey'})
    for axis, label in zip(axes, ('a)', 'b)')):
        axis.set_axis_off()
        axis.text(-0.06, 1.02, label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D4_water_balance_spatial_validation.jpeg')
    uncertainty_mean = gdf_01_pred.groupby('cell_01', as_index=False).agg(mean_prediction_std_cm=('y_pred_final_std', 'mean'))
    uncertainty_geometry = gdf_01_pred[['cell_01', 'geometry_01']].drop_duplicates('cell_01')
    uncertainty_map = uncertainty_geometry.merge(uncertainty_mean, on='cell_01', how='left')
    uncertainty_map = gpd.GeoDataFrame(uncertainty_map, geometry='geometry_01', crs=gdf_01_pred.crs)
    fig, ax = plt.subplots(figsize=(7, 6))
    uncertainty_map.plot(column='mean_prediction_std_cm', ax=ax, cmap='magma', edgecolor='none', legend=True, legend_kwds={'label': 'Mean prediction SD, cm', 'shrink': 0.8}, missing_kwds={'color': 'lightgrey'})
    ax.set_axis_off()
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D5_downscaled_uncertainty_map.jpeg')
    del uncertainty_mean, uncertainty_geometry, uncertainty_map
    gc.collect()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    imputation_cell_metrics_gdf.plot(column='mean_prediction_std', ax=axes[0], cmap='magma', edgecolor='none', legend=True, legend_kwds={'label': 'Mean prediction SD, cm', 'shrink': 0.85}, missing_kwds={'color': 'lightgrey'})
    imputation_cell_metrics_gdf.plot(column='interval_coverage', ax=axes[1], cmap='viridis', vmin=0, vmax=1, edgecolor='none', legend=True, legend_kwds={'label': 'Empirical interval coverage', 'shrink': 0.85}, missing_kwds={'color': 'lightgrey'})
    for axis, label in zip(axes, ('a)', 'b)')):
        axis.set_axis_off()
        axis.text(-0.06, 1.02, label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D6_imputation_uncertainty_maps.jpeg')
    pivot_prediction_std = spatiotemporal_cv_df.pivot(index='spatial_block', columns='year', values='mean_prediction_std_cm')
    pivot_coverage = spatiotemporal_cv_df.pivot(index='spatial_block', columns='year', values='empirical_95_interval_coverage')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    image_std = axes[0].imshow(pivot_prediction_std.values, aspect='auto', origin='lower', cmap='magma')
    image_coverage = axes[1].imshow(pivot_coverage.values, aspect='auto', origin='lower', cmap='viridis', vmin=0, vmax=1)
    for axis, pivot, label in [(axes[0], pivot_prediction_std, 'a)'), (axes[1], pivot_coverage, 'b)')]:
        axis.set_yticks(np.arange(pivot.shape[0]))
        axis.set_yticklabels(pivot.index)
        axis.set_xticks(np.arange(pivot.shape[1]))
        axis.set_xticklabels(pivot.columns, rotation=45, ha='right')
        axis.set_xlabel('Year')
        axis.set_ylabel('Spatial block')
        axis.text(-0.1, 1.02, label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    colorbar_std = fig.colorbar(image_std, ax=axes[0], fraction=0.046, pad=0.02)
    colorbar_std.set_label('Mean prediction SD, cm')
    colorbar_coverage = fig.colorbar(image_coverage, ax=axes[1], fraction=0.046, pad=0.02)
    colorbar_coverage.set_label('Empirical 95% coverage')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D7_spatiotemporal_cv_uncertainty.jpeg')
    del pivot_prediction_std, pivot_coverage
    gc.collect()
    validation_plot = outer_cv_comparison_table.copy()
    validation_plot['plot_label'] = validation_plot['validation_design'].replace({'Grouped CV out-of-fold': 'Grouped CV OOF', 'Repeated outer tests, pooled': 'Repeated outer pooled', 'Complete temporal holdout': 'Full temporal holdout'})
    x = np.arange(len(validation_plot))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    metric_specs = [('r2', 'r2_95_CI_lower', 'r2_95_CI_upper', 'R²', 'a)'), ('mae_cm', 'mae_95_CI_lower_cm', 'mae_95_CI_upper_cm', 'MAE, cm', 'b)'), ('rmse_cm', 'rmse_95_CI_lower_cm', 'rmse_95_CI_upper_cm', 'RMSE, cm', 'c)')]
    for axis, (metric, lower_column, upper_column, ylabel, panel_label) in zip(axes, metric_specs):
        lower_error = np.maximum(validation_plot[metric] - validation_plot[lower_column], 0)
        upper_error = np.maximum(validation_plot[upper_column] - validation_plot[metric], 0)
        axis.errorbar(x, validation_plot[metric], yerr=np.vstack([lower_error, upper_error]), fmt='o', capsize=4)
        axis.set_xticks(x)
        axis.set_xticklabels(validation_plot['plot_label'], rotation=25, ha='right')
        axis.set_ylabel(ylabel)
        axis.grid(True, alpha=0.3)
        axis.text(-0.1, 1.02, panel_label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D8_cv_outer_holdout_comparison.jpeg')
    gap_summary = pseudo_gap_results_table.sort_values('gap_length_months').reset_index(drop=True)
    x = np.arange(len(gap_summary))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for metric, persistence_metric, axis, ylabel, panel_label in [('R2', 'persistence_R2', axes[0], 'R²', 'a)'), ('MAE_cm', 'persistence_MAE_cm', axes[1], 'MAE, cm', 'b)')]:
        axis.plot(gap_summary['gap_length_months'], gap_summary[metric], marker='o', label='Random Forest')
        axis.plot(gap_summary['gap_length_months'], gap_summary[persistence_metric], marker='s', linestyle='--', label='Persistence')
        axis.set_xlabel('Artificial gap length, months')
        axis.set_ylabel(ylabel)
        axis.grid(True, alpha=0.3)
        axis.legend()
        axis.text(-0.1, 1.02, panel_label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    axes[2].plot(gap_summary['gap_length_months'], gap_summary['MAE_skill_vs_persistence'], marker='o', label='MAE skill')
    axes[2].axhline(0, linestyle='--', linewidth=1.2, label='No improvement over persistence')
    axes[2].set_xlabel('Artificial gap length, months')
    axes[2].set_ylabel('Skill relative to persistence')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    axes[2].text(-0.1, 1.02, 'c)', transform=axes[2].transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D9_strict_pseudo_gap_performance.jpeg')
    fig, axes = plt.subplots(2, 2, figsize=(12, 11), constrained_layout=True)
    for axis, gap_length, panel_label in zip(axes.flatten(), PSEUDO_GAP_LENGTHS, ('a)', 'b)', 'c)', 'd)')):
        gap_data = pseudo_gap_predictions_df.loc[pseudo_gap_predictions_df['gap_length_months'].eq(gap_length)].dropna(subset=['observed_cm', 'reconstructed_cm'])
        summary_row = pseudo_gap_results_table.loc[pseudo_gap_results_table['gap_length_months'].eq(gap_length)].iloc[0]
        axis.scatter(gap_data['observed_cm'], gap_data['reconstructed_cm'], alpha=0.45, s=18)
        if len(gap_data) > 180:
            sample = gap_data.sample(n=180, random_state=RANDOM_STATE + gap_length)
        else:
            sample = gap_data
        axis.errorbar(sample['observed_cm'], sample['reconstructed_cm'], yerr=np.vstack([sample['reconstructed_cm'] - sample['prediction_lower_cm'], sample['prediction_upper_cm'] - sample['reconstructed_cm']]), fmt='none', alpha=0.18, linewidth=0.5)
        minimum = float(np.nanmin([gap_data['observed_cm'].min(), gap_data['reconstructed_cm'].min()]))
        maximum = float(np.nanmax([gap_data['observed_cm'].max(), gap_data['reconstructed_cm'].max()]))
        axis.plot([minimum, maximum], [minimum, maximum], linestyle='--', linewidth=1.2)
        axis.set_xlabel('Observed GRACE TWS, cm')
        axis.set_ylabel('Reconstructed GRACE TWS, cm')
        axis.set_title(f'{gap_length}-month pseudo-gap')
        axis.grid(True, alpha=0.3)
        axis.text(0.04, 0.96, f"RF R² = {summary_row['R2']:.2f}\nRF MAE = {summary_row['MAE_cm']:.2f} cm\nPersistence MAE = {summary_row['persistence_MAE_cm']:.2f} cm\nN = {int(summary_row['withheld_observations'])}", transform=axis.transAxes, va='top', bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'})
        axis.text(-0.1, 1.02, panel_label, transform=axis.transAxes, fontsize=16, fontweight='bold')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D10_pseudo_gap_observed_vs_reconstructed.jpeg')
    fig, ax = plt.subplots(figsize=(11, 6))
    for gap_length in PSEUDO_GAP_LENGTHS:
        position_data = pseudo_gap_position_table.loc[pseudo_gap_position_table['gap_length_months'].eq(gap_length)].sort_values('gap_position')
        rf_line = ax.plot(position_data['gap_position'], position_data['MAE_cm'], marker='o', label=f'RF: {gap_length} months')[0]
        ax.fill_between(position_data['gap_position'], position_data['MAE_95_CI_lower_cm'], position_data['MAE_95_CI_upper_cm'], alpha=0.1, color=rf_line.get_color())
        ax.plot(position_data['gap_position'], position_data['persistence_MAE_cm'], linestyle='--', marker='s', color=rf_line.get_color(), label=f'Persistence: {gap_length} months')
    ax.set_xlabel('Position within artificial gap, month')
    ax.set_ylabel('MAE, cm')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, title='Method and gap length')
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D11_strict_pseudo_gap_error_growth.jpeg')
    del validation_plot, gap_summary
    gc.collect()
    metrics = external_interpolation_qc_metrics.set_index('product').loc[['GRACE-SeDA', 'WGHM']].reset_index()
    positions = np.arange(len(metrics))
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    for axis, metric, lower, upper, ylabel, label in [(axes[0], 'r2', 'r2_95_CI_lower', 'r2_95_CI_upper', 'R²: aggregated 0.1° versus native 0.5°', 'a)'), (axes[1], 'mae_cm', 'mae_cm_95_CI_lower', 'mae_cm_95_CI_upper', 'MAE after aggregation, cm', 'b)')]:
        errors = np.vstack([np.maximum(metrics[metric] - metrics[lower], 0), np.maximum(metrics[upper] - metrics[metric], 0)])
        axis.errorbar(positions, metrics[metric], yerr=errors, fmt='o', capsize=4)
        axis.set_xticks(positions, metrics['product'])
        axis.set_ylabel(ylabel)
        axis.grid(True, alpha=0.3)
        add_panel_label(axis, label, x=-0.1)
    save_show_close(fig, DIAGNOSTIC_DIR / 'diag_D12_external_interpolation_qc.jpeg')


## Output manifests

In [ ]:
# Supplementary and diagnostic output manifest
supplementary_manifest = pd.DataFrame([
    ['Table S1', 'table_S1_original_GRACE_gap_months.csv',
     'Original months containing complete or partial GRACE gaps.'],
    ['Table S2', 'table_S2_representative_025_locations.csv',
     'Coordinates, availability, and agreement metrics for the five representative 0.25° grid cells used in Figure S3.'],
    ['Figure S1', 'fig_S1_hydroclimatic_spatial_statistics.jpg',
     'Temporal minimum, maximum, standard deviation, and seasonal amplitude of ET, P, runoff, temperature, soil moisture, and GRACE TWS.'],
    ['Figure S2', 'fig_S2_GRACE_missingness_and_completed_timeseries.jpeg',
     'Monthly GRACE spatial coverage and the completed observed–imputed TWS record.'],
    ['Figure S3', 'fig_S3_representative_025_observed_vs_predicted_timeseries.jpeg',
     'Original observed GRACE and raw uncorrected 0.25° ML TWS time series at five spatially distinct grid cells, with a location map.'],
    ['Figure S3 data', 'figure_S3_representative_025_timeseries_data.csv',
     'Monthly original observed GRACE and raw 0.25° ML predictions plotted in Figure S3.'],
    ['Video S1', 'videos/video_S1_monthly_downscaling_sequence.mp4',
     'Monthly sequence of the four Figure 15 downscaling stages.'],
    ['Video S2', 'videos/video_S2_monthly_model_comparison.mp4',
     'Monthly comparison of raw GRACE, the present product, GRACE-SeDA, and WGHM corresponding to Figure 20.'],
], columns=['Item', 'File', 'Description'])
display(supplementary_manifest)
supplementary_manifest.to_csv(SUPPLEMENT_DIR / 'supplementary_manifest.csv', index=False)

diagnostic_files = sorted(path.relative_to(DIAGNOSTIC_DIR) for path in DIAGNOSTIC_DIR.rglob('*') if path.is_file())
display(pd.DataFrame({'Diagnostic output': [str(path) for path in diagnostic_files]}))


## Generated files

In [ ]:
produced_files = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file())
display(pd.DataFrame({'Generated file': [str(path.relative_to(OUTPUT_DIR)) for path in produced_files]}))
